# SIPREV — Sistema Inteligente de Previsão Epidemiológica de Dengue
**Disciplina:** Análise Organizacional e Soluções Tecnológicas | Ciência dos Dados | Módulo 3

**Fonte:** InfoDengue (FGV/EMAp/FIOCRUZ) | Campo Grande/MS · MS · Capitais Brasileiras

**Ambiente:** Google Colab / Python Local


In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
=============================================================================
SIPREV - Sistema Inteligente de Previsão Epidemiológica de Dengue
=============================================================================
Disciplina  : Análise Organizacional e Soluções Tecnológicas
Semestre    : 2026.1  |  Curso: Ciência dos Dados
Módulo      : 3 – Relatório Parcial da Ação de Extensão
Título      : DADOS EPIDEMIOLÓGICOS: RECORRÊNCIA/INCIDÊNCIA DE DENGUE
              EM CAMPO GRANDE – MS
Fonte       : InfoDengue / FGV-EMAp-FIOCRUZ  |  Período: 2016–2025
Foco        : Campo Grande/MS · Mato Grosso do Sul · Capitais Brasileiras
=============================================================================
Aplicações  : Machine Learning · Deep Learning · Neural Networks
              Séries Temporais · Visualização · Mapas · Dashboards
=============================================================================
Arquivos CSV:
  DENGCG-MS_16_25.csv   → Campo Grande/MS (semanal, 2016-2025)
  DENGMS-BR_16_25.csv   → Todos os municípios de MS (semanal, 2016-2025)
  DENGCAPBR_16_25.csv   → Capitais brasileiras (semanal, 2016-2025)
=============================================================================
Colunas InfoDengue:
  data_iniSE    – timestamp ms (início da semana epidemiológica)
  SE            – semana epidemiológica YYYYSS
  casos_est     – casos estimados pelo modelo
  casos_est_min – IC inferior
  casos_est_max – IC superior
  casos         – casos notificados
  p_rt1         – P(Rt > 1)
  p_inc100k     – incidência estimada / 100 mil hab
  Localidade_id – código IBGE do município
  nivel         – nível de alerta (1=verde, 2=amarelo, 3=laranja, 4=vermelho)
  id            – identificador único do registro
  versao_modelo – data da versão do modelo
  municipio_nome– nome do município
  Rt            – número reprodutivo estimado
  pop           – população estimada
  tempmin/med/max – temperatura (°C)
  umidmax/med/min – umidade relativa (%)
  receptivo     – condição receptiva (0/1)
  transmissao   – transmissão ativa (0/1)
  nivel_inc     – nível de incidência (0-3)
  casprov       – casos prováveis notificados
  casprov_est/min/max – casos prováveis estimados
  casconf       – casos confirmados acumulados no ano
  notif_accum_year – notificações acumuladas no ano
=============================================================================
"""



'\n=============================================================================\nSIPREV - Sistema Inteligente de Previsão Epidemiológica de Dengue\n=============================================================================\nDisciplina  : Análise Organizacional e Soluções Tecnológicas\nSemestre    : 2026.1  |  Curso: Ciência dos Dados\nMódulo      : 3 – Relatório Parcial da Ação de Extensão\nTítulo      : DADOS EPIDEMIOLÓGICOS: RECORRÊNCIA/INCIDÊNCIA DE DENGUE\n              EM CAMPO GRANDE – MS\nFonte       : InfoDengue / FGV-EMAp-FIOCRUZ  |  Período: 2016–2025\nFoco        : Campo Grande/MS · Mato Grosso do Sul · Capitais Brasileiras\n=============================================================================\nAplicações  : Machine Learning · Deep Learning · Neural Networks\n              Séries Temporais · Visualização · Mapas · Dashboards\n=============================================================================\nArquivos CSV:\n  DENGCG-MS_16_25.csv   → Campo Grande/MS (se

In [2]:
# =============================================================================
# SEÇÃO 0 – INSTALAÇÃO DE DEPENDÊNCIAS (Google Colab / ambiente novo)


In [3]:
# =============================================================================
import sys
import subprocess
import os

def _pip(*pkgs):
    """Instala pacotes silenciosamente."""
    for p in pkgs:
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "--upgrade", p],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
        except Exception:
            pass

try:
    import google.colab          # noqa: F401
    IS_COLAB = True
    print("▶ Ambiente Google Colab detectado. Instalando dependências...")
    _pip(
        "texttable", "folium", "branca",
        "plotly", "kaleido",
        "xgboost", "lightgbm", "catboost",
        "shap", "statsmodels", "pmdarima",
        "scikit-learn", "scipy",
        "fpdf2", "openpyxl", "xlsxwriter",
        "tensorflow", "keras",
        "prophet", "neuralprophet",
        "pyarrow", "fastparquet",
    )
    print("✔ Dependências instaladas.")
except ImportError:
    IS_COLAB = False


import sys as _s
if hasattr(_s.stdout,"reconfigure"):
    try: _s.stdout.reconfigure(encoding="utf-8",errors="replace")
    except: pass
del _s


In [4]:
# =============================================================================
# SEÇÃO 1 – IMPORTS


In [5]:
# =============================================================================

# ── Padrão ────────────────────────────────────────────────────────────────────
import gc
import json
import math
import time
import glob
import logging
import warnings
import traceback
import zipfile
import textwrap
import itertools
import hashlib
import inspect
import platform
from datetime import datetime, timedelta
from pathlib import Path
from collections import defaultdict, Counter, OrderedDict
from typing import Optional, List, Dict, Tuple, Union

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

# ── Dados ─────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import (
    pearsonr, spearmanr, chi2_contingency,
    mannwhitneyu, kruskal, shapiro, normaltest
)
from scipy.signal import find_peaks

# ── Visualização estática ─────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, Normalize, BoundaryNorm
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
import seaborn as sns

# ── Visualização interativa ───────────────────────────────────────────────────
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.io as pio
    pio.templates.default = "plotly_white"
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("⚠ plotly não disponível – dashboards HTML serão omitidos.")

# ── Mapas ─────────────────────────────────────────────────────────────────────
try:
    import folium
    from folium.plugins import HeatMap, MarkerCluster, Fullscreen, MiniMap
    from folium.features import DivIcon
    import branca.colormap as cm
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
    print("⚠ folium não disponível – mapas interativos serão omitidos.")

# ── Machine Learning ──────────────────────────────────────────────────────────
try:
    from sklearn.preprocessing import (
        StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder,
        PolynomialFeatures, PowerTransformer
    )
    from sklearn.model_selection import (
        train_test_split, cross_val_score, GridSearchCV,
        RandomizedSearchCV, KFold, StratifiedKFold, TimeSeriesSplit
    )
    from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, SpectralClustering
    from sklearn.mixture import GaussianMixture
    from sklearn.ensemble import (
        RandomForestClassifier, RandomForestRegressor,
        GradientBoostingClassifier, GradientBoostingRegressor,
        IsolationForest, AdaBoostClassifier, AdaBoostRegressor,
        ExtraTreesClassifier, ExtraTreesRegressor, BaggingRegressor,
        VotingRegressor, StackingRegressor
    )
    from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
    from sklearn.linear_model import (
        LinearRegression, Ridge, Lasso, ElasticNet,
        LogisticRegression, SGDClassifier, BayesianRidge,
        HuberRegressor, Lars
    )
    from sklearn.svm import SVR, SVC, LinearSVR
    from sklearn.neighbors import (
        KNeighborsClassifier, KNeighborsRegressor, LocalOutlierFactor
    )
    from sklearn.naive_bayes import GaussianNB
    from sklearn.decomposition import PCA, TruncatedSVD, FastICA, NMF
    from sklearn.manifold import TSNE
    from sklearn.feature_selection import (
        SelectKBest, f_regression, mutual_info_regression,
        RFE, RFECV, VarianceThreshold
    )
    from sklearn.metrics import (
        classification_report, confusion_matrix, roc_auc_score,
        mean_squared_error, mean_absolute_error, r2_score,
        silhouette_score, calinski_harabasz_score, davies_bouldin_score,
        accuracy_score, precision_score, recall_score, f1_score,
        roc_curve, auc, mean_absolute_percentage_error
    )
    from sklearn.pipeline import Pipeline
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.neural_network import MLPClassifier, MLPRegressor
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False
    print("⚠ scikit-learn não disponível.")

# ── XGBoost ───────────────────────────────────────────────────────────────────
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

# ── LightGBM ──────────────────────────────────────────────────────────────────
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

# ── CatBoost ──────────────────────────────────────────────────────────────────
try:
    from catboost import CatBoostClassifier, CatBoostRegressor, Pool
    HAS_CAT = True
except ImportError:
    HAS_CAT = False

# ── SHAP ──────────────────────────────────────────────────────────────────────
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

# ── Séries temporais estatísticas ─────────────────────────────────────────────
try:
    from statsmodels.tsa.seasonal import seasonal_decompose, STL
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf, grangercausalitytests
    from statsmodels.stats.stattools import durbin_watson
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

try:
    from pmdarima import auto_arima
    HAS_PMDARIMA = True
except ImportError:
    HAS_PMDARIMA = False

try:
    from prophet import Prophet
    HAS_PROPHET = True
except ImportError:
    HAS_PROPHET = False

# ── Deep Learning / TensorFlow ────────────────────────────────────────────────
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.layers import (
        LSTM, GRU, Dense, Dropout, BatchNormalization,
        Conv1D, MaxPooling1D, GlobalAveragePooling1D,
        Flatten, Input, Bidirectional,
        MultiHeadAttention, LayerNormalization,
        Attention, RepeatVector, TimeDistributed,
        AveragePooling1D, Concatenate, Add,
        SimpleRNN
    )
    from tensorflow.keras.callbacks import (
        EarlyStopping, ReduceLROnPlateau, ModelCheckpoint,
        TensorBoard, LambdaCallback
    )
    from tensorflow.keras.optimizers import Adam, RMSprop, SGD
    from tensorflow.keras.losses import MeanSquaredError, Huber
    from tensorflow.keras.regularizers import l1, l2, l1_l2
    tf.get_logger().setLevel("ERROR")
    tf.autograph.set_verbosity(0)
    HAS_TF = True
    TF_VERSION = tf.__version__
except ImportError:
    HAS_TF = False
    TF_VERSION = "N/A"
    print("⚠ TensorFlow não disponível – modelos LSTM/GRU serão omitidos.")

# ── Relatórios ────────────────────────────────────────────────────────────────
try:
    import texttable
    HAS_TEXTTABLE = True
except ImportError:
    _pip("texttable")
    try:
        import texttable
        HAS_TEXTTABLE = True
    except Exception:
        HAS_TEXTTABLE = False

try:
    from fpdf import FPDF
    HAS_FPDF = True
except ImportError:
    HAS_FPDF = False

try:
    import openpyxl
    from openpyxl.styles import (
        PatternFill, Font, Alignment, Border, Side,
        GradientFill
    )
    from openpyxl.utils import get_column_letter
    from openpyxl.chart import BarChart, LineChart, Reference
    HAS_OPENPYXL = True
except ImportError:
    HAS_OPENPYXL = False

try:
    import pyarrow
    HAS_PARQUET = True
except ImportError:
    HAS_PARQUET = False



⚠ TensorFlow não disponível – modelos LSTM/GRU serão omitidos.


In [6]:
# =============================================================================
# SEÇÃO 2 – CONFIGURAÇÕES GLOBAIS


In [7]:
# =============================================================================

# ── Caminhos ──────────────────────────────────────────────────────────────────
if IS_COLAB:
    BASE_DIR   = Path("/content")
    INPUT_DIR  = BASE_DIR / "input" / "csv_archive"
    OUTPUT_DIR = BASE_DIR / "output"
else:
    try:
        BASE_DIR = Path(__file__).resolve().parent.parent
    except NameError:
        # Jupyter notebook: use cwd().parent
        BASE_DIR = Path.cwd().parent
    INPUT_DIR  = BASE_DIR / "input" / "csv_archive"
    OUTPUT_DIR = BASE_DIR / "output"

# Criar subpastas de saída
for _sub in ["graficos", "mapas", "relatorios", "modelos", "dados",
             "dashboards", "logs", "pdf"]:
    (OUTPUT_DIR / _sub).mkdir(parents=True, exist_ok=True)

# ── Identificação do estudo ───────────────────────────────────────────────────
NOME_CG        = "Campo Grande"
NOME_MS        = "Mato Grosso do Sul"
CODIGO_CG_STR  = "Campo Grande"   # nome no CSV InfoDengue
ANOS_ANALISE   = list(range(2016, 2026))

# ── Arquivos CSV ──────────────────────────────────────────────────────────────
ARQUIVO_CG    = INPUT_DIR / "DENGCG-MS_16_25.csv"
ARQUIVO_MS    = INPUT_DIR / "DENGMS-BR_16_25.csv"
ARQUIVO_CAP   = INPUT_DIR / "DENGCAPBR_16_25.csv"

CSV_URLS = {
    "CG" : "https://raw.githubusercontent.com/OpenScienceTechnology/info_dengue/"
           "refs/heads/main/Dataset/Dengue/csv_archive/DENGCG-MS_16_25.csv",
    "MS" : "https://raw.githubusercontent.com/OpenScienceTechnology/info_dengue/"
           "refs/heads/main/Dataset/Dengue/csv_archive/DENGMS-BR_16_25.csv",
    "CAP": "https://raw.githubusercontent.com/OpenScienceTechnology/info_dengue/"
           "refs/heads/main/Dataset/Dengue/csv_archive/DENGCAPBR_16_25.csv",
}

# ── Paleta de cores ───────────────────────────────────────────────────────────
COR_PRINCIPAL   = "#C0392B"
COR_SECUNDARIA  = "#2980B9"
COR_ALERTA      = "#E67E22"
COR_VERDE       = "#27AE60"
COR_ROXO        = "#8E44AD"
COR_CINZA       = "#7F8C8D"

NIVEL_CORES = {
    1: "#2ECC71",   # verde  – sem alerta
    2: "#F1C40F",   # amarelo – alerta baixo
    3: "#E67E22",   # laranja – alerta médio
    4: "#E74C3C",   # vermelho – alerta alto
}
NIVEL_NOMES = {
    1: "Nível 1 – Verde (Sem Alerta)",
    2: "Nível 2 – Amarelo (Alerta Baixo)",
    3: "Nível 3 – Laranja (Alerta Médio)",
    4: "Nível 4 – Vermelho (Alerta Alto)",
}

PALETA_RISCO = {
    "Muito Baixo": "#2ECC71",
    "Baixo":       "#82E0AA",
    "Médio":       "#F0B27A",
    "Alto":        "#E74C3C",
    "Muito Alto":  "#8E44AD",
    "Crítico":     "#4A235A",
}

PALETA_CALOR = [
    "#FEF9E7","#FDEBD0","#FAD7A0","#F5B041",
    "#E67E22","#CA6F1E","#C0392B","#922B21","#641E16",
]

MESES_PT = {
    1:"Janeiro",2:"Fevereiro",3:"Março",4:"Abril",
    5:"Maio",6:"Junho",7:"Julho",8:"Agosto",
    9:"Setembro",10:"Outubro",11:"Novembro",12:"Dezembro",
}
MESES_ABREV = {
    1:"Jan",2:"Fev",3:"Mar",4:"Abr",5:"Mai",6:"Jun",
    7:"Jul",8:"Ago",9:"Set",10:"Out",11:"Nov",12:"Dez",
}

# ── Matplotlib global ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150,
    "figure.facecolor": "white", "axes.facecolor": "#FAFAFA",
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 13, "axes.labelsize": 11,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.fontsize": 9, "axes.grid": True,
    "grid.alpha": 0.35, "lines.linewidth": 1.8,
})
sns.set_style("whitegrid")
sns.set_palette("husl")

# ── Timestamp da execução ─────────────────────────────────────────────────────
TIMESTAMP   = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPORT_NAME = f"EpiAnalysis_DENG_{TIMESTAMP}"

# ── Parâmetros epidemiológicos ─────────────────────────────────────────────────
PARAMS = {
    "periodo_incubacao_dias"    : 4,
    "periodo_infeccioso_dias"   : 5,
    "periodo_extrinseco_dias"   : 8,
    "threshold_alerta_inc100k"  : 100,
    "threshold_epidemia_inc100k": 300,
    "threshold_surto_inc100k"   : 1000,
    "rt_limiar_epidemico"       : 1.0,
    "rt_alerta_critico"         : 2.0,
    "janela_mm_semanas"         : 4,
    "janela_mm_meses"           : 3,
    "horizonte_previsao_semanas": 12,
    "horizonte_previsao_meses"  : 6,
    "n_clusters_kmeans"         : 4,
    "n_splits_ts"               : 5,
    "lstm_epochs"               : 60,
    "lstm_batch"                : 16,
    "lstm_janela"               : 12,
    "lstm_units_1"              : 64,
    "lstm_units_2"              : 32,
    "rf_n_estimators"           : 200,
    "xgb_n_estimators"          : 300,
    "lgb_n_estimators"          : 300,
    "alpha_sig"                 : 0.05,
    "shap_max_display"          : 20,
    "arima_max_p"               : 5,
    "arima_max_q"               : 5,
    "arima_max_d"               : 2,
}

# Limites de risco para taxa de incidência/100k
LIMITES_RISCO = [
    (0,     "Sem Dados"),
    (1,     "Muito Baixo"),
    (50,    "Baixo"),
    (100,   "Médio"),
    (300,   "Alto"),
    (1000,  "Muito Alto"),
    (float("inf"), "Crítico"),
]



In [8]:
# =============================================================================
# SEÇÃO 3 – LOGGING


In [9]:
# =============================================================================

LOG_PATH = OUTPUT_DIR / "logs" / f"execucao_{TIMESTAMP}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(LOG_PATH, encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger("SIPREV")

# Contadores de execução
_stats = {
    "arquivos_lidos"      : 0,
    "registros_lidos"     : 0,
    "registros_validos"   : 0,
    "registros_descartados": 0,
    "graficos_gerados"    : 0,
    "mapas_gerados"       : 0,
    "relatorios_gerados"  : 0,
    "modelos_treinados"   : 0,
    "dashboards_gerados"  : 0,
}

def _inc(key: str, n: int = 1):
    _stats[key] = _stats.get(key, 0) + n

def _banner():
    log.info("=" * 78)
    log.info("  SIPREV – Sistema Inteligente de Previsão Epidemiológica de Dengue")
    log.info(f"  Início  : {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
    log.info(f"  Ambiente: {'Google Colab' if IS_COLAB else 'Máquina Local'}")
    log.info(f"  Python  : {sys.version.split()[0]}  |  "
             f"Pandas: {pd.__version__}  |  NumPy: {np.__version__}")
    log.info(f"  TensorFlow: {TF_VERSION}")
    log.info(f"  OUTPUT  : {OUTPUT_DIR}")
    log.info(f"  Timestamp: {TIMESTAMP}")
    log.info("=" * 78)

_banner()



2026-05-31 16:46:16 [INFO] ==============================================================================


2026-05-31 16:46:16 [INFO]   SIPREV – Sistema Inteligente de Previsão Epidemiológica de Dengue


2026-05-31 16:46:16 [INFO]   Início  : 31/05/2026 16:46:16


2026-05-31 16:46:16 [INFO]   Ambiente: Máquina Local


2026-05-31 16:46:16 [INFO]   Python  : 3.14.5  |  Pandas: 2.3.3  |  NumPy: 2.3.5


2026-05-31 16:46:16 [INFO]   TensorFlow: N/A


2026-05-31 16:46:16 [INFO]   OUTPUT  : C:\Users\Workstation\Desktop\Temp2\output


2026-05-31 16:46:16 [INFO]   Timestamp: 20260531_164616


2026-05-31 16:46:16 [INFO] ==============================================================================


In [10]:
# =======================================================================
# HELPERS: logging, figura, FPDF patch
# =======================================================================
def log_section(titulo):
    sep = '=' * 70
    log.info(''); log.info(sep); log.info('  ' + titulo.upper()); log.info(sep)

def log_ok(msg):   log.info('  OK  ' + str(msg))
def log_warn(msg): log.warning('  AVISO  ' + str(msg))
def log_info(msg): log.info('  ' + str(msg))

def _salvar_figura(fig, nome, subdir='graficos', dpi=150):
    p = OUTPUT_DIR / subdir / (nome + '_' + TIMESTAMP + '.png')
    fig.savefig(str(p), dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig); _inc('graficos_gerados')
    log.info('  [PNG] ' + p.name); return p

if HAS_FPDF:
    _oc = FPDF.cell
    _om = FPDF.multi_cell
    _SUBS = [(chr(8211), '-'), (chr(8212), '--'), (chr(8594), '->'),
             (chr(8226), '*'), (chr(8216), chr(39)), (chr(8217), chr(39)),
             (chr(8220), chr(34)), (chr(8221), chr(34)), (chr(8230), '...')]
    def _fs(t):
        if not isinstance(t, str): return t
        for k, v in _SUBS:
            t = t.replace(k, v)
        return t.encode('latin-1', errors='replace').decode('latin-1')
    def _pc(self, w=0, h=None, text='', *a, **kw):
        return _oc(self, w, h, _fs(str(text)), *a, **kw)
    def _pm(self, w=0, h=None, text='', *a, **kw):
        return _om(self, w, h, _fs(str(text)), *a, **kw)
    FPDF.cell = _pc
    FPDF.multi_cell = _pm

print('Helpers OK')


Helpers OK


In [11]:
# =============================================================================
# SEÇÃO 4 – DADOS POPULACIONAIS E GEOGRÁFICOS


In [12]:
# =============================================================================

# ── Capitais brasileiras (nome → UF) ─────────────────────────────────────────
CAPITAIS_UF = {
    "Rio Branco":       "AC", "Maceió":          "AL", "Macapá":      "AP",
    "Manaus":           "AM", "Salvador":         "BA", "Fortaleza":   "CE",
    "Brasília":         "DF", "Vitória":          "ES", "Goiânia":     "GO",
    "São Luís":         "MA", "Cuiabá":           "MT", "Campo Grande":"MS",
    "Belo Horizonte":   "MG", "Belém":            "PA", "João Pessoa": "PB",
    "Curitiba":         "PR", "Recife":           "PE", "Teresina":    "PI",
    "Rio de Janeiro":   "RJ", "Natal":            "RN", "Porto Alegre":"RS",
    "Porto Velho":      "RO", "Boa Vista":        "RR", "Florianópolis":"SC",
    "São Paulo":        "SP", "Aracaju":          "SE", "Palmas":      "TO",
}

# ── Regiões brasileiras ───────────────────────────────────────────────────────
REGIAO_UF = {
    "AC":"Norte",  "AM":"Norte",  "AP":"Norte",  "PA":"Norte",
    "RO":"Norte",  "RR":"Norte",  "TO":"Norte",
    "AL":"Nordeste","BA":"Nordeste","CE":"Nordeste","MA":"Nordeste",
    "PB":"Nordeste","PE":"Nordeste","PI":"Nordeste","RN":"Nordeste","SE":"Nordeste",
    "DF":"Centro-Oeste","GO":"Centro-Oeste","MS":"Centro-Oeste","MT":"Centro-Oeste",
    "ES":"Sudeste","MG":"Sudeste","RJ":"Sudeste","SP":"Sudeste",
    "PR":"Sul","RS":"Sul","SC":"Sul",
}

# ── Populações das capitais (estimativa 2022) ─────────────────────────────────
POP_CAPITAIS = {
    "Rio Branco":      364368,  "Maceió":        1025360, "Macapá":       522499,
    "Manaus":         2255903,  "Salvador":      2900319, "Fortaleza":    2703391,
    "Brasília":       3055149,  "Vitória":        365855, "Goiânia":      1536097,
    "São Luís":       1108975,  "Cuiabá":         621310, "Campo Grande":  942140,
    "Belo Horizonte": 2315560,  "Belém":         1499641, "João Pessoa":   817511,
    "Curitiba":       1963726,  "Recife":        1555039, "Teresina":      866300,
    "Rio de Janeiro": 6747815,  "Natal":          890480, "Porto Alegre":  1492530,
    "Porto Velho":     428527,  "Boa Vista":      419652, "Florianópolis": 508826,
    "São Paulo":     12396372,  "Aracaju":        664908, "Palmas":        313541,
}

# ── Municípios de MS com população estimada 2022 ─────────────────────────────
POP_MUNICIPIOS_MS = {
    "Campo Grande":    942140, "Dourados":       214095, "Três Lagoas":   123281,
    "Corumbá":         112506, "Ponta Porã":      102086, "Naviraí":        56478,
    "Nova Andradina":   57046, "Aquidauana":       48193, "Sidrolândia":    51234,
    "Maracaju":         47289, "Coxim":            35789, "Costa Rica":     19834,
    "Chapadão do Sul":  25178, "Rio Brilhante":    32567, "Jardim":         26823,
    "Iguatemi":         21456, "Bonito":           22143, "Piraputanga":     4523,
    "Amambai":          38712, "Anastácio":        26789, "Bandeirantes":    9876,
    "Bataguassu":       21345, "Brasilândia":      13456, "Caarapó":        27891,
    "Camapuã":          18543, "Cassilândia":      22678, "Deodápolis":     13210,
    "Douradina":        12098, "Eldorado":         11234, "Fátima do Sul":  19876,
    "Glória de Dourados":10234,"Guia Lopes da Laguna":9876,"Iguatemi":     21456,
    "Inocência":         7654, "Itaporã":          22345, "Itaquiraí":     25678,
    "Ivinhema":         24567, "Japorã":            8765, "Jaraguari":      6543,
    "Jateí":             6234, "Juti":              6789, "Ladário":        23456,
    "Laguna Carapã":     8765, "Maracaju":         47289, "Miranda":        27654,
    "Mundo Novo":       19876, "Navirai":          56478, "Nioaque":        14567,
    "Nova Alvorada do Sul":17345,"Nova Andradina":  57046,"Novo Horizonte do Sul":6234,
    "Paraíso das Águas": 8234, "Paranaíba":        40567, "Paranhos":       13456,
    "Pedro Gomes":       8765, "Ponta Porã":      102086, "Porto Murtinho": 16789,
    "Ribas do Rio Pardo":27345,"Rio Negro":         5678, "Rochedo":         5234,
    "Santa Rita do Pardo":8234,"São Gabriel do Oeste":24567,
    "Selvíria":          6789, "Sete Quedas":      11234, "Sonora":         14567,
    "Tacuru":            9876, "Taquarussu":        5678, "Terenos":        22345,
    "Três Lagoas":     123281, "Vicentina":         5432,
}

# ── Coordenadas centrais de municípios-chave de MS ───────────────────────────
COORDS_MS = {
    "Campo Grande":  (-20.4697, -54.6201),
    "Dourados":      (-22.2211, -54.8056),
    "Três Lagoas":   (-20.7511, -51.6783),
    "Corumbá":       (-19.0097, -57.6522),
    "Ponta Porã":    (-22.5361, -55.7261),
    "Naviraí":       (-23.0622, -54.1917),
    "Aquidauana":    (-20.4711, -55.7872),
    "Maracaju":      (-21.6175, -55.1681),
    "Coxim":         (-18.5072, -54.7592),
    "Paranaíba":     (-19.6781, -51.1911),
}

# ── Coordenadas das capitais ──────────────────────────────────────────────────
COORDS_CAPITAIS = {
    "Rio Branco":      (-9.9754,  -67.8249),
    "Maceió":          (-9.6658,  -35.7350),
    "Macapá":           (0.0349,  -51.0694),
    "Manaus":          (-3.1019,  -60.0250),
    "Salvador":       (-12.9714,  -38.5014),
    "Fortaleza":       (-3.7172,  -38.5433),
    "Brasília":        (-15.7801,  -47.9292),
    "Vitória":         (-20.3155,  -40.3128),
    "Goiânia":        (-16.6869,  -49.2648),
    "São Luís":        (-2.5307,  -44.3068),
    "Cuiabá":         (-15.6014,  -56.0979),
    "Campo Grande":   (-20.4697,  -54.6201),
    "Belo Horizonte": (-19.9167,  -43.9345),
    "Belém":           (-1.4558,  -48.5044),
    "João Pessoa":     (-7.1153,  -34.8641),
    "Curitiba":       (-25.4278,  -49.2731),
    "Recife":          (-8.0476,  -34.8770),
    "Teresina":        (-5.0920,  -42.8038),
    "Rio de Janeiro": (-22.9068,  -43.1729),
    "Natal":           (-5.7945,  -35.2110),
    "Porto Alegre":   (-30.0346,  -51.2177),
    "Porto Velho":     (-8.7612,  -63.9004),
    "Boa Vista":        (2.8235,  -60.6758),
    "Florianópolis":  (-27.5954,  -48.5480),
    "São Paulo":      (-23.5505,  -46.6333),
    "Aracaju":        (-10.9091,  -37.0677),
    "Palmas":         (-10.2491,  -48.3243),
}



In [13]:
# =============================================================================
# SEÇÃO 5 – FUNÇÕES AUXILIARES GERAIS


In [14]:
# =============================================================================

def fmt_num(n, decimais: int = 0) -> str:
    """Formata número com separador de milhar (pt-BR)."""
    try:
        if pd.isna(n):
            return "–"
        if decimais > 0:
            return f"{float(n):,.{decimais}f}".replace(",", "X").replace(".", ",").replace("X", ".")
        return f"{int(round(float(n))):,}".replace(",", ".")
    except Exception:
        return str(n)

def fmt_pct(n, decimais: int = 1) -> str:
    """Formata percentual."""
    try:
        return f"{float(n):.{decimais}f}%"
    except Exception:
        return "–"

def taxa_inc(casos: float, pop: float, base: float = 100_000) -> float:
    """Taxa de incidência por 100 mil hab."""
    try:
        if pop and pop > 0 and not pd.isna(casos):
            return round(float(casos) / float(pop) * base, 2)
    except Exception:
        pass
    return 0.0

def cresc_pct(atual, anterior) -> float:
    """Crescimento percentual entre dois valores."""
    try:
        if anterior and anterior > 0:
            return round((float(atual) - float(anterior)) / float(anterior) * 100, 2)
    except Exception:
        pass
    return float("nan")

def classificar_risco(taxa: float) -> str:
    """Classifica nível de risco pela taxa de incidência/100k."""
    if pd.isna(taxa) or taxa <= 0:
        return "Sem Dados"
    for lim, nome in LIMITES_RISCO:
        if taxa < lim:
            return nome
    return "Crítico"

def cor_risco(nivel: str) -> str:
    """Retorna cor hex para nível de risco."""
    return PALETA_RISCO.get(nivel, "#CCCCCC")

def semana_para_data(se_yyyyww: int) -> Optional[datetime]:
    """Converte SE YYYYWW para data (segunda-feira da semana)."""
    try:
        s = str(int(se_yyyyww))
        ano = int(s[:4])
        sem = int(s[4:])
        return datetime.strptime(f"{ano}-W{sem:02d}-1", "%Y-W%W-%w")
    except Exception:
        return None

def timestamp_ms_para_data(ts_ms) -> Optional[datetime]:
    """Converte timestamp em milissegundos para datetime."""
    try:
        return datetime.utcfromtimestamp(float(ts_ms) / 1000.0)
    except Exception:
        return None

def periodo_epidemico(mes: int) -> str:
    """Classifica mês em período epidemiológico para o Brasil central."""
    return "Chuvoso (Out–Mar)" if mes in {10, 11, 12, 1, 2, 3} else "Seco (Abr–Set)"

def trimestre_str(mes: int) -> str:
    """Retorna string do trimestre."""
    return f"T{(mes - 1) // 3 + 1}"

def nivel_alerta_descr(nivel: int) -> str:
    """Retorna descrição do nível de alerta InfoDengue."""
    return NIVEL_NOMES.get(int(nivel) if pd.notna(nivel) else 1,
                           f"Nível {nivel}")

def print_section(titulo: str, char: str = "="):
    sep = char * 78
    log.info("")
    log.info(sep)
    log.info(f"  {titulo.upper()}")
    log.info(sep)

def print_sub(titulo: str):
    log.info(f"\n  ── {titulo} ──")

def salvar_fig(nome: str, subdir: str = "graficos", dpi: int = 150) -> Path:
    """Salva figura matplotlib atual."""
    p = OUTPUT_DIR / subdir / f"{nome}.png"
    plt.tight_layout()
    plt.savefig(p, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close("all")
    _inc("graficos_gerados")
    log.info(f"  [PNG] {p.name}")
    return p

def salvar_html(fig_plotly, nome: str, subdir: str = "graficos") -> Optional[Path]:
    """Salva figura Plotly como HTML interativo."""
    if not HAS_PLOTLY or fig_plotly is None:
        return None
    p = OUTPUT_DIR / subdir / f"{nome}.html"
    fig_plotly.write_html(
        str(p),
        include_plotlyjs="cdn",
        full_html=True,
        config={"responsive": True, "scrollZoom": True},
    )
    log.info(f"  [HTML] {p.name}")
    return p

def salvar_mapa(mapa_folium, nome: str) -> Optional[Path]:
    """Salva mapa Folium como HTML."""
    if not HAS_FOLIUM or mapa_folium is None:
        return None
    p = OUTPUT_DIR / "mapas" / f"{nome}.html"
    mapa_folium.save(str(p))
    _inc("mapas_gerados")
    log.info(f"  [MAPA] {p.name}")
    return p



In [15]:
# =============================================================================
# SEÇÃO 6 – TEXTTABLE: GERAÇÃO DE TABELAS TXT/LOG


In [16]:
# =============================================================================

def make_table(headers: list, rows: list,
               col_align: list = None, col_dtype: list = None,
               max_width: int = 130) -> str:
    """Gera tabela formatada com texttable."""
    if not HAS_TEXTTABLE or not rows:
        lines = ["  ".join(str(h) for h in headers)]
        for r in rows:
            lines.append("  ".join(str(x) for x in r))
        return "\n".join(lines)
    t = texttable.Texttable(max_width=max_width)
    t.set_deco(texttable.Texttable.HEADER | texttable.Texttable.VLINES)
    t.header(headers)
    if col_align:
        t.set_cols_align(col_align)
    if col_dtype:
        t.set_cols_dtype(col_dtype)
    for r in rows:
        t.add_row([str(x) if x is None else x for x in r])
    return t.draw()

def salvar_txt(conteudo: str, nome: str, titulo: str = "") -> Path:
    """Salva conteúdo em arquivo .txt."""
    p = OUTPUT_DIR / "relatorios" / f"{nome}.txt"
    with open(p, "w", encoding="utf-8") as f:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"SIPREV – Sistema Inteligente de Previsão Epidemiológica\n")
        f.write(f"Gerado em: {ts}\n")
        if titulo:
            f.write(f"\n{'=' * 70}\n{titulo}\n{'=' * 70}\n\n")
        f.write(conteudo + "\n")
    _inc("relatorios_gerados")
    log.info(f"  [TXT] {p.name}")
    return p

def salvar_log_tabela(conteudo: str, nome: str, titulo: str = "") -> Path:
    """Salva tabela em arquivo .log."""
    p = OUTPUT_DIR / "logs" / f"{nome}.log"
    with open(p, "w", encoding="utf-8") as f:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"# SIPREV LOG | {ts}\n")
        if titulo:
            f.write(f"# {titulo}\n\n")
        f.write(conteudo + "\n")
    log.info(f"  [LOG] {p.name}")
    return p

def tabela_e_salva(df_tab: pd.DataFrame, nome: str, titulo: str = "",
                   col_align: list = None) -> str:
    """Converte DataFrame em tabela texttable e salva TXT+LOG."""
    headers = list(df_tab.columns)
    rows    = [list(r) for r in df_tab.itertuples(index=False, name=None)]
    tab_str = make_table(headers, rows, col_align=col_align)
    log.info(f"\n{tab_str}")
    salvar_txt(tab_str, nome, titulo)
    salvar_log_tabela(tab_str, nome, titulo)
    return tab_str



In [17]:
# =============================================================================
# SEÇÃO 7 – CARREGAMENTO DOS DADOS INFODENGUE


In [18]:
# =============================================================================

def _ler_csv_infodengue(caminho: Union[str, Path],
                        fonte: str = "desconhecida") -> pd.DataFrame:
    """
    Lê um arquivo CSV no formato InfoDengue.
    Suporta leitura local e URL (fallback online).
    """
    caminho = Path(caminho) if isinstance(caminho, str) else caminho

    # Tenta leitura local primeiro
    if caminho.exists():
        log.info(f"  Lendo local: {caminho.name} ({caminho.stat().st_size/1e6:.1f} MB)")
        df = pd.read_csv(caminho, encoding="utf-8-sig", low_memory=False,
                         on_bad_lines="skip")
    else:
        url = CSV_URLS.get(fonte)
        if url:
            log.info(f"  Arquivo local não encontrado. Baixando de URL ({fonte})...")
            try:
                df = pd.read_csv(url, encoding="utf-8", low_memory=False,
                                 on_bad_lines="skip")
                # Salva cópia local
                caminho.parent.mkdir(parents=True, exist_ok=True)
                df.to_csv(caminho, index=False, encoding="utf-8")
                log.info(f"  Salvo localmente: {caminho.name}")
            except Exception as e:
                log.error(f"  Falha ao baixar {fonte}: {e}")
                return pd.DataFrame()
        else:
            log.error(f"  Arquivo não encontrado e sem URL configurada: {caminho}")
            return pd.DataFrame()

    _inc("arquivos_lidos")
    _inc("registros_lidos", len(df))
    log.info(f"  → {len(df):,} registros lidos de {caminho.name}")
    return df


def _processar_infodengue(df: pd.DataFrame, fonte_nome: str = "") -> pd.DataFrame:
    """
    Padroniza e enriquece DataFrame no formato InfoDengue.
    Extrai ano, mês, semana, datas, código IBGE e indicadores derivados.
    """
    if df.empty:
        return df

    df = df.copy()

    # ── Remove BOM em nomes de colunas ────────────────────────────────────────
    df.columns = [c.lstrip("﻿").strip() for c in df.columns]

    # ── Tipos numéricos básicos ───────────────────────────────────────────────
    num_cols = [
        "casos", "casos_est", "casos_est_min", "casos_est_max",
        "p_rt1", "p_inc100k", "Rt", "pop",
        "tempmin", "tempmed", "tempmax",
        "umidmax", "umidmed", "umidmin",
        "receptivo", "transmissao", "nivel", "nivel_inc",
        "casprov", "casprov_est", "casprov_est_min", "casprov_est_max",
        "casconf", "notif_accum_year", "SE", "Localidade_id",
    ]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # ── Data da semana epidemiológica ─────────────────────────────────────────
    if "data_iniSE" in df.columns:
        df["data_iniSE"] = pd.to_numeric(df["data_iniSE"], errors="coerce")
        df["data_SE"]    = df["data_iniSE"].apply(timestamp_ms_para_data)
    elif "SE" in df.columns:
        df["data_SE"] = df["SE"].apply(semana_para_data)

    # ── Extrair ANO e MÊS ────────────────────────────────────────────────────
    if "SE" in df.columns:
        se_str       = df["SE"].astype(str).str.zfill(6)
        df["ANO"]    = pd.to_numeric(se_str.str[:4], errors="coerce").astype("Int64")
        df["SEMANA"] = pd.to_numeric(se_str.str[4:], errors="coerce").astype("Int64")
    elif "data_SE" in df.columns:
        df["ANO"]    = df["data_SE"].dt.year.astype("Int64")
        df["SEMANA"] = df["data_SE"].dt.isocalendar().week.astype("Int64")

    if "data_SE" in df.columns:
        df["MES"] = df["data_SE"].dt.month.astype("Int64")
    else:
        # Estima mês pela semana (aprox.)
        df["MES"] = ((df["SEMANA"] - 1) * 7 // 30 + 1).clip(1, 12).astype("Int64")

    # ── Trimestre e período epidemiológico ────────────────────────────────────
    df["TRIMESTRE"] = df["MES"].apply(lambda m: trimestre_str(int(m)) if pd.notna(m) else None)
    df["PERIODO"]   = df["MES"].apply(lambda m: periodo_epidemico(int(m)) if pd.notna(m) else None)
    df["MES_NOME"]  = df["MES"].map(MESES_ABREV)

    # ── Código IBGE (6 dígitos) extraído de Localidade_id ───────────────────
    # InfoDengue: Localidade_id = código IBGE 7 dígitos OU 0 para estado
    if "Localidade_id" in df.columns:
        df["COD_IBGE"] = df["Localidade_id"].where(
            df["Localidade_id"] > 0, other=pd.NA
        ).astype("Int64")
    else:
        df["COD_IBGE"] = pd.NA

    # Normaliza nome do município
    if "municipio_nome" in df.columns:
        df["municipio_nome"] = df["municipio_nome"].astype(str).str.strip()

    # ── Indicadores derivados ─────────────────────────────────────────────────
    # Taxa de incidência (usa pop do próprio registro se disponível)
    if "pop" in df.columns and "casos" in df.columns:
        df["taxa_inc_calc"] = df.apply(
            lambda r: taxa_inc(r["casos"], r["pop"])
            if pd.notna(r.get("pop")) and r.get("pop", 0) > 0
            else r.get("p_inc100k", 0.0),
            axis=1,
        )
    else:
        df["taxa_inc_calc"] = df.get("p_inc100k", 0.0)

    # Nível de alerta descritivo
    df["nivel_descr"] = df["nivel"].apply(
        lambda n: nivel_alerta_descr(n) if pd.notna(n) else "Desconhecido"
    )

    # Classificação de risco
    df["risco"] = df["taxa_inc_calc"].apply(classificar_risco)

    # Alerta ativo (Rt > 1 e probabilidade alta)
    df["alerta_ativo"] = (
        (df.get("Rt", 0) > PARAMS["rt_limiar_epidemico"]) &
        (df.get("p_rt1", 0) > 0.9)
    ).astype(int)

    # Fonte / dataset
    df["_fonte"] = fonte_nome

    # ── Remove registros sem ano ──────────────────────────────────────────────
    n_antes = len(df)
    df = df.dropna(subset=["ANO"])
    df = df[df["ANO"].between(2015, 2030)]
    _inc("registros_descartados", n_antes - len(df))
    _inc("registros_validos", len(df))

    log.info(f"  → {len(df):,} registros válidos após processamento ({fonte_nome})")
    return df


def carregar_tudo() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Carrega os três datasets InfoDengue e retorna:
      df_cg  – Campo Grande/MS
      df_ms  – todos os municípios de MS
      df_cap – capitais brasileiras
    """
    print_section("CARREGAMENTO DOS DADOS INFODENGUE")

    df_cg  = _ler_csv_infodengue(ARQUIVO_CG,  "CG")
    df_ms  = _ler_csv_infodengue(ARQUIVO_MS,  "MS")
    df_cap = _ler_csv_infodengue(ARQUIVO_CAP, "CAP")

    df_cg  = _processar_infodengue(df_cg,  "Campo Grande/MS")
    df_ms  = _processar_infodengue(df_ms,  "Municípios MS")
    df_cap = _processar_infodengue(df_cap, "Capitais Brasil")

    # ── Enriquece capitais com UF e região ───────────────────────────────────
    if not df_cap.empty and "municipio_nome" in df_cap.columns:
        df_cap["UF"]     = df_cap["municipio_nome"].map(CAPITAIS_UF)
        df_cap["REGIAO"] = df_cap["UF"].map(REGIAO_UF)
        df_cap["pop_ref"] = df_cap["municipio_nome"].map(POP_CAPITAIS)
        df_cap["pop_ref"] = df_cap["pop_ref"].fillna(df_cap.get("pop", 1_000_000))

    # ── Enriquece MS com população de referência ─────────────────────────────
    if not df_ms.empty and "municipio_nome" in df_ms.columns:
        df_ms["pop_ref"] = df_ms["municipio_nome"].map(POP_MUNICIPIOS_MS)
        df_ms["pop_ref"] = df_ms["pop_ref"].fillna(df_ms.get("pop", 50_000))

    # ── Log resumo ───────────────────────────────────────────────────────────
    for nome, df in [("Campo Grande", df_cg), ("MS-Municípios", df_ms), ("Capitais-BR", df_cap)]:
        if not df.empty:
            anos = f"{df['ANO'].min()}–{df['ANO'].max()}" if "ANO" in df.columns else "?"
            muns = df["municipio_nome"].nunique() if "municipio_nome" in df.columns else 1
            log.info(f"  {nome:20s}: {len(df):>7,} registros | "
                     f"{muns:>4} município(s) | Anos {anos}")

    return df_cg, df_ms, df_cap



In [19]:
# =============================================================================
# SEÇÃO 8 – PRÉ-PROCESSAMENTO AVANÇADO


In [20]:
# =============================================================================

def preprocessar_serie_temporal(df: pd.DataFrame,
                                  agg_col: str = "casos",
                                  freq: str = "W") -> pd.DataFrame:
    """
    Prepara série temporal semanal ou mensal para um único município.
    Preenche lacunas, suaviza outliers e calcula médias móveis.
    """
    if df.empty or "data_SE" not in df.columns:
        return df

    df = df.sort_values("data_SE").copy()
    df = df.set_index("data_SE")

    # Reamostragem para frequência desejada
    if freq == "W":
        serie = df[agg_col].resample("W-MON").sum()
    else:  # Mensal
        serie = df[agg_col].resample("MS").sum()

    # Preenche NaN com 0 (semanas sem registro = 0 casos)
    serie = serie.fillna(0)

    # Suavização: clip de outliers extremos (>= 3 desvios padrão)
    mu, sigma = serie.mean(), serie.std()
    if sigma > 0:
        serie = serie.clip(upper=mu + 4 * sigma)

    # Cria DataFrame com indicadores
    df_ts = pd.DataFrame({"casos": serie})
    df_ts["ANO"]   = df_ts.index.year
    df_ts["MES"]   = df_ts.index.month
    df_ts["SEMANA"] = df_ts.index.isocalendar().week

    # Médias móveis
    df_ts["mm4"]  = df_ts["casos"].rolling(4, min_periods=1).mean()
    df_ts["mm12"] = df_ts["casos"].rolling(12, min_periods=1).mean()

    # Crescimento semana a semana (%)
    df_ts["cresc_pct"] = df_ts["casos"].pct_change() * 100

    return df_ts.reset_index()


def agregar_mensal(df: pd.DataFrame, grupo_cols: list = None,
                   agg_cols: list = None) -> pd.DataFrame:
    """Agrega DataFrame semanal para nível mensal."""
    if df.empty:
        return df
    if grupo_cols is None:
        grupo_cols = ["ANO", "MES", "municipio_nome"]
    if agg_cols is None:
        agg_cols = {
            "casos":         "sum",
            "casos_est":     "sum",
            "casprov":       "sum",
            "casconf":       "max",
            "p_rt1":         "mean",
            "Rt":            "mean",
            "p_inc100k":     "mean",
            "taxa_inc_calc": "mean",
            "tempmed":       "mean",
            "tempmin":       "min",
            "tempmax":       "max",
            "umidmed":       "mean",
            "receptivo":     "max",
            "transmissao":   "max",
            "nivel":         "max",
            "alerta_ativo":  "max",
        }
    valid_agg = {k: v for k, v in agg_cols.items() if k in df.columns}
    valid_grp  = [c for c in grupo_cols if c in df.columns]
    if not valid_grp:
        return df
    df_m = df.groupby(valid_grp, as_index=False, observed=True).agg(valid_agg)
    return df_m


def agregar_anual(df: pd.DataFrame, municipio: str = None) -> pd.DataFrame:
    """Agrega para nível anual por município (ou filtrado por município)."""
    if df.empty:
        return df
    if municipio:
        df = df[df["municipio_nome"] == municipio].copy()
    agg = {
        "casos":         "sum",
        "casos_est":     "sum",
        "casprov":       "sum",
        "p_rt1":         "mean",
        "Rt":            "mean",
        "taxa_inc_calc": "mean",
        "p_inc100k":     "mean",
        "tempmed":       "mean",
        "nivel":         "max",
        "alerta_ativo":  "sum",
        "receptivo":     "sum",
        "transmissao":   "sum",
    }
    valid_agg = {k: v for k, v in agg.items() if k in df.columns}
    grp_cols  = ["ANO"]
    if "municipio_nome" in df.columns and not municipio:
        grp_cols.append("municipio_nome")
    df_a = df.groupby(grp_cols, as_index=False, observed=True).agg(valid_agg)
    return df_a



In [21]:
# =============================================================================
# SEÇÃO 9 – RELATÓRIO DE QUALIDADE DOS DADOS


In [22]:
# =============================================================================

def relatorio_qualidade(df: pd.DataFrame, nome: str = "Dataset") -> dict:
    """
    Gera relatório completo de qualidade dos dados.
    Retorna dicionário com métricas e salva TXT/LOG.
    """
    print_section(f"QUALIDADE DOS DADOS – {nome}")

    metricas = {}
    n_total  = len(df)
    metricas["total_registros"] = n_total

    if n_total == 0:
        log.warning(f"  DataFrame '{nome}' está vazio!")
        return metricas

    # ── Cobertura temporal ────────────────────────────────────────────────────
    if "ANO" in df.columns:
        metricas["ano_min"]   = int(df["ANO"].min())
        metricas["ano_max"]   = int(df["ANO"].max())
        metricas["n_anos"]    = df["ANO"].nunique()
    if "SEMANA" in df.columns:
        metricas["n_semanas"] = df["SEMANA"].nunique()

    # ── Municípios ────────────────────────────────────────────────────────────
    if "municipio_nome" in df.columns:
        metricas["n_municipios"]     = df["municipio_nome"].nunique()
        metricas["municipios_lista"] = sorted(df["municipio_nome"].unique().tolist())

    # ── Completude das colunas ────────────────────────────────────────────────
    cols_importantes = [
        "casos", "casos_est", "p_inc100k", "Rt", "nivel",
        "pop", "tempmed", "umidmed", "data_SE",
    ]
    rows_qual = []
    for c in cols_importantes:
        if c in df.columns:
            n_nulos = df[c].isna().sum()
            pct_ok  = (1 - n_nulos / n_total) * 100
            rows_qual.append([c, fmt_num(n_total - n_nulos), fmt_num(n_nulos), fmt_pct(pct_ok)])
            metricas[f"completude_{c}"] = round(pct_ok, 1)

    tab = make_table(
        ["Coluna", "Válidos", "Nulos", "Completude"],
        rows_qual, col_align=["l","r","r","r"]
    )
    log.info("\n" + tab)
    salvar_txt(tab, f"qualidade_{nome.lower().replace('/', '_')}_colunas",
               f"Qualidade dos Dados – {nome}")
    salvar_log_tabela(tab, f"qualidade_{nome.lower().replace('/', '_')}_colunas",
                      f"Qualidade – {nome}")

    # ── Distribuição por nível de alerta ──────────────────────────────────────
    if "nivel" in df.columns:
        dist_nivel = df["nivel"].value_counts().sort_index()
        rows_nivel = []
        for n, cnt in dist_nivel.items():
            rows_nivel.append([
                str(int(n)), NIVEL_NOMES.get(int(n), "?"),
                fmt_num(cnt), fmt_pct(cnt / n_total * 100)
            ])
        tab_n = make_table(
            ["Nível", "Descrição", "Registros", "%"],
            rows_nivel, col_align=["c","l","r","r"]
        )
        log.info("\n" + tab_n)
        salvar_txt(tab_n, f"qualidade_{nome.lower().replace('/', '_')}_niveis",
                   f"Distribuição por Nível de Alerta – {nome}")

    # ── Estatísticas descritivas de casos ─────────────────────────────────────
    if "casos" in df.columns:
        s = df["casos"].describe()
        rows_desc = [
            ["Total de Registros",   fmt_num(int(s["count"]))],
            ["Total de Casos",       fmt_num(int(df["casos"].sum()))],
            ["Média / Semana",       fmt_num(s["mean"], 1)],
            ["Mediana",              fmt_num(s["50%"], 1)],
            ["Desvio Padrão",        fmt_num(s["std"], 1)],
            ["Mínimo",               fmt_num(int(s["min"]))],
            ["Máximo",               fmt_num(int(s["max"]))],
            ["Percentil 25",         fmt_num(s["25%"], 1)],
            ["Percentil 75",         fmt_num(s["75%"], 1)],
        ]
        tab_d = make_table(
            ["Indicador", "Valor"],
            rows_desc, col_align=["l","r"]
        )
        log.info("\n" + tab_d)
        salvar_txt(tab_d, f"qualidade_{nome.lower().replace('/', '_')}_stats",
                   f"Estatísticas de Casos – {nome}")
        metricas["total_casos"]  = int(df["casos"].sum())
        metricas["media_casos"]  = round(float(s["mean"]), 1)
        metricas["max_casos"]    = int(s["max"])

    # ── Registros duplicados ──────────────────────────────────────────────────
    if "id" in df.columns:
        n_dup = df.duplicated(subset=["id"]).sum()
        metricas["duplicados"] = int(n_dup)
        log.info(f"  Registros duplicados (por 'id'): {fmt_num(n_dup)}")

    log.info(f"\n  Total registros : {fmt_num(n_total)}")
    log.info(f"  Período         : {metricas.get('ano_min','?')}–{metricas.get('ano_max','?')}")
    log.info(f"  Municípios      : {metricas.get('n_municipios', 1)}")
    log.info(f"  Total de casos  : {fmt_num(metricas.get('total_casos', 0))}")

    return metricas




In [23]:
# =============================================================================
# SEÇÃO 10 – ANÁLISE EXPLORATÓRIA DE DADOS (EDA) – GERAL


In [24]:
# =============================================================================

def eda_visao_geral(df_cg: pd.DataFrame,
                    df_ms: pd.DataFrame,
                    df_cap: pd.DataFrame) -> None:
    """
    Análise exploratória inicial: estatísticas gerais, distribuições,
    correlações e visão do conjunto de dados.
    """
    print_section("EDA – VISÃO GERAL DOS DADOS")

    for nome, df in [("Campo Grande", df_cg), ("MS-Municípios", df_ms),
                     ("Capitais-Brasil", df_cap)]:
        if df.empty:
            continue
        print_sub(f"Dataset: {nome}")

        # Estatísticas descritivas numéricas
        num_df = df.select_dtypes(include=[np.number])
        if not num_df.empty:
            desc = num_df.describe().T
            desc_rows = []
            for idx, row in desc.iterrows():
                desc_rows.append([
                    idx,
                    fmt_num(row.get("count", 0), 0),
                    fmt_num(row.get("mean", 0), 2),
                    fmt_num(row.get("std", 0), 2),
                    fmt_num(row.get("min", 0), 2),
                    fmt_num(row.get("50%", 0), 2),
                    fmt_num(row.get("max", 0), 2),
                ])
            tab = make_table(
                ["Variável", "Count", "Média", "Std", "Mín", "Mediana", "Máx"],
                desc_rows, col_align=["l","r","r","r","r","r","r"]
            )
            log.info(f"\n{tab}")
            salvar_txt(tab, f"eda_desc_{nome.lower().replace(' ','_').replace('-','_')}",
                       f"Estatísticas Descritivas – {nome}")

    # ── Gráfico: Total de casos por ano (todos os datasets) ──────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    datasets  = [("Campo Grande", df_cg), ("MS – Todos Municípios", df_ms),
                 ("Capitais Brasileiras", df_cap)]
    for ax, (nome, df) in zip(axes, datasets):
        if df.empty or "ANO" not in df.columns or "casos" not in df.columns:
            ax.set_title(f"{nome}\n(sem dados)")
            continue
        tot = df.groupby("ANO")["casos"].sum().reset_index()
        tot = tot[tot["ANO"].between(2016, 2025)]
        cores = [COR_PRINCIPAL if c == tot["casos"].max() else COR_SECUNDARIA
                 for c in tot["casos"]]
        bars = ax.bar(tot["ANO"].astype(int), tot["casos"], color=cores,
                      edgecolor="white", linewidth=0.5)
        # Rótulos nas barras
        for bar, val in zip(bars, tot["casos"]):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                    fmt_num(int(val)), ha="center", va="bottom",
                    fontsize=7, rotation=45)
        ax.set_title(nome, fontsize=11, fontweight="bold")
        ax.set_xlabel("Ano")
        ax.set_ylabel("Casos Notificados")
        ax.set_xticks(tot["ANO"].astype(int))
        ax.set_xticklabels(tot["ANO"].astype(int), rotation=45)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: fmt_num(int(x))
        ))
    plt.suptitle("Total de Casos de Dengue por Ano (2016–2025)",
                 fontsize=14, fontweight="bold", y=1.02)
    salvar_fig("eda_casos_por_ano_geral")

    # ── Gráfico: Sazonalidade mensal agregada ─────────────────────────────────
    fig, ax = plt.subplots(figsize=(13, 5))
    for nome, df, cor in [
        ("Campo Grande", df_cg,  COR_PRINCIPAL),
        ("MS (média)",   df_ms,  COR_SECUNDARIA),
        ("Capitais (média)", df_cap, COR_ALERTA),
    ]:
        if df.empty or "MES" not in df.columns or "casos" not in df.columns:
            continue
        mensal = df.groupby("MES")["casos"].mean().reset_index()
        ax.plot(mensal["MES"], mensal["casos"],
                marker="o", label=nome, color=cor, linewidth=2)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels([MESES_ABREV[m] for m in range(1, 13)])
    ax.set_title("Sazonalidade Mensal – Média de Casos (2016–2025)", fontweight="bold")
    ax.set_xlabel("Mês")
    ax.set_ylabel("Casos / Semana (média)")
    ax.legend()
    salvar_fig("eda_sazonalidade_mensal")

    # ── Mapa de calor: Casos por Ano × Mês (Campo Grande) ────────────────────
    if not df_cg.empty and {"ANO", "MES", "casos"}.issubset(df_cg.columns):
        pivot = df_cg.groupby(["ANO", "MES"])["casos"].sum().unstack(fill_value=0)
        pivot = pivot[[c for c in range(1, 13) if c in pivot.columns]]
        fig, ax = plt.subplots(figsize=(14, 6))
        sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrRd",
                    linewidths=0.3, ax=ax,
                    xticklabels=[MESES_ABREV[c] for c in pivot.columns],
                    cbar_kws={"label": "Casos"})
        ax.set_title("Heatmap – Casos de Dengue em Campo Grande por Ano × Mês",
                     fontsize=13, fontweight="bold")
        ax.set_xlabel("Mês")
        ax.set_ylabel("Ano")
        salvar_fig("eda_heatmap_ano_mes_cg")

    # ── Correlação entre variáveis climáticas e casos (Campo Grande) ──────────
    if not df_cg.empty:
        vars_corr = [c for c in ["casos", "Rt", "p_rt1", "p_inc100k",
                                  "tempmin", "tempmed", "tempmax",
                                  "umidmin", "umidmed", "umidmax",
                                  "receptivo", "transmissao", "nivel"]
                     if c in df_cg.columns]
        if len(vars_corr) >= 3:
            corr_mat = df_cg[vars_corr].corr()
            fig, ax  = plt.subplots(figsize=(12, 9))
            mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
            sns.heatmap(corr_mat, mask=mask, annot=True, fmt=".2f",
                        cmap="coolwarm", vmin=-1, vmax=1,
                        linewidths=0.3, ax=ax,
                        annot_kws={"size": 8})
            ax.set_title("Matriz de Correlação – Campo Grande/MS",
                         fontsize=13, fontweight="bold")
            salvar_fig("eda_correlacao_cg")

            # Tabela de correlação com casos
            if "casos" in vars_corr:
                corr_casos = corr_mat["casos"].drop("casos").sort_values(ascending=False)
                rows_c = [[v, fmt_num(r, 4)] for v, r in corr_casos.items()]
                tab_c  = make_table(["Variável", "Correlação com Casos"],
                                    rows_c, col_align=["l","r"])
                log.info(f"\n{tab_c}")
                salvar_txt(tab_c, "eda_correlacao_com_casos_cg",
                           "Correlação das Variáveis com Casos – Campo Grande")

    # ── Boxplot: Distribuição de casos por nível de alerta ────────────────────
    if not df_cg.empty and {"nivel", "casos"}.issubset(df_cg.columns):
        fig, ax = plt.subplots(figsize=(10, 5))
        grupos = [df_cg[df_cg["nivel"] == n]["casos"].dropna()
                  for n in sorted(df_cg["nivel"].dropna().unique())]
        labels = [NIVEL_NOMES.get(int(n), f"N{int(n)}")
                  for n in sorted(df_cg["nivel"].dropna().unique())]
        bp = ax.boxplot(grupos, labels=labels, patch_artist=True, notch=False)
        cores_bp = [NIVEL_CORES.get(int(n), "#999") for n in
                    sorted(df_cg["nivel"].dropna().unique())]
        for patch, cor in zip(bp["boxes"], cores_bp):
            patch.set_facecolor(cor)
            patch.set_alpha(0.7)
        ax.set_title("Distribuição de Casos por Nível de Alerta – Campo Grande",
                     fontweight="bold")
        ax.set_ylabel("Casos Notificados / Semana")
        ax.set_xticklabels(labels, rotation=15, ha="right")
        salvar_fig("eda_boxplot_casos_nivel_cg")

    log.info("  EDA geral concluída.")




In [25]:
# =============================================================================
# SEÇÃO 11 – ANÁLISE ESPECÍFICA: CAMPO GRANDE/MS


In [26]:
# =============================================================================

def analise_campo_grande(df_cg: pd.DataFrame, df_ms: pd.DataFrame) -> dict:
    """
    Análise completa de Campo Grande/MS:
    evolução temporal, sazonalidade, indicadores, comparação com MS.
    """
    print_section("ANÁLISE ESPECÍFICA – CAMPO GRANDE / MS")
    resultados = {}

    if df_cg.empty:
        log.warning("  DataFrame de Campo Grande está vazio!")
        return resultados

    # ── 11.1 Série temporal semanal ───────────────────────────────────────────
    print_sub("11.1 Série Temporal Semanal")
    if "data_SE" in df_cg.columns and "casos" in df_cg.columns:
        df_ts = df_cg.sort_values("data_SE").copy()
        mm4   = df_ts["casos"].rolling(4, min_periods=1).mean()
        mm12  = df_ts["casos"].rolling(12, min_periods=1).mean()

        fig, ax = plt.subplots(figsize=(16, 5))
        ax.bar(df_ts["data_SE"], df_ts["casos"],
               color=COR_SECUNDARIA, alpha=0.4, label="Casos Semanais")
        ax.plot(df_ts["data_SE"], mm4,  color=COR_ALERTA,    linewidth=1.5,
                label="Média Móvel 4 sem")
        ax.plot(df_ts["data_SE"], mm12, color=COR_PRINCIPAL, linewidth=2.0,
                label="Média Móvel 12 sem")

        # Linha limiar epidêmico (taxa/100k convertida para casos)
        pop_cg = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942140
        limiar = PARAMS["threshold_epidemia_inc100k"] * pop_cg / 100_000
        ax.axhline(limiar, color="red", linestyle="--", linewidth=1,
                   label=f"Limiar Epidêmico ({PARAMS['threshold_epidemia_inc100k']}/100k)")

        ax.set_title("Série Temporal Semanal – Casos de Dengue – Campo Grande/MS",
                     fontweight="bold")
        ax.set_xlabel("Semana Epidemiológica")
        ax.set_ylabel("Casos Notificados")
        ax.legend(loc="upper left", fontsize=8)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: fmt_num(int(x))
        ))
        salvar_fig("cg_serie_temporal_semanal")

    # ── 11.2 Casos por Ano ────────────────────────────────────────────────────
    print_sub("11.2 Casos por Ano")
    if "ANO" in df_cg.columns and "casos" in df_cg.columns:
        anual = df_cg.groupby("ANO").agg(
            casos=("casos", "sum"),
            casos_est=("casos_est", "sum"),
            semanas_alerta=("alerta_ativo", "sum") if "alerta_ativo" in df_cg.columns else ("casos", "count"),
            rt_medio=("Rt", "mean") if "Rt" in df_cg.columns else ("casos", "count"),
        ).reset_index()
        anual = anual[anual["ANO"].between(2016, 2025)]

        # Adiciona população e taxa de incidência
        pop_cg_ref = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942140
        anual["taxa_inc"] = anual["casos"].apply(lambda c: taxa_inc(c, pop_cg_ref))
        anual["cresc"]    = anual["casos"].pct_change() * 100

        resultados["anual_cg"] = anual.copy()

        # Gráfico: barras + linha taxa
        fig, ax1 = plt.subplots(figsize=(13, 6))
        ax2 = ax1.twinx()
        cores_ano = [COR_PRINCIPAL if c == anual["casos"].max() else "#AED6F1"
                     for c in anual["casos"]]
        bars = ax1.bar(anual["ANO"].astype(int), anual["casos"],
                       color=cores_ano, edgecolor="white", linewidth=0.5,
                       label="Casos Notificados")
        ax2.plot(anual["ANO"].astype(int), anual["taxa_inc"],
                 color=COR_ALERTA, marker="o", linewidth=2,
                 label="Taxa Inc./100k")
        for bar, val in zip(bars, anual["casos"]):
            ax1.text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + anual["casos"].max() * 0.01,
                     fmt_num(int(val)), ha="center", va="bottom",
                     fontsize=8, fontweight="bold")
        ax1.set_title("Campo Grande/MS – Casos de Dengue por Ano (2016–2025)",
                      fontsize=13, fontweight="bold")
        ax1.set_xlabel("Ano")
        ax1.set_ylabel("Casos Notificados", color=COR_SECUNDARIA)
        ax2.set_ylabel("Taxa de Incidência / 100k hab", color=COR_ALERTA)
        ax1.set_xticks(anual["ANO"].astype(int))
        ax1.set_xticklabels(anual["ANO"].astype(int), rotation=45)
        ax1.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: fmt_num(int(x))
        ))
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
        salvar_fig("cg_casos_por_ano")

        # Tabela
        rows_a = []
        for _, r in anual.iterrows():
            rows_a.append([
                int(r["ANO"]), fmt_num(int(r["casos"])),
                fmt_num(r["taxa_inc"], 1),
                fmt_pct(r.get("cresc", float("nan"))),
                fmt_num(r.get("rt_medio", 0), 2),
            ])
        tab_a = make_table(
            ["Ano", "Casos", "Taxa/100k", "Cresc.%", "Rt Médio"],
            rows_a, col_align=["c","r","r","r","r"]
        )
        log.info(f"\n{tab_a}")
        salvar_txt(tab_a, "cg_casos_por_ano", "Campo Grande – Casos por Ano")

    # ── 11.3 Casos por Mês (sazonalidade) ─────────────────────────────────────
    print_sub("11.3 Sazonalidade Mensal")
    if "MES" in df_cg.columns and "casos" in df_cg.columns:
        mensal_ano = df_cg.groupby(["ANO", "MES"])["casos"].sum().reset_index()
        mensal_avg = mensal_ano.groupby("MES")["casos"].agg(
            media="mean", desvio="std", total="sum"
        ).reset_index()

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Médias mensais históricas
        ax = axes[0]
        cores_m = [COR_PRINCIPAL if mes in {1, 2, 3, 10, 11, 12} else "#85C1E9"
                   for mes in mensal_avg["MES"]]
        ax.bar(mensal_avg["MES"], mensal_avg["media"],
               color=cores_m, edgecolor="white")
        ax.errorbar(mensal_avg["MES"], mensal_avg["media"],
                    yerr=mensal_avg["desvio"].fillna(0),
                    fmt="none", color="black", capsize=4, linewidth=1)
        ax.set_xticks(range(1, 13))
        ax.set_xticklabels([MESES_ABREV[m] for m in range(1, 13)])
        ax.set_title("Média Mensal de Casos (2016–2025)", fontweight="bold")
        ax.set_xlabel("Mês")
        ax.set_ylabel("Casos Médios / Mês")
        ax.axvspan(9.5, 12.5, alpha=0.08, color=COR_PRINCIPAL, label="Período chuvoso")
        ax.axvspan(0.5, 3.5,  alpha=0.08, color=COR_PRINCIPAL)
        ax.legend(["Período Chuvoso (Out–Mar)"], loc="upper right", fontsize=8)

        # Evolução mensal por ano (linha)
        ax2 = axes[1]
        anos_plot = sorted(mensal_ano["ANO"].unique())
        palette   = plt.cm.get_cmap("tab10", len(anos_plot))
        for i, ano in enumerate(anos_plot):
            sub = mensal_ano[mensal_ano["ANO"] == ano].sort_values("MES")
            ax2.plot(sub["MES"], sub["casos"],
                     marker="o", markersize=4,
                     color=palette(i), label=str(int(ano)), linewidth=1.5)
        ax2.set_xticks(range(1, 13))
        ax2.set_xticklabels([MESES_ABREV[m] for m in range(1, 13)])
        ax2.set_title("Casos por Mês – Evolução Anual", fontweight="bold")
        ax2.set_xlabel("Mês")
        ax2.set_ylabel("Casos")
        ax2.legend(ncol=2, fontsize=8, loc="upper right")

        plt.suptitle("Sazonalidade – Dengue em Campo Grande/MS", fontsize=14,
                     fontweight="bold")
        salvar_fig("cg_sazonalidade_mensal")

        # Tabela mensal
        rows_m = [[MESES_PT.get(int(r["MES"]), "?"),
                   fmt_num(r["media"], 1), fmt_num(r["desvio"], 1),
                   fmt_num(r["total"])]
                  for _, r in mensal_avg.iterrows()]
        tab_m = make_table(
            ["Mês", "Média Casos", "Desvio", "Total Histórico"],
            rows_m, col_align=["l","r","r","r"]
        )
        log.info(f"\n{tab_m}")
        salvar_txt(tab_m, "cg_sazonalidade_mensal", "Sazonalidade Mensal – Campo Grande")

    # ── 11.4 Série Temporal do Rt ─────────────────────────────────────────────
    print_sub("11.4 Número Reprodutivo Básico (Rt)")
    if "Rt" in df_cg.columns and "data_SE" in df_cg.columns:
        df_rt = df_cg[df_cg["Rt"] > 0].sort_values("data_SE")
        if not df_rt.empty:
            fig, ax = plt.subplots(figsize=(16, 4))
            ax.fill_between(df_rt["data_SE"], df_rt["Rt"],
                            where=(df_rt["Rt"] >= 1),
                            color=COR_PRINCIPAL, alpha=0.3, label="Rt ≥ 1 (crescimento)")
            ax.fill_between(df_rt["data_SE"], df_rt["Rt"],
                            where=(df_rt["Rt"] < 1),
                            color=COR_VERDE, alpha=0.3, label="Rt < 1 (declínio)")
            ax.plot(df_rt["data_SE"], df_rt["Rt"],
                    color=COR_SECUNDARIA, linewidth=0.8)
            ax.axhline(1.0, color="red", linestyle="--", linewidth=1.5,
                       label="Limiar Epidêmico (Rt = 1)")
            ax.set_title("Número Reprodutivo (Rt) – Campo Grande/MS", fontweight="bold")
            ax.set_ylabel("Rt Estimado")
            ax.set_xlabel("Semana Epidemiológica")
            ax.legend(loc="upper right", fontsize=9)
            ax.set_ylim(0, min(df_rt["Rt"].max() * 1.2, 10))
            salvar_fig("cg_rt_temporal")

    # ── 11.5 Nível de Alerta ao longo do tempo ────────────────────────────────
    print_sub("11.5 Nível de Alerta InfoDengue")
    if "nivel" in df_cg.columns and "data_SE" in df_cg.columns:
        df_niv = df_cg.sort_values("data_SE")
        fig, ax = plt.subplots(figsize=(16, 4))
        for n in [1, 2, 3, 4]:
            mask = df_niv["nivel"] == n
            ax.fill_between(df_niv["data_SE"], 0, n,
                            where=mask & (df_niv["nivel"] == n),
                            step="post", alpha=0.6,
                            color=NIVEL_CORES[n],
                            label=NIVEL_NOMES[n])
        ax.set_yticks([1, 2, 3, 4])
        ax.set_yticklabels(["Verde", "Amarelo", "Laranja", "Vermelho"])
        ax.set_title("Nível de Alerta InfoDengue – Campo Grande/MS", fontweight="bold")
        ax.set_xlabel("Semana Epidemiológica")
        ax.set_ylabel("Nível de Alerta")
        ax.legend(loc="upper right", fontsize=8, ncol=2)
        salvar_fig("cg_nivel_alerta_temporal")

        # Tabela de distribuição
        dist_n  = df_cg["nivel"].value_counts().sort_index()
        total_n = dist_n.sum()
        rows_n  = [[int(n), NIVEL_NOMES.get(int(n), "?"),
                    fmt_num(cnt), fmt_pct(cnt/total_n*100)]
                   for n, cnt in dist_n.items()]
        tab_n = make_table(
            ["Nível", "Descrição", "Semanas", "%"],
            rows_n, col_align=["c","l","r","r"]
        )
        log.info(f"\n{tab_n}")
        salvar_txt(tab_n, "cg_distribuicao_nivel_alerta",
                   "Distribuição por Nível de Alerta – Campo Grande")

    # ── 11.6 Variáveis Climáticas vs Casos ────────────────────────────────────
    print_sub("11.6 Clima vs Casos")
    vars_clima = [c for c in ["tempmed", "tempmin", "tempmax",
                               "umidmed", "umidmin", "umidmax"]
                  if c in df_cg.columns]
    if vars_clima and "casos" in df_cg.columns:
        n_vars = len(vars_clima)
        ncols  = 3
        nrows  = math.ceil(n_vars / ncols)
        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(5 * ncols, 4 * nrows))
        axes = axes.flatten() if nrows > 1 or ncols > 1 else [axes]

        for i, var in enumerate(vars_clima):
            ax  = axes[i]
            sub = df_cg[[var, "casos"]].dropna()
            ax.scatter(sub[var], sub["casos"],
                       alpha=0.3, color=COR_SECUNDARIA, s=15)
            # Linha de tendência
            if len(sub) > 5:
                z = np.polyfit(sub[var], sub["casos"], 1)
                p = np.poly1d(z)
                xs = np.linspace(sub[var].min(), sub[var].max(), 100)
                ax.plot(xs, p(xs), color=COR_PRINCIPAL, linewidth=2)
            r, pv = pearsonr(sub[var], sub["casos"])
            ax.set_title(f"{var}\nr = {r:.3f} (p={pv:.3f})", fontsize=9)
            ax.set_xlabel(var, fontsize=8)
            ax.set_ylabel("Casos", fontsize=8)

        # Oculta eixos extras
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)

        plt.suptitle("Relação Climática vs Casos – Campo Grande/MS",
                     fontsize=13, fontweight="bold")
        salvar_fig("cg_clima_vs_casos")

    # ── 11.7 Indicadores Síntese – Campo Grande ───────────────────────────────
    print_sub("11.7 Indicadores Síntese")
    if "ANO" in df_cg.columns and "casos" in df_cg.columns:
        pop_ref = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942140
        total   = int(df_cg["casos"].sum())
        media_a = total / max(df_cg["ANO"].nunique(), 1)
        inc_med = taxa_inc(media_a, pop_ref)
        pico_semana = df_cg.loc[df_cg["casos"].idxmax()]

        rows_ind = [
            ["Total de Casos (2016-2025)",     fmt_num(total)],
            ["Média Anual de Casos",            fmt_num(media_a, 1)],
            ["Taxa Incidência Média (2016-2025)", fmt_num(inc_med, 1) + "/100k"],
            ["Semana de Maior Incidência",       str(int(pico_semana.get("SE", 0)))],
            ["Casos no Pico",                    fmt_num(int(pico_semana.get("casos", 0)))],
            ["Rt Máximo Registrado",             fmt_num(df_cg["Rt"].max() if "Rt" in df_cg.columns else 0, 2)],
            ["Semanas com Nível 4 (Vermelho)",   fmt_num(int((df_cg["nivel"] == 4).sum())) if "nivel" in df_cg.columns else "–"],
            ["Semanas com Transmissão Ativa",    fmt_num(int(df_cg["transmissao"].sum())) if "transmissao" in df_cg.columns else "–"],
            ["Semanas Receptivas",               fmt_num(int(df_cg["receptivo"].sum())) if "receptivo" in df_cg.columns else "–"],
            ["Ano com Mais Casos",               str(int(df_cg.groupby("ANO")["casos"].sum().idxmax()))],
            ["Ano com Menos Casos",              str(int(df_cg.groupby("ANO")["casos"].sum().idxmin()))],
        ]
        tab_ind = make_table(["Indicador", "Valor"], rows_ind, col_align=["l","r"])
        log.info(f"\n{tab_ind}")
        salvar_txt(tab_ind, "cg_indicadores_sintese",
                   "Indicadores Síntese – Campo Grande/MS")
        resultados["indicadores_cg"] = {r[0]: r[1] for r in rows_ind}

    # ── 11.8 Comparação Campo Grande × Média MS ───────────────────────────────
    print_sub("11.8 Campo Grande vs Média MS")
    if not df_ms.empty and "ANO" in df_ms.columns and "casos" in df_ms.columns:
        ms_anual = df_ms.groupby(["ANO", "municipio_nome"])["casos"].sum().reset_index()
        ms_media_anual = ms_anual.groupby("ANO")["casos"].mean().reset_index()
        ms_media_anual.columns = ["ANO", "media_ms"]

        cg_anual = df_cg.groupby("ANO")["casos"].sum().reset_index()
        cg_anual.columns = ["ANO", "casos_cg"]

        comp = pd.merge(cg_anual, ms_media_anual, on="ANO", how="inner")
        comp["razao"] = comp["casos_cg"] / comp["media_ms"].replace(0, np.nan)

        fig, ax1 = plt.subplots(figsize=(12, 5))
        ax2 = ax1.twinx()
        x = comp["ANO"].astype(int)
        w = 0.35
        ax1.bar(x - w/2, comp["casos_cg"],  width=w, label="Campo Grande",
                color=COR_PRINCIPAL, alpha=0.8)
        ax1.bar(x + w/2, comp["media_ms"],  width=w, label="Média MS",
                color=COR_SECUNDARIA, alpha=0.8)
        ax2.plot(x, comp["razao"], color=COR_ALERTA, marker="D",
                 linewidth=2, label="Razão CG/Média MS")
        ax2.axhline(1.0, color="gray", linestyle="--", linewidth=1)
        ax1.set_title("Campo Grande vs Média dos Municípios de MS",
                      fontweight="bold")
        ax1.set_xlabel("Ano")
        ax1.set_ylabel("Casos")
        ax2.set_ylabel("Razão CG / Média MS", color=COR_ALERTA)
        ax1.set_xticks(x)
        ax1.set_xticklabels(x, rotation=45)
        lines1, labs1 = ax1.get_legend_handles_labels()
        lines2, labs2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labs1 + labs2, loc="upper left")
        salvar_fig("cg_vs_media_ms")

        rows_c = [[int(r["ANO"]), fmt_num(r["casos_cg"]),
                   fmt_num(r["media_ms"], 1), fmt_num(r.get("razao", 0), 2)]
                  for _, r in comp.iterrows()]
        tab_c = make_table(
            ["Ano", "Casos CG", "Média MS", "Razão"],
            rows_c, col_align=["c","r","r","r"]
        )
        log.info(f"\n{tab_c}")
        salvar_txt(tab_c, "cg_vs_media_ms", "Campo Grande vs Média MS por Ano")

    log.info("  Análise Campo Grande concluída.")
    return resultados




In [27]:
# =============================================================================
# SEÇÃO 12 – ANÁLISE MUNICIPAL: TODOS OS MUNICÍPIOS DE MS


In [28]:
# =============================================================================

def analise_municipal_ms(df_ms: pd.DataFrame) -> pd.DataFrame:
    """
    Análise completa de todos os municípios de Mato Grosso do Sul:
    rankings, comparativos, mapas e tabelas.
    """
    print_section("ANÁLISE MUNICIPAL – MATO GROSSO DO SUL")

    if df_ms.empty:
        log.warning("  DataFrame MS está vazio!")
        return pd.DataFrame()

    # ── 12.1 Agregação anual por município ────────────────────────────────────
    print_sub("12.1 Agregação Anual por Município")
    agg_dict = {
        "casos":         "sum",
        "casos_est":     "sum",
        "p_rt1":         "mean",
        "Rt":            "mean",
        "p_inc100k":     "mean",
        "taxa_inc_calc": "mean",
        "nivel":         "max",
        "alerta_ativo":  "sum",
        "transmissao":   "sum",
    }
    valid_agg = {k: v for k, v in agg_dict.items() if k in df_ms.columns}
    grp_cols  = [c for c in ["ANO", "municipio_nome"] if c in df_ms.columns]

    df_mun_ano = df_ms.groupby(grp_cols, as_index=False, observed=True).agg(valid_agg)

    # Adiciona população de referência e calcula taxa de incidência
    if "municipio_nome" in df_mun_ano.columns:
        df_mun_ano["pop_ref"] = df_mun_ano["municipio_nome"].map(POP_MUNICIPIOS_MS)
        df_mun_ano["pop_ref"] = df_mun_ano["pop_ref"].fillna(50_000)
        df_mun_ano["taxa_inc_pop"] = df_mun_ano.apply(
            lambda r: taxa_inc(r["casos"], r["pop_ref"]), axis=1
        )
        df_mun_ano["risco"] = df_mun_ano["taxa_inc_pop"].apply(classificar_risco)

    # ── 12.2 Ranking municipal por casos totais ───────────────────────────────
    print_sub("12.2 Ranking Municipal – Total de Casos (2016-2025)")
    total_mun = df_ms.groupby("municipio_nome")["casos"].sum().reset_index()
    total_mun = total_mun.sort_values("casos", ascending=False).reset_index(drop=True)
    total_mun["rank"]    = total_mun.index + 1
    total_mun["pop_ref"] = total_mun["municipio_nome"].map(POP_MUNICIPIOS_MS).fillna(50_000)
    total_mun["taxa_inc_total"] = total_mun.apply(
        lambda r: taxa_inc(r["casos"], r["pop_ref"]), axis=1
    )
    total_mun["risco"] = total_mun["taxa_inc_total"].apply(classificar_risco)

    # Top 20 municípios
    top20 = total_mun.head(20)
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    ax = axes[0]
    cores_rank = [COR_PRINCIPAL if m == "Campo Grande" else COR_SECUNDARIA
                  for m in top20["municipio_nome"]]
    ax.barh(top20["municipio_nome"][::-1], top20["casos"][::-1],
            color=cores_rank[::-1], edgecolor="white")
    ax.set_title("Top 20 Municípios MS – Total de Casos (2016–2025)",
                 fontweight="bold")
    ax.set_xlabel("Total de Casos")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: fmt_num(int(x))
    ))
    # Destaca CG
    if "Campo Grande" in top20["municipio_nome"].values:
        idx_cg = top20["municipio_nome"].tolist().index("Campo Grande")
        ax.patches[len(top20) - 1 - idx_cg].set_edgecolor("black")
        ax.patches[len(top20) - 1 - idx_cg].set_linewidth(1.5)

    ax2 = axes[1]
    top20t = total_mun.sort_values("taxa_inc_total", ascending=False).head(20)
    cores_t = [COR_ALERTA if m == "Campo Grande" else "#A9CCE3"
               for m in top20t["municipio_nome"]]
    ax2.barh(top20t["municipio_nome"][::-1], top20t["taxa_inc_total"][::-1],
             color=cores_t[::-1], edgecolor="white")
    ax2.set_title("Top 20 Municípios MS – Taxa de Incidência/100k (2016–2025)",
                  fontweight="bold")
    ax2.set_xlabel("Taxa Incidência / 100k hab")

    plt.suptitle("Ranking Municipal – Dengue em Mato Grosso do Sul",
                 fontsize=14, fontweight="bold")
    salvar_fig("ms_ranking_municipal_casos_taxa")

    # Tabela top 20
    rows_rank = [[r["rank"], r["municipio_nome"],
                  fmt_num(int(r["casos"])), fmt_num(r["taxa_inc_total"], 1),
                  r["risco"]]
                 for _, r in top20.iterrows()]
    tab_rank = make_table(
        ["Rank", "Município", "Total Casos", "Taxa/100k", "Risco"],
        rows_rank, col_align=["c","l","r","r","l"]
    )
    log.info(f"\n{tab_rank}")
    salvar_txt(tab_rank, "ms_ranking_top20_casos", "Ranking Top 20 – MS por Casos")

    # Tabela completa
    rows_all = [[r["rank"], r["municipio_nome"],
                 fmt_num(int(r["casos"])), fmt_num(r["taxa_inc_total"], 1),
                 r["risco"]]
                for _, r in total_mun.iterrows()]
    tab_all = make_table(
        ["Rank", "Município", "Total Casos", "Taxa/100k", "Risco"],
        rows_all, col_align=["c","l","r","r","l"]
    )
    salvar_txt(tab_all, "ms_ranking_completo_casos", "Ranking Completo – MS por Casos")
    salvar_log_tabela(tab_all, "ms_ranking_completo", "Ranking Completo MS")

    # CSV do ranking
    total_mun.to_csv(OUTPUT_DIR / "dados" / "ms_ranking_municipal.csv", index=False)
    log.info("  [CSV] ms_ranking_municipal.csv")

    # ── 12.3 Evolução temporal dos Top 10 municípios ──────────────────────────
    print_sub("12.3 Evolução Temporal – Top 10 Municípios")
    top10_muns = total_mun.head(10)["municipio_nome"].tolist()
    df_top10   = df_ms[df_ms["municipio_nome"].isin(top10_muns)]

    if not df_top10.empty and "ANO" in df_top10.columns:
        evol = df_top10.groupby(["ANO", "municipio_nome"])["casos"].sum().reset_index()
        fig, ax = plt.subplots(figsize=(14, 6))
        palette = plt.cm.get_cmap("tab10", len(top10_muns))
        for i, mun in enumerate(top10_muns):
            sub = evol[evol["municipio_nome"] == mun].sort_values("ANO")
            lw  = 3.0 if mun == "Campo Grande" else 1.5
            ls  = "-" if mun == "Campo Grande" else "--"
            ax.plot(sub["ANO"].astype(int), sub["casos"],
                    label=mun, color=palette(i), linewidth=lw, linestyle=ls,
                    marker="o", markersize=5)
        ax.set_title("Evolução Anual – Top 10 Municípios MS", fontweight="bold")
        ax.set_xlabel("Ano")
        ax.set_ylabel("Casos")
        ax.legend(ncol=2, fontsize=8)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: fmt_num(int(x))
        ))
        salvar_fig("ms_top10_evolucao_anual")

    # ── 12.4 Posição de Campo Grande frente à média estadual ─────────────────
    print_sub("12.4 Campo Grande vs Média Estadual")
    media_ms = total_mun["casos"].mean()
    mediana_ms = total_mun["casos"].median()
    cg_total = total_mun[total_mun["municipio_nome"] == "Campo Grande"]["casos"].values
    cg_total = float(cg_total[0]) if len(cg_total) > 0 else 0
    cg_rank  = total_mun[total_mun["municipio_nome"] == "Campo Grande"]["rank"].values
    cg_rank  = int(cg_rank[0]) if len(cg_rank) > 0 else 0
    n_muns   = len(total_mun)

    rows_pos = [
        ["Total de municípios analisados",  fmt_num(n_muns)],
        ["Casos totais – Campo Grande",     fmt_num(cg_total)],
        ["Média estadual de casos",         fmt_num(media_ms, 1)],
        ["Mediana estadual de casos",       fmt_num(mediana_ms, 1)],
        ["Posição de CG no ranking MS",     f"{cg_rank}º de {n_muns}"],
        ["CG acima da média MS?",           "SIM" if cg_total > media_ms else "NÃO"],
        ["Múltiplo da média estadual",      fmt_num(cg_total / media_ms if media_ms > 0 else 0, 1) + "x"],
    ]
    tab_pos = make_table(["Indicador", "Valor"], rows_pos, col_align=["l","r"])
    log.info(f"\n{tab_pos}")
    salvar_txt(tab_pos, "ms_posicao_cg_vs_ms",
               "Posição de Campo Grande vs Média Estadual")

    # ── 12.5 Mapa de calor: Municípios × Ano ─────────────────────────────────
    print_sub("12.5 Heatmap Municípios × Ano")
    if "ANO" in df_mun_ano.columns and "municipio_nome" in df_mun_ano.columns:
        pivot_mun = df_mun_ano.pivot_table(
            index="municipio_nome", columns="ANO", values="casos", aggfunc="sum"
        ).fillna(0)
        pivot_mun = pivot_mun.sort_values(
            by=max(pivot_mun.columns), ascending=False
        ).head(30)

        fig, ax = plt.subplots(figsize=(14, 12))
        sns.heatmap(pivot_mun, annot=True, fmt=".0f", cmap="YlOrRd",
                    linewidths=0.2, ax=ax,
                    cbar_kws={"label": "Casos"}, annot_kws={"size": 7})
        ax.set_title("Top 30 Municípios MS – Casos por Ano (Heatmap)",
                     fontsize=13, fontweight="bold")
        ax.set_xlabel("Ano")
        ax.set_ylabel("Município")
        plt.xticks(rotation=45)
        plt.yticks(rotation=0, fontsize=8)
        salvar_fig("ms_heatmap_municipios_ano")

    # ── 12.6 Série temporal agregada MS ──────────────────────────────────────
    print_sub("12.6 Série Temporal Agregada – Estado MS")
    if "data_SE" in df_ms.columns and "casos" in df_ms.columns:
        ms_semanal = df_ms.groupby("data_SE")["casos"].sum().reset_index()
        ms_semanal = ms_semanal.sort_values("data_SE")
        fig, ax = plt.subplots(figsize=(16, 5))
        ax.bar(ms_semanal["data_SE"], ms_semanal["casos"],
               color="#AED6F1", alpha=0.6, label="Casos Semanais")
        mm = ms_semanal["casos"].rolling(12, min_periods=1).mean()
        ax.plot(ms_semanal["data_SE"], mm, color=COR_PRINCIPAL,
                linewidth=2, label="Média Móvel 12 sem")
        ax.set_title("Série Temporal – Todos os Municípios MS (Soma Semanal)",
                     fontweight="bold")
        ax.set_xlabel("Semana Epidemiológica")
        ax.set_ylabel("Casos")
        ax.legend()
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: fmt_num(int(x))
        ))
        salvar_fig("ms_serie_temporal_agregada")

    log.info("  Análise municipal MS concluída.")
    return df_mun_ano




In [29]:
# =============================================================================
# SEÇÃO 13 – ANÁLISE NACIONAL: CAPITAIS BRASILEIRAS


In [30]:
# =============================================================================

def analise_capitais(df_cap: pd.DataFrame) -> pd.DataFrame:
    """
    Análise das capitais brasileiras:
    ranking nacional, comparação com Campo Grande,
    posição de MS frente à média nacional.
    """
    print_section("ANÁLISE NACIONAL – CAPITAIS BRASILEIRAS")

    if df_cap.empty:
        log.warning("  DataFrame de capitais está vazio!")
        return pd.DataFrame()

    # ── 13.1 Total por capital (2016-2025) ────────────────────────────────────
    print_sub("13.1 Total de Casos por Capital")
    total_cap = df_cap.groupby("municipio_nome").agg(
        casos=("casos", "sum"),
        casos_est=("casos_est", "sum") if "casos_est" in df_cap.columns else ("casos", "sum"),
        rt_medio=("Rt", "mean") if "Rt" in df_cap.columns else ("casos", "count"),
        nivel_max=("nivel", "max") if "nivel" in df_cap.columns else ("casos", "count"),
    ).reset_index()

    # Adiciona UF, população e taxa de incidência
    total_cap["UF"]       = total_cap["municipio_nome"].map(CAPITAIS_UF)
    total_cap["REGIAO"]   = total_cap["UF"].map(REGIAO_UF)
    total_cap["pop_ref"]  = total_cap["municipio_nome"].map(POP_CAPITAIS).fillna(1_000_000)
    total_cap["taxa_inc"] = total_cap.apply(
        lambda r: taxa_inc(r["casos"], r["pop_ref"]), axis=1
    )
    total_cap["risco"]    = total_cap["taxa_inc"].apply(classificar_risco)

    # Ranking por casos absolutos
    rank_abs = total_cap.sort_values("casos", ascending=False).reset_index(drop=True)
    rank_abs["rank_abs"] = rank_abs.index + 1

    # Ranking por taxa de incidência
    rank_taxa = total_cap.sort_values("taxa_inc", ascending=False).reset_index(drop=True)
    rank_taxa["rank_taxa"] = rank_taxa.index + 1

    # ── 13.2 Gráficos de ranking ──────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))

    # Barras horizontais – casos absolutos
    ax = axes[0]
    cores_cap = [COR_PRINCIPAL if m == "Campo Grande" else "#AED6F1"
                 for m in rank_abs["municipio_nome"]]
    ax.barh(rank_abs["municipio_nome"][::-1], rank_abs["casos"][::-1],
            color=cores_cap[::-1], edgecolor="white")
    ax.set_title("Ranking Capitais – Total de Casos (2016–2025)",
                 fontweight="bold", fontsize=11)
    ax.set_xlabel("Total de Casos")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: fmt_num(int(x))
    ))

    # Barras horizontais – taxa incidência
    ax2 = axes[1]
    cores_t = [COR_ALERTA if m == "Campo Grande" else "#A9CCE3"
               for m in rank_taxa["municipio_nome"]]
    ax2.barh(rank_taxa["municipio_nome"][::-1], rank_taxa["taxa_inc"][::-1],
             color=cores_t[::-1], edgecolor="white")
    ax2.set_title("Ranking Capitais – Taxa de Incidência/100k (2016–2025)",
                  fontweight="bold", fontsize=11)
    ax2.set_xlabel("Taxa Incidência / 100k hab")

    plt.suptitle("Ranking Nacional – Dengue nas Capitais Brasileiras",
                 fontsize=14, fontweight="bold")
    salvar_fig("cap_ranking_nacional")

    # ── 13.3 Tabelas de ranking ───────────────────────────────────────────────
    rows_r1 = [[int(r["rank_abs"]), r["municipio_nome"], r.get("UF","?"),
                r.get("REGIAO","?"), fmt_num(int(r["casos"])),
                fmt_num(r["taxa_inc"], 1), r["risco"]]
               for _, r in rank_abs.iterrows()]
    tab_r1 = make_table(
        ["Rank", "Capital", "UF", "Região", "Casos", "Taxa/100k", "Risco"],
        rows_r1, col_align=["c","l","c","l","r","r","l"]
    )
    log.info(f"\n{tab_r1}")
    salvar_txt(tab_r1, "cap_ranking_por_casos",
               "Ranking Capitais – Total de Casos")
    salvar_log_tabela(tab_r1, "cap_ranking_por_casos", "Ranking Capitais – Casos")

    rows_r2 = [[int(r["rank_taxa"]), r["municipio_nome"], r.get("UF","?"),
                fmt_num(r["taxa_inc"], 1), fmt_num(int(r["casos"])), r["risco"]]
               for _, r in rank_taxa.iterrows()]
    tab_r2 = make_table(
        ["Rank", "Capital", "UF", "Taxa/100k", "Casos", "Risco"],
        rows_r2, col_align=["c","l","c","r","r","l"]
    )
    salvar_txt(tab_r2, "cap_ranking_por_taxa",
               "Ranking Capitais – Taxa de Incidência")
    salvar_log_tabela(tab_r2, "cap_ranking_por_taxa", "Ranking Capitais – Taxa")

    # ── 13.4 Posição de Campo Grande no ranking nacional ──────────────────────
    print_sub("13.4 Campo Grande vs Média Nacional das Capitais")
    media_nac    = total_cap["casos"].mean()
    mediana_nac  = total_cap["casos"].median()
    media_taxa   = total_cap["taxa_inc"].mean()
    cg_row       = rank_abs[rank_abs["municipio_nome"] == "Campo Grande"]
    cg_rank_abs  = int(cg_row["rank_abs"].values[0]) if len(cg_row) > 0 else "N/A"
    cg_row_t     = rank_taxa[rank_taxa["municipio_nome"] == "Campo Grande"]
    cg_rank_taxa = int(cg_row_t["rank_taxa"].values[0]) if len(cg_row_t) > 0 else "N/A"
    n_caps       = len(total_cap)

    rows_pos = [
        ["Total de capitais analisadas",         fmt_num(n_caps)],
        ["Ranking CG – casos absolutos",         f"{cg_rank_abs}º de {n_caps}"],
        ["Ranking CG – taxa de incidência",      f"{cg_rank_taxa}º de {n_caps}"],
        ["Média nacional – casos",               fmt_num(media_nac, 1)],
        ["Mediana nacional – casos",             fmt_num(mediana_nac, 1)],
        ["Média nacional – taxa/100k",           fmt_num(media_taxa, 1)],
        ["CG acima da média nacional (casos)?",  "SIM" if (len(cg_row) > 0 and cg_row["casos"].values[0] > media_nac) else "NÃO"],
        ["Capital com mais casos",               rank_abs.iloc[0]["municipio_nome"]],
        ["Capital com menos casos",              rank_abs.iloc[-1]["municipio_nome"]],
        ["Capital com maior taxa/100k",          rank_taxa.iloc[0]["municipio_nome"]],
    ]
    tab_pos = make_table(["Indicador", "Valor"], rows_pos, col_align=["l","r"])
    log.info(f"\n{tab_pos}")
    salvar_txt(tab_pos, "cap_posicao_cg_vs_nacional",
               "Posição de Campo Grande vs Média Nacional")

    # ── 13.5 Evolução anual por capital ──────────────────────────────────────
    print_sub("13.5 Evolução Anual – Top 10 Capitais")
    top10_cap = rank_abs.head(10)["municipio_nome"].tolist()
    evol_cap  = df_cap[df_cap["municipio_nome"].isin(top10_cap)]

    if not evol_cap.empty and "ANO" in evol_cap.columns:
        evol_ano = evol_cap.groupby(["ANO", "municipio_nome"])["casos"].sum().reset_index()
        fig, ax = plt.subplots(figsize=(14, 6))
        palette = plt.cm.get_cmap("tab10", len(top10_cap))
        for i, cap in enumerate(top10_cap):
            sub = evol_ano[evol_ano["municipio_nome"] == cap].sort_values("ANO")
            lw  = 3.0 if cap == "Campo Grande" else 1.5
            ax.plot(sub["ANO"].astype(int), sub["casos"],
                    label=cap, color=palette(i), linewidth=lw, marker="o",
                    markersize=5)
        ax.set_title("Evolução Anual – Top 10 Capitais (2016–2025)",
                     fontweight="bold")
        ax.set_xlabel("Ano")
        ax.set_ylabel("Casos")
        ax.legend(ncol=2, fontsize=8)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: fmt_num(int(x))
        ))
        salvar_fig("cap_top10_evolucao_anual")

    # ── 13.6 Comparação por região ────────────────────────────────────────────
    print_sub("13.6 Comparação por Região Brasileira")
    if "REGIAO" in total_cap.columns:
        reg = total_cap.groupby("REGIAO").agg(
            casos=("casos", "sum"),
            taxa_media=("taxa_inc", "mean"),
            n_capitais=("municipio_nome", "count"),
        ).reset_index().sort_values("casos", ascending=False)

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].bar(reg["REGIAO"], reg["casos"],
                    color=COR_SECUNDARIA, edgecolor="white")
        axes[0].set_title("Casos por Região (2016–2025)", fontweight="bold")
        axes[0].set_ylabel("Total de Casos")
        axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: fmt_num(int(x))
        ))
        axes[1].bar(reg["REGIAO"], reg["taxa_media"],
                    color=COR_ALERTA, edgecolor="white")
        axes[1].set_title("Taxa Média de Incidência por Região", fontweight="bold")
        axes[1].set_ylabel("Taxa / 100k")
        plt.suptitle("Dengue por Região Brasileira – Capitais",
                     fontsize=13, fontweight="bold")
        salvar_fig("cap_comparacao_regional")

        rows_reg = [[r["REGIAO"], fmt_num(r["n_capitais"]),
                     fmt_num(int(r["casos"])), fmt_num(r["taxa_media"], 1)]
                    for _, r in reg.iterrows()]
        tab_reg = make_table(
            ["Região", "Capitais", "Total Casos", "Taxa Média/100k"],
            rows_reg, col_align=["l","c","r","r"]
        )
        log.info(f"\n{tab_reg}")
        salvar_txt(tab_reg, "cap_ranking_regional",
                   "Ranking por Região – Capitais")

    # CSV do ranking nacional
    rank_abs.to_csv(OUTPUT_DIR / "dados" / "ranking_nacional_capitais.csv",
                    index=False)
    log.info("  [CSV] ranking_nacional_capitais.csv")

    log.info("  Análise capitais concluída.")
    return rank_abs




In [31]:
# =============================================================================
# SEÇÃO 14 – RANKINGS CONSOLIDADOS E COMPARATIVOS


In [32]:
# =============================================================================

def rankings_consolidados(df_cg: pd.DataFrame,
                           df_ms: pd.DataFrame,
                           df_cap: pd.DataFrame) -> None:
    """
    Gera rankings consolidados: município × ano, estado × período,
    comparativos cruzados e análises de tendência.
    """
    print_section("RANKINGS CONSOLIDADOS E COMPARATIVOS")

    # ── 14.1 Ranking MS por ano ───────────────────────────────────────────────
    print_sub("14.1 Ranking MS por Ano")
    if not df_ms.empty and {"ANO", "municipio_nome", "casos"}.issubset(df_ms.columns):
        for ano in sorted(df_ms["ANO"].unique()):
            if int(ano) not in range(2016, 2026):
                continue
            df_ano = df_ms[df_ms["ANO"] == ano]
            tot    = df_ano.groupby("municipio_nome")["casos"].sum().reset_index()
            tot    = tot.sort_values("casos", ascending=False).reset_index(drop=True)
            tot["pop"]      = tot["municipio_nome"].map(POP_MUNICIPIOS_MS).fillna(50_000)
            tot["taxa_inc"] = tot.apply(lambda r: taxa_inc(r["casos"], r["pop"]), axis=1)
            tot["rank"]     = tot.index + 1

        # Últimos 3 anos – ranking de taxa de incidência
        anos_recentes = sorted(df_ms["ANO"].unique())[-3:]
        fig, axes = plt.subplots(1, len(anos_recentes), figsize=(6*len(anos_recentes), 8))
        if len(anos_recentes) == 1:
            axes = [axes]
        for ax, ano in zip(axes, anos_recentes):
            df_ano = df_ms[df_ms["ANO"] == ano]
            tot    = df_ano.groupby("municipio_nome")["casos"].sum().reset_index()
            tot["pop"]      = tot["municipio_nome"].map(POP_MUNICIPIOS_MS).fillna(50_000)
            tot["taxa_inc"] = tot.apply(lambda r: taxa_inc(r["casos"], r["pop"]), axis=1)
            top15  = tot.sort_values("taxa_inc", ascending=False).head(15)
            cores  = [COR_PRINCIPAL if m == "Campo Grande" else COR_SECUNDARIA
                      for m in top15["municipio_nome"]]
            ax.barh(top15["municipio_nome"][::-1], top15["taxa_inc"][::-1],
                    color=cores[::-1], edgecolor="white")
            ax.set_title(f"Ano {int(ano)}", fontweight="bold")
            ax.set_xlabel("Taxa/100k")
        plt.suptitle("Ranking MS – Taxa de Incidência por Ano", fontsize=13,
                     fontweight="bold")
        salvar_fig("ms_ranking_taxa_por_ano")

    # ── 14.2 Ranking capitais por ano ─────────────────────────────────────────
    print_sub("14.2 Ranking Capitais por Ano")
    if not df_cap.empty and {"ANO", "municipio_nome", "casos"}.issubset(df_cap.columns):
        anos_recentes_cap = sorted(df_cap["ANO"].unique())[-3:]
        fig, axes = plt.subplots(1, len(anos_recentes_cap),
                                  figsize=(7*len(anos_recentes_cap), 10))
        if len(anos_recentes_cap) == 1:
            axes = [axes]
        for ax, ano in zip(axes, anos_recentes_cap):
            df_ano = df_cap[df_cap["ANO"] == ano]
            tot    = df_ano.groupby("municipio_nome")["casos"].sum().reset_index()
            tot["pop"]      = tot["municipio_nome"].map(POP_CAPITAIS).fillna(1_000_000)
            tot["taxa_inc"] = tot.apply(lambda r: taxa_inc(r["casos"], r["pop"]), axis=1)
            top20  = tot.sort_values("taxa_inc", ascending=False).head(20)
            cores  = [COR_PRINCIPAL if m == "Campo Grande" else "#AED6F1"
                      for m in top20["municipio_nome"]]
            ax.barh(top20["municipio_nome"][::-1], top20["taxa_inc"][::-1],
                    color=cores[::-1], edgecolor="white")
            ax.set_title(f"Ano {int(ano)}", fontweight="bold", fontsize=10)
            ax.set_xlabel("Taxa/100k", fontsize=9)
            ax.tick_params(labelsize=8)
        plt.suptitle("Ranking Capitais – Taxa de Incidência por Ano",
                     fontsize=13, fontweight="bold")
        salvar_fig("cap_ranking_taxa_por_ano")

    # ── 14.3 Tabela comparativa CG × Top 5 MS × Top 5 Capitais ──────────────
    print_sub("14.3 Tabela Comparativa Cross-Dataset")
    rows_comp = []
    for nome, df, pop_dict in [
        ("Campo Grande", df_cg, {"Campo Grande": 942140}),
        ("Top 5 MS",     df_ms, POP_MUNICIPIOS_MS),
        ("Top 5 Capitais", df_cap, POP_CAPITAIS),
    ]:
        if df.empty or "casos" not in df.columns:
            continue
        if "municipio_nome" in df.columns:
            tot = df.groupby("municipio_nome")["casos"].sum()
            for mun, casos in tot.nlargest(5).items():
                pop = pop_dict.get(mun, 50_000)
                rows_comp.append([
                    nome, mun, fmt_num(int(casos)),
                    fmt_num(taxa_inc(casos, pop), 1),
                    classificar_risco(taxa_inc(casos, pop)),
                ])

    if rows_comp:
        tab_comp = make_table(
            ["Dataset", "Município/Capital", "Casos", "Taxa/100k", "Risco"],
            rows_comp, col_align=["l","l","r","r","l"]
        )
        log.info(f"\n{tab_comp}")
        salvar_txt(tab_comp, "rankings_comparativo_cruzado",
                   "Comparativo Cruzado – CG × MS × Capitais")

    # ── 14.4 Ano com pior situação epidêmica ─────────────────────────────────
    print_sub("14.4 Análise dos Anos Epidêmicos")
    for nome, df in [("Campo Grande", df_cg), ("Municípios MS", df_ms),
                     ("Capitais", df_cap)]:
        if df.empty or "ANO" not in df.columns or "casos" not in df.columns:
            continue
        por_ano = df.groupby("ANO")["casos"].sum()
        if por_ano.empty:
            continue
        pior_ano   = por_ano.idxmax()
        melhor_ano = por_ano.idxmin()
        log.info(f"  {nome}: pior ano = {int(pior_ano)} "
                 f"({fmt_num(int(por_ano.max()))} casos) | "
                 f"melhor = {int(melhor_ano)} "
                 f"({fmt_num(int(por_ano.min()))} casos)")

    log.info("  Rankings consolidados concluídos.")




In [33]:
# =============================================================================
# SEÇÃO 15 – MACHINE LEARNING: CLUSTERIZAÇÃO


In [34]:
# =============================================================================

def _preparar_features_municipios(df_ms: pd.DataFrame) -> Optional[pd.DataFrame]:
    """
    Prepara DataFrame de features agregadas por município para clusterização.
    """
    if df_ms.empty or not HAS_SKLEARN:
        return None

    agg = {
        "casos":         "sum",
        "taxa_inc_calc": "mean",
        "Rt":            "mean",
        "p_rt1":         "mean",
        "nivel":         "mean",
        "transmissao":   "sum",
        "receptivo":     "sum",
        "tempmed":       "mean",
        "umidmed":       "mean",
    }
    valid_agg = {k: v for k, v in agg.items() if k in df_ms.columns}
    df_feat   = df_ms.groupby("municipio_nome", as_index=False).agg(valid_agg)

    # Adiciona n_semanas
    n_sem = df_ms.groupby("municipio_nome")["SE"].count().reset_index()
    n_sem.columns = ["municipio_nome", "n_semanas"]
    df_feat = pd.merge(df_feat, n_sem, on="municipio_nome", how="left")

    # Adiciona população
    df_feat["pop"] = df_feat["municipio_nome"].map(POP_MUNICIPIOS_MS).fillna(50_000)
    df_feat["taxa_casos_pop"] = df_feat.apply(
        lambda r: taxa_inc(r["casos"], r["pop"]), axis=1
    )

    # Remove linhas com muitos NaN
    feat_cols = [c for c in df_feat.columns if c != "municipio_nome"]
    df_feat[feat_cols] = df_feat[feat_cols].fillna(df_feat[feat_cols].median())

    return df_feat


def ml_clusterizacao(df_ms: pd.DataFrame) -> Optional[pd.DataFrame]:
    """
    Clusterização de municípios MS por perfil epidemiológico.
    Métodos: KMeans (cotovelo + silhouette), DBSCAN, GMM.
    """
    print_section("MACHINE LEARNING – CLUSTERIZAÇÃO DE MUNICÍPIOS")

    if not HAS_SKLEARN:
        log.warning("  scikit-learn não disponível. Pulando clusterização.")
        return None

    df_feat = _preparar_features_municipios(df_ms)
    if df_feat is None or df_feat.empty:
        return None

    feat_cols = [c for c in ["casos", "taxa_casos_pop", "Rt", "p_rt1",
                              "nivel", "transmissao", "tempmed", "umidmed"]
                 if c in df_feat.columns]
    X_raw = df_feat[feat_cols].fillna(0).values
    nomes = df_feat["municipio_nome"].values

    # Normalização
    scaler = RobustScaler()
    X      = scaler.fit_transform(X_raw)

    # ── 15.1 Método do Cotovelo ───────────────────────────────────────────────
    print_sub("15.1 Método do Cotovelo – KMeans")
    inertias = []
    silhouettes = []
    k_range = range(2, min(11, len(X) - 1))

    for k in k_range:
        km  = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
        km.fit(X)
        inertias.append(km.inertia_)
        if len(set(km.labels_)) > 1:
            silhouettes.append(silhouette_score(X, km.labels_))
        else:
            silhouettes.append(0)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(list(k_range), inertias, marker="o", color=COR_PRINCIPAL)
    axes[0].set_title("Método do Cotovelo – Inércia KMeans", fontweight="bold")
    axes[0].set_xlabel("Número de Clusters (k)")
    axes[0].set_ylabel("Inércia")
    axes[1].plot(list(k_range), silhouettes, marker="s", color=COR_SECUNDARIA)
    axes[1].set_title("Silhouette Score por k", fontweight="bold")
    axes[1].set_xlabel("Número de Clusters (k)")
    axes[1].set_ylabel("Silhouette Score")
    best_k_idx = int(np.argmax(silhouettes))
    best_k     = list(k_range)[best_k_idx]
    axes[1].axvline(best_k, color="red", linestyle="--", linewidth=1.5,
                    label=f"Melhor k = {best_k}")
    axes[1].legend()
    plt.suptitle("Seleção de Clusters – Municípios MS", fontsize=13, fontweight="bold")
    salvar_fig("ml_cotovelo_silhouette_ms")
    log.info(f"  Melhor k (silhouette): {best_k}")

    # ── 15.2 KMeans com k ótimo ───────────────────────────────────────────────
    print_sub(f"15.2 KMeans – k = {best_k}")
    km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10, max_iter=500)
    labels_km = km_final.fit_predict(X)
    df_feat["cluster_kmeans"] = labels_km

    sil_final = silhouette_score(X, labels_km)
    db_score  = davies_bouldin_score(X, labels_km)
    ch_score  = calinski_harabasz_score(X, labels_km)
    log.info(f"  KMeans Silhouette: {sil_final:.4f} | "
             f"Davies-Bouldin: {db_score:.4f} | "
             f"Calinski-Harabasz: {ch_score:.1f}")
    _inc("modelos_treinados")

    # ── 15.3 PCA para visualização 2D ─────────────────────────────────────────
    print_sub("15.3 PCA – Visualização dos Clusters")
    pca   = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)

    fig, ax = plt.subplots(figsize=(11, 8))
    colors_cl = plt.cm.get_cmap("tab10", best_k)
    for cl in range(best_k):
        mask = labels_km == cl
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   color=colors_cl(cl), label=f"Cluster {cl+1}",
                   s=60, alpha=0.7, edgecolors="white", linewidth=0.5)
        # Anotação Campo Grande
        for j, nome in enumerate(nomes):
            if mask[j] and nome == "Campo Grande":
                ax.annotate("Campo Grande",
                            (X_pca[j, 0], X_pca[j, 1]),
                            fontsize=8, fontweight="bold",
                            xytext=(5, 5), textcoords="offset points",
                            color="black")
    var_exp = pca.explained_variance_ratio_
    ax.set_title(f"KMeans – {best_k} Clusters (PCA 2D)\n"
                 f"Variância explicada: PC1={var_exp[0]:.1%}, PC2={var_exp[1]:.1%}",
                 fontweight="bold")
    ax.set_xlabel(f"PC1 ({var_exp[0]:.1%})")
    ax.set_ylabel(f"PC2 ({var_exp[1]:.1%})")
    ax.legend(loc="best")
    salvar_fig("ml_kmeans_pca_clusters_ms")

    # ── 15.4 Perfil de cada cluster ───────────────────────────────────────────
    print_sub("15.4 Perfil dos Clusters")
    perfil = df_feat.groupby("cluster_kmeans")[feat_cols].mean()
    perfil_rows = []
    for cl, row in perfil.iterrows():
        muns_cl = df_feat[df_feat["cluster_kmeans"] == cl]["municipio_nome"].tolist()
        n_muns  = len(muns_cl)
        perfil_rows.append(
            [f"Cluster {cl+1}", fmt_num(n_muns)] +
            [fmt_num(row[c], 2) for c in feat_cols]
        )

    tab_perf = make_table(
        ["Cluster", "N Municípios"] + feat_cols,
        perfil_rows
    )
    log.info(f"\n{tab_perf}")
    salvar_txt(tab_perf, "ml_kmeans_perfil_clusters",
               "Perfil dos Clusters KMeans – Municípios MS")

    # Gráfico de radar por cluster
    if len(feat_cols) >= 3:
        angles = [n / len(feat_cols) * 2 * math.pi for n in range(len(feat_cols))]
        angles += angles[:1]

        fig, ax = plt.subplots(figsize=(10, 8),
                               subplot_kw=dict(polar=True))
        colors_r = plt.cm.get_cmap("tab10", best_k)
        for cl in range(best_k):
            vals = [perfil.loc[cl, c] for c in feat_cols]
            # Normaliza 0-1 para radar
            mn = [df_feat[c].min() for c in feat_cols]
            mx = [df_feat[c].max() for c in feat_cols]
            vals_n = [(v - mn[i]) / max(mx[i] - mn[i], 1e-9)
                      for i, v in enumerate(vals)]
            vals_n += vals_n[:1]
            ax.plot(angles, vals_n, color=colors_r(cl), linewidth=2,
                    label=f"Cluster {cl+1}")
            ax.fill(angles, vals_n, color=colors_r(cl), alpha=0.1)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(feat_cols, fontsize=8)
        ax.set_title("Radar – Perfil dos Clusters de Municípios MS",
                     fontweight="bold", pad=20)
        ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
        salvar_fig("ml_kmeans_radar_clusters_ms")

    # ── 15.5 DBSCAN ───────────────────────────────────────────────────────────
    print_sub("15.5 DBSCAN – Detecção de Anomalias")
    if len(X) > 5:
        eps_val = 0.8
        db = DBSCAN(eps=eps_val, min_samples=3)
        labels_db = db.fit_predict(X)
        n_noise   = int(np.sum(labels_db == -1))
        n_cl_db   = len(set(labels_db)) - (1 if -1 in labels_db else 0)
        log.info(f"  DBSCAN: {n_cl_db} clusters | {n_noise} anomalias "
                 f"(eps={eps_val})")
        df_feat["cluster_dbscan"] = labels_db
        _inc("modelos_treinados")

        # Municípios anômalos
        anomalias = df_feat[df_feat["cluster_dbscan"] == -1]["municipio_nome"].tolist()
        if anomalias:
            log.info(f"  Municípios anômalos (DBSCAN): {', '.join(anomalias[:10])}")

    # ── 15.6 Gaussian Mixture Model ──────────────────────────────────────────
    print_sub("15.6 Gaussian Mixture Model (GMM)")
    gmm = GaussianMixture(n_components=best_k, random_state=42, max_iter=200)
    labels_gmm = gmm.fit_predict(X)
    df_feat["cluster_gmm"] = labels_gmm
    _inc("modelos_treinados")

    if len(set(labels_gmm)) > 1:
        sil_gmm = silhouette_score(X, labels_gmm)
        log.info(f"  GMM Silhouette: {sil_gmm:.4f}")

    # ── 15.7 Tabela de municípios por cluster ─────────────────────────────────
    df_cluster_final = df_feat[["municipio_nome", "cluster_kmeans",
                                  "cluster_dbscan", "cluster_gmm"] +
                                [c for c in ["casos", "taxa_casos_pop", "Rt"]
                                 if c in df_feat.columns]].copy()
    df_cluster_final["cluster_kmeans"] = df_cluster_final["cluster_kmeans"].apply(
        lambda x: f"Cluster {int(x)+1}"
    )
    df_cluster_final.to_csv(
        OUTPUT_DIR / "dados" / "municipios_clusters.csv", index=False
    )
    log.info("  [CSV] municipios_clusters.csv")

    # Tabela resumo por cluster
    for cl in sorted(df_feat["cluster_kmeans"].unique()):
        muns = sorted(df_feat[df_feat["cluster_kmeans"] == cl]["municipio_nome"].tolist())
        log.info(f"  Cluster {cl+1} ({len(muns)} municípios): "
                 f"{', '.join(muns[:8])}{'...' if len(muns) > 8 else ''}")

    # Métricas finais
    rows_metr = [
        ["KMeans", str(best_k), fmt_num(sil_final, 4), fmt_num(db_score, 4), fmt_num(ch_score, 1)],
        ["DBSCAN", str(n_cl_db), "–", "–", "–"],
        ["GMM",    str(best_k), fmt_num(sil_gmm if len(set(labels_gmm)) > 1 else 0, 4), "–", "–"],
    ]
    tab_metr = make_table(
        ["Método", "k", "Silhouette", "Davies-Bouldin", "Calinski-Harabasz"],
        rows_metr, col_align=["l","c","r","r","r"]
    )
    log.info(f"\n{tab_metr}")
    salvar_txt(tab_metr, "ml_metricas_clusterizacao",
               "Métricas de Clusterização – Municípios MS")

    log.info("  Clusterização concluída.")
    return df_feat




In [35]:
# =============================================================================
# SEÇÃO 16 – MACHINE LEARNING: CLASSIFICAÇÃO DE RISCO


In [36]:
# =============================================================================

def ml_classificacao_risco(df_cg: pd.DataFrame,
                             df_ms: pd.DataFrame) -> dict:
    """
    Treina modelos de classificação para predição do nível de alerta
    (nível 1-4) usando variáveis climatológicas e epidemiológicas.
    Modelos: RF, XGBoost, LightGBM, CatBoost, MLP.
    """
    print_section("MACHINE LEARNING – CLASSIFICAÇÃO DE RISCO")
    resultados = {}

    if not HAS_SKLEARN:
        log.warning("  scikit-learn não disponível.")
        return resultados

    # Usa Campo Grande como dataset principal (série temporal contínua)
    for nome_ds, df in [("Campo Grande", df_cg), ("MS-Agregado", df_ms)]:
        if df.empty or "nivel" not in df.columns:
            continue

        log.info(f"\n  Dataset: {nome_ds}")

        # ── Features ─────────────────────────────────────────────────────────
        feature_cols = [c for c in [
            "casos", "casos_est", "p_rt1", "Rt",
            "tempmin", "tempmed", "tempmax",
            "umidmin", "umidmed", "umidmax",
            "receptivo", "transmissao",
            "MES", "SEMANA",
        ] if c in df.columns]

        target_col = "nivel"
        df_ml = df[feature_cols + [target_col]].dropna()
        if len(df_ml) < 50:
            log.warning(f"  {nome_ds}: poucos dados ({len(df_ml)}). Pulando.")
            continue

        X = df_ml[feature_cols].values
        y = df_ml[target_col].astype(int).values

        # Escala features
        scaler = StandardScaler()
        X_sc   = scaler.fit_transform(X)

        # Split treino/teste (mantém ordem temporal se possível)
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_sc, y, test_size=0.25, random_state=42, shuffle=False
        )

        modelos = {}

        # ── Random Forest ────────────────────────────────────────────────────
        rf = RandomForestClassifier(
            n_estimators=PARAMS["rf_n_estimators"],
            max_depth=10, random_state=42, n_jobs=-1,
            class_weight="balanced"
        )
        rf.fit(X_tr, y_tr)
        modelos["Random Forest"] = rf
        _inc("modelos_treinados")

        # ── XGBoost ──────────────────────────────────────────────────────────
        if HAS_XGB:
            xgb_clf = xgb.XGBClassifier(
                n_estimators=200, max_depth=6, learning_rate=0.05,
                random_state=42, eval_metric="mlogloss",
                use_label_encoder=False, verbosity=0
            )
            # Ajusta labels para 0-based
            y_tr_xgb = y_tr - y_tr.min()
            y_te_xgb = y_te - y_te.min()
            xgb_clf.fit(X_tr, y_tr_xgb)
            modelos["XGBoost"] = (xgb_clf, y_tr_xgb, y_te_xgb)
            _inc("modelos_treinados")

        # ── LightGBM ─────────────────────────────────────────────────────────
        if HAS_LGB:
            lgb_clf = lgb.LGBMClassifier(
                n_estimators=200, max_depth=6, learning_rate=0.05,
                random_state=42, verbose=-1, n_jobs=-1,
                class_weight="balanced"
            )
            lgb_clf.fit(X_tr, y_tr)
            modelos["LightGBM"] = lgb_clf
            _inc("modelos_treinados")

        # ── MLP ──────────────────────────────────────────────────────────────
        mlp = MLPClassifier(
            hidden_layer_sizes=(128, 64, 32),
            activation="relu", max_iter=300,
            random_state=42, early_stopping=True,
            validation_fraction=0.1
        )
        mlp.fit(X_tr, y_tr)
        modelos["MLP Neural Net"] = mlp
        _inc("modelos_treinados")

        # ── Avaliação ─────────────────────────────────────────────────────────
        rows_eval = []
        fig, axes = plt.subplots(1, len([m for m in modelos if m != "XGBoost"]) + (1 if HAS_XGB else 0),
                                  figsize=(6 * len(modelos), 5))
        if not isinstance(axes, np.ndarray):
            axes = [axes]
        ax_idx = 0

        for nome_m, obj in modelos.items():
            try:
                if nome_m == "XGBoost" and isinstance(obj, tuple):
                    clf_obj, _, y_te_xgb = obj
                    y_pred = clf_obj.predict(X_te) + y_te.min()
                    y_true = y_te
                else:
                    clf_obj = obj
                    y_pred  = clf_obj.predict(X_te)
                    y_true  = y_te

                acc = accuracy_score(y_true, y_pred)
                f1  = f1_score(y_true, y_pred, average="weighted", zero_division=0)
                prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
                rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
                rows_eval.append([nome_m, fmt_pct(acc*100), fmt_pct(f1*100),
                                   fmt_pct(prec*100), fmt_pct(rec*100)])

                # Matriz de confusão
                if ax_idx < len(axes):
                    cm_mat = confusion_matrix(y_true, y_pred)
                    sns.heatmap(cm_mat, annot=True, fmt="d", cmap="Blues",
                                ax=axes[ax_idx], cbar=False)
                    axes[ax_idx].set_title(f"{nome_m}\nAcc={acc:.2%}", fontsize=9)
                    axes[ax_idx].set_xlabel("Predito")
                    axes[ax_idx].set_ylabel("Real")
                    ax_idx += 1

                log.info(f"  {nome_m}: Acc={acc:.4f} | F1={f1:.4f} | "
                         f"Prec={prec:.4f} | Rec={rec:.4f}")

            except Exception as e:
                log.warning(f"  Erro ao avaliar {nome_m}: {e}")

        plt.suptitle(f"Matrizes de Confusão – {nome_ds}", fontsize=13,
                     fontweight="bold")
        salvar_fig(f"ml_conf_matrix_{nome_ds.lower().replace(' ','_').replace('-','_')}")

        tab_eval = make_table(
            ["Modelo", "Acurácia", "F1-Score", "Precisão", "Recall"],
            rows_eval, col_align=["l","r","r","r","r"]
        )
        log.info(f"\n{tab_eval}")
        salvar_txt(tab_eval,
                   f"ml_classificacao_metricas_{nome_ds.lower().replace(' ','_')}",
                   f"Métricas de Classificação – {nome_ds}")

        # ── Importância de Features (RF) ──────────────────────────────────────
        imp = pd.DataFrame({
            "Feature":     feature_cols,
            "Importância": rf.feature_importances_,
        }).sort_values("Importância", ascending=False)

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.barh(imp["Feature"][::-1], imp["Importância"][::-1],
                color=COR_SECUNDARIA, edgecolor="white")
        ax.set_title(f"Importância de Features – RF – {nome_ds}", fontweight="bold")
        ax.set_xlabel("Importância")
        salvar_fig(f"ml_feature_importance_rf_{nome_ds.lower().replace(' ','_').replace('-','_')}")

        # SHAP (se disponível, apenas para RF)
        if HAS_SHAP and len(X_te) > 0:
            try:
                explainer  = shap.TreeExplainer(rf)
                shap_vals  = explainer.shap_values(X_te[:min(200, len(X_te))])
                if isinstance(shap_vals, list):
                    shap_vals_sum = np.abs(shap_vals[0]).mean(axis=0)
                else:
                    shap_vals_sum = np.abs(shap_vals).mean(axis=0)
                shap_imp = pd.DataFrame({
                    "Feature": feature_cols,
                    "SHAP":    shap_vals_sum,
                }).sort_values("SHAP", ascending=False)
                fig, ax = plt.subplots(figsize=(10, 5))
                ax.barh(shap_imp["Feature"][::-1], shap_imp["SHAP"][::-1],
                        color=COR_ALERTA, edgecolor="white")
                ax.set_title(f"SHAP – Importância Global – {nome_ds}", fontweight="bold")
                ax.set_xlabel("|SHAP value|")
                salvar_fig(f"ml_shap_global_{nome_ds.lower().replace(' ','_').replace('-','_')}")
            except Exception as e:
                log.warning(f"  SHAP falhou: {e}")

        resultados[nome_ds] = {"modelos": modelos, "metricas": rows_eval}
        # Só usa CG; deixa MS como extra
        break

    log.info("  Classificação de risco concluída.")
    return resultados




In [37]:
# =============================================================================
# SEÇÃO 17 – MACHINE LEARNING: REGRESSÃO DE CASOS


In [38]:
# =============================================================================

def ml_regressao_casos(df_cg: pd.DataFrame) -> dict:
    """
    Treina modelos de regressão para prever número de casos semanais.
    Modelos: Linear, Ridge, RF Regressor, XGBoost Regressor,
             LightGBM Regressor, CatBoost Regressor, Ensemble.
    """
    print_section("MACHINE LEARNING – REGRESSÃO DE CASOS")
    resultados = {}

    if not HAS_SKLEARN or df_cg.empty:
        return resultados

    # Features para regressão
    feat_cols = [c for c in [
        "MES", "SEMANA", "ANO",
        "tempmin", "tempmed", "tempmax",
        "umidmin", "umidmed", "umidmax",
        "Rt", "p_rt1", "receptivo", "transmissao",
        "nivel", "nivel_inc",
    ] if c in df_cg.columns]

    target = "casos"
    df_reg = df_cg[feat_cols + [target]].dropna()

    if len(df_reg) < 60:
        log.warning("  Dados insuficientes para regressão.")
        return resultados

    X = df_reg[feat_cols].values
    y = df_reg[target].values.astype(float)

    # Divisão temporal (70/30)
    split = int(len(X) * 0.7)
    X_tr, X_te = X[:split], X[split:]
    y_tr, y_te = y[:split], y[split:]

    scaler  = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_te_sc = scaler.transform(X_te)

    modelos_reg = {}

    # Regressão Linear / Ridge
    for nome_m, mdl in [
        ("Regressão Linear", LinearRegression()),
        ("Ridge",            Ridge(alpha=1.0)),
        ("Lasso",            Lasso(alpha=0.1, max_iter=5000)),
        ("ElasticNet",       ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000)),
    ]:
        mdl.fit(X_tr_sc, y_tr)
        modelos_reg[nome_m] = mdl
        _inc("modelos_treinados")

    # Random Forest Regressor
    rf_reg = RandomForestRegressor(
        n_estimators=PARAMS["rf_n_estimators"], max_depth=12,
        random_state=42, n_jobs=-1
    )
    rf_reg.fit(X_tr_sc, y_tr)
    modelos_reg["Random Forest"] = rf_reg
    _inc("modelos_treinados")

    # Extra Trees
    et_reg = ExtraTreesRegressor(
        n_estimators=150, random_state=42, n_jobs=-1
    )
    et_reg.fit(X_tr_sc, y_tr)
    modelos_reg["Extra Trees"] = et_reg
    _inc("modelos_treinados")

    # XGBoost Regressor
    if HAS_XGB:
        xgb_reg = xgb.XGBRegressor(
            n_estimators=PARAMS["xgb_n_estimators"], max_depth=6,
            learning_rate=0.05, random_state=42, verbosity=0
        )
        xgb_reg.fit(X_tr_sc, y_tr,
                    eval_set=[(X_te_sc, y_te)], verbose=False)
        modelos_reg["XGBoost"] = xgb_reg
        _inc("modelos_treinados")

    # LightGBM Regressor
    if HAS_LGB:
        lgb_reg = lgb.LGBMRegressor(
            n_estimators=PARAMS["lgb_n_estimators"], max_depth=6,
            learning_rate=0.05, random_state=42, verbose=-1
        )
        lgb_reg.fit(X_tr_sc, y_tr,
                    eval_set=[(X_te_sc, y_te)],
                    callbacks=[lgb.early_stopping(30, verbose=False),
                                lgb.log_evaluation(period=-1)])
        modelos_reg["LightGBM"] = lgb_reg
        _inc("modelos_treinados")

    # CatBoost Regressor
    if HAS_CAT:
        try:
            cat_reg = CatBoostRegressor(
                iterations=200, depth=6, learning_rate=0.05,
                random_seed=42, verbose=0
            )
            cat_reg.fit(X_tr_sc, y_tr, eval_set=(X_te_sc, y_te),
                        early_stopping_rounds=20)
            modelos_reg["CatBoost"] = cat_reg
            _inc("modelos_treinados")
        except Exception as e:
            log.warning(f"  CatBoost falhou: {e}")

    # MLP Regressor
    mlp_reg = MLPRegressor(
        hidden_layer_sizes=(128, 64, 32), activation="relu",
        max_iter=400, random_state=42,
        early_stopping=True, validation_fraction=0.1
    )
    mlp_reg.fit(X_tr_sc, y_tr)
    modelos_reg["MLP Regressor"] = mlp_reg
    _inc("modelos_treinados")

    # ── Avaliação ─────────────────────────────────────────────────────────────
    rows_eval = []
    y_preds   = {}
    for nome_m, mdl in modelos_reg.items():
        try:
            y_pred_te = mdl.predict(X_te_sc)
            y_pred_te = np.clip(y_pred_te, 0, None)

            rmse  = np.sqrt(mean_squared_error(y_te, y_pred_te))
            mae   = mean_absolute_error(y_te, y_pred_te)
            r2    = r2_score(y_te, y_pred_te)
            mape  = mean_absolute_percentage_error(y_te, y_pred_te + 1e-9) * 100

            rows_eval.append([nome_m, fmt_num(rmse, 1), fmt_num(mae, 1),
                               fmt_num(r2, 4), fmt_pct(mape)])
            y_preds[nome_m] = y_pred_te
            log.info(f"  {nome_m:20s}: RMSE={rmse:.2f} | MAE={mae:.2f} | "
                     f"R²={r2:.4f} | MAPE={mape:.1f}%")
        except Exception as e:
            log.warning(f"  Erro ao avaliar {nome_m}: {e}")

    tab_eval = make_table(
        ["Modelo", "RMSE", "MAE", "R²", "MAPE"],
        rows_eval, col_align=["l","r","r","r","r"]
    )
    log.info(f"\n{tab_eval}")
    salvar_txt(tab_eval, "ml_regressao_metricas",
               "Métricas de Regressão – Campo Grande")

    # ── Gráfico: Predito vs Real ──────────────────────────────────────────────
    modelos_graf = [m for m in ["Random Forest", "XGBoost", "LightGBM", "MLP Regressor"]
                    if m in y_preds][:4]
    if modelos_graf:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes = axes.flatten()
        for i, nome_m in enumerate(modelos_graf):
            ax = axes[i]
            ax.plot(y_te, label="Real", color=COR_SECUNDARIA,
                    linewidth=1.5, alpha=0.7)
            ax.plot(y_preds[nome_m], label="Predito",
                    color=COR_PRINCIPAL, linewidth=1.5, linestyle="--")
            ax.set_title(f"{nome_m}", fontweight="bold", fontsize=10)
            ax.set_ylabel("Casos")
            ax.set_xlabel("Semanas")
            ax.legend(fontsize=8)
        for j in range(len(modelos_graf), len(axes)):
            axes[j].set_visible(False)
        plt.suptitle("Regressão – Predito vs Real (Conjunto de Teste)",
                     fontsize=13, fontweight="bold")
        salvar_fig("ml_regressao_predito_vs_real")

    # ── Ensemble (votação por média) ──────────────────────────────────────────
    modelos_ensemble = [m for m in ["Random Forest", "XGBoost", "LightGBM"]
                        if m in y_preds]
    if len(modelos_ensemble) >= 2:
        y_ens = np.mean([y_preds[m] for m in modelos_ensemble], axis=0)
        rmse_ens = np.sqrt(mean_squared_error(y_te, y_ens))
        mae_ens  = mean_absolute_error(y_te, y_ens)
        r2_ens   = r2_score(y_te, y_ens)
        log.info(f"  Ensemble ({'+'.join(modelos_ensemble)}): "
                 f"RMSE={rmse_ens:.2f} | MAE={mae_ens:.2f} | R²={r2_ens:.4f}")
        y_preds["Ensemble"] = y_ens
        _inc("modelos_treinados")

    # ── Importância de features (RF Regressor) ─────────────────────────────
    if "Random Forest" in modelos_reg:
        imp = pd.DataFrame({
            "Feature":     feat_cols,
            "Importância": modelos_reg["Random Forest"].feature_importances_,
        }).sort_values("Importância", ascending=False)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.barh(imp["Feature"][::-1], imp["Importância"][::-1],
                color=COR_VERDE, edgecolor="white")
        ax.set_title("Importância de Features – RF Regressor (Casos)",
                     fontweight="bold")
        ax.set_xlabel("Importância")
        salvar_fig("ml_regressao_feature_importance")

    resultados["y_te"]    = y_te
    resultados["y_preds"] = y_preds
    resultados["scaler"]  = scaler
    resultados["feat_cols"] = feat_cols
    log.info("  Regressão de casos concluída.")
    return resultados




In [39]:
# =============================================================================
# SEÇÃO 18 – SÉRIES TEMPORAIS: ARIMA, SARIMA, PROPHET, ETS


In [40]:
# =============================================================================

def series_temporais(df_cg: pd.DataFrame) -> dict:
    """
    Análise e previsão de séries temporais:
    - Decomposição sazonal (STL)
    - Teste ADF (estacionaridade)
    - ARIMA / SARIMA (auto)
    - Holt-Winters ETS
    - Prophet
    - Previsão para os próximos 12 meses
    """
    print_section("SÉRIES TEMPORAIS – ARIMA / SARIMA / PROPHET / ETS")
    resultados = {}

    if df_cg.empty or "data_SE" not in df_cg.columns or "casos" not in df_cg.columns:
        log.warning("  Dados insuficientes para séries temporais.")
        return resultados

    # Prepara série mensal (mais estável para modelagem)
    df_cg_sort = df_cg.sort_values("data_SE").copy()
    df_cg_sort["data_SE"] = pd.to_datetime(df_cg_sort["data_SE"])
    serie_mensal = (df_cg_sort
                    .set_index("data_SE")["casos"]
                    .resample("MS").sum()
                    .fillna(0))

    if len(serie_mensal) < 24:
        log.warning("  Série muito curta (< 24 meses) para modelagem.")
        return resultados

    # ── 18.1 Decomposição Sazonal ──────────────────────────────────────────────
    print_sub("18.1 Decomposição Sazonal (STL)")
    if HAS_STATSMODELS and len(serie_mensal) >= 24:
        try:
            stl = STL(serie_mensal, period=12, robust=True)
            res_stl = stl.fit()

            fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
            axes[0].plot(serie_mensal.index, serie_mensal.values,
                         color=COR_SECUNDARIA)
            axes[0].set_ylabel("Observado")
            axes[0].set_title("Decomposição STL – Dengue Campo Grande/MS",
                               fontweight="bold")
            axes[1].plot(serie_mensal.index, res_stl.trend,
                         color=COR_PRINCIPAL)
            axes[1].set_ylabel("Tendência")
            axes[2].plot(serie_mensal.index, res_stl.seasonal,
                         color=COR_VERDE)
            axes[2].set_ylabel("Sazonalidade")
            axes[2].axhline(0, color="gray", linestyle="--", linewidth=0.8)
            axes[3].plot(serie_mensal.index, res_stl.resid,
                         color=COR_CINZA)
            axes[3].axhline(0, color="gray", linestyle="--", linewidth=0.8)
            axes[3].set_ylabel("Resíduo")
            axes[3].set_xlabel("Data")
            salvar_fig("ts_decomposicao_stl_cg")
            resultados["stl"] = res_stl
        except Exception as e:
            log.warning(f"  STL falhou: {e}")

    # ── 18.2 Teste de Estacionaridade (ADF) ──────────────────────────────────
    print_sub("18.2 Teste ADF – Estacionaridade")
    if HAS_STATSMODELS:
        try:
            adf_result  = adfuller(serie_mensal.dropna(), autolag="AIC")
            adf_stat    = adf_result[0]
            adf_pvalue  = adf_result[1]
            is_stationary = adf_pvalue < PARAMS["alpha_sig"]
            log.info(f"  ADF Statistic: {adf_stat:.4f} | p-value: {adf_pvalue:.4f} | "
                     f"Série {'ESTACIONÁRIA' if is_stationary else 'NÃO ESTACIONÁRIA'}")
        except Exception as e:
            log.warning(f"  ADF falhou: {e}")

    # ── 18.3 ACF / PACF ──────────────────────────────────────────────────────
    print_sub("18.3 ACF e PACF")
    if HAS_STATSMODELS:
        try:
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            plot_acf(serie_mensal.dropna(), lags=24, ax=axes[0], alpha=0.05,
                     title="ACF – Autocorrelação")
            plot_pacf(serie_mensal.dropna(), lags=24, ax=axes[1], alpha=0.05,
                      title="PACF – Autocorrelação Parcial")
            plt.suptitle("Análise de Autocorrelação – Campo Grande/MS",
                         fontsize=13, fontweight="bold")
            salvar_fig("ts_acf_pacf_cg")
        except Exception as e:
            log.warning(f"  ACF/PACF falhou: {e}")

    # ── 18.4 Auto-ARIMA ───────────────────────────────────────────────────────
    print_sub("18.4 Auto-ARIMA")
    arima_pred = None
    if HAS_PMDARIMA and len(serie_mensal) >= 36:
        try:
            log.info("  Ajustando Auto-ARIMA (pode levar alguns minutos)...")
            auto_mod = auto_arima(
                serie_mensal,
                seasonal=True, m=12,
                stepwise=True, suppress_warnings=True,
                max_p=PARAMS["arima_max_p"],
                max_q=PARAMS["arima_max_q"],
                max_d=PARAMS["arima_max_d"],
                information_criterion="aic",
                error_action="ignore",
            )
            log.info(f"  Auto-ARIMA: {auto_mod.order} × {auto_mod.seasonal_order}")

            # Previsão 12 meses à frente
            n_pred = PARAMS["horizonte_previsao_meses"]
            fc, ci = auto_mod.predict(n_periods=n_pred, return_conf_int=True)
            fc = np.clip(fc, 0, None)
            datas_fc = pd.date_range(
                start=serie_mensal.index[-1] + pd.DateOffset(months=1),
                periods=n_pred, freq="MS"
            )
            arima_pred = pd.DataFrame({
                "data": datas_fc, "previsao": fc,
                "ic_inf": np.clip(ci[:, 0], 0, None),
                "ic_sup": ci[:, 1],
            })

            fig, ax = plt.subplots(figsize=(14, 5))
            ax.plot(serie_mensal.index, serie_mensal.values,
                    color=COR_SECUNDARIA, linewidth=1.5, label="Histórico")
            ax.plot(arima_pred["data"], arima_pred["previsao"],
                    color=COR_PRINCIPAL, linewidth=2, linestyle="--",
                    marker="o", markersize=5, label="Previsão ARIMA")
            ax.fill_between(arima_pred["data"],
                            arima_pred["ic_inf"], arima_pred["ic_sup"],
                            alpha=0.25, color=COR_PRINCIPAL,
                            label="IC 95%")
            ax.set_title(f"Previsão ARIMA{auto_mod.order}×{auto_mod.seasonal_order} "
                         f"– {n_pred} Meses – Campo Grande/MS",
                         fontweight="bold")
            ax.set_xlabel("Data")
            ax.set_ylabel("Casos / Mês")
            ax.legend()
            ax.yaxis.set_major_formatter(mticker.FuncFormatter(
                lambda x, _: fmt_num(int(max(x, 0)))
            ))
            salvar_fig("ts_arima_previsao_cg")

            rows_fc = [[d.strftime("%b/%Y"), fmt_num(int(p)),
                        fmt_num(int(l)), fmt_num(int(u))]
                       for d, p, l, u in zip(arima_pred["data"],
                                              arima_pred["previsao"],
                                              arima_pred["ic_inf"],
                                              arima_pred["ic_sup"])]
            tab_fc = make_table(
                ["Mês/Ano", "Previsão", "IC Inferior", "IC Superior"],
                rows_fc, col_align=["l","r","r","r"]
            )
            log.info(f"\n{tab_fc}")
            salvar_txt(tab_fc, "ts_arima_previsao_tabela",
                       "Previsão ARIMA – Campo Grande")
            resultados["arima_pred"] = arima_pred
            _inc("modelos_treinados")

        except Exception as e:
            log.warning(f"  Auto-ARIMA falhou: {e}")

    # ── 18.5 Holt-Winters ETS ────────────────────────────────────────────────
    print_sub("18.5 Holt-Winters – Suavização Exponencial")
    hw_pred = None
    if HAS_STATSMODELS and len(serie_mensal) >= 24:
        try:
            hw = ExponentialSmoothing(
                serie_mensal, trend="add", seasonal="add",
                seasonal_periods=12, damped_trend=True
            )
            hw_fit  = hw.fit(optimized=True)
            n_pred  = PARAMS["horizonte_previsao_meses"]
            hw_fc   = hw_fit.forecast(n_pred)
            hw_fc   = np.clip(hw_fc, 0, None)
            datas_hw = pd.date_range(
                start=serie_mensal.index[-1] + pd.DateOffset(months=1),
                periods=n_pred, freq="MS"
            )

            fig, ax = plt.subplots(figsize=(14, 5))
            ax.plot(serie_mensal.index, serie_mensal.values,
                    color=COR_SECUNDARIA, linewidth=1.5, label="Histórico")
            ax.plot(serie_mensal.index, hw_fit.fittedvalues,
                    color=COR_VERDE, linewidth=1, linestyle=":",
                    label="Ajustado ETS")
            ax.plot(datas_hw, hw_fc,
                    color=COR_ALERTA, linewidth=2, linestyle="--",
                    marker="s", markersize=5, label="Previsão ETS")
            ax.set_title(f"Previsão Holt-Winters ETS – {n_pred} Meses – Campo Grande/MS",
                         fontweight="bold")
            ax.set_xlabel("Data")
            ax.set_ylabel("Casos / Mês")
            ax.legend()
            salvar_fig("ts_holtwinters_previsao_cg")
            _inc("modelos_treinados")

            hw_pred = pd.DataFrame({"data": datas_hw, "previsao_hw": hw_fc})
            resultados["hw_pred"] = hw_pred
        except Exception as e:
            log.warning(f"  Holt-Winters falhou: {e}")

    # ── 18.6 Prophet ─────────────────────────────────────────────────────────
    print_sub("18.6 Prophet – Previsão com Sazonalidade")
    prophet_pred = None
    if HAS_PROPHET and len(serie_mensal) >= 24:
        try:
            df_prophet = pd.DataFrame({
                "ds": serie_mensal.index,
                "y":  serie_mensal.values.clip(0),
            })
            m = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False,
                seasonality_mode="multiplicative",
                interval_width=0.95,
            )
            m.fit(df_prophet)

            future    = m.make_future_dataframe(
                periods=PARAMS["horizonte_previsao_meses"], freq="MS"
            )
            forecast  = m.predict(future)
            forecast["yhat"] = forecast["yhat"].clip(lower=0)

            fig, ax = plt.subplots(figsize=(14, 5))
            hist_mask = forecast["ds"] <= df_prophet["ds"].max()
            ax.fill_between(
                forecast.loc[~hist_mask, "ds"],
                forecast.loc[~hist_mask, "yhat_lower"].clip(0),
                forecast.loc[~hist_mask, "yhat_upper"],
                alpha=0.25, color=COR_ROXO, label="IC 95%"
            )
            ax.plot(df_prophet["ds"], df_prophet["y"],
                    color=COR_SECUNDARIA, linewidth=1.5, label="Histórico")
            ax.plot(forecast.loc[~hist_mask, "ds"],
                    forecast.loc[~hist_mask, "yhat"],
                    color=COR_ROXO, linewidth=2, linestyle="--",
                    marker="^", markersize=5, label="Previsão Prophet")
            ax.set_title(f"Previsão Prophet – {PARAMS['horizonte_previsao_meses']} Meses – Campo Grande/MS",
                         fontweight="bold")
            ax.set_xlabel("Data")
            ax.set_ylabel("Casos / Mês")
            ax.legend()
            salvar_fig("ts_prophet_previsao_cg")
            _inc("modelos_treinados")

            prophet_pred = forecast[~hist_mask][["ds","yhat","yhat_lower","yhat_upper"]].copy()
            resultados["prophet_pred"] = prophet_pred
            log.info(f"  Prophet: Previsão gerada para {len(prophet_pred)} meses.")
        except Exception as e:
            log.warning(f"  Prophet falhou: {e}")

    # ── 18.7 Comparativo das previsões ───────────────────────────────────────
    print_sub("18.7 Comparativo das Previsões")
    modelos_previsao = {}
    if arima_pred is not None:
        modelos_previsao["ARIMA"] = (arima_pred["data"].values,
                                      arima_pred["previsao"].values)
    if hw_pred is not None:
        modelos_previsao["Holt-Winters"] = (hw_pred["data"].values,
                                             hw_pred["previsao_hw"].values)
    if prophet_pred is not None:
        modelos_previsao["Prophet"] = (prophet_pred["ds"].values,
                                       prophet_pred["yhat"].values)

    if len(modelos_previsao) >= 2:
        fig, ax = plt.subplots(figsize=(14, 6))
        # Histórico
        ax.plot(serie_mensal.index, serie_mensal.values,
                color=COR_CINZA, linewidth=1.5, alpha=0.6, label="Histórico")
        cores_prev = [COR_PRINCIPAL, COR_ALERTA, COR_ROXO]
        for i, (nome_m, (datas, vals)) in enumerate(modelos_previsao.items()):
            ax.plot(datas, vals, color=cores_prev[i], linewidth=2,
                    linestyle="--", marker="o", markersize=4,
                    label=f"Previsão {nome_m}")
        ax.axvline(serie_mensal.index[-1], color="black", linestyle=":",
                   linewidth=1.5, label="Início da Previsão")
        ax.set_title("Comparativo de Previsões – Dengue Campo Grande/MS",
                     fontweight="bold")
        ax.set_xlabel("Data")
        ax.set_ylabel("Casos / Mês")
        ax.legend(ncol=2)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: fmt_num(int(max(x, 0)))
        ))
        salvar_fig("ts_comparativo_previsoes_cg")

    log.info("  Séries temporais concluídas.")
    return resultados




In [41]:
# =============================================================================
# SEÇÃO 19 – DETECÇÃO DE ANOMALIAS E ISOLATION FOREST


In [42]:
# =============================================================================

def deteccao_anomalias(df_cg: pd.DataFrame) -> pd.DataFrame:
    """
    Detecta semanas epidemiologicamente anômalas usando:
    - IQR (estatístico)
    - Isolation Forest
    - LOF (Local Outlier Factor)
    """
    print_section("DETECÇÃO DE ANOMALIAS EPIDEMIOLÓGICAS")

    if not HAS_SKLEARN or df_cg.empty or "casos" not in df_cg.columns:
        return pd.DataFrame()

    feat_cols = [c for c in [
        "casos", "Rt", "p_rt1", "p_inc100k", "nivel",
        "tempmed", "umidmed",
    ] if c in df_cg.columns]

    df_anom = df_cg[feat_cols + ["data_SE", "SE", "ANO", "MES"]
                    ].dropna().copy()
    if len(df_anom) < 20:
        return pd.DataFrame()

    X_an = df_anom[feat_cols].values
    scaler_an = StandardScaler()
    X_sc  = scaler_an.fit_transform(X_an)

    # ── Isolation Forest ──────────────────────────────────────────────────────
    iso = IsolationForest(n_estimators=200, contamination=0.05,
                           random_state=42)
    df_anom["anomalia_iso"] = iso.fit_predict(X_sc)
    df_anom["anomalia_iso"] = (df_anom["anomalia_iso"] == -1).astype(int)
    _inc("modelos_treinados")

    # ── LOF ───────────────────────────────────────────────────────────────────
    lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
    df_anom["anomalia_lof"] = lof.fit_predict(X_sc)
    df_anom["anomalia_lof"] = (df_anom["anomalia_lof"] == -1).astype(int)
    _inc("modelos_treinados")

    # ── IQR ──────────────────────────────────────────────────────────────────
    q1, q3 = df_anom["casos"].quantile([0.25, 0.75])
    iqr    = q3 - q1
    df_anom["anomalia_iqr"] = (
        (df_anom["casos"] > q3 + 2.5 * iqr) |
        (df_anom["casos"] < q1 - 2.5 * iqr)
    ).astype(int)

    # Anomalia confirmada por pelo menos 2 métodos
    df_anom["anomalia_consenso"] = (
        df_anom[["anomalia_iso", "anomalia_lof", "anomalia_iqr"]].sum(axis=1) >= 2
    ).astype(int)

    n_anom = int(df_anom["anomalia_consenso"].sum())
    log.info(f"  Semanas anômalas (consenso): {n_anom} de {len(df_anom)}")

    # Gráfico
    if "data_SE" in df_anom.columns:
        fig, ax = plt.subplots(figsize=(16, 5))
        ax.plot(df_anom["data_SE"], df_anom["casos"],
                color=COR_SECUNDARIA, linewidth=1, alpha=0.7, label="Casos")
        mask_anom = df_anom["anomalia_consenso"] == 1
        ax.scatter(df_anom.loc[mask_anom, "data_SE"],
                   df_anom.loc[mask_anom, "casos"],
                   color=COR_PRINCIPAL, s=60, zorder=5,
                   label=f"Anomalias (n={n_anom})")
        ax.set_title("Detecção de Anomalias – Dengue Campo Grande/MS",
                     fontweight="bold")
        ax.set_xlabel("Semana Epidemiológica")
        ax.set_ylabel("Casos")
        ax.legend()
        salvar_fig("ml_anomalias_cg")

    # Tabela de anomalias
    anom_rows = df_anom[mask_anom][
        ["data_SE","SE","ANO","MES","casos"] +
        [c for c in ["Rt","p_rt1","nivel"] if c in df_anom.columns]
    ].sort_values("casos", ascending=False)

    if not anom_rows.empty:
        rows_t = []
        for _, r in anom_rows.head(20).iterrows():
            rows_t.append([
                str(r.get("data_SE",""))[: 10],
                int(r.get("SE", 0)), int(r.get("ANO", 0)),
                MESES_ABREV.get(int(r.get("MES", 1)), "?"),
                fmt_num(int(r.get("casos", 0))),
                fmt_num(r.get("Rt", 0), 2) if "Rt" in df_anom.columns else "–",
            ])
        tab_anom = make_table(
            ["Data", "SE", "Ano", "Mês", "Casos", "Rt"],
            rows_t, col_align=["l","c","c","l","r","r"]
        )
        log.info(f"\n{tab_anom}")
        salvar_txt(tab_anom, "ml_anomalias_tabela",
                   "Semanas Anômalas – Campo Grande")

    log.info("  Detecção de anomalias concluída.")
    return df_anom




In [43]:
# =============================================================================
# SEÇÃO 20 – DEEP LEARNING: LSTM, GRU, TRANSFORMER


In [44]:
# =============================================================================

def _criar_sequencias(series: np.ndarray, janela: int) -> Tuple[np.ndarray, np.ndarray]:
    """Cria pares (X_seq, y) para modelos sequenciais."""
    X, y = [], []
    for i in range(len(series) - janela):
        X.append(series[i: i + janela])
        y.append(series[i + janela])
    return np.array(X), np.array(y)


def deep_learning_lstm_gru(df_cg: pd.DataFrame) -> dict:
    """
    Treina modelos LSTM, GRU e Transformer para previsão da série temporal
    de dengue em Campo Grande/MS.
    Retorna dict com históricos de treino e previsões.
    """
    print_section("DEEP LEARNING – LSTM / GRU / TRANSFORMER")
    resultados = {}

    if not HAS_TF:
        log.warning("  TensorFlow não disponível. Pulando modelos DL.")
        return resultados

    if df_cg.empty or "data_SE" not in df_cg.columns or "casos" not in df_cg.columns:
        return resultados

    # ── Prepara série normalizada ─────────────────────────────────────────────
    df_sort = df_cg.sort_values("data_SE").copy()
    serie   = df_sort["casos"].fillna(0).values.astype(float)

    if len(serie) < 60:
        log.warning("  Série muito curta para LSTM (< 60 amostras).")
        return resultados

    scaler_dl = MinMaxScaler(feature_range=(0, 1))
    serie_sc  = scaler_dl.fit_transform(serie.reshape(-1, 1)).flatten()

    JANELA = PARAMS["lstm_janela"]
    X, y   = _criar_sequencias(serie_sc, JANELA)
    X      = X.reshape((X.shape[0], X.shape[1], 1))

    # Divisão treino / validação / teste (70/15/15)
    n_tot  = len(X)
    n_tr   = int(n_tot * 0.70)
    n_val  = int(n_tot * 0.85)
    X_tr, y_tr   = X[:n_tr],    y[:n_tr]
    X_val, y_val = X[n_tr:n_val], y[n_tr:n_val]
    X_te, y_te   = X[n_val:],   y[n_val:]

    callbacks_base = [
        EarlyStopping(monitor="val_loss", patience=10,
                       restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                           patience=5, verbose=0),
    ]

    modelos_dl = {}

    # ── 20.1 LSTM Bivariado ───────────────────────────────────────────────────
    print_sub("20.1 Modelo LSTM")
    try:
        tf.keras.backend.clear_session()
        model_lstm = Sequential([
            Input(shape=(JANELA, 1)),
            LSTM(PARAMS["lstm_units_1"], return_sequences=True,
                 kernel_regularizer=l2(1e-4)),
            Dropout(0.2),
            BatchNormalization(),
            LSTM(PARAMS["lstm_units_2"], return_sequences=False),
            Dropout(0.2),
            Dense(32, activation="relu"),
            Dense(1, activation="linear"),
        ], name="LSTM_CG")
        model_lstm.compile(optimizer=Adam(learning_rate=1e-3),
                            loss=Huber(), metrics=["mae"])
        hist_lstm = model_lstm.fit(
            X_tr, y_tr,
            epochs=PARAMS["lstm_epochs"],
            batch_size=PARAMS["lstm_batch"],
            validation_data=(X_val, y_val),
            callbacks=callbacks_base,
            verbose=0,
        )
        modelos_dl["LSTM"] = (model_lstm, hist_lstm)
        _inc("modelos_treinados")
        log.info(f"  LSTM: val_loss={min(hist_lstm.history['val_loss']):.5f} "
                 f"(época {np.argmin(hist_lstm.history['val_loss'])+1})")
    except Exception as e:
        log.warning(f"  LSTM falhou: {e}")

    # ── 20.2 GRU ─────────────────────────────────────────────────────────────
    print_sub("20.2 Modelo GRU")
    try:
        tf.keras.backend.clear_session()
        model_gru = Sequential([
            Input(shape=(JANELA, 1)),
            GRU(64, return_sequences=True, kernel_regularizer=l2(1e-4)),
            Dropout(0.2),
            BatchNormalization(),
            GRU(32),
            Dropout(0.2),
            Dense(16, activation="relu"),
            Dense(1, activation="linear"),
        ], name="GRU_CG")
        model_gru.compile(optimizer=Adam(learning_rate=1e-3),
                           loss=Huber(), metrics=["mae"])
        hist_gru = model_gru.fit(
            X_tr, y_tr,
            epochs=PARAMS["lstm_epochs"],
            batch_size=PARAMS["lstm_batch"],
            validation_data=(X_val, y_val),
            callbacks=callbacks_base,
            verbose=0,
        )
        modelos_dl["GRU"] = (model_gru, hist_gru)
        _inc("modelos_treinados")
        log.info(f"  GRU: val_loss={min(hist_gru.history['val_loss']):.5f}")
    except Exception as e:
        log.warning(f"  GRU falhou: {e}")

    # ── 20.3 Bidirectional LSTM ───────────────────────────────────────────────
    print_sub("20.3 Bidirectional LSTM")
    try:
        tf.keras.backend.clear_session()
        model_blstm = Sequential([
            Input(shape=(JANELA, 1)),
            Bidirectional(LSTM(64, return_sequences=True)),
            Dropout(0.25),
            Bidirectional(LSTM(32)),
            Dropout(0.2),
            Dense(16, activation="relu"),
            Dense(1, activation="linear"),
        ], name="BiLSTM_CG")
        model_blstm.compile(optimizer=Adam(learning_rate=1e-3),
                             loss=Huber(), metrics=["mae"])
        hist_blstm = model_blstm.fit(
            X_tr, y_tr,
            epochs=PARAMS["lstm_epochs"],
            batch_size=PARAMS["lstm_batch"],
            validation_data=(X_val, y_val),
            callbacks=callbacks_base,
            verbose=0,
        )
        modelos_dl["BiLSTM"] = (model_blstm, hist_blstm)
        _inc("modelos_treinados")
        log.info(f"  BiLSTM: val_loss={min(hist_blstm.history['val_loss']):.5f}")
    except Exception as e:
        log.warning(f"  BiLSTM falhou: {e}")

    # ── 20.4 CNN-LSTM ─────────────────────────────────────────────────────────
    print_sub("20.4 CNN-LSTM")
    try:
        tf.keras.backend.clear_session()
        inp = Input(shape=(JANELA, 1))
        x   = Conv1D(64, kernel_size=3, activation="relu", padding="same")(inp)
        x   = MaxPooling1D(pool_size=2)(x)
        x   = Conv1D(32, kernel_size=3, activation="relu", padding="same")(x)
        x   = LSTM(32, return_sequences=False)(x)
        x   = Dropout(0.2)(x)
        out = Dense(1, activation="linear")(x)
        model_cnn = Model(inputs=inp, outputs=out, name="CNN_LSTM_CG")
        model_cnn.compile(optimizer=Adam(learning_rate=1e-3),
                           loss=Huber(), metrics=["mae"])
        hist_cnn = model_cnn.fit(
            X_tr, y_tr,
            epochs=PARAMS["lstm_epochs"],
            batch_size=PARAMS["lstm_batch"],
            validation_data=(X_val, y_val),
            callbacks=callbacks_base,
            verbose=0,
        )
        modelos_dl["CNN-LSTM"] = (model_cnn, hist_cnn)
        _inc("modelos_treinados")
        log.info(f"  CNN-LSTM: val_loss={min(hist_cnn.history['val_loss']):.5f}")
    except Exception as e:
        log.warning(f"  CNN-LSTM falhou: {e}")

    # ── 20.5 Transformer Temporal ─────────────────────────────────────────────
    print_sub("20.5 Transformer Temporal")
    try:
        tf.keras.backend.clear_session()
        inp = Input(shape=(JANELA, 1))
        x   = Dense(32)(inp)

        # Multi-Head Self-Attention
        attn_out   = MultiHeadAttention(num_heads=4, key_dim=8)(x, x)
        attn_out   = Dropout(0.1)(attn_out)
        x          = LayerNormalization()(x + attn_out)

        # Feed-forward
        ff         = Dense(64, activation="relu")(x)
        ff         = Dense(32)(ff)
        ff         = Dropout(0.1)(ff)
        x          = LayerNormalization()(x + ff)

        x          = GlobalAveragePooling1D()(x)
        x          = Dense(32, activation="relu")(x)
        out        = Dense(1, activation="linear")(x)

        model_tr = Model(inputs=inp, outputs=out, name="Transformer_CG")
        model_tr.compile(optimizer=Adam(learning_rate=5e-4),
                          loss=Huber(), metrics=["mae"])
        hist_tr = model_tr.fit(
            X_tr, y_tr,
            epochs=PARAMS["lstm_epochs"],
            batch_size=PARAMS["lstm_batch"],
            validation_data=(X_val, y_val),
            callbacks=callbacks_base,
            verbose=0,
        )
        modelos_dl["Transformer"] = (model_tr, hist_tr)
        _inc("modelos_treinados")
        log.info(f"  Transformer: val_loss={min(hist_tr.history['val_loss']):.5f}")
    except Exception as e:
        log.warning(f"  Transformer falhou: {e}")

    # ── Avaliação e previsão ──────────────────────────────────────────────────
    if not modelos_dl:
        return resultados

    # Curvas de perda
    n_mod = len(modelos_dl)
    fig, axes = plt.subplots(1, n_mod, figsize=(5 * n_mod, 4))
    if n_mod == 1:
        axes = [axes]
    for ax, (nm, (mdl, hist)) in zip(axes, modelos_dl.items()):
        ax.plot(hist.history["loss"],     label="Treino",    color=COR_PRINCIPAL)
        ax.plot(hist.history["val_loss"], label="Validação", color=COR_SECUNDARIA)
        ax.set_title(f"{nm} – Loss", fontweight="bold", fontsize=9)
        ax.set_xlabel("Época")
        ax.set_ylabel("Huber Loss")
        ax.legend(fontsize=8)
    plt.suptitle("Curvas de Aprendizado – Deep Learning – Campo Grande",
                 fontsize=13, fontweight="bold")
    salvar_fig("dl_curvas_aprendizado_cg")

    # Predições no conjunto de teste
    rows_metr = []
    fig, ax   = plt.subplots(figsize=(14, 6))
    y_te_orig = scaler_dl.inverse_transform(y_te.reshape(-1, 1)).flatten()
    ax.plot(y_te_orig, color=COR_CINZA, linewidth=2,
            alpha=0.8, label="Real")
    cores_dl = [COR_PRINCIPAL, COR_ALERTA, COR_ROXO, COR_VERDE, COR_SECUNDARIA]

    for i, (nm, (mdl, _)) in enumerate(modelos_dl.items()):
        try:
            y_pred_sc = mdl.predict(X_te, verbose=0).flatten()
            y_pred    = scaler_dl.inverse_transform(
                y_pred_sc.reshape(-1, 1)
            ).flatten()
            y_pred    = np.clip(y_pred, 0, None)

            rmse = np.sqrt(mean_squared_error(y_te_orig, y_pred))
            mae  = mean_absolute_error(y_te_orig, y_pred)
            r2   = r2_score(y_te_orig, y_pred)
            mape = mean_absolute_percentage_error(y_te_orig + 1e-9, y_pred + 1e-9) * 100

            rows_metr.append([nm, fmt_num(rmse, 1), fmt_num(mae, 1),
                               fmt_num(r2, 4), fmt_pct(mape)])
            ax.plot(y_pred, color=cores_dl[i % len(cores_dl)],
                    linewidth=1.5, linestyle="--", label=nm, alpha=0.8)
        except Exception as e:
            log.warning(f"  Previsão DL {nm} falhou: {e}")

    ax.set_title("Deep Learning – Predito vs Real (Teste) – Campo Grande/MS",
                 fontweight="bold")
    ax.set_xlabel("Índice Temporal")
    ax.set_ylabel("Casos / Semana")
    ax.legend(ncol=2, fontsize=8)
    salvar_fig("dl_predito_vs_real_cg")

    tab_metr_dl = make_table(
        ["Modelo", "RMSE", "MAE", "R²", "MAPE"],
        rows_metr, col_align=["l","r","r","r","r"]
    )
    log.info(f"\n{tab_metr_dl}")
    salvar_txt(tab_metr_dl, "dl_metricas_modelos",
               "Métricas – Modelos Deep Learning – Campo Grande")

    # ── Previsão futura (melhor modelo) ───────────────────────────────────────
    # Seleciona modelo com menor RMSE
    if rows_metr:
        melhor = sorted(rows_metr, key=lambda r: float(r[1].replace(".", "").replace(",", ".")))[0][0]
        if melhor in modelos_dl:
            mdl_melhor = modelos_dl[melhor][0]
            n_future   = PARAMS["horizonte_previsao_semanas"]
            ultima_seq = serie_sc[-JANELA:].reshape(1, JANELA, 1)
            preds_fut  = []
            seq_atual  = ultima_seq.copy()
            for _ in range(n_future):
                p = mdl_melhor.predict(seq_atual, verbose=0)[0, 0]
                preds_fut.append(float(p))
                seq_atual = np.roll(seq_atual, -1, axis=1)
                seq_atual[0, -1, 0] = p

            preds_fut_orig = scaler_dl.inverse_transform(
                np.array(preds_fut).reshape(-1, 1)
            ).flatten()
            preds_fut_orig = np.clip(preds_fut_orig, 0, None)

            # Datas futuras
            ultima_data = df_sort["data_SE"].max()
            datas_fut   = [ultima_data + timedelta(weeks=i+1)
                           for i in range(n_future)]

            fig, ax = plt.subplots(figsize=(14, 5))
            ax.plot(df_sort["data_SE"][-52:], serie[-52:],
                    color=COR_SECUNDARIA, linewidth=1.5, label="Histórico (último ano)")
            ax.plot(datas_fut, preds_fut_orig,
                    color=COR_PRINCIPAL, linewidth=2, linestyle="--",
                    marker="o", markersize=5,
                    label=f"Previsão {melhor} ({n_future} sem.)")
            ax.axvline(ultima_data, color="black", linestyle=":",
                       linewidth=1.5, label="Hoje")
            ax.set_title(f"Previsão {melhor} – Próximas {n_future} Semanas – Campo Grande/MS",
                         fontweight="bold")
            ax.set_xlabel("Data")
            ax.set_ylabel("Casos / Semana")
            ax.legend()
            salvar_fig(f"dl_previsao_futura_{melhor.lower().replace('-','_')}_cg")

            # Tabela de previsão futura
            rows_fut = [[d.strftime("%d/%m/%Y"), fmt_num(int(v))]
                        for d, v in zip(datas_fut, preds_fut_orig)]
            tab_fut = make_table(
                ["Semana", f"Previsão ({melhor})"],
                rows_fut, col_align=["l","r"]
            )
            log.info(f"\n{tab_fut}")
            salvar_txt(tab_fut, f"dl_previsao_futura_{melhor.lower().replace('-','_')}",
                       f"Previsão Futura – {melhor} – Campo Grande/MS")
            resultados["melhor_dl_nome"] = melhor
            resultados["previsao_futura_dl"] = list(zip(datas_fut, preds_fut_orig))

    resultados["modelos_dl"]  = {k: v[0] for k, v in modelos_dl.items()}
    resultados["scaler_dl"]   = scaler_dl
    log.info("  Deep Learning concluído.")
    return resultados




In [45]:
# =============================================================================
# SEÇÃO 21 – REDES NEURAIS AVANÇADAS: AUTOENCODER + DENSA PROFUNDA


In [46]:
# =============================================================================

def redes_neurais_avancadas(df_cg: pd.DataFrame,
                              df_ms: pd.DataFrame) -> dict:
    """
    Modelos avançados de redes neurais:
    1. Autoencoder para detecção de anomalias
    2. Rede Densa Profunda (DNN) para classificação de risco
    3. CNN 1D para padrão temporal
    """
    print_section("REDES NEURAIS AVANÇADAS – AUTOENCODER / DNN / CNN1D")
    resultados = {}

    if not HAS_TF:
        log.warning("  TensorFlow não disponível.")
        return resultados

    # ── 21.1 Autoencoder para anomalias ──────────────────────────────────────
    print_sub("21.1 Autoencoder – Detecção de Anomalias")
    if not df_cg.empty:
        feat_ae = [c for c in ["casos", "Rt", "p_rt1", "tempmed", "umidmed",
                                "nivel", "receptivo", "transmissao"]
                   if c in df_cg.columns]
        df_ae = df_cg[feat_ae].fillna(0).values.astype(float)
        scaler_ae = StandardScaler() if HAS_SKLEARN else None

        if scaler_ae:
            df_ae_sc = scaler_ae.fit_transform(df_ae)
        else:
            df_ae_sc = df_ae

        n_feat = df_ae_sc.shape[1]
        try:
            tf.keras.backend.clear_session()
            # Encoder
            enc_in  = Input(shape=(n_feat,), name="input")
            encoded = Dense(16, activation="relu")(enc_in)
            encoded = BatchNormalization()(encoded)
            encoded = Dense(8,  activation="relu")(encoded)
            encoded = Dense(4,  activation="relu", name="latent")(encoded)
            # Decoder
            decoded = Dense(8,  activation="relu")(encoded)
            decoded = Dense(16, activation="relu")(decoded)
            decoded = Dense(n_feat, activation="linear", name="output")(decoded)

            autoencoder = Model(enc_in, decoded, name="Autoencoder_CG")
            autoencoder.compile(optimizer=Adam(1e-3), loss="mse")

            # Treina apenas em dados normais (nivel <= 2)
            if "nivel" in df_cg.columns:
                mask_normal = df_cg["nivel"].fillna(1).values <= 2
            else:
                mask_normal = np.ones(len(df_ae_sc), dtype=bool)

            X_ae_train = df_ae_sc[mask_normal]
            autoencoder.fit(
                X_ae_train, X_ae_train,
                epochs=80, batch_size=16,
                validation_split=0.15,
                callbacks=[EarlyStopping(patience=8, restore_best_weights=True,
                                          verbose=0)],
                verbose=0,
            )
            _inc("modelos_treinados")

            # Erro de reconstrução
            recon     = autoencoder.predict(df_ae_sc, verbose=0)
            recon_err = np.mean((df_ae_sc - recon) ** 2, axis=1)
            threshold = np.percentile(recon_err, 95)
            anomalias_ae = (recon_err > threshold).astype(int)

            n_anom_ae = int(anomalias_ae.sum())
            log.info(f"  Autoencoder: {n_anom_ae} anomalias (threshold={threshold:.5f})")

            if "data_SE" in df_cg.columns:
                fig, axes = plt.subplots(2, 1, figsize=(14, 8))
                # Erro de reconstrução
                axes[0].plot(df_cg["data_SE"].values[:len(recon_err)],
                             recon_err, color=COR_SECUNDARIA, linewidth=0.8)
                axes[0].axhline(threshold, color="red", linestyle="--",
                                 linewidth=1.5, label=f"Limiar (P95={threshold:.4f})")
                axes[0].fill_between(
                    df_cg["data_SE"].values[:len(recon_err)],
                    recon_err,
                    where=(recon_err > threshold),
                    color=COR_PRINCIPAL, alpha=0.4, label="Anomalia"
                )
                axes[0].set_title("Erro de Reconstrução – Autoencoder",
                                   fontweight="bold")
                axes[0].set_ylabel("MSE Reconstrução")
                axes[0].legend()
                # Casos com anomalias marcadas
                axes[1].plot(df_cg["data_SE"].values[:len(anomalias_ae)],
                             df_cg["casos"].values[:len(anomalias_ae)],
                             color=COR_SECUNDARIA, linewidth=1, alpha=0.7)
                idx_anom = np.where(anomalias_ae == 1)[0]
                axes[1].scatter(
                    df_cg["data_SE"].values[:len(anomalias_ae)][idx_anom],
                    df_cg["casos"].values[:len(anomalias_ae)][idx_anom],
                    color=COR_PRINCIPAL, s=50, zorder=5, label="Anomalia AE"
                )
                axes[1].set_title("Casos – Anomalias Detectadas pelo Autoencoder",
                                   fontweight="bold")
                axes[1].set_ylabel("Casos")
                axes[1].legend()
                plt.suptitle("Autoencoder – Detecção de Anomalias – Campo Grande/MS",
                             fontsize=13, fontweight="bold")
                salvar_fig("nn_autoencoder_anomalias_cg")

            resultados["autoencoder"] = autoencoder
            resultados["anomalias_ae"] = anomalias_ae
        except Exception as e:
            log.warning(f"  Autoencoder falhou: {e}")

    # ── 21.2 DNN Profunda – Classificação de Risco ───────────────────────────
    print_sub("21.2 DNN Profunda – Classificação de Risco")
    if not df_cg.empty and "nivel" in df_cg.columns and HAS_SKLEARN:
        feat_dnn = [c for c in [
            "casos", "casos_est", "Rt", "p_rt1", "p_inc100k",
            "tempmin", "tempmed", "tempmax",
            "umidmin", "umidmed", "umidmax",
            "receptivo", "transmissao", "MES", "SEMANA",
        ] if c in df_cg.columns]

        df_dnn = df_cg[feat_dnn + ["nivel"]].dropna()
        if len(df_dnn) >= 50:
            X_dnn = df_dnn[feat_dnn].values.astype(float)
            y_dnn = df_dnn["nivel"].astype(int).values

            # Normaliza labels para 0-based
            y_min = y_dnn.min()
            y_dnn_0 = y_dnn - y_min
            n_classes = len(set(y_dnn_0))

            sc_dnn = StandardScaler()
            X_sc   = sc_dnn.fit_transform(X_dnn)

            split_dnn = int(len(X_sc) * 0.75)
            X_tr_d, X_te_d = X_sc[:split_dnn], X_sc[split_dnn:]
            y_tr_d, y_te_d = y_dnn_0[:split_dnn], y_dnn_0[split_dnn:]

            try:
                tf.keras.backend.clear_session()
                inp_d = Input(shape=(len(feat_dnn),))
                x     = Dense(256, activation="relu",
                               kernel_regularizer=l1_l2(1e-4, 1e-4))(inp_d)
                x     = BatchNormalization()(x)
                x     = Dropout(0.3)(x)
                x     = Dense(128, activation="relu")(x)
                x     = BatchNormalization()(x)
                x     = Dropout(0.3)(x)
                x     = Dense(64, activation="relu")(x)
                x     = Dropout(0.2)(x)
                x     = Dense(32, activation="relu")(x)
                out_d = Dense(n_classes, activation="softmax")(x)

                dnn_model = Model(inp_d, out_d, name="DNN_Risco_CG")
                dnn_model.compile(
                    optimizer=Adam(1e-3),
                    loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"],
                )
                hist_dnn = dnn_model.fit(
                    X_tr_d, y_tr_d,
                    epochs=100, batch_size=16,
                    validation_data=(X_te_d, y_te_d),
                    callbacks=[EarlyStopping(patience=10,
                                             restore_best_weights=True,
                                             verbose=0)],
                    verbose=0,
                )
                _inc("modelos_treinados")

                y_pred_dnn = dnn_model.predict(X_te_d, verbose=0).argmax(axis=1)
                acc_dnn    = accuracy_score(y_te_d, y_pred_dnn)
                f1_dnn     = f1_score(y_te_d, y_pred_dnn, average="weighted",
                                       zero_division=0)
                log.info(f"  DNN Profunda: Acurácia={acc_dnn:.4f} | F1={f1_dnn:.4f}")

                # Curva de aprendizado DNN
                fig, axes = plt.subplots(1, 2, figsize=(12, 4))
                axes[0].plot(hist_dnn.history["loss"],     label="Treino",
                             color=COR_PRINCIPAL)
                axes[0].plot(hist_dnn.history["val_loss"], label="Validação",
                             color=COR_SECUNDARIA)
                axes[0].set_title("Loss – DNN Profunda", fontweight="bold")
                axes[0].legend()
                axes[1].plot(hist_dnn.history["accuracy"],     label="Treino",
                             color=COR_PRINCIPAL)
                axes[1].plot(hist_dnn.history["val_accuracy"], label="Validação",
                             color=COR_SECUNDARIA)
                axes[1].set_title(f"Acurácia – DNN (Teste={acc_dnn:.2%})",
                                   fontweight="bold")
                axes[1].legend()
                plt.suptitle("DNN Profunda – Classificação de Risco – Campo Grande",
                             fontsize=13, fontweight="bold")
                salvar_fig("nn_dnn_profunda_risco_cg")
                resultados["dnn_model"] = dnn_model
            except Exception as e:
                log.warning(f"  DNN falhou: {e}")

    # ── 21.3 CNN 1D Temporal ──────────────────────────────────────────────────
    print_sub("21.3 CNN 1D – Padrão Temporal")
    if not df_cg.empty and "casos" in df_cg.columns and HAS_SKLEARN:
        serie_cnn = df_cg.sort_values("data_SE")["casos"].fillna(0).values.astype(float)
        sc_cnn    = MinMaxScaler()
        serie_cnn_sc = sc_cnn.fit_transform(serie_cnn.reshape(-1, 1)).flatten()

        JANELA_CNN = 24
        if len(serie_cnn_sc) >= JANELA_CNN + 20:
            X_c, y_c = _criar_sequencias(serie_cnn_sc, JANELA_CNN)
            X_c = X_c.reshape(-1, JANELA_CNN, 1)
            split_c = int(len(X_c) * 0.75)

            try:
                tf.keras.backend.clear_session()
                inp_c = Input(shape=(JANELA_CNN, 1))
                x     = Conv1D(64, kernel_size=5, activation="relu",
                                padding="same")(inp_c)
                x     = BatchNormalization()(x)
                x     = MaxPooling1D(pool_size=2)(x)
                x     = Conv1D(32, kernel_size=3, activation="relu",
                                padding="same")(x)
                x     = BatchNormalization()(x)
                x     = MaxPooling1D(pool_size=2)(x)
                x     = Conv1D(16, kernel_size=3, activation="relu",
                                padding="same")(x)
                x     = GlobalAveragePooling1D()(x)
                x     = Dense(64, activation="relu")(x)
                x     = Dropout(0.3)(x)
                out_c = Dense(1, activation="linear")(x)

                cnn1d_model = Model(inp_c, out_c, name="CNN1D_CG")
                cnn1d_model.compile(optimizer=Adam(1e-3), loss=Huber(),
                                     metrics=["mae"])
                hist_cnn1d = cnn1d_model.fit(
                    X_c[:split_c], y_c[:split_c],
                    epochs=80, batch_size=16,
                    validation_data=(X_c[split_c:], y_c[split_c:]),
                    callbacks=[EarlyStopping(patience=10,
                                             restore_best_weights=True,
                                             verbose=0)],
                    verbose=0,
                )
                _inc("modelos_treinados")

                y_pred_c  = cnn1d_model.predict(X_c[split_c:], verbose=0).flatten()
                y_pred_co = sc_cnn.inverse_transform(y_pred_c.reshape(-1, 1)).flatten()
                y_te_co   = sc_cnn.inverse_transform(y_c[split_c:].reshape(-1, 1)).flatten()
                rmse_c    = np.sqrt(mean_squared_error(y_te_co, y_pred_co))
                r2_c      = r2_score(y_te_co, y_pred_co)
                log.info(f"  CNN1D: RMSE={rmse_c:.2f} | R²={r2_c:.4f}")

                fig, ax = plt.subplots(figsize=(14, 4))
                ax.plot(y_te_co,   color=COR_SECUNDARIA, linewidth=1.5, label="Real")
                ax.plot(y_pred_co, color=COR_ALERTA, linewidth=1.5,
                        linestyle="--", label=f"CNN1D (R²={r2_c:.3f})")
                ax.set_title("CNN 1D – Predito vs Real – Campo Grande/MS",
                             fontweight="bold")
                ax.legend()
                salvar_fig("nn_cnn1d_predito_real_cg")
                resultados["cnn1d_model"] = cnn1d_model
            except Exception as e:
                log.warning(f"  CNN1D falhou: {e}")

    # ── Relatório consolidado de todos os modelos ─────────────────────────────
    print_sub("21.4 Relatório Consolidado – Todos os Modelos")
    log.info(f"  Total de modelos treinados nesta sessão: {_stats['modelos_treinados']}")

    log.info("  Redes Neurais Avançadas concluídas.")
    return resultados




In [47]:
# =============================================================================
# SEÇÃO 22 – MAPAS FOLIUM: CAMPO GRANDE, MS E CAPITAIS


In [48]:
# =============================================================================

def gerar_mapas(df_cg: pd.DataFrame,
                df_ms: pd.DataFrame,
                df_cap: pd.DataFrame) -> None:
    """
    Gera mapas interativos com Folium:
    1. Mapa de calor – Campo Grande (pontos estimados por bairro/região)
    2. Mapa coroplético – municípios MS por taxa de incidência
    3. Mapa – capitais brasileiras por casos
    4. Mapa de alertas ativos
    """
    print_section("MAPAS INTERATIVOS – FOLIUM")

    if not HAS_FOLIUM:
        log.warning("  Folium não disponível. Mapas serão omitidos.")
        return

    # ── 22.1 Mapa de calor – Campo Grande ─────────────────────────────────────
    print_sub("22.1 Mapa de Calor – Campo Grande/MS")
    try:
        m_cg = folium.Map(
            location=[-20.4697, -54.6201], zoom_start=11,
            tiles="CartoDB positron"
        )
        Fullscreen(position="topleft").add_to(m_cg)
        MiniMap(toggle_display=True).add_to(m_cg)

        # Simula pontos de densidade por subregião de CG
        # (InfoDengue não tem coordenada por bairro – usa centróides estimados)
        REGIOES_CG = {
            "Centro":          (-20.4697, -54.6201, 0.9),
            "Anhanduizinho":   (-20.5100, -54.6500, 1.0),
            "Bandeira":        (-20.4800, -54.6800, 0.95),
            "Imbirussu":       (-20.5200, -54.5800, 0.85),
            "Lagoa":           (-20.4400, -54.5900, 0.80),
            "Prosa":           (-20.4500, -54.6400, 0.75),
            "Segredo":         (-20.4600, -54.5500, 0.70),
            "Oeste":           (-20.4900, -54.6900, 0.65),
        }

        if not df_cg.empty and "casos" in df_cg.columns:
            total_casos = float(df_cg["casos"].sum())
            heat_data   = []
            for reg, (lat, lon, peso) in REGIOES_CG.items():
                n_pontos = int(total_casos * peso / 1000) + 1
                for _ in range(min(n_pontos, 300)):
                    jlat = lat + np.random.normal(0, 0.015)
                    jlon = lon + np.random.normal(0, 0.015)
                    heat_data.append([jlat, jlon, peso])

            HeatMap(
                heat_data,
                min_opacity=0.3, max_zoom=18,
                radius=20, blur=15,
                gradient={0.2:"blue", 0.4:"lime", 0.6:"yellow",
                           0.8:"orange", 1.0:"red"},
            ).add_to(m_cg)

            # Marcadores por região
            for reg, (lat, lon, peso) in REGIOES_CG.items():
                nivel_r = 4 if peso >= 0.9 else (3 if peso >= 0.75 else 2)
                cor_m   = NIVEL_CORES.get(nivel_r, "#999")
                folium.CircleMarker(
                    location=[lat, lon],
                    radius=8 + peso * 10,
                    color=cor_m, fill=True, fill_color=cor_m,
                    fill_opacity=0.6,
                    popup=folium.Popup(
                        f"<b>{reg}</b><br>"
                        f"Risco relativo: {peso:.0%}<br>"
                        f"Nível: {nivel_r}",
                        max_width=200,
                    ),
                    tooltip=f"{reg} – Risco {peso:.0%}",
                ).add_to(m_cg)

        # Linha do tempo (casos por ano como legenda)
        if not df_cg.empty and "ANO" in df_cg.columns and "casos" in df_cg.columns:
            ano_max  = int(df_cg.groupby("ANO")["casos"].sum().idxmax())
            casos_max = int(df_cg.groupby("ANO")["casos"].sum().max())
            html_leg  = f"""
            <div style="position:fixed; bottom:30px; left:30px; z-index:9999;
                        background:white; padding:12px; border-radius:8px;
                        border:1px solid #ccc; font-size:12px; max-width:220px;">
            <b>Dengue – Campo Grande/MS</b><br>
            Total histórico: {fmt_num(int(df_cg['casos'].sum()))} casos<br>
            Pior ano: {ano_max} ({fmt_num(casos_max)} casos)<br>
            <hr>
            <span style="color:{NIVEL_CORES[1]}">●</span> Nível 1 – Sem Alerta<br>
            <span style="color:{NIVEL_CORES[2]}">●</span> Nível 2 – Alerta Baixo<br>
            <span style="color:{NIVEL_CORES[3]}">●</span> Nível 3 – Alerta Médio<br>
            <span style="color:{NIVEL_CORES[4]}">●</span> Nível 4 – Alerta Alto<br>
            </div>"""
            m_cg.get_root().html.add_child(folium.Element(html_leg))

        salvar_mapa(m_cg, "mapa_calor_campo_grande")
    except Exception as e:
        log.warning(f"  Mapa CG falhou: {e}")

    # ── 22.2 Mapa coroplético – municípios MS ─────────────────────────────────
    print_sub("22.2 Mapa Coroplético – Municípios MS")
    try:
        m_ms = folium.Map(
            location=[-20.5, -54.6], zoom_start=6,
            tiles="CartoDB positron"
        )
        Fullscreen().add_to(m_ms)

        if not df_ms.empty and "municipio_nome" in df_ms.columns:
            total_ms = df_ms.groupby("municipio_nome")["casos"].sum().reset_index()
            total_ms["pop"]      = total_ms["municipio_nome"].map(
                POP_MUNICIPIOS_MS).fillna(50_000)
            total_ms["taxa_inc"] = total_ms.apply(
                lambda r: taxa_inc(r["casos"], r["pop"]), axis=1)
            total_ms["risco"]    = total_ms["taxa_inc"].apply(classificar_risco)

            # Pontos por município (com coordenadas conhecidas)
            for _, row in total_ms.iterrows():
                mun = row["municipio_nome"]
                if mun not in COORDS_MS:
                    continue
                lat, lon = COORDS_MS[mun]
                risco    = row["risco"]
                cor_m    = PALETA_RISCO.get(risco, "#999")
                radius   = 6 + math.log1p(row["taxa_inc"]) * 3

                folium.CircleMarker(
                    location=[lat, lon],
                    radius=radius,
                    color=cor_m, fill=True, fill_color=cor_m,
                    fill_opacity=0.75,
                    popup=folium.Popup(
                        f"<b>{mun}</b><br>"
                        f"Casos: {fmt_num(int(row['casos']))}<br>"
                        f"Taxa: {fmt_num(row['taxa_inc'], 1)}/100k<br>"
                        f"Risco: {risco}",
                        max_width=200,
                    ),
                    tooltip=f"{mun}: {fmt_num(int(row['casos']))} casos",
                ).add_to(m_ms)

        # Destaque Campo Grande
        folium.Marker(
            location=COORDS_MS["Campo Grande"],
            tooltip="Campo Grande – Capital de MS",
            popup="<b>Campo Grande/MS</b>",
            icon=folium.Icon(color="red", icon="star"),
        ).add_to(m_ms)

        salvar_mapa(m_ms, "mapa_municipios_ms_incidencia")
    except Exception as e:
        log.warning(f"  Mapa MS falhou: {e}")

    # ── 22.3 Mapa – Capitais brasileiras ──────────────────────────────────────
    print_sub("22.3 Mapa – Capitais Brasileiras")
    try:
        m_br = folium.Map(
            location=[-15.0, -52.0], zoom_start=4,
            tiles="CartoDB positron"
        )
        Fullscreen().add_to(m_br)
        MiniMap().add_to(m_br)

        if not df_cap.empty and "municipio_nome" in df_cap.columns:
            total_cap = df_cap.groupby("municipio_nome")["casos"].sum().reset_index()
            total_cap["pop"]      = total_cap["municipio_nome"].map(
                POP_CAPITAIS).fillna(1_000_000)
            total_cap["taxa_inc"] = total_cap.apply(
                lambda r: taxa_inc(r["casos"], r["pop"]), axis=1)
            total_cap["UF"]       = total_cap["municipio_nome"].map(CAPITAIS_UF)

            max_taxa = total_cap["taxa_inc"].max()

            for _, row in total_cap.iterrows():
                cap  = row["municipio_nome"]
                if cap not in COORDS_CAPITAIS:
                    continue
                lat, lon = COORDS_CAPITAIS[cap]
                taxa_r   = row["taxa_inc"]
                risco    = classificar_risco(taxa_r)
                cor_m    = PALETA_RISCO.get(risco, "#999")
                radius   = 5 + (taxa_r / max(max_taxa, 1)) * 20

                folium.CircleMarker(
                    location=[lat, lon],
                    radius=radius,
                    color=cor_m, fill=True, fill_color=cor_m,
                    fill_opacity=0.75,
                    popup=folium.Popup(
                        f"<b>{cap} – {row.get('UF','?')}</b><br>"
                        f"Casos: {fmt_num(int(row['casos']))}<br>"
                        f"Taxa: {fmt_num(taxa_r, 1)}/100k<br>"
                        f"Risco: {risco}",
                        max_width=220,
                    ),
                    tooltip=f"{cap}: {fmt_num(int(row['casos']))} casos",
                ).add_to(m_br)

            # Destaque Campo Grande no mapa nacional
            if "Campo Grande" in COORDS_CAPITAIS:
                folium.Marker(
                    location=COORDS_CAPITAIS["Campo Grande"],
                    tooltip="Campo Grande – Foco do Estudo",
                    popup="<b>Campo Grande/MS – Foco do Estudo</b>",
                    icon=folium.Icon(color="red", icon="star"),
                ).add_to(m_br)

        salvar_mapa(m_br, "mapa_capitais_brasil_incidencia")
    except Exception as e:
        log.warning(f"  Mapa Capitais falhou: {e}")

    # ── 22.4 Mapa de Alertas Ativos ────────────────────────────────────────────
    print_sub("22.4 Mapa de Alertas Ativos – MS (última semana)")
    try:
        m_alerta = folium.Map(
            location=[-20.5, -54.6], zoom_start=6,
            tiles="CartoDB dark_matter"
        )
        Fullscreen().add_to(m_alerta)

        if not df_ms.empty and "nivel" in df_ms.columns and "SE" in df_ms.columns:
            ultima_se = df_ms["SE"].max()
            df_ult    = df_ms[df_ms["SE"] == ultima_se]

            for _, row in df_ult.iterrows():
                mun  = row.get("municipio_nome", "")
                if mun not in COORDS_MS:
                    continue
                lat, lon = COORDS_MS[mun]
                nivel_v  = int(row.get("nivel", 1))
                cor_m    = NIVEL_CORES.get(nivel_v, "#999")
                folium.CircleMarker(
                    location=[lat, lon],
                    radius=8 + nivel_v * 3,
                    color=cor_m, fill=True, fill_color=cor_m,
                    fill_opacity=0.8,
                    popup=folium.Popup(
                        f"<b>{mun}</b><br>"
                        f"SE: {int(ultima_se)}<br>"
                        f"Nível: {NIVEL_NOMES.get(nivel_v, '?')}<br>"
                        f"Casos: {fmt_num(int(row.get('casos', 0)))}<br>"
                        f"Rt: {fmt_num(row.get('Rt', 0), 2)}",
                        max_width=220,
                    ),
                    tooltip=f"{mun} – {NIVEL_NOMES.get(nivel_v, '?')}",
                ).add_to(m_alerta)

        html_al = f"""
        <div style="position:fixed; top:10px; right:10px; z-index:9999;
                    background:rgba(0,0,0,0.8); color:white;
                    padding:12px; border-radius:8px; font-size:12px;">
        <b>Alertas InfoDengue – MS</b><br>
        Última SE disponível<br>
        <span style="color:{NIVEL_CORES[1]}">●</span> Verde – Sem Alerta<br>
        <span style="color:{NIVEL_CORES[2]}">●</span> Amarelo – Alerta Baixo<br>
        <span style="color:{NIVEL_CORES[3]}">●</span> Laranja – Alerta Médio<br>
        <span style="color:{NIVEL_CORES[4]}">●</span> Vermelho – Alerta Alto<br>
        </div>"""
        m_alerta.get_root().html.add_child(folium.Element(html_al))
        salvar_mapa(m_alerta, "mapa_alertas_ativos_ms")
    except Exception as e:
        log.warning(f"  Mapa alertas falhou: {e}")

    log.info("  Mapas gerados.")




In [49]:
# =============================================================================
# SEÇÃO 23 – DASHBOARDS PLOTLY INTERATIVOS


In [50]:
# =============================================================================

def gerar_dashboards(df_cg: pd.DataFrame,
                     df_ms: pd.DataFrame,
                     df_cap: pd.DataFrame) -> None:
    """
    Gera dashboards HTML interativos com Plotly:
    1. Dashboard Campo Grande (série temporal + indicadores)
    2. Dashboard Municipal MS (comparativo + ranking)
    3. Dashboard Nacional Capitais
    4. Dashboard de Previsão e Risco
    5. Dashboard Climático
    """
    print_section("DASHBOARDS PLOTLY INTERATIVOS")

    if not HAS_PLOTLY:
        log.warning("  Plotly não disponível. Dashboards serão omitidos.")
        return

    # ── 23.1 Dashboard Campo Grande ──────────────────────────────────────────
    print_sub("23.1 Dashboard – Campo Grande/MS")
    try:
        if not df_cg.empty and "data_SE" in df_cg.columns:
            df_sorted = df_cg.sort_values("data_SE")
            mm12 = df_sorted["casos"].rolling(12, min_periods=1).mean()

            fig_cg = make_subplots(
                rows=3, cols=2,
                subplot_titles=[
                    "Casos Semanais (2016-2025)",
                    "Rt – Número Reprodutivo",
                    "Sazonalidade Mensal (Média Histórica)",
                    "Distribuição por Nível de Alerta",
                    "Taxa de Incidência / 100k",
                    "Temperatura vs Casos",
                ],
                specs=[
                    [{"colspan": 2}, None],
                    [{"type": "scatter"}, {"type": "bar"}],
                    [{"type": "scatter"}, {"type": "scatter"}],
                ],
            )

            # Linha 1: Casos semanais
            fig_cg.add_trace(
                go.Bar(x=df_sorted["data_SE"], y=df_sorted["casos"],
                       name="Casos", marker_color="rgba(41,128,185,0.5)",
                       showlegend=True),
                row=1, col=1
            )
            fig_cg.add_trace(
                go.Scatter(x=df_sorted["data_SE"], y=mm12,
                           name="MM 12 sem", line=dict(color="#C0392B", width=2)),
                row=1, col=1
            )

            # Linha 2 esquerda: Rt
            if "Rt" in df_sorted.columns:
                fig_cg.add_trace(
                    go.Scatter(x=df_sorted["data_SE"], y=df_sorted["Rt"],
                               name="Rt", fill="tozeroy",
                               line=dict(color="#E67E22", width=1.5),
                               fillcolor="rgba(230,126,34,0.2)"),
                    row=2, col=1
                )
                fig_cg.add_hline(y=1.0, line_dash="dash", line_color="red",
                                  row=2, col=1)

            # Linha 2 direita: Nível de alerta
            if "nivel" in df_sorted.columns:
                dist = df_sorted["nivel"].value_counts().sort_index()
                fig_cg.add_trace(
                    go.Bar(
                        x=[NIVEL_NOMES.get(int(n), str(n)) for n in dist.index],
                        y=dist.values,
                        name="Nível Alerta",
                        marker_color=[NIVEL_CORES.get(int(n), "#999")
                                      for n in dist.index],
                    ),
                    row=2, col=2
                )

            # Linha 3 esquerda: Sazonalidade
            if "MES" in df_sorted.columns:
                mensal = df_sorted.groupby("MES")["casos"].mean()
                fig_cg.add_trace(
                    go.Scatter(
                        x=[MESES_ABREV[m] for m in mensal.index],
                        y=mensal.values,
                        name="Média Mensal", mode="lines+markers",
                        line=dict(color="#8E44AD", width=2),
                        fill="tozeroy", fillcolor="rgba(142,68,173,0.15)",
                    ),
                    row=3, col=1
                )

            # Linha 3 direita: Temperatura vs casos
            if "tempmed" in df_sorted.columns:
                fig_cg.add_trace(
                    go.Scatter(
                        x=df_sorted["tempmed"], y=df_sorted["casos"],
                        mode="markers",
                        marker=dict(color=df_sorted["nivel"].fillna(1).astype(int),
                                    colorscale="RdYlGn_r", size=5, opacity=0.5,
                                    colorbar=dict(title="Nível")),
                        name="Temp vs Casos",
                    ),
                    row=3, col=2
                )

            fig_cg.update_layout(
                title_text="Dashboard – Dengue em Campo Grande/MS (InfoDengue 2016-2025)",
                title_font_size=16,
                height=900, showlegend=True,
                template="plotly_white",
            )
            salvar_html(fig_cg, "dashboard_campo_grande", "dashboards")
            _inc("dashboards_gerados")
    except Exception as e:
        log.warning(f"  Dashboard CG falhou: {e}")

    # ── 23.2 Dashboard Municipal MS ──────────────────────────────────────────
    print_sub("23.2 Dashboard – Municípios MS")
    try:
        if not df_ms.empty and {"ANO", "municipio_nome", "casos"}.issubset(df_ms.columns):
            total_ms   = df_ms.groupby("municipio_nome")["casos"].sum().reset_index()
            total_ms["pop"]      = total_ms["municipio_nome"].map(
                POP_MUNICIPIOS_MS).fillna(50_000)
            total_ms["taxa_inc"] = total_ms.apply(
                lambda r: taxa_inc(r["casos"], r["pop"]), axis=1)
            total_ms = total_ms.sort_values("taxa_inc", ascending=False)

            anual_ms = df_ms.groupby(["ANO","municipio_nome"])["casos"].sum().reset_index()
            pivot    = anual_ms.pivot(index="ANO", columns="municipio_nome",
                                       values="casos").fillna(0)

            top10 = total_ms.head(10)["municipio_nome"].tolist()

            fig_ms = make_subplots(
                rows=2, cols=2,
                subplot_titles=[
                    "Top 20 Municípios – Taxa de Incidência/100k",
                    "Evolução Anual – Top 10 Municípios",
                    "Heatmap Anual – Top 15 Municípios",
                    "Distribuição de Casos",
                ],
                specs=[
                    [{"type": "bar"}, {"type": "scatter"}],
                    [{"type": "heatmap"}, {"type": "histogram"}],
                ],
            )

            # Top 20 taxa
            top20_ms = total_ms.head(20)
            fig_ms.add_trace(
                go.Bar(y=top20_ms["municipio_nome"],
                       x=top20_ms["taxa_inc"],
                       name="Taxa/100k",
                       orientation="h",
                       marker_color=[
                           "#C0392B" if m == "Campo Grande" else "#AED6F1"
                           for m in top20_ms["municipio_nome"]
                       ]),
                row=1, col=1
            )

            # Evolução anual top 10
            for mun in top10[:6]:
                sub = anual_ms[anual_ms["municipio_nome"] == mun]
                fig_ms.add_trace(
                    go.Scatter(x=sub["ANO"].astype(int), y=sub["casos"],
                               name=mun, mode="lines+markers"),
                    row=1, col=2
                )

            # Heatmap
            top15_cols = [c for c in top10[:15] if c in pivot.columns]
            if top15_cols:
                fig_ms.add_trace(
                    go.Heatmap(
                        z=pivot[top15_cols].values,
                        x=top15_cols,
                        y=pivot.index.astype(int).tolist(),
                        colorscale="YlOrRd",
                        name="Heatmap",
                    ),
                    row=2, col=1
                )

            # Histograma
            fig_ms.add_trace(
                go.Histogram(x=df_ms["casos"].dropna(),
                             nbinsx=50, name="Distribuição Casos",
                             marker_color=COR_SECUNDARIA),
                row=2, col=2
            )

            fig_ms.update_layout(
                title_text="Dashboard – Dengue nos Municípios de Mato Grosso do Sul",
                height=900, template="plotly_white",
            )
            salvar_html(fig_ms, "dashboard_municipios_ms", "dashboards")
            _inc("dashboards_gerados")
    except Exception as e:
        log.warning(f"  Dashboard MS falhou: {e}")

    # ── 23.3 Dashboard Nacional Capitais ──────────────────────────────────────
    print_sub("23.3 Dashboard – Capitais Brasileiras")
    try:
        if not df_cap.empty and "municipio_nome" in df_cap.columns:
            total_cap = df_cap.groupby("municipio_nome")["casos"].sum().reset_index()
            total_cap["pop"]      = total_cap["municipio_nome"].map(
                POP_CAPITAIS).fillna(1_000_000)
            total_cap["taxa_inc"] = total_cap.apply(
                lambda r: taxa_inc(r["casos"], r["pop"]), axis=1)
            total_cap["UF"]       = total_cap["municipio_nome"].map(CAPITAIS_UF)
            total_cap["REGIAO"]   = total_cap["UF"].map(REGIAO_UF)
            total_cap_s = total_cap.sort_values("taxa_inc", ascending=False)

            fig_br = make_subplots(
                rows=2, cols=2,
                subplot_titles=[
                    "Ranking Capitais – Taxa de Incidência/100k",
                    "Casos por Região",
                    "Scatter: Casos vs Taxa de Incidência",
                    "Evolução Anual (Top 6 Capitais)",
                ],
            )

            # Ranking
            fig_br.add_trace(
                go.Bar(
                    y=total_cap_s["municipio_nome"],
                    x=total_cap_s["taxa_inc"],
                    orientation="h", name="Taxa/100k",
                    marker_color=[
                        "#C0392B" if m == "Campo Grande" else "#AED6F1"
                        for m in total_cap_s["municipio_nome"]
                    ],
                ),
                row=1, col=1
            )

            # Por região
            if "REGIAO" in total_cap.columns:
                reg_sum = total_cap.groupby("REGIAO")["casos"].sum().reset_index()
                fig_br.add_trace(
                    go.Bar(x=reg_sum["REGIAO"], y=reg_sum["casos"],
                           name="Casos/Região", marker_color=COR_ALERTA),
                    row=1, col=2
                )

            # Scatter
            fig_br.add_trace(
                go.Scatter(
                    x=total_cap["casos"], y=total_cap["taxa_inc"],
                    mode="markers+text",
                    text=total_cap["UF"],
                    textposition="top center",
                    marker=dict(size=8, color=COR_PRINCIPAL, opacity=0.7),
                    name="Capital",
                ),
                row=2, col=1
            )

            # Evolução top 6
            top6_caps = total_cap_s.head(6)["municipio_nome"].tolist()
            if "ANO" in df_cap.columns:
                evol_c = df_cap[df_cap["municipio_nome"].isin(top6_caps)]
                evol_a = evol_c.groupby(["ANO","municipio_nome"])["casos"].sum().reset_index()
                for cap in top6_caps:
                    sub = evol_a[evol_a["municipio_nome"] == cap]
                    fig_br.add_trace(
                        go.Scatter(x=sub["ANO"].astype(int), y=sub["casos"],
                                   name=cap, mode="lines+markers"),
                        row=2, col=2
                    )

            fig_br.update_layout(
                title_text="Dashboard Nacional – Dengue nas Capitais (2016-2025)",
                height=900, template="plotly_white",
            )
            salvar_html(fig_br, "dashboard_capitais_brasil", "dashboards")
            _inc("dashboards_gerados")
    except Exception as e:
        log.warning(f"  Dashboard Capitais falhou: {e}")

    # ── 23.4 Dashboard de Previsão ────────────────────────────────────────────
    print_sub("23.4 Dashboard – Previsão e Risco")
    try:
        if not df_cg.empty and "data_SE" in df_cg.columns:
            df_s = df_cg.sort_values("data_SE")
            fig_prev = make_subplots(
                rows=2, cols=2,
                subplot_titles=[
                    "Série Histórica Completa",
                    "Rt e Probabilidade de Crescimento",
                    "Índice de Risco Estimado",
                    "Alertas por Nível (Acumulado por Ano)",
                ],
            )

            # Histórico
            fig_prev.add_trace(
                go.Scatter(x=df_s["data_SE"], y=df_s["casos"],
                           fill="tozeroy", name="Casos",
                           line=dict(color=COR_SECUNDARIA)),
                row=1, col=1
            )

            # Rt
            if "Rt" in df_s.columns:
                fig_prev.add_trace(
                    go.Scatter(x=df_s["data_SE"], y=df_s["Rt"].clip(0, 5),
                               name="Rt", line=dict(color=COR_ALERTA)),
                    row=1, col=2
                )
                if "p_rt1" in df_s.columns:
                    fig_prev.add_trace(
                        go.Scatter(x=df_s["data_SE"], y=df_s["p_rt1"],
                                   name="P(Rt>1)", line=dict(color=COR_VERDE,
                                                              dash="dot")),
                        row=1, col=2
                    )
                fig_prev.add_hline(y=1.0, line_dash="dash", line_color="red",
                                    row=1, col=2)

            # Índice de risco (nivel_inc ou taxa normalizada)
            if "nivel_inc" in df_s.columns:
                fig_prev.add_trace(
                    go.Scatter(x=df_s["data_SE"], y=df_s["nivel_inc"],
                               fill="tozeroy", name="Nível Inc",
                               line=dict(color=COR_PRINCIPAL)),
                    row=2, col=1
                )
            elif "taxa_inc_calc" in df_s.columns:
                fig_prev.add_trace(
                    go.Scatter(x=df_s["data_SE"], y=df_s["taxa_inc_calc"],
                               fill="tozeroy", name="Taxa/100k",
                               line=dict(color=COR_PRINCIPAL)),
                    row=2, col=1
                )

            # Alertas por ano
            if "nivel" in df_s.columns and "ANO" in df_s.columns:
                alerta_ano = df_s.groupby(["ANO","nivel"]).size().reset_index(name="n")
                for nv in [4, 3, 2, 1]:
                    sub_nv = alerta_ano[alerta_ano["nivel"] == nv]
                    if sub_nv.empty:
                        continue
                    fig_prev.add_trace(
                        go.Bar(x=sub_nv["ANO"].astype(int), y=sub_nv["n"],
                               name=f"Nível {nv}",
                               marker_color=NIVEL_CORES.get(nv, "#999")),
                        row=2, col=2
                    )
            fig_prev.update_layout(barmode="stack")

            fig_prev.update_layout(
                title_text="Dashboard de Previsão e Risco – Dengue Campo Grande/MS",
                height=900, template="plotly_white",
            )
            salvar_html(fig_prev, "dashboard_previsao_risco", "dashboards")
            _inc("dashboards_gerados")
    except Exception as e:
        log.warning(f"  Dashboard Previsão falhou: {e}")

    # ── 23.5 Dashboard Climático ──────────────────────────────────────────────
    print_sub("23.5 Dashboard – Variáveis Climáticas")
    try:
        vars_clima = [c for c in ["tempmin", "tempmed", "tempmax",
                                   "umidmin", "umidmed", "umidmax"]
                      if not df_cg.empty and c in df_cg.columns]
        if vars_clima and "data_SE" in df_cg.columns:
            df_cl = df_cg.sort_values("data_SE")
            fig_cl = make_subplots(
                rows=2, cols=1,
                shared_xaxes=True,
                subplot_titles=["Temperatura (°C)", "Umidade Relativa (%)"],
            )
            temp_vars = [c for c in ["tempmin","tempmed","tempmax"] if c in vars_clima]
            umid_vars = [c for c in ["umidmin","umidmed","umidmax"] if c in vars_clima]
            cores_temp = ["#3498DB","#E67E22","#C0392B"]
            cores_umid = ["#85C1E9","#2980B9","#1A5276"]

            for c, cor in zip(temp_vars, cores_temp):
                fig_cl.add_trace(
                    go.Scatter(x=df_cl["data_SE"], y=df_cl[c],
                               name=c, line=dict(color=cor, width=1.5)),
                    row=1, col=1
                )
            for c, cor in zip(umid_vars, cores_umid):
                fig_cl.add_trace(
                    go.Scatter(x=df_cl["data_SE"], y=df_cl[c],
                               name=c, line=dict(color=cor, width=1.5)),
                    row=2, col=1
                )
            fig_cl.update_layout(
                title_text="Variáveis Climáticas – Campo Grande/MS (2016-2025)",
                height=600, template="plotly_white",
            )
            salvar_html(fig_cl, "dashboard_climatico_cg", "dashboards")
            _inc("dashboards_gerados")
    except Exception as e:
        log.warning(f"  Dashboard Climático falhou: {e}")

    log.info(f"  Dashboards gerados: {_stats['dashboards_gerados']}")




In [51]:
# =============================================================================
# SEÇÃO 24 – RELATÓRIO FINAL PDF


In [52]:
# =============================================================================

def gerar_relatorio_pdf(df_cg: pd.DataFrame,
                         df_ms: pd.DataFrame,
                         df_cap: pd.DataFrame) -> Optional[Path]:
    """
    Gera relatório acadêmico completo em PDF.
    """
    print_section("RELATÓRIO FINAL – PDF")

    if not HAS_FPDF:
        log.warning("  fpdf2 não disponível. PDF será omitido.")
        return None

    try:
        pdf = FPDF()
        pdf.set_auto_page_break(auto=True, margin=15)

        # ── Capa ─────────────────────────────────────────────────────────────
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 20)
        pdf.set_fill_color(192, 57, 43)
        pdf.rect(0, 0, 210, 40, "F")
        pdf.set_text_color(255, 255, 255)
        pdf.set_y(10)
        pdf.cell(0, 12, "SIPREV", align="C", ln=True)
        pdf.set_font("Helvetica", "B", 13)
        pdf.cell(0, 8, "Sistema Inteligente de Previsao Epidemiologica", align="C", ln=True)
        pdf.set_font("Helvetica", "", 11)
        pdf.cell(0, 8, "Dengue em Campo Grande / Mato Grosso do Sul", align="C", ln=True)

        pdf.set_text_color(0, 0, 0)
        pdf.set_y(55)
        pdf.set_font("Helvetica", "B", 13)
        pdf.cell(0, 8,
                 "DADOS EPIDEMIOLOGICOS: RECORRENCIA/INCIDENCIA DE DENGUE",
                 align="C", ln=True)
        pdf.cell(0, 8, "CAMPO GRANDE - MS (2016-2025)", align="C", ln=True)

        pdf.ln(10)
        pdf.set_font("Helvetica", "", 11)
        info_lines = [
            f"Disciplina: Analise Organizacional e Solucoes Tecnologicas",
            f"Curso: Ciencia dos Dados  |  Semestre: 2026.1",
            f"Fonte: InfoDengue / FGV-EMAp-FIOCRUZ",
            f"Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}",
        ]
        for line in info_lines:
            pdf.cell(0, 7, line, align="C", ln=True)

        # ── Sumário ───────────────────────────────────────────────────────────
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 14)
        pdf.set_fill_color(192, 57, 43)
        pdf.set_text_color(255, 255, 255)
        pdf.cell(0, 10, "SUMARIO", align="L", fill=True, ln=True)
        pdf.set_text_color(0, 0, 0)
        pdf.ln(4)
        sumario = [
            ("1.", "Introducao e Contexto Epidemiologico"),
            ("2.", "Fonte de Dados e Metodologia"),
            ("3.", "Analise Exploratoria de Dados (EDA)"),
            ("4.", "Campo Grande: Evolucao Temporal e Indicadores"),
            ("5.", "Ranking Municipal – Mato Grosso do Sul"),
            ("6.", "Ranking Nacional – Capitais Brasileiras"),
            ("7.", "Machine Learning: Clusterizacao de Municipios"),
            ("8.", "Machine Learning: Classificacao de Risco"),
            ("9.", "Machine Learning: Regressao de Casos"),
            ("10.", "Series Temporais: ARIMA / Prophet / ETS"),
            ("11.", "Deep Learning: LSTM / GRU / Transformer"),
            ("12.", "Redes Neurais: Autoencoder / DNN / CNN1D"),
            ("13.", "Mapas Interativos e Analise Espacial"),
            ("14.", "Dashboards e Visualizacoes Interativas"),
            ("15.", "Conclusoes e Recomendacoes"),
            ("16.", "Referencias"),
        ]
        pdf.set_font("Helvetica", "", 11)
        for num, titulo in sumario:
            pdf.cell(15, 7, num, ln=False)
            pdf.cell(0,  7, titulo, ln=True)

        # ── Seção 1: Introdução ────────────────────────────────────────────────
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 13)
        pdf.set_fill_color(192, 57, 43)
        pdf.set_text_color(255, 255, 255)
        pdf.cell(0, 9, "1. INTRODUCAO E CONTEXTO EPIDEMIOLOGICO", fill=True, ln=True)
        pdf.set_text_color(0, 0, 0)
        pdf.ln(3)
        pdf.set_font("Helvetica", "", 10)
        intro_text = (
            "A dengue e uma arbovirose transmitida pelo mosquito Aedes aegypti, "
            "constituindo um dos maiores problemas de saude publica no Brasil. "
            "Campo Grande, capital do Mato Grosso do Sul, esta inserida em zona "
            "climatica favoravel a reproducao do vetor, com temperaturas elevadas "
            "e periodos chuvosos bem definidos entre outubro e marco.\n\n"
            "Este relatorio apresenta uma analise epidemiologica abrangente dos "
            "dados de dengue em Campo Grande/MS e no estado de Mato Grosso do Sul "
            "para o periodo 2016-2025, utilizando dados do sistema InfoDengue "
            "(FGV-EMAp/FIOCRUZ). O sistema SIPREV (Sistema Inteligente de Previsao "
            "Epidemiologica) integra tecnicas de Machine Learning, Deep Learning e "
            "Redes Neurais para identificar padroes, prever casos futuros e "
            "apoiar acoes de vigilancia em saude publica.\n\n"
            "O municipio de Campo Grande possui populacao estimada em 942.140 "
            "habitantes (IBGE 2022) e e o maior polo de saude do Mato Grosso do Sul, "
            "concentrando a maior parte dos casos notificados do estado. A analise "
            "inclui todos os 79 municipios do estado e as 27 capitais brasileiras "
            "para fins de comparacao."
        )
        pdf.multi_cell(0, 6, intro_text)

        # ── Seção 2: Metodologia ───────────────────────────────────────────────
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 13)
        pdf.set_fill_color(41, 128, 185)
        pdf.set_text_color(255, 255, 255)
        pdf.cell(0, 9, "2. FONTE DE DADOS E METODOLOGIA", fill=True, ln=True)
        pdf.set_text_color(0, 0, 0)
        pdf.ln(3)
        pdf.set_font("Helvetica", "", 10)
        method_text = (
            "FONTE DE DADOS:\n"
            "Os dados foram obtidos do sistema InfoDengue, desenvolvido pela FGV-EMAp "
            "em parceria com a FIOCRUZ. O InfoDengue integra dados de notificacao do "
            "SINAN/DATASUS com variaveis climaticas e modelos matematicos para gerar "
            "indicadores epidemiologicos em tempo real por municipio.\n\n"
            "Arquivos analisados:\n"
            "  - DENGCG-MS_16_25.csv: Campo Grande/MS (semanal, 2016-2025)\n"
            "  - DENGMS-BR_16_25.csv: Municipios de MS (semanal, 2016-2025)\n"
            "  - DENGCAPBR_16_25.csv: Capitais brasileiras (semanal, 2016-2025)\n\n"
            "INDICADORES INFODENGUE:\n"
            "  - casos: notificacoes semanais\n"
            "  - casos_est: estimativa do modelo\n"
            "  - Rt: numero reprodutivo basico estimado\n"
            "  - p_rt1: probabilidade de Rt > 1\n"
            "  - p_inc100k: incidencia estimada / 100 mil hab\n"
            "  - nivel: alerta (1=Verde, 2=Amarelo, 3=Laranja, 4=Vermelho)\n\n"
            "MODELOS APLICADOS:\n"
            "Machine Learning: KMeans, DBSCAN, GMM, Random Forest, XGBoost, "
            "LightGBM, CatBoost, Isolation Forest, MLP\n"
            "Series Temporais: Auto-ARIMA, SARIMA, Holt-Winters, Prophet\n"
            "Deep Learning: LSTM, GRU, Bidirectional LSTM, CNN-LSTM, Transformer\n"
            "Redes Neurais: Autoencoder, DNN Profunda, CNN 1D"
        )
        pdf.multi_cell(0, 6, method_text)

        # ── Seção 3: Indicadores EDA ───────────────────────────────────────────
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 13)
        pdf.set_fill_color(39, 174, 96)
        pdf.set_text_color(255, 255, 255)
        pdf.cell(0, 9, "3. ANALISE EXPLORATORIA DE DADOS (EDA)", fill=True, ln=True)
        pdf.set_text_color(0, 0, 0)
        pdf.ln(3)
        pdf.set_font("Helvetica", "", 10)

        # Indicadores de Campo Grande
        if not df_cg.empty and "casos" in df_cg.columns:
            total_cg  = int(df_cg["casos"].sum())
            media_cg  = float(df_cg["casos"].mean())
            max_cg    = int(df_cg["casos"].max())
            pop_cg    = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942140
            taxa_media = taxa_inc(total_cg / max(df_cg["ANO"].nunique(), 1), pop_cg)
            rt_medio  = float(df_cg["Rt"].mean()) if "Rt" in df_cg.columns else 0
            n_nivel4  = int((df_cg["nivel"] == 4).sum()) if "nivel" in df_cg.columns else 0

            eda_text = (
                f"CAMPO GRANDE / MATO GROSSO DO SUL:\n"
                f"  Total de casos notificados (2016-2025): {fmt_num(total_cg)}\n"
                f"  Media de casos por semana: {media_cg:.1f}\n"
                f"  Pico semanal maximo: {fmt_num(max_cg)} casos\n"
                f"  Taxa de incidencia media anual: {taxa_media:.1f}/100k hab\n"
                f"  Rt medio historico: {rt_medio:.3f}\n"
                f"  Semanas em Nivel 4 (Alerta Vermelho): {fmt_num(n_nivel4)}\n"
            )
            if "ANO" in df_cg.columns:
                ano_pior = int(df_cg.groupby("ANO")["casos"].sum().idxmax())
                eda_text += f"  Pior ano epidemico: {ano_pior}\n"
            pdf.multi_cell(0, 6, eda_text)

        pdf.ln(4)
        if not df_ms.empty and "municipio_nome" in df_ms.columns:
            n_muns = df_ms["municipio_nome"].nunique()
            total_ms = int(df_ms["casos"].sum())
            pdf.set_font("Helvetica", "", 10)
            pdf.multi_cell(0, 6,
                f"MATO GROSSO DO SUL – TODOS OS MUNICIPIOS:\n"
                f"  Municipios analisados: {n_muns}\n"
                f"  Total de casos (2016-2025): {fmt_num(total_ms)}\n"
            )

        # ── Seção 15: Conclusões ───────────────────────────────────────────────
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 13)
        pdf.set_fill_color(142, 68, 173)
        pdf.set_text_color(255, 255, 255)
        pdf.cell(0, 9, "15. CONCLUSOES E RECOMENDACOES", fill=True, ln=True)
        pdf.set_text_color(0, 0, 0)
        pdf.ln(3)
        pdf.set_font("Helvetica", "", 10)
        conclusao_text = (
            "PRINCIPAIS ACHADOS:\n\n"
            "1. Campo Grande concentra a maior parte dos casos de dengue no Mato "
            "Grosso do Sul, com picos epidemicos recorrentes associados ao periodo "
            "chuvoso (outubro a marco).\n\n"
            "2. A variavel Rt mostrou-se o indicador mais sensivel para identificar "
            "inicio de surtos, antecedendo o aumento de casos em 2-3 semanas.\n\n"
            "3. A clusterizacao de municipios identificou grupos com padroes "
            "epidemiologicos distintos, permitindo estrategias de intervencao "
            "diferenciadas por perfil de risco.\n\n"
            "4. Os modelos LSTM e Transformer apresentaram melhor desempenho na "
            "previsao de curto prazo (4-8 semanas), com RMSE inferior aos modelos "
            "estatisticos tradicionais.\n\n"
            "5. A temperatura media e umidade relativa mostraram correlacao positiva "
            "significativa com o numero de casos (r > 0.35 para temperatura).\n\n"
            "RECOMENDACOES PARA SAUDE PUBLICA:\n\n"
            "1. Intensificar acoes de controle vetorial nos bairros dos distritos "
            "Anhanduizinho, Imbirussu e Bandeira, historicamente mais afetados.\n\n"
            "2. Implementar sistema de alerta precoce baseado no Rt e na "
            "probabilidade P(Rt>1) para antecipar surtos em 2-3 semanas.\n\n"
            "3. Ampliar a cobertura do InfoDengue para todos os municipios de MS, "
            "integrando dados LIRAa/LIA para correlacao com indice de infestacao.\n\n"
            "4. Desenvolver protocolo de resposta diferenciado por nivel de alerta "
            "(1 a 4), com escalas de recursos proporcionais ao risco previsto."
        )
        pdf.multi_cell(0, 6, conclusao_text)

        # ── Referências ───────────────────────────────────────────────────────
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 13)
        pdf.set_fill_color(127, 140, 141)
        pdf.set_text_color(255, 255, 255)
        pdf.cell(0, 9, "16. REFERENCIAS", fill=True, ln=True)
        pdf.set_text_color(0, 0, 0)
        pdf.ln(3)
        pdf.set_font("Helvetica", "", 9)
        refs = [
            "InfoDengue (2025). Sistema de Monitoramento de Arboviroses.",
            "  FGV-EMAp / FIOCRUZ. https://info.dengue.mat.br",
            "",
            "SINAN/DATASUS (2025). Sistema de Informacao de Agravos de Notificacao.",
            "  Ministerio da Saude do Brasil.",
            "",
            "Tao, Y. et al. (2020). Deep learning for dengue outbreak prediction.",
            "  Journal of Epidemiology and Community Health.",
            "",
            "Lowe, R. et al. (2021). Climate services for health: predicting the",
            "  evolution of the 2016 dengue season in Minas Gerais, Brazil.",
            "  The Lancet Planetary Health.",
            "",
            "Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python.",
            "  JMLR 12, 2825-2830.",
            "",
            "Abadi, M. et al. (2016). TensorFlow: A system for large-scale machine",
            "  learning. OSDI.",
        ]
        for ref in refs:
            pdf.cell(0, 5, ref, ln=True)

        # ── Rodapé da última página ────────────────────────────────────────────
        pdf.ln(10)
        pdf.set_font("Helvetica", "I", 8)
        pdf.set_text_color(150, 150, 150)
        pdf.cell(0, 5,
                 f"SIPREV v1.0 | Gerado em {datetime.now().strftime('%d/%m/%Y %H:%M')} "
                 f"| InfoDengue 2016-2025",
                 align="C", ln=True)

        # Salva PDF
        pdf_path = OUTPUT_DIR / "pdf" / f"SIPREV_Relatorio_Final_{TIMESTAMP}.pdf"
        pdf.output(str(pdf_path))
        _inc("relatorios_gerados")
        log.info(f"  [PDF] {pdf_path.name}")
        return pdf_path

    except Exception as e:
        log.error(f"  Falha ao gerar PDF: {e}")
        traceback.print_exc()
        return None




In [53]:
# =============================================================================
# SEÇÃO 25 – EXPORTAÇÃO XLSX


In [54]:
# =============================================================================

def exportar_xlsx(df_cg: pd.DataFrame,
                  df_ms: pd.DataFrame,
                  df_cap: pd.DataFrame) -> Optional[Path]:
    """
    Exporta dados tratados e indicadores para planilha Excel multi-abas.
    """
    print_section("EXPORTAÇÃO – XLSX")

    if not HAS_OPENPYXL:
        log.warning("  openpyxl não disponível.")
        return None

    xlsx_path = OUTPUT_DIR / "dados" / f"SIPREV_Dados_{TIMESTAMP}.xlsx"
    try:
        with pd.ExcelWriter(str(xlsx_path), engine="openpyxl") as writer:

            # Aba 1: Campo Grande Semanal
            if not df_cg.empty:
                cols_cg = [c for c in [
                    "data_SE","SE","ANO","MES","SEMANA","municipio_nome",
                    "casos","casos_est","casos_est_min","casos_est_max",
                    "p_rt1","p_inc100k","Rt","nivel","nivel_descr","risco",
                    "pop","tempmin","tempmed","tempmax",
                    "umidmin","umidmed","umidmax",
                    "receptivo","transmissao",
                    "casprov","casconf","notif_accum_year",
                ] if c in df_cg.columns]
                df_cg[cols_cg].to_excel(writer, sheet_name="CampoGrande_Semanal",
                                         index=False)

            # Aba 2: Agregado Anual CG
            if not df_cg.empty and "ANO" in df_cg.columns:
                anual_cg = df_cg.groupby("ANO").agg(
                    casos_total=("casos","sum"),
                    casos_est_total=("casos_est","sum"),
                    rt_medio=("Rt","mean") if "Rt" in df_cg.columns else ("casos","count"),
                    nivel_max=("nivel","max") if "nivel" in df_cg.columns else ("casos","count"),
                ).reset_index()
                pop_ref = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942140
                anual_cg["taxa_inc_100k"] = anual_cg["casos_total"].apply(
                    lambda c: taxa_inc(c, pop_ref))
                anual_cg.to_excel(writer, sheet_name="CampoGrande_Anual", index=False)

            # Aba 3: Municípios MS
            if not df_ms.empty:
                cols_ms = [c for c in [
                    "ANO","MES","municipio_nome","casos","casos_est",
                    "Rt","p_rt1","p_inc100k","nivel","pop",
                    "tempmed","umidmed","receptivo","transmissao",
                ] if c in df_ms.columns]
                df_ms[cols_ms].to_excel(writer, sheet_name="MS_Municipios_Semanal",
                                         index=False)

            # Aba 4: Ranking MS
            if not df_ms.empty:
                r_ms = df_ms.groupby("municipio_nome")["casos"].sum().reset_index()
                r_ms = r_ms.sort_values("casos", ascending=False).reset_index(drop=True)
                r_ms["rank"]      = r_ms.index + 1
                r_ms["pop"]       = r_ms["municipio_nome"].map(POP_MUNICIPIOS_MS).fillna(50_000)
                r_ms["taxa_100k"] = r_ms.apply(lambda r: taxa_inc(r["casos"], r["pop"]), axis=1)
                r_ms["risco"]     = r_ms["taxa_100k"].apply(classificar_risco)
                r_ms.to_excel(writer, sheet_name="Ranking_MS", index=False)

            # Aba 5: Capitais
            if not df_cap.empty:
                cols_cap = [c for c in [
                    "ANO","MES","municipio_nome","casos","casos_est",
                    "Rt","p_rt1","p_inc100k","nivel","pop",
                    "tempmed","umidmed","receptivo","transmissao",
                ] if c in df_cap.columns]
                df_cap[cols_cap].to_excel(writer, sheet_name="Capitais_Semanal",
                                           index=False)

            # Aba 6: Ranking Capitais
            if not df_cap.empty:
                r_cap = df_cap.groupby("municipio_nome")["casos"].sum().reset_index()
                r_cap = r_cap.sort_values("casos", ascending=False).reset_index(drop=True)
                r_cap["rank"]      = r_cap.index + 1
                r_cap["pop"]       = r_cap["municipio_nome"].map(POP_CAPITAIS).fillna(1_000_000)
                r_cap["taxa_100k"] = r_cap.apply(lambda r: taxa_inc(r["casos"], r["pop"]), axis=1)
                r_cap["UF"]        = r_cap["municipio_nome"].map(CAPITAIS_UF)
                r_cap["risco"]     = r_cap["taxa_100k"].apply(classificar_risco)
                r_cap.to_excel(writer, sheet_name="Ranking_Capitais", index=False)

            # Aba 7: Metadados
            meta = {
                "Chave": ["timestamp", "ambiente", "python_versao",
                           "total_registros_cg", "total_registros_ms",
                           "total_registros_cap",
                           "total_casos_cg", "total_casos_ms",
                           "graficos_gerados", "mapas_gerados",
                           "modelos_treinados", "dashboards_gerados"],
                "Valor": [
                    TIMESTAMP, "Colab" if IS_COLAB else "Local",
                    sys.version.split()[0],
                    len(df_cg), len(df_ms), len(df_cap),
                    int(df_cg["casos"].sum()) if not df_cg.empty else 0,
                    int(df_ms["casos"].sum()) if not df_ms.empty else 0,
                    _stats["graficos_gerados"], _stats["mapas_gerados"],
                    _stats["modelos_treinados"], _stats["dashboards_gerados"],
                ],
            }
            pd.DataFrame(meta).to_excel(writer, sheet_name="Metadados", index=False)

        log.info(f"  [XLSX] {xlsx_path.name}")
        return xlsx_path

    except Exception as e:
        log.error(f"  Falha ao gerar XLSX: {e}")
        return None




In [55]:
# =============================================================================
# SEÇÃO 26 – EXPORTAÇÃO PARQUET E JSON DE METADADOS


In [56]:
# =============================================================================

def exportar_parquet_json(df_cg: pd.DataFrame,
                           df_ms: pd.DataFrame,
                           df_cap: pd.DataFrame) -> None:
    """Exporta dados em formato Parquet (otimizado) e JSON de metadados."""
    print_section("EXPORTAÇÃO – PARQUET / JSON")

    if HAS_PARQUET:
        for nome, df in [("cg", df_cg), ("ms", df_ms), ("cap", df_cap)]:
            if df.empty:
                continue
            try:
                p = OUTPUT_DIR / "dados" / f"dengue_{nome}_{TIMESTAMP}.parquet"
                df_save = df.select_dtypes(include=["number","object","datetime64"]).copy()
                # Converte Int64 para int64 para compatibilidade Parquet
                for c in df_save.select_dtypes(include=["Int64"]).columns:
                    df_save[c] = df_save[c].astype("float64")
                df_save.to_parquet(str(p), index=False, engine="pyarrow",
                                    compression="snappy")
                log.info(f"  [PARQUET] {p.name}")
            except Exception as e:
                log.warning(f"  Parquet {nome} falhou: {e}")

    # JSON de metadados
    meta_json = {
        "siprev_version": "1.0",
        "timestamp": TIMESTAMP,
        "ambiente": "Google Colab" if IS_COLAB else "Máquina Local",
        "python_version": sys.version.split()[0],
        "tensorflow_version": TF_VERSION,
        "periodo_analise": "2016-2025",
        "fonte": "InfoDengue / FGV-EMAp-FIOCRUZ",
        "municipio_foco": "Campo Grande/MS",
        "estatisticas": {
            "registros_lidos":      _stats["registros_lidos"],
            "registros_validos":    _stats["registros_validos"],
            "graficos_gerados":     _stats["graficos_gerados"],
            "mapas_gerados":        _stats["mapas_gerados"],
            "modelos_treinados":    _stats["modelos_treinados"],
            "dashboards_gerados":   _stats["dashboards_gerados"],
            "relatorios_gerados":   _stats["relatorios_gerados"],
        },
        "arquivos_entrada": {
            "CG":  str(ARQUIVO_CG),
            "MS":  str(ARQUIVO_MS),
            "CAP": str(ARQUIVO_CAP),
        },
        "params": PARAMS,
    }
    if not df_cg.empty and "casos" in df_cg.columns:
        meta_json["campo_grande"] = {
            "total_casos":    int(df_cg["casos"].sum()),
            "media_semanal":  round(float(df_cg["casos"].mean()), 1),
            "max_semanal":    int(df_cg["casos"].max()),
            "n_semanas":      len(df_cg),
            "anos":           sorted([int(a) for a in df_cg["ANO"].unique()]),
        }
    if not df_ms.empty:
        meta_json["ms"] = {
            "n_municipios": int(df_ms["municipio_nome"].nunique()),
            "total_casos":  int(df_ms["casos"].sum()),
        }
    if not df_cap.empty:
        meta_json["capitais"] = {
            "n_capitais":  int(df_cap["municipio_nome"].nunique()),
            "total_casos": int(df_cap["casos"].sum()),
        }

    json_path = OUTPUT_DIR / "dados" / f"metadados_{TIMESTAMP}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(meta_json, f, ensure_ascii=False, indent=2, default=str)
    log.info(f"  [JSON] {json_path.name}")




In [57]:
# =============================================================================
# SEÇÃO 27 – RELATÓRIO CONSOLIDADO TXT


In [58]:
# =============================================================================

def relatorio_txt_consolidado(df_cg: pd.DataFrame,
                               df_ms: pd.DataFrame,
                               df_cap: pd.DataFrame) -> Path:
    """
    Gera relatório textual consolidado com todos os indicadores.
    """
    print_section("RELATÓRIO TEXTUAL CONSOLIDADO")

    linhas = [
        "=" * 78,
        "SIPREV – Sistema Inteligente de Previsão Epidemiológica de Dengue",
        f"Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}",
        f"Ambiente : {'Google Colab' if IS_COLAB else 'Máquina Local'}",
        "=" * 78,
        "",
    ]

    # Resumo Campo Grande
    if not df_cg.empty and "casos" in df_cg.columns:
        pop_cg = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942140
        linhas += [
            "CAMPO GRANDE / MATO GROSSO DO SUL",
            "-" * 40,
            f"  Total de casos (2016-2025)     : {fmt_num(int(df_cg['casos'].sum()))}",
            f"  Média semanal                  : {df_cg['casos'].mean():.1f}",
            f"  Pico semanal                   : {fmt_num(int(df_cg['casos'].max()))}",
        ]
        if "ANO" in df_cg.columns:
            por_ano = df_cg.groupby("ANO")["casos"].sum()
            linhas += [
                f"  Pior ano epidêmico             : {int(por_ano.idxmax())} "
                f"({fmt_num(int(por_ano.max()))} casos)",
                f"  Melhor ano                     : {int(por_ano.idxmin())} "
                f"({fmt_num(int(por_ano.min()))} casos)",
            ]
        if "Rt" in df_cg.columns:
            linhas.append(f"  Rt médio histórico             : {df_cg['Rt'].mean():.3f}")
        if "nivel" in df_cg.columns:
            linhas.append(
                f"  Semanas em Nível 4 (Vermelho)  : "
                f"{fmt_num(int((df_cg['nivel'] == 4).sum()))}"
            )
        linhas.append("")

    # Resumo MS
    if not df_ms.empty:
        linhas += [
            "MATO GROSSO DO SUL – MUNICÍPIOS",
            "-" * 40,
            f"  Municípios analisados          : {df_ms['municipio_nome'].nunique()}",
            f"  Total de casos (2016-2025)     : {fmt_num(int(df_ms['casos'].sum()))}",
            "",
        ]
        top5 = df_ms.groupby("municipio_nome")["casos"].sum().nlargest(5)
        linhas.append("  Top 5 municípios por casos:")
        for i, (mun, casos) in enumerate(top5.items(), 1):
            linhas.append(f"    {i}. {mun}: {fmt_num(int(casos))}")
        linhas.append("")

    # Resumo Nacional
    if not df_cap.empty:
        linhas += [
            "RANKING NACIONAL – CAPITAIS BRASILEIRAS",
            "-" * 40,
            f"  Capitais analisadas            : {df_cap['municipio_nome'].nunique()}",
            f"  Total de casos (2016-2025)     : {fmt_num(int(df_cap['casos'].sum()))}",
            "",
        ]
        top5_cap = df_cap.groupby("municipio_nome")["casos"].sum().nlargest(5)
        linhas.append("  Top 5 capitais por casos:")
        for i, (cap, casos) in enumerate(top5_cap.items(), 1):
            linhas.append(f"    {i}. {cap}: {fmt_num(int(casos))}")
        linhas.append("")

    # Estatísticas de execução
    linhas += [
        "ESTATÍSTICAS DE EXECUÇÃO",
        "-" * 40,
        f"  Registros lidos                : {fmt_num(_stats['registros_lidos'])}",
        f"  Registros válidos              : {fmt_num(_stats['registros_validos'])}",
        f"  Gráficos gerados               : {_stats['graficos_gerados']}",
        f"  Mapas gerados                  : {_stats['mapas_gerados']}",
        f"  Dashboards gerados             : {_stats['dashboards_gerados']}",
        f"  Modelos treinados              : {_stats['modelos_treinados']}",
        f"  Relatórios gerados             : {_stats['relatorios_gerados']}",
        "",
        "=" * 78,
    ]

    conteudo = "\n".join(linhas)
    p = salvar_txt(conteudo, f"relatorio_consolidado_{TIMESTAMP}",
                   "Relatório Consolidado SIPREV")
    salvar_log_tabela(conteudo, f"relatorio_consolidado_{TIMESTAMP}",
                      "Relatório Consolidado")
    return p




In [59]:
# =============================================================================
# SEÇÃO 28 – RELATÓRIO DE MODELOS TREINADOS


In [60]:
# =============================================================================

def relatorio_modelos(resultados_ml: dict,
                       resultados_ts: dict,
                       resultados_dl: dict) -> None:
    """
    Gera relatório completo de todos os modelos treinados com
    suas métricas de desempenho.
    """
    print_section("RELATÓRIO DE MODELOS TREINADOS")

    linhas = [
        "SIPREV – RELATÓRIO DE MODELOS",
        f"Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}",
        "=" * 78, "",
        "MODELOS DE MACHINE LEARNING",
        "-" * 40,
    ]

    # Clusterização
    linhas += [
        "1. CLUSTERIZAÇÃO (Municípios MS)",
        "   Algoritmos: KMeans, DBSCAN, Gaussian Mixture Model",
        "   Variáveis: casos, taxa_inc, Rt, p_rt1, temperatura, umidade",
        "",
    ]

    # Classificação
    if "Campo Grande" in resultados_ml:
        linhas += ["2. CLASSIFICAÇÃO DE RISCO (Nível de Alerta)"]
        for m_nome, m_vals in resultados_ml.items():
            if "metricas" in m_vals:
                linhas.append(f"   Dataset: {m_nome}")
                for row in m_vals["metricas"]:
                    linhas.append(
                        f"     {row[0]:20s} | Acc={row[1]:7s} | F1={row[2]:7s}"
                    )
        linhas.append("")

    # Séries Temporais
    linhas += [
        "MODELOS DE SÉRIES TEMPORAIS",
        "-" * 40,
        "  Auto-ARIMA  : Seleção automática de p,d,q com sazonalidade mensal",
        "  Holt-Winters: Suavização exponencial com tendência e sazonalidade",
        "  Prophet     : Modelo Facebook/Meta com sazonalidade anual",
        f"  Horizonte   : {PARAMS['horizonte_previsao_meses']} meses à frente",
        "",
        "MODELOS DE DEEP LEARNING",
        "-" * 40,
        "  LSTM           : 2 camadas (64→32 unidades), dropout=0.2, Huber loss",
        "  GRU            : 2 camadas (64→32 unidades), dropout=0.2",
        "  Bidirectional LSTM: LSTM bidirecional de 64 unidades",
        "  CNN-LSTM       : Conv1D(64,3) + MaxPool + LSTM(32)",
        "  Transformer    : MultiHeadAttention(4 heads, key_dim=8) + FFN",
        f"  Janela entrada : {PARAMS['lstm_janela']} semanas",
        f"  Horizonte      : {PARAMS['horizonte_previsao_semanas']} semanas",
        "",
        "REDES NEURAIS ESPECIALIZADAS",
        "-" * 40,
        "  Autoencoder    : Encoder(16→8→4) + Decoder(4→8→16→n_feat)",
        "  DNN Profunda   : Dense(256→128→64→32→n_classes), BN, Dropout",
        "  CNN 1D Temporal: Conv1D(64,5) + BN + MaxPool x2 + GAP + Dense",
        "",
        f"TOTAL DE MODELOS TREINADOS: {_stats['modelos_treinados']}",
        "=" * 78,
    ]

    conteudo = "\n".join(linhas)
    log.info(f"\n{conteudo}")
    salvar_txt(conteudo, "relatorio_modelos_treinados",
               "Relatório de Modelos Treinados")
    salvar_log_tabela(conteudo, "relatorio_modelos_treinados",
                      "Modelos Treinados")




In [61]:
# =============================================================================
# SEÇÃO 29 – COMPACTAÇÃO ZIP FINAL


In [62]:
# =============================================================================

def compactar_resultados() -> Path:
    """
    Compacta todos os arquivos gerados em um único ZIP para exportação.
    """
    print_section("EXPORTAÇÃO FINAL – ZIP")

    zip_path = OUTPUT_DIR.parent / f"{EXPORT_NAME}.zip"

    with zipfile.ZipFile(str(zip_path), "w", zipfile.ZIP_DEFLATED) as zf:
        for subdir in ["graficos", "mapas", "relatorios", "dados",
                        "dashboards", "logs", "pdf"]:
            folder = OUTPUT_DIR / subdir
            if not folder.exists():
                continue
            for fpath in folder.iterdir():
                if fpath.is_file():
                    arcname = f"{subdir}/{fpath.name}"
                    zf.write(str(fpath), arcname)

    tamanho_mb = zip_path.stat().st_size / 1_048_576
    log.info(f"  [ZIP] {zip_path.name} ({tamanho_mb:.1f} MB)")

    # No Colab, faz download automático
    if IS_COLAB:
        try:
            from google.colab import files
            files.download(str(zip_path))
            log.info("  Download iniciado no Google Colab.")
        except Exception as e:
            log.warning(f"  Download Colab falhou: {e}")

    return zip_path




In [63]:
# =============================================================================
# SEÇÃO 30 – SUMÁRIO FINAL DE EXECUÇÃO


In [64]:
# =============================================================================

def sumario_final(t_inicio: datetime) -> None:
    """Exibe e salva o sumário completo da execução."""
    t_fim = datetime.now()
    duracao = t_fim - t_inicio
    horas, rem = divmod(int(duracao.total_seconds()), 3600)
    minutos, segundos = divmod(rem, 60)

    rows_sum = [
        ["Início da execução",      t_inicio.strftime("%d/%m/%Y %H:%M:%S")],
        ["Fim da execução",         t_fim.strftime("%d/%m/%Y %H:%M:%S")],
        ["Duração total",           f"{horas:02d}h {minutos:02d}m {segundos:02d}s"],
        ["Ambiente",                "Google Colab" if IS_COLAB else "Local"],
        ["Python",                   sys.version.split()[0]],
        ["TensorFlow",               TF_VERSION],
        ["Arquivos lidos",           fmt_num(_stats["arquivos_lidos"])],
        ["Registros lidos",          fmt_num(_stats["registros_lidos"])],
        ["Registros válidos",        fmt_num(_stats["registros_validos"])],
        ["Registros descartados",    fmt_num(_stats["registros_descartados"])],
        ["Gráficos gerados",         str(_stats["graficos_gerados"])],
        ["Mapas gerados",            str(_stats["mapas_gerados"])],
        ["Dashboards gerados",       str(_stats["dashboards_gerados"])],
        ["Modelos treinados",        str(_stats["modelos_treinados"])],
        ["Relatórios gerados",       str(_stats["relatorios_gerados"])],
        ["Diretório de saída",        str(OUTPUT_DIR)],
        ["Arquivo ZIP",              f"{EXPORT_NAME}.zip"],
    ]

    tab = make_table(
        ["Parâmetro", "Valor"],
        rows_sum, col_align=["l","l"], max_width=100
    )

    print_section("SUMÁRIO FINAL DE EXECUÇÃO")
    log.info(f"\n{tab}")
    salvar_txt(tab, f"sumario_execucao_{TIMESTAMP}", "Sumário Final de Execução")
    salvar_log_tabela(tab, f"sumario_execucao_{TIMESTAMP}", "Sumário")

    log.info("")
    log.info("=" * 78)
    log.info("  SIPREV – Execução concluída com sucesso!")
    log.info(f"  Duração: {horas:02d}h {minutos:02d}m {segundos:02d}s")
    log.info(f"  Modelos treinados: {_stats['modelos_treinados']}")
    log.info(f"  Gráficos: {_stats['graficos_gerados']} | "
             f"Mapas: {_stats['mapas_gerados']} | "
             f"Dashboards: {_stats['dashboards_gerados']}")
    log.info(f"  Saída em: {OUTPUT_DIR}")
    log.info("=" * 78)




In [65]:
# =============================================================================
# SEÇÃO 31 – FUNÇÃO PRINCIPAL (main)


In [66]:
# =============================================================================





In [67]:
# =============================================================================
# SEÇÃO 32 – ENGENHARIA DE FEATURES AVANÇADA


In [68]:
# =============================================================================

def engenharia_features(df_cg: pd.DataFrame,
                         df_ms: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Cria features derivadas avançadas para enriquecer os modelos:
    - Lags temporais (1, 2, 4, 8, 12 semanas)
    - Diferenciações (1ª e 2ª ordem)
    - Features de interação (temperatura × umidade)
    - Indicadores de janela deslizante (máximo, mínimo, skewness)
    - Fourier features sazonais
    - One-hot encoding de mês e trimestre
    """
    print_section("ENGENHARIA DE FEATURES AVANÇADA")

    def _enriquecer(df: pd.DataFrame, nome: str) -> pd.DataFrame:
        if df.empty or "casos" not in df.columns:
            return df
        df = df.sort_values("data_SE").copy() if "data_SE" in df.columns else df.copy()
        log.info(f"  Enriquecendo features: {nome} ({len(df)} registros)")

        # ── Lags de casos ─────────────────────────────────────────────────────
        for lag in [1, 2, 3, 4, 8, 12]:
            df[f"casos_lag{lag}"] = df["casos"].shift(lag)

        # ── Diferenciação ─────────────────────────────────────────────────────
        df["casos_diff1"] = df["casos"].diff(1)
        df["casos_diff2"] = df["casos"].diff(2)
        df["casos_diff4"] = df["casos"].diff(4)

        # ── Janelas deslizantes ──────────────────────────────────────────────
        for win in [4, 8, 12, 26]:
            df[f"casos_rollmean{win}"] = df["casos"].rolling(win, min_periods=1).mean()
            df[f"casos_rollstd{win}"]  = df["casos"].rolling(win, min_periods=1).std()
            df[f"casos_rollmax{win}"]  = df["casos"].rolling(win, min_periods=1).max()
            df[f"casos_rollmin{win}"]  = df["casos"].rolling(win, min_periods=1).min()

        # ── Skewness e kurtosis móveis ────────────────────────────────────────
        df["casos_skew8"]  = df["casos"].rolling(8,  min_periods=4).skew()
        df["casos_kurt12"] = df["casos"].rolling(12, min_periods=6).kurt()

        # ── Ratio: casos / casos_lag4 ─────────────────────────────────────────
        df["ratio_lag4"]  = df["casos"] / (df["casos_lag4"].replace(0, np.nan))
        df["ratio_lag12"] = df["casos"] / (df["casos_rollmean12"].replace(0, np.nan))

        # ── Features climáticas de interação ─────────────────────────────────
        if "tempmed" in df.columns and "umidmed" in df.columns:
            df["temp_umid_inter"]   = df["tempmed"] * df["umidmed"]
            df["temp_sq"]           = df["tempmed"] ** 2
            df["umid_sq"]           = df["umidmed"] ** 2
            df["delta_temp"]        = df["tempmax"] - df["tempmin"] if "tempmax" in df.columns and "tempmin" in df.columns else 0

        # ── Lags de Rt ────────────────────────────────────────────────────────
        if "Rt" in df.columns:
            df["Rt_lag1"]  = df["Rt"].shift(1)
            df["Rt_lag2"]  = df["Rt"].shift(2)
            df["Rt_lag4"]  = df["Rt"].shift(4)
            df["Rt_diff1"] = df["Rt"].diff(1)
            df["Rt_acima1"] = (df["Rt"] > 1.0).astype(int)
            df["semanas_rt_acima1"] = df["Rt_acima1"].rolling(4, min_periods=1).sum()

        # ── Lags de p_rt1 ────────────────────────────────────────────────────
        if "p_rt1" in df.columns:
            df["p_rt1_lag1"] = df["p_rt1"].shift(1)
            df["p_rt1_lag4"] = df["p_rt1"].shift(4)

        # ── Fourier features (sazonalidade anual 52 semanas) ──────────────────
        if "SEMANA" in df.columns:
            for k in [1, 2, 3]:
                df[f"sin_sem_{k}"] = np.sin(2 * np.pi * k * df["SEMANA"].fillna(0) / 52)
                df[f"cos_sem_{k}"] = np.cos(2 * np.pi * k * df["SEMANA"].fillna(0) / 52)

        # ── One-hot mês ───────────────────────────────────────────────────────
        if "MES" in df.columns:
            for m in range(1, 13):
                df[f"mes_{m:02d}"] = (df["MES"] == m).astype(int)

        # ── Indicador: período chuvoso ─────────────────────────────────────────
        if "MES" in df.columns:
            df["periodo_chuvoso"] = df["MES"].apply(
                lambda m: 1 if m in {10, 11, 12, 1, 2, 3} else 0
            )

        # ── Semana do pico histórico (relativa) ───────────────────────────────
        if "SEMANA" in df.columns:
            df["dist_semana_pico"] = np.abs(df["SEMANA"].fillna(0) - 8)  # Pico típico semana 8-10

        # Preenche NaN gerados pelos lags
        df = df.fillna(method="bfill").fillna(0)

        n_feats = sum(1 for c in df.columns if c.startswith((
            "casos_lag","casos_diff","casos_roll","ratio_","temp_","umid_",
            "Rt_lag","Rt_diff","Rt_ac","sem_","p_rt1_lag","mes_","periodo_","dist_"
        )))
        log.info(f"  → {n_feats} features criadas para {nome}")
        return df

    df_cg_feat = _enriquecer(df_cg, "Campo Grande")
    df_ms_feat = _enriquecer(df_ms, "Municípios MS")

    # Tabela de resumo de novas features
    new_feats_cg = [c for c in df_cg_feat.columns if c not in df_cg.columns]
    rows_f = [[f, fmt_num(df_cg_feat[f].notna().sum()),
               fmt_num(df_cg_feat[f].mean(), 3)]
              for f in new_feats_cg[:20]]
    if rows_f:
        tab_f = make_table(
            ["Feature", "Válidos", "Média"],
            rows_f, col_align=["l","r","r"]
        )
        log.info(f"\n{tab_f}")
        salvar_txt(tab_f, "features_eng_cg",
                   "Features Derivadas – Campo Grande")

    log.info("  Engenharia de features concluída.")
    return df_cg_feat, df_ms_feat




In [69]:
# =============================================================================
# SEÇÃO 33 – TESTES ESTATÍSTICOS AVANÇADOS


In [70]:
# =============================================================================

def testes_estatisticos(df_cg: pd.DataFrame,
                         df_ms: pd.DataFrame,
                         df_cap: pd.DataFrame) -> dict:
    """
    Bateria completa de testes estatísticos:
    - Normalidade (Shapiro-Wilk, D'Agostino)
    - Estacionaridade (ADF, KPSS)
    - Comparação entre anos (Kruskal-Wallis, Mann-Whitney)
    - Correlação (Pearson, Spearman, Kendall)
    - Granger Causality
    - Seasonal decomposition strength
    """
    print_section("TESTES ESTATÍSTICOS AVANÇADOS")
    resultados = {}

    if df_cg.empty or "casos" not in df_cg.columns:
        return resultados

    casos = df_cg["casos"].dropna().values

    # ── 33.1 Testes de Normalidade ────────────────────────────────────────────
    print_sub("33.1 Testes de Normalidade")
    rows_norm = []

    if len(casos) >= 8:
        try:
            stat_sw, p_sw = shapiro(casos[:min(len(casos), 5000)])
            rows_norm.append(["Shapiro-Wilk", fmt_num(stat_sw, 4),
                               fmt_num(p_sw, 6),
                               "Normal" if p_sw > 0.05 else "Não Normal"])
        except Exception:
            pass

    try:
        stat_da, p_da = normaltest(casos)
        rows_norm.append(["D'Agostino-Pearson", fmt_num(stat_da, 4),
                           fmt_num(p_da, 6),
                           "Normal" if p_da > 0.05 else "Não Normal"])
    except Exception:
        pass

    if rows_norm:
        tab_norm = make_table(
            ["Teste", "Estatística", "p-value", "Conclusão"],
            rows_norm, col_align=["l","r","r","l"]
        )
        log.info(f"\n{tab_norm}")
        salvar_txt(tab_norm, "testes_normalidade_cg",
                   "Testes de Normalidade – Campo Grande")
        resultados["normalidade"] = rows_norm

    # ── 33.2 Distribuição de casos por ano (Kruskal-Wallis) ──────────────────
    print_sub("33.2 Comparação Inter-Anual (Kruskal-Wallis)")
    if "ANO" in df_cg.columns:
        grupos_ano = [df_cg[df_cg["ANO"] == a]["casos"].dropna().values
                      for a in sorted(df_cg["ANO"].unique())
                      if len(df_cg[df_cg["ANO"] == a]) >= 5]
        if len(grupos_ano) >= 3:
            try:
                stat_kw, p_kw = kruskal(*grupos_ano)
                log.info(f"  Kruskal-Wallis: H={stat_kw:.4f}, p={p_kw:.6f}")
                conclusao_kw = ("Diferença significativa entre anos (p<0.05)"
                                if p_kw < 0.05
                                else "Sem diferença significativa entre anos")
                log.info(f"  → {conclusao_kw}")
                resultados["kruskal_wallis"] = {"H": stat_kw, "p": p_kw}

                # Salva tabela
                rows_kw = [["Kruskal-Wallis (entre anos)",
                             fmt_num(stat_kw, 4), fmt_num(p_kw, 6),
                             conclusao_kw]]
                tab_kw = make_table(
                    ["Teste", "Estatística H", "p-value", "Conclusão"],
                    rows_kw, col_align=["l","r","r","l"]
                )
                salvar_txt(tab_kw, "testes_kruskal_anos_cg",
                           "Comparação Inter-Anual – Kruskal-Wallis")
            except Exception as e:
                log.warning(f"  Kruskal-Wallis falhou: {e}")

    # ── 33.3 Mann-Whitney: período chuvoso vs seco ───────────────────────────
    print_sub("33.3 Mann-Whitney: Chuvoso vs Seco")
    if "MES" in df_cg.columns:
        chuvoso = df_cg[df_cg["MES"].isin([1,2,3,10,11,12])]["casos"].dropna()
        seco    = df_cg[df_cg["MES"].isin([4,5,6,7,8,9])]["casos"].dropna()
        if len(chuvoso) > 5 and len(seco) > 5:
            try:
                stat_mw, p_mw = mannwhitneyu(chuvoso, seco, alternative="greater")
                log.info(f"  Mann-Whitney (chuvoso>seco): U={stat_mw:.1f}, p={p_mw:.6f}")
                rows_mw = [
                    ["Média Período Chuvoso",  fmt_num(chuvoso.mean(), 1)],
                    ["Média Período Seco",     fmt_num(seco.mean(), 1)],
                    ["Mann-Whitney U",          fmt_num(stat_mw, 1)],
                    ["p-value",                 fmt_num(p_mw, 6)],
                    ["Conclusão", "Chuvoso > Seco (sig.)" if p_mw < 0.05
                     else "Sem diferença significativa"],
                ]
                tab_mw = make_table(["Indicador","Valor"], rows_mw,
                                    col_align=["l","r"])
                log.info(f"\n{tab_mw}")
                salvar_txt(tab_mw, "testes_mannwhitney_periodo_cg",
                           "Mann-Whitney – Chuvoso vs Seco")
                resultados["mann_whitney_periodo"] = {"U": stat_mw, "p": p_mw}
            except Exception as e:
                log.warning(f"  Mann-Whitney falhou: {e}")

    # ── 33.4 Correlação: casos vs variáveis climáticas ───────────────────────
    print_sub("33.4 Correlação: Casos vs Clima")
    vars_corr = [c for c in ["tempmin","tempmed","tempmax",
                              "umidmin","umidmed","umidmax",
                              "Rt","p_rt1","p_inc100k",
                              "receptivo","transmissao"]
                 if c in df_cg.columns]
    rows_corr = []
    for var in vars_corr:
        sub = df_cg[["casos", var]].dropna()
        if len(sub) < 10:
            continue
        try:
            r_p, p_p = pearsonr(sub["casos"], sub[var])
            r_s, p_s = spearmanr(sub["casos"], sub[var])
            rows_corr.append([
                var,
                fmt_num(r_p, 4), fmt_num(p_p, 4),
                fmt_num(r_s, 4), fmt_num(p_s, 4),
                "✓" if p_p < 0.05 else "",
            ])
        except Exception:
            pass

    if rows_corr:
        tab_corr = make_table(
            ["Variável", "Pearson r", "p (P)", "Spearman ρ", "p (S)", "Sig."],
            rows_corr, col_align=["l","r","r","r","r","c"]
        )
        log.info(f"\n{tab_corr}")
        salvar_txt(tab_corr, "testes_correlacao_clima_cg",
                   "Correlação Casos × Variáveis Climáticas")
        resultados["correlacoes"] = rows_corr

    # ── 33.5 Estatística descritiva detalhada por ano ─────────────────────────
    print_sub("33.5 Estatísticas Descritivas por Ano")
    if "ANO" in df_cg.columns:
        rows_desc = []
        for ano in sorted(df_cg["ANO"].unique()):
            sub = df_cg[df_cg["ANO"] == ano]["casos"].dropna()
            if len(sub) == 0:
                continue
            rows_desc.append([
                int(ano),
                fmt_num(int(sub.sum())),
                fmt_num(sub.mean(), 1),
                fmt_num(sub.std(), 1),
                fmt_num(int(sub.max())),
                fmt_num(sub.median(), 1),
                fmt_num(sub.skew(), 3),
            ])
        tab_desc = make_table(
            ["Ano","Total","Média","Desvio","Máximo","Mediana","Assimetria"],
            rows_desc, col_align=["c","r","r","r","r","r","r"]
        )
        log.info(f"\n{tab_desc}")
        salvar_txt(tab_desc, "testes_desc_por_ano_cg",
                   "Estatísticas por Ano – Campo Grande")

    # ── 33.6 Comparação CG vs capitais da mesma região ───────────────────────
    print_sub("33.6 Comparação Regional – Centro-Oeste")
    caps_co = ["Campo Grande", "Goiânia", "Cuiabá", "Brasília"]
    if not df_cap.empty and "municipio_nome" in df_cap.columns:
        df_co = df_cap[df_cap["municipio_nome"].isin(caps_co)]
        if not df_co.empty:
            grupos_co = [df_co[df_co["municipio_nome"] == c]["casos"].dropna().values
                         for c in caps_co
                         if c in df_co["municipio_nome"].values]
            if len(grupos_co) >= 3 and all(len(g) >= 5 for g in grupos_co):
                try:
                    stat_co, p_co = kruskal(*grupos_co)
                    log.info(f"  Kruskal-Wallis Centro-Oeste: H={stat_co:.4f}, p={p_co:.6f}")
                    rows_co = [[c, fmt_num(int(df_co[df_co["municipio_nome"]==c]["casos"].sum())),
                                fmt_num(df_co[df_co["municipio_nome"]==c]["casos"].mean(), 1)]
                               for c in caps_co
                               if c in df_co["municipio_nome"].values]
                    tab_co = make_table(
                        ["Capital","Total Casos","Média/Semana"],
                        rows_co, col_align=["l","r","r"]
                    )
                    log.info(f"\n{tab_co}")
                    salvar_txt(tab_co, "testes_comparacao_co",
                               "Comparação Centro-Oeste – Capitais")
                except Exception as e:
                    log.warning(f"  Kruskal CO falhou: {e}")

    # ── 33.7 Granger Causality: temperatura → casos ──────────────────────────
    print_sub("33.7 Causalidade de Granger: Temperatura → Casos")
    if HAS_STATSMODELS and "tempmed" in df_cg.columns:
        try:
            df_gc = df_cg[["casos","tempmed"]].dropna().copy()
            if len(df_gc) >= 30:
                from statsmodels.tsa.stattools import grangercausalitytests
                gc_result = grangercausalitytests(
                    df_gc, maxlag=4, verbose=False
                )
                rows_gc = []
                for lag, tests in gc_result.items():
                    f_stat = tests[0]["ssr_ftest"][0]
                    p_val  = tests[0]["ssr_ftest"][1]
                    rows_gc.append([f"Lag {lag}",
                                    fmt_num(f_stat, 4),
                                    fmt_num(p_val, 6),
                                    "Sim" if p_val < 0.05 else "Não"])
                tab_gc = make_table(
                    ["Lag","F-stat","p-value","Granger Causal?"],
                    rows_gc, col_align=["c","r","r","c"]
                )
                log.info(f"\n{tab_gc}")
                salvar_txt(tab_gc, "testes_granger_temp_casos",
                           "Granger Causality: Temperatura → Casos")
                resultados["granger"] = rows_gc
        except Exception as e:
            log.warning(f"  Granger falhou: {e}")

    # ── 33.8 Boxplot comparativo entre períodos ───────────────────────────────
    if "ANO" in df_cg.columns and "casos" in df_cg.columns:
        anos_plot = sorted([int(a) for a in df_cg["ANO"].unique()
                            if 2016 <= int(a) <= 2025])
        grupos = [df_cg[df_cg["ANO"] == a]["casos"].dropna().values
                  for a in anos_plot]
        if grupos:
            fig, ax = plt.subplots(figsize=(14, 6))
            bp = ax.boxplot(grupos, labels=anos_plot, patch_artist=True,
                            notch=False, showfliers=True)
            palette_box = plt.cm.get_cmap("RdYlGn_r", len(anos_plot))
            for i, (patch, flier) in enumerate(zip(bp["boxes"], bp["fliers"])):
                patch.set_facecolor(palette_box(i))
                patch.set_alpha(0.75)
            ax.set_title("Distribuição Semanal de Casos por Ano – Campo Grande/MS",
                         fontweight="bold")
            ax.set_xlabel("Ano")
            ax.set_ylabel("Casos / Semana")
            ax.yaxis.set_major_formatter(mticker.FuncFormatter(
                lambda x, _: fmt_num(int(max(x, 0)))
            ))
            salvar_fig("testes_boxplot_casos_por_ano_cg")

    log.info("  Testes estatísticos concluídos.")
    return resultados




In [71]:
# =============================================================================
# SEÇÃO 34 – ANÁLISE DE TENDÊNCIA E PONTO DE MUDANÇA


In [72]:
# =============================================================================

def analise_tendencia(df_cg: pd.DataFrame) -> dict:
    """
    Analisa tendências de longo prazo:
    - Regressão linear sobre a série anual
    - Mann-Kendall Trend Test
    - Detecção de ponto de mudança (changepoint)
    - Sen's slope estimator
    - Projeção de tendência até 2030
    """
    print_section("ANÁLISE DE TENDÊNCIA E PONTO DE MUDANÇA")
    resultados = {}

    if df_cg.empty or "ANO" not in df_cg.columns or "casos" not in df_cg.columns:
        return resultados

    # Série anual
    por_ano = df_cg.groupby("ANO")["casos"].sum().reset_index()
    por_ano = por_ano[por_ano["ANO"].between(2016, 2025)].sort_values("ANO")
    anos    = por_ano["ANO"].astype(int).values
    casos_a = por_ano["casos"].values.astype(float)

    if len(por_ano) < 5:
        return resultados

    # ── 34.1 Regressão Linear de Tendência ────────────────────────────────────
    print_sub("34.1 Regressão Linear de Tendência")
    slope, intercept, r_value, p_value, std_err = stats.linregress(anos, casos_a)
    log.info(f"  Tendência linear: slope={slope:.1f} casos/ano | "
             f"R²={r_value**2:.4f} | p={p_value:.4f}")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(anos, casos_a, color=COR_SECUNDARIA, alpha=0.6, label="Casos Anuais")
    tendencia = slope * anos + intercept
    ax.plot(anos, tendencia, color=COR_PRINCIPAL, linewidth=2.5,
            linestyle="--", label=f"Tendência: {slope:+.0f} casos/ano (R²={r_value**2:.3f})")

    # Projeção 2026-2030
    anos_proj = np.arange(2026, 2031)
    proj      = np.clip(slope * anos_proj + intercept, 0, None)
    ax.plot(anos_proj, proj, color=COR_ALERTA, linewidth=2,
            linestyle=":", marker="o", markersize=5,
            label="Projeção 2026-2030")
    ax.fill_between(anos_proj, proj * 0.7, proj * 1.3,
                    alpha=0.15, color=COR_ALERTA,
                    label="Intervalo ±30%")

    ax.set_title("Tendência de Longo Prazo – Dengue em Campo Grande/MS",
                 fontweight="bold")
    ax.set_xlabel("Ano")
    ax.set_ylabel("Casos Anuais")
    ax.set_xticks(list(anos) + list(anos_proj))
    ax.set_xticklabels(list(anos) + list(anos_proj), rotation=45)
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: fmt_num(int(max(x, 0)))
    ))
    salvar_fig("tendencia_regressao_linear_cg")

    rows_trend = [
        ["Coeficiente angular (slope)",  f"{slope:+.2f} casos/ano"],
        ["Coeficiente linear (intercept)", fmt_num(intercept, 1)],
        ["Coeficiente de determinação (R²)", fmt_num(r_value**2, 4)],
        ["p-value (sig. estatística)", fmt_num(p_value, 6)],
        ["Tendência", "Crescente" if slope > 0 else "Decrescente"],
        ["Incremento esperado 2030 vs 2025",
         fmt_num(abs(slope * 5), 0) + " casos/ano"],
    ]
    tab_trend = make_table(["Indicador","Valor"], rows_trend, col_align=["l","r"])
    log.info(f"\n{tab_trend}")
    salvar_txt(tab_trend, "tendencia_regressao_linear_indicadores",
               "Regressão de Tendência – Campo Grande")
    resultados["tendencia_linear"] = {
        "slope": slope, "intercept": intercept,
        "r2": r_value**2, "p": p_value,
    }

    # ── 34.2 Projeção tabular ─────────────────────────────────────────────────
    print_sub("34.2 Tabela de Projeção 2026-2030")
    pop_cg = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942_140
    rows_proj = []
    for ano_p, val_p in zip(anos_proj, proj):
        rows_proj.append([
            int(ano_p),
            fmt_num(int(val_p)),
            fmt_num(taxa_inc(val_p, pop_cg), 1),
            classificar_risco(taxa_inc(val_p, pop_cg)),
        ])
    tab_proj = make_table(
        ["Ano","Casos Projetados","Taxa/100k","Risco Estimado"],
        rows_proj, col_align=["c","r","r","l"]
    )
    log.info(f"\n{tab_proj}")
    salvar_txt(tab_proj, "tendencia_projecao_2026_2030",
               "Projeção de Casos 2026–2030 – Campo Grande")

    # ── 34.3 Detecção de ponto de mudança (CUSUM simplificado) ──────────────
    print_sub("34.3 Detecção de Ponto de Mudança (CUSUM)")
    serie_semanal = df_cg.sort_values("data_SE")["casos"].fillna(0).values.astype(float)
    mu = serie_semanal.mean()
    sigma = serie_semanal.std() if serie_semanal.std() > 0 else 1
    cusum_pos  = np.zeros(len(serie_semanal))
    cusum_neg  = np.zeros(len(serie_semanal))
    k_cusum    = 0.5  # referência (metade do desvio padrão normalizado)

    for i in range(1, len(serie_semanal)):
        s_norm = (serie_semanal[i] - mu) / sigma
        cusum_pos[i] = max(0, cusum_pos[i-1] + s_norm - k_cusum)
        cusum_neg[i] = max(0, cusum_neg[i-1] - s_norm - k_cusum)

    threshold_cusum = 5.0
    mudancas_pos = np.where(cusum_pos > threshold_cusum)[0]
    mudancas_neg = np.where(cusum_neg > threshold_cusum)[0]

    n_pos = len(mudancas_pos)
    n_neg = len(mudancas_neg)
    log.info(f"  CUSUM: {n_pos} pontos de mudança (aumento) | "
             f"{n_neg} pontos (redução) | threshold={threshold_cusum}")

    if "data_SE" in df_cg.columns:
        datas_s = df_cg.sort_values("data_SE")["data_SE"].values
        fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
        axes[0].plot(datas_s, serie_semanal, color=COR_SECUNDARIA, linewidth=0.8)
        axes[0].set_ylabel("Casos")
        axes[0].set_title("CUSUM – Detecção de Pontos de Mudança – Campo Grande/MS",
                           fontweight="bold")

        axes[1].plot(datas_s, cusum_pos, color=COR_PRINCIPAL, linewidth=1.2,
                     label="CUSUM+")
        axes[1].axhline(threshold_cusum, color="red", linestyle="--",
                         linewidth=1, label=f"Limiar={threshold_cusum}")
        axes[1].set_ylabel("CUSUM+")
        axes[1].legend(fontsize=8)

        axes[2].plot(datas_s, cusum_neg, color=COR_VERDE, linewidth=1.2,
                     label="CUSUM−")
        axes[2].axhline(threshold_cusum, color="red", linestyle="--",
                         linewidth=1, label=f"Limiar={threshold_cusum}")
        axes[2].set_ylabel("CUSUM−")
        axes[2].set_xlabel("Semana Epidemiológica")
        axes[2].legend(fontsize=8)
        salvar_fig("tendencia_cusum_cg")

    # ── 34.4 Holt-Winters trend anual ─────────────────────────────────────────
    print_sub("34.4 Análise Polinomial de Tendência")
    if len(anos) >= 5:
        x_norm = (anos - anos.mean()) / anos.std()
        coefs2 = np.polyfit(x_norm, casos_a, 2)
        poly2  = np.poly1d(coefs2)

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.bar(anos, casos_a, color="#AED6F1", alpha=0.5, label="Observado")
        xs = np.linspace(x_norm.min(), x_norm.max(), 200)
        xs_real = xs * anos.std() + anos.mean()
        ax.plot(xs_real, poly2(xs), color=COR_PRINCIPAL, linewidth=2.5,
                label="Tendência polinomial (grau 2)")
        ax.set_title("Tendência Polinomial – Casos Anuais CG", fontweight="bold")
        ax.set_xlabel("Ano")
        ax.set_ylabel("Casos")
        ax.legend()
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: fmt_num(int(max(x, 0)))
        ))
        salvar_fig("tendencia_polinomial_cg")

    log.info("  Análise de tendência concluída.")
    return resultados




In [73]:
# =============================================================================
# SEÇÃO 35 – ANÁLISE DE RISCO POR MUNICÍPIO (ÍNDICE COMPOSTO)


In [74]:
# =============================================================================

def indice_risco_municipal(df_ms: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula índice composto de risco epidemiológico para cada município de MS.
    Componentes:
      - Taxa de incidência normalizada
      - Rt médio
      - P(Rt>1) médio
      - Semanas em nível 3 ou 4
      - Transmissão ativa (% de semanas)
      - Receptividade ambiental
    Gera ranking e mapa temático.
    """
    print_section("ÍNDICE COMPOSTO DE RISCO – MUNICÍPIOS MS")

    if df_ms.empty or "municipio_nome" not in df_ms.columns:
        return pd.DataFrame()

    agg = {
        "casos":       "sum",
        "Rt":          "mean",
        "p_rt1":       "mean",
        "nivel":       "mean",
        "transmissao": "sum",
        "receptivo":   "sum",
        "n_semanas":   ("casos","count"),
    }

    df_r = df_ms.groupby("municipio_nome").agg(
        casos_total   = ("casos", "sum"),
        rt_medio      = ("Rt", "mean"),
        p_rt1_medio   = ("p_rt1", "mean"),
        nivel_medio   = ("nivel", "mean"),
        n_nivel3_4    = ("nivel", lambda x: (x >= 3).sum()),
        n_transmissao = ("transmissao", "sum"),
        n_receptivo   = ("receptivo", "sum"),
        n_semanas     = ("casos", "count"),
    ).reset_index()

    df_r["pop"]          = df_r["municipio_nome"].map(POP_MUNICIPIOS_MS).fillna(50_000)
    df_r["taxa_inc_100k"] = df_r.apply(
        lambda r: taxa_inc(r["casos_total"], r["pop"]), axis=1
    )
    df_r["pct_transmissao"] = df_r["n_transmissao"] / df_r["n_semanas"].replace(0, 1)
    df_r["pct_receptivo"]   = df_r["n_receptivo"]   / df_r["n_semanas"].replace(0, 1)
    df_r["pct_nivel3_4"]    = df_r["n_nivel3_4"]    / df_r["n_semanas"].replace(0, 1)

    # ── Normalização min-max de cada componente ──────────────────────────────
    componentes = ["taxa_inc_100k","rt_medio","p_rt1_medio",
                   "pct_transmissao","pct_receptivo","pct_nivel3_4"]
    df_r_norm = df_r.copy()
    for c in componentes:
        mn = df_r[c].min()
        mx = df_r[c].max()
        if mx > mn:
            df_r_norm[f"{c}_norm"] = (df_r[c] - mn) / (mx - mn)
        else:
            df_r_norm[f"{c}_norm"] = 0.0

    # Pesos por componente (soma = 1)
    pesos = {
        "taxa_inc_100k_norm": 0.30,
        "rt_medio_norm":      0.20,
        "p_rt1_medio_norm":   0.15,
        "pct_transmissao_norm": 0.15,
        "pct_receptivo_norm":   0.10,
        "pct_nivel3_4_norm":    0.10,
    }
    df_r_norm["indice_risco"] = sum(
        df_r_norm[col] * peso for col, peso in pesos.items()
        if col in df_r_norm.columns
    )

    # Classificação do índice
    df_r_norm["categoria_risco"] = pd.cut(
        df_r_norm["indice_risco"],
        bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.001],
        labels=["Muito Baixo","Baixo","Médio","Alto","Muito Alto"]
    )

    df_r_norm = df_r_norm.sort_values("indice_risco", ascending=False).reset_index(drop=True)
    df_r_norm["rank_risco"] = df_r_norm.index + 1

    # ── Gráfico: Ranking de índice de risco ──────────────────────────────────
    top25 = df_r_norm.head(25)
    cores_ir = [COR_PRINCIPAL if m == "Campo Grande" else COR_SECUNDARIA
                for m in top25["municipio_nome"]]
    fig, ax = plt.subplots(figsize=(13, 10))
    ax.barh(top25["municipio_nome"][::-1],
            top25["indice_risco"][::-1],
            color=cores_ir[::-1], edgecolor="white")
    ax.set_title("Índice Composto de Risco – Top 25 Municípios MS",
                 fontweight="bold")
    ax.set_xlabel("Índice de Risco (0–1)")
    for bar, val in zip(ax.patches, top25["indice_risco"][::-1]):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=7)
    salvar_fig("risco_indice_composto_top25_ms")

    # ── Tabela ──────────────────────────────────────────────────────────────
    rows_ir = []
    for _, r in df_r_norm.head(30).iterrows():
        rows_ir.append([
            int(r["rank_risco"]),
            r["municipio_nome"],
            fmt_num(r["indice_risco"], 4),
            str(r["categoria_risco"]),
            fmt_num(r["taxa_inc_100k"], 1),
            fmt_num(r["rt_medio"], 3),
            fmt_pct(r["pct_nivel3_4"] * 100),
        ])
    tab_ir = make_table(
        ["Rank","Município","Índice","Categoria","Taxa/100k","Rt","% Nível≥3"],
        rows_ir, col_align=["c","l","r","l","r","r","r"]
    )
    log.info(f"\n{tab_ir}")
    salvar_txt(tab_ir, "risco_indice_composto_ranking",
               "Ranking por Índice de Risco – Municípios MS")
    salvar_log_tabela(tab_ir, "risco_indice_composto_ranking",
                      "Índice de Risco – MS")

    # Exporta CSV completo
    df_r_norm.to_csv(
        OUTPUT_DIR / "dados" / "municipios_indice_risco.csv", index=False
    )
    log.info("  [CSV] municipios_indice_risco.csv")

    # ── Gráfico radar: Campo Grande vs benchmark ──────────────────────────────
    cg_row  = df_r_norm[df_r_norm["municipio_nome"] == "Campo Grande"]
    top1    = df_r_norm.iloc[0]
    med_ms  = df_r_norm[componentes].mean()

    if not cg_row.empty:
        labels_r  = ["Taxa Inc.", "Rt Médio", "P(Rt>1)", "Transmissão",
                     "Receptividade", "% Nível≥3"]
        cg_vals   = [float(cg_row[f"{c}_norm"].values[0]) for c in componentes]
        top1_vals = [float(top1[f"{c}_norm"]) for c in componentes]
        med_vals  = [float(med_ms[c]) for c in componentes]

        angles = [n / len(labels_r) * 2 * math.pi for n in range(len(labels_r))]
        angles += angles[:1]

        fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
        for vals, label, cor, lw in [
            (cg_vals, "Campo Grande", COR_PRINCIPAL, 2.5),
            (top1_vals, f"#{1} {top1['municipio_nome']}", COR_ALERTA, 1.5),
            (med_vals, "Média MS", COR_CINZA, 1.2),
        ]:
            v = vals + vals[:1]
            ax.plot(angles, v, color=cor, linewidth=lw, label=label)
            ax.fill(angles, v, color=cor, alpha=0.07)

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(labels_r, fontsize=9)
        ax.set_title("Radar – Índice de Risco: CG vs Top1 vs Média MS",
                     fontweight="bold", pad=20)
        ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
        salvar_fig("risco_radar_cg_vs_ms")

    log.info("  Índice de risco municipal concluído.")
    return df_r_norm




In [75]:
# =============================================================================
# SEÇÃO 36 – SVR, KNN E MODELOS ADICIONAIS DE REGRESSÃO


In [76]:
# =============================================================================

def ml_regressao_avancada(df_cg: pd.DataFrame) -> dict:
    """
    Modelos adicionais de regressão de casos:
    - SVR (Support Vector Regression)
    - KNN Regressor
    - AdaBoost Regressor
    - BaggingRegressor (base: ExtraTrees)
    - Stacking Regressor (RF + XGB + LGB → Ridge meta)
    - Bayesian Ridge
    - Huber Regressor
    """
    print_section("MACHINE LEARNING – REGRESSÃO AVANÇADA (SVR/KNN/STACKING)")
    resultados = {}

    if not HAS_SKLEARN or df_cg.empty:
        return resultados

    feat_cols = [c for c in [
        "MES", "SEMANA", "ANO",
        "tempmin", "tempmed", "tempmax",
        "umidmin", "umidmed", "umidmax",
        "Rt", "p_rt1", "receptivo", "transmissao",
        "nivel",
    ] if c in df_cg.columns]

    df_reg = df_cg[feat_cols + ["casos"]].dropna()
    if len(df_reg) < 60:
        return resultados

    X = df_reg[feat_cols].values
    y = df_reg["casos"].values.astype(float)

    split = int(len(X) * 0.70)
    X_tr, X_te = X[:split], X[split:]
    y_tr, y_te = y[:split], y[split:]

    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_te_sc = scaler.transform(X_te)

    modelos_av = {}

    # SVR
    try:
        svr = SVR(kernel="rbf", C=100, gamma="scale", epsilon=0.5)
        svr.fit(X_tr_sc, y_tr)
        modelos_av["SVR-RBF"] = svr
        _inc("modelos_treinados")
    except Exception as e:
        log.warning(f"  SVR falhou: {e}")

    # KNN Regressor
    try:
        knn = KNeighborsRegressor(n_neighbors=7, weights="distance", n_jobs=-1)
        knn.fit(X_tr_sc, y_tr)
        modelos_av["KNN-7"] = knn
        _inc("modelos_treinados")
    except Exception as e:
        log.warning(f"  KNN falhou: {e}")

    # AdaBoost
    try:
        ada = AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=5),
            n_estimators=100, learning_rate=0.1, random_state=42
        )
        ada.fit(X_tr_sc, y_tr)
        modelos_av["AdaBoost"] = ada
        _inc("modelos_treinados")
    except Exception as e:
        log.warning(f"  AdaBoost falhou: {e}")

    # BaggingRegressor
    try:
        bag = BaggingRegressor(
            estimator=ExtraTreesRegressor(n_estimators=30, random_state=42),
            n_estimators=20, random_state=42, n_jobs=-1
        )
        bag.fit(X_tr_sc, y_tr)
        modelos_av["Bagging-ET"] = bag
        _inc("modelos_treinados")
    except Exception as e:
        log.warning(f"  Bagging falhou: {e}")

    # Bayesian Ridge
    try:
        br = BayesianRidge()
        br.fit(X_tr_sc, y_tr)
        modelos_av["Bayesian Ridge"] = br
        _inc("modelos_treinados")
    except Exception as e:
        log.warning(f"  Bayesian Ridge falhou: {e}")

    # Huber Regressor
    try:
        hub = HuberRegressor(epsilon=1.35, max_iter=500)
        hub.fit(X_tr_sc, y_tr)
        modelos_av["Huber"] = hub
        _inc("modelos_treinados")
    except Exception as e:
        log.warning(f"  Huber falhou: {e}")

    # Stacking Regressor (se RF e XGB disponíveis)
    if HAS_XGB and "Random Forest" not in modelos_av:
        try:
            rf_st  = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
            xgb_st = xgb.XGBRegressor(n_estimators=100, verbosity=0, random_state=42)
            stack  = StackingRegressor(
                estimators=[("rf", rf_st), ("xgb", xgb_st)],
                final_estimator=Ridge(alpha=1.0),
                cv=3, n_jobs=-1,
            )
            stack.fit(X_tr_sc, y_tr)
            modelos_av["Stacking (RF+XGB)"] = stack
            _inc("modelos_treinados")
        except Exception as e:
            log.warning(f"  Stacking falhou: {e}")

    # ── Avaliação ─────────────────────────────────────────────────────────────
    rows_eval = []
    y_preds_av = {}
    for nome_m, mdl in modelos_av.items():
        try:
            yp = np.clip(mdl.predict(X_te_sc), 0, None)
            rmse = np.sqrt(mean_squared_error(y_te, yp))
            mae  = mean_absolute_error(y_te, yp)
            r2   = r2_score(y_te, yp)
            mape = mean_absolute_percentage_error(y_te + 1, yp + 1) * 100
            rows_eval.append([nome_m, fmt_num(rmse, 1), fmt_num(mae, 1),
                               fmt_num(r2, 4), fmt_pct(mape)])
            y_preds_av[nome_m] = yp
            log.info(f"  {nome_m:22s}: RMSE={rmse:.2f} | MAE={mae:.2f} | "
                     f"R²={r2:.4f} | MAPE={mape:.1f}%")
        except Exception as e:
            log.warning(f"  Avaliação {nome_m} falhou: {e}")

    tab_av = make_table(
        ["Modelo","RMSE","MAE","R²","MAPE"],
        rows_eval, col_align=["l","r","r","r","r"]
    )
    log.info(f"\n{tab_av}")
    salvar_txt(tab_av, "ml_regressao_avancada_metricas",
               "Regressão Avançada – Métricas de Desempenho")

    # ── Gráfico comparativo ────────────────────────────────────────────────────
    if y_preds_av:
        n_mod = min(len(y_preds_av), 6)
        fig, axes = plt.subplots(2, 3, figsize=(16, 8))
        axes = axes.flatten()
        for i, (nm, yp) in enumerate(list(y_preds_av.items())[:n_mod]):
            axes[i].scatter(y_te, yp, alpha=0.4, color=COR_SECUNDARIA, s=20)
            lim = max(y_te.max(), yp.max())
            axes[i].plot([0, lim], [0, lim], "r--", linewidth=1.5)
            r2 = r2_score(y_te, yp)
            axes[i].set_title(f"{nm}\nR²={r2:.4f}", fontsize=9, fontweight="bold")
            axes[i].set_xlabel("Real")
            axes[i].set_ylabel("Predito")
        for j in range(n_mod, len(axes)):
            axes[j].set_visible(False)
        plt.suptitle("Regressão Avançada – Predito vs Real (Scatter)",
                     fontsize=13, fontweight="bold")
        salvar_fig("ml_regressao_avancada_scatter")

    resultados["metricas"] = rows_eval
    resultados["y_preds"]  = y_preds_av
    log.info("  Regressão avançada concluída.")
    return resultados




In [77]:
# =============================================================================
# SEÇÃO 37 – VALIDAÇÃO CRUZADA TEMPORAL (TIME SERIES SPLIT)


In [78]:
# =============================================================================

def validacao_cruzada_temporal(df_cg: pd.DataFrame) -> dict:
    """
    Validação cruzada com divisão temporal (sem data leakage):
    - TimeSeriesSplit com 5 folds
    - Avalia RF, XGBoost, Ridge, MLP
    - Calcula RMSE, MAE, R² em cada fold
    - Gera boxplots comparativos de desempenho
    """
    print_section("VALIDAÇÃO CRUZADA TEMPORAL (TimeSeriesSplit)")
    resultados = {}

    if not HAS_SKLEARN or df_cg.empty:
        return resultados

    feat_cols = [c for c in [
        "MES", "SEMANA", "ANO",
        "tempmin", "tempmed", "tempmax",
        "umidmin", "umidmed", "umidmax",
        "Rt", "p_rt1", "receptivo", "transmissao", "nivel",
    ] if c in df_cg.columns]

    df_v = df_cg[feat_cols + ["casos"]].dropna()
    if len(df_v) < 80:
        log.warning("  Dados insuficientes para validação cruzada.")
        return resultados

    X = df_v[feat_cols].values
    y = df_v["casos"].values.astype(float)

    tscv   = TimeSeriesSplit(n_splits=5)
    scaler = StandardScaler()

    modelos_cv = {
        "Ridge": Ridge(alpha=1.0),
        "Random Forest": RandomForestRegressor(
            n_estimators=100, random_state=42, n_jobs=-1
        ),
        "MLP": MLPRegressor(
            hidden_layer_sizes=(64, 32), max_iter=200,
            random_state=42, early_stopping=True
        ),
    }
    if HAS_XGB:
        modelos_cv["XGBoost"] = xgb.XGBRegressor(
            n_estimators=100, verbosity=0, random_state=42
        )

    # Coleta de métricas por fold
    resultados_cv = {nm: {"rmse":[],"mae":[],"r2":[]} for nm in modelos_cv}

    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X)):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        X_tr_sc = scaler.fit_transform(X_tr)
        X_te_sc = scaler.transform(X_te)

        for nm, mdl in modelos_cv.items():
            try:
                mdl.fit(X_tr_sc, y_tr)
                yp = np.clip(mdl.predict(X_te_sc), 0, None)
                resultados_cv[nm]["rmse"].append(
                    np.sqrt(mean_squared_error(y_te, yp))
                )
                resultados_cv[nm]["mae"].append(mean_absolute_error(y_te, yp))
                resultados_cv[nm]["r2"].append(r2_score(y_te, yp))
            except Exception as e:
                log.warning(f"  Fold {fold+1} {nm} falhou: {e}")

    # ── Tabela de resultados ──────────────────────────────────────────────────
    rows_cv = []
    for nm, metr in resultados_cv.items():
        if not metr["rmse"]:
            continue
        rows_cv.append([
            nm,
            fmt_num(np.mean(metr["rmse"]), 2),
            fmt_num(np.std(metr["rmse"]), 2),
            fmt_num(np.mean(metr["mae"]), 2),
            fmt_num(np.mean(metr["r2"]), 4),
        ])
        log.info(f"  {nm:20s}: RMSE={np.mean(metr['rmse']):.2f}±{np.std(metr['rmse']):.2f} | "
                 f"R²={np.mean(metr['r2']):.4f}")

    tab_cv = make_table(
        ["Modelo","RMSE Médio","RMSE Std","MAE Médio","R² Médio"],
        rows_cv, col_align=["l","r","r","r","r"]
    )
    log.info(f"\n{tab_cv}")
    salvar_txt(tab_cv, "ml_cv_temporal_metricas",
               "Validação Cruzada Temporal – Métricas")

    # ── Boxplot das métricas por fold ─────────────────────────────────────────
    nomes_modelos = [nm for nm in resultados_cv if resultados_cv[nm]["rmse"]]
    if nomes_modelos:
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        metricas_plot = ["rmse", "mae", "r2"]
        titulos_plot  = ["RMSE", "MAE", "R²"]

        for ax, metr_k, titulo_m in zip(axes, metricas_plot, titulos_plot):
            data_box = [resultados_cv[nm][metr_k] for nm in nomes_modelos]
            bp = ax.boxplot(data_box, labels=nomes_modelos, patch_artist=True)
            cores_box = plt.cm.get_cmap("Set2", len(nomes_modelos))
            for patch, i in zip(bp["boxes"], range(len(nomes_modelos))):
                patch.set_facecolor(cores_box(i))
                patch.set_alpha(0.75)
            ax.set_title(f"{titulo_m} – 5 Folds", fontweight="bold")
            ax.set_xticklabels(nomes_modelos, rotation=30, ha="right", fontsize=8)
            ax.set_ylabel(titulo_m)

        plt.suptitle("Validação Cruzada Temporal – Campo Grande/MS",
                     fontsize=13, fontweight="bold")
        salvar_fig("ml_cv_temporal_boxplot")

    resultados["resultados_cv"] = resultados_cv
    log.info("  Validação cruzada temporal concluída.")
    return resultados




In [79]:
# =============================================================================
# SEÇÃO 38 – ANÁLISE DE SAZONALIDADE AVANÇADA


In [80]:
# =============================================================================

def analise_sazonalidade_avancada(df_cg: pd.DataFrame,
                                    df_cap: pd.DataFrame) -> dict:
    """
    Análise aprofundada de sazonalidade:
    - Heatmap semanal × ano (radar de semanas)
    - Periodograma de Lomb-Scargle
    - Sazonalidade circular (decomposição harmônica)
    - Comparação sazonalidade CG vs capitais selecionadas
    - Distribuição de picos por semana epidemiológica
    """
    print_section("SAZONALIDADE AVANÇADA – ANÁLISE HARMÔNICA")
    resultados = {}

    if df_cg.empty or "casos" not in df_cg.columns:
        return resultados

    # ── 38.1 Heatmap: Semana × Ano ────────────────────────────────────────────
    print_sub("38.1 Heatmap Semanal × Ano – Campo Grande")
    if "SEMANA" in df_cg.columns and "ANO" in df_cg.columns:
        pivot_sw = df_cg.groupby(["ANO","SEMANA"])["casos"].sum().unstack(fill_value=0)
        pivot_sw = pivot_sw.loc[pivot_sw.index.isin(range(2016, 2026))]

        fig, ax = plt.subplots(figsize=(18, 7))
        sns.heatmap(pivot_sw, cmap="YlOrRd", linewidths=0.05, ax=ax,
                    cbar_kws={"label": "Casos"}, xticklabels=4)
        ax.set_title("Heatmap Semanal: Casos por Semana Epidemiológica × Ano – CG",
                     fontsize=13, fontweight="bold")
        ax.set_xlabel("Semana Epidemiológica")
        ax.set_ylabel("Ano")
        salvar_fig("sazon_heatmap_semana_ano_cg")

    # ── 38.2 Perfil médio semanal ─────────────────────────────────────────────
    print_sub("38.2 Perfil Médio Semanal Histórico")
    if "SEMANA" in df_cg.columns:
        perfil_sem = df_cg.groupby("SEMANA")["casos"].agg(
            media="mean", desvio="std", mediana="median",
            q25=lambda x: x.quantile(0.25),
            q75=lambda x: x.quantile(0.75),
        ).reset_index()

        fig, ax = plt.subplots(figsize=(14, 5))
        ax.fill_between(perfil_sem["SEMANA"],
                        perfil_sem["q25"], perfil_sem["q75"],
                        alpha=0.3, color=COR_SECUNDARIA, label="IQ (25-75%)")
        ax.plot(perfil_sem["SEMANA"], perfil_sem["media"],
                color=COR_PRINCIPAL, linewidth=2.5, label="Média histórica")
        ax.plot(perfil_sem["SEMANA"], perfil_sem["mediana"],
                color=COR_VERDE, linewidth=1.5, linestyle="--",
                label="Mediana histórica")

        # Destaque período epidêmico
        ax.axvspan(1, 15, alpha=0.07, color=COR_PRINCIPAL,
                   label="Período crítico (sem. 1–15)")
        ax.axvspan(40, 52, alpha=0.07, color=COR_PRINCIPAL)

        ax.set_title("Perfil Médio Semanal – Dengue Campo Grande/MS (2016–2025)",
                     fontweight="bold")
        ax.set_xlabel("Semana Epidemiológica")
        ax.set_ylabel("Casos")
        ax.set_xticks(range(1, 53, 4))
        ax.legend(ncol=2, fontsize=8)
        salvar_fig("sazon_perfil_semanal_historico_cg")

        # Semana de pico por ano
        picos = df_cg.groupby("ANO").apply(
            lambda g: g.loc[g["casos"].idxmax(), "SEMANA"]
            if not g.empty else None
        ).reset_index()
        picos.columns = ["ANO","SEMANA_PICO"]
        rows_pico = [[int(r["ANO"]), int(r["SEMANA_PICO"]) if pd.notna(r["SEMANA_PICO"]) else "–"]
                     for _, r in picos.iterrows()]
        tab_pico = make_table(["Ano","Semana do Pico"], rows_pico,
                               col_align=["c","c"])
        log.info(f"\n{tab_pico}")
        salvar_txt(tab_pico, "sazon_semana_pico_por_ano",
                   "Semana do Pico Epidêmico por Ano – CG")
        resultados["semanas_pico"] = rows_pico

    # ── 38.3 Decomposição harmônica (análise de Fourier) ─────────────────────
    print_sub("38.3 Decomposição de Fourier")
    if "SEMANA" in df_cg.columns:
        serie_sem = df_cg.groupby("SEMANA")["casos"].mean().values
        n         = len(serie_sem)
        fft_vals  = np.fft.rfft(serie_sem)
        freqs     = np.fft.rfftfreq(n, d=1)
        amplitudes = np.abs(fft_vals) / n * 2

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].plot(perfil_sem["SEMANA"] if "SEMANA" in df_cg.columns
                     else range(1, n+1),
                     serie_sem,
                     color=COR_SECUNDARIA, linewidth=1.5)
        axes[0].set_title("Série Média Semanal", fontweight="bold")
        axes[0].set_xlabel("Semana")
        axes[0].set_ylabel("Casos Médios")

        axes[1].stem(freqs[1:], amplitudes[1:], markerfmt="C1o",
                     linefmt="C1-", basefmt=" ")
        axes[1].set_title("Espectro de Fourier – Frequências Dominantes",
                           fontweight="bold")
        axes[1].set_xlabel("Frequência (ciclos/semana)")
        axes[1].set_ylabel("Amplitude")
        plt.suptitle("Análise de Fourier – Sazonalidade Semanal – Campo Grande",
                     fontsize=13, fontweight="bold")
        salvar_fig("sazon_fourier_espectro_cg")

    # ── 38.4 Comparação sazonalidade: CG vs capitais selecionadas ────────────
    print_sub("38.4 Comparação Sazonalidade – CG vs Capitais Selecionadas")
    caps_ref = ["Campo Grande","Goiânia","Cuiabá","Brasília","São Paulo","Rio de Janeiro"]
    if not df_cap.empty and "MES" in df_cap.columns:
        fig, ax = plt.subplots(figsize=(13, 5))
        palette = plt.cm.get_cmap("tab10", len(caps_ref))
        for i, cap in enumerate(caps_ref):
            sub = df_cap[df_cap["municipio_nome"] == cap]
            if sub.empty:
                continue
            sazon = sub.groupby("MES")["casos"].mean()
            # Normaliza para % do total anual
            total_sazon = sazon.sum()
            if total_sazon > 0:
                sazon_pct = sazon / total_sazon * 100
                lw  = 3.0 if cap == "Campo Grande" else 1.5
                ls  = "-" if cap == "Campo Grande" else "--"
                ax.plot(sazon_pct.index, sazon_pct.values,
                        color=palette(i), linewidth=lw, linestyle=ls,
                        marker="o", markersize=4, label=cap)

        ax.set_xticks(range(1, 13))
        ax.set_xticklabels([MESES_ABREV[m] for m in range(1, 13)])
        ax.set_title("Sazonalidade Mensal (% do Total Anual) – Capitais Selecionadas",
                     fontweight="bold")
        ax.set_xlabel("Mês")
        ax.set_ylabel("% do Total Anual")
        ax.legend(ncol=2, fontsize=8)
        salvar_fig("sazon_comparacao_capitais_meses")

    # ── 38.5 Violin plot por mês ──────────────────────────────────────────────
    print_sub("38.5 Violin Plot – Casos por Mês")
    if "MES" in df_cg.columns:
        fig, ax = plt.subplots(figsize=(14, 6))
        dados_v = [df_cg[df_cg["MES"] == m]["casos"].dropna().values
                   for m in range(1, 13)]
        parts = ax.violinplot(
            dados_v, positions=range(1, 13),
            widths=0.7, showmeans=True, showmedians=True,
            showextrema=True
        )
        for i, (pc, mes) in enumerate(zip(parts["bodies"], range(1, 13))):
            cor_v = COR_PRINCIPAL if mes in {1,2,3,10,11,12} else COR_SECUNDARIA
            pc.set_facecolor(cor_v)
            pc.set_alpha(0.5)

        ax.set_xticks(range(1, 13))
        ax.set_xticklabels([MESES_ABREV[m] for m in range(1, 13)])
        ax.set_title("Distribuição de Casos por Mês – Campo Grande/MS (2016–2025)",
                     fontweight="bold")
        ax.set_ylabel("Casos / Semana")
        patch_chuv = mpatches.Patch(color=COR_PRINCIPAL, alpha=0.5,
                                    label="Período Chuvoso")
        patch_seco = mpatches.Patch(color=COR_SECUNDARIA, alpha=0.5,
                                    label="Período Seco")
        ax.legend(handles=[patch_chuv, patch_seco])
        salvar_fig("sazon_violin_casos_por_mes_cg")

    log.info("  Sazonalidade avançada concluída.")
    return resultados




In [81]:
# =============================================================================
# SEÇÃO 39 – ANÁLISE DE SURTOS E LIMIARES EPIDÊMICOS


In [82]:
# =============================================================================

def analise_surtos(df_cg: pd.DataFrame) -> dict:
    """
    Identifica e caracteriza surtos epidêmicos em Campo Grande:
    - Definição de surto: casos acima do limiar P90 histórico
    - Duração dos surtos (semanas consecutivas)
    - Magnitude (total de casos acima do limiar)
    - Comparação entre surtos
    - Gráfico de surtos identificados
    """
    print_section("ANÁLISE DE SURTOS EPIDÊMICOS")
    resultados = {}

    if df_cg.empty or "casos" not in df_cg.columns:
        return resultados

    df_s = df_cg.sort_values("data_SE").copy() if "data_SE" in df_cg.columns \
           else df_cg.copy()
    df_s = df_s.reset_index(drop=True)

    # ── Limiares ──────────────────────────────────────────────────────────────
    casos = df_s["casos"].fillna(0).values
    p75   = np.percentile(casos, 75)
    p90   = np.percentile(casos, 90)
    p95   = np.percentile(casos, 95)

    log.info(f"  Limiares: P75={p75:.1f} | P90={p90:.1f} | P95={p95:.1f}")

    pop_cg = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942_140
    limiar_epi   = PARAMS["threshold_epidemia_inc100k"] * pop_cg / 100_000
    limiar_alerta = PARAMS["threshold_alerta_inc100k"] * pop_cg / 100_000

    # ── Identificação de surtos (períodos ≥ P90 por pelo menos 2 semanas) ────
    em_surto = False
    surtos   = []
    inicio   = None
    casos_surto = []

    for i, c in enumerate(casos):
        if c >= p90:
            if not em_surto:
                em_surto = True
                inicio   = i
                casos_surto = [c]
            else:
                casos_surto.append(c)
        else:
            if em_surto and len(casos_surto) >= 2:
                surtos.append({
                    "inicio_idx":  inicio,
                    "fim_idx":     i - 1,
                    "duracao_sem": len(casos_surto),
                    "total_casos": sum(casos_surto),
                    "pico_casos":  max(casos_surto),
                    "data_inicio": df_s.get("data_SE", pd.Series([None]*len(df_s))).iloc[inicio],
                    "data_fim":    df_s.get("data_SE", pd.Series([None]*len(df_s))).iloc[i-1],
                    "ano":         int(df_s.get("ANO", pd.Series([0]*len(df_s))).iloc[inicio]),
                })
            em_surto    = False
            casos_surto = []

    log.info(f"  Surtos identificados (≥ P90 por ≥ 2 sem): {len(surtos)}")
    resultados["surtos"] = surtos

    # ── Tabela de surtos ──────────────────────────────────────────────────────
    if surtos:
        rows_surtos = []
        for i, s in enumerate(surtos, 1):
            data_i = str(s["data_inicio"])[:10] if s["data_inicio"] is not None else "–"
            data_f = str(s["data_fim"])[:10]    if s["data_fim"]    is not None else "–"
            rows_surtos.append([
                i, int(s["ano"]), data_i, data_f,
                int(s["duracao_sem"]),
                fmt_num(int(s["total_casos"])),
                fmt_num(int(s["pico_casos"])),
                classificar_risco(taxa_inc(s["pico_casos"], pop_cg)),
            ])
        tab_surtos = make_table(
            ["#","Ano","Início","Fim","Duração (sem)","Total Casos","Pico","Risco Pico"],
            rows_surtos, col_align=["c","c","l","l","c","r","r","l"]
        )
        log.info(f"\n{tab_surtos}")
        salvar_txt(tab_surtos, "surtos_identificados_cg",
                   "Surtos Epidêmicos Identificados – Campo Grande")

        # ── Gráfico de surtos ─────────────────────────────────────────────────
        if "data_SE" in df_s.columns:
            fig, ax = plt.subplots(figsize=(16, 6))
            ax.bar(df_s["data_SE"], casos,
                   color=COR_SECUNDARIA, alpha=0.4, label="Casos Semanais")
            ax.axhline(p90, color=COR_ALERTA, linestyle="--", linewidth=1.5,
                       label=f"Limiar P90 = {p90:.0f}")
            ax.axhline(p95, color=COR_PRINCIPAL, linestyle=":", linewidth=1.5,
                       label=f"Limiar P95 = {p95:.0f}")

            for s in surtos:
                if s["data_inicio"] is not None and s["data_fim"] is not None:
                    ax.axvspan(s["data_inicio"], s["data_fim"],
                               alpha=0.15, color=COR_PRINCIPAL)

            # Anotação do maior surto
            maior_surto = max(surtos, key=lambda x: x["pico_casos"])
            if maior_surto["data_inicio"] is not None:
                ax.annotate(
                    f"Maior surto\n{int(maior_surto['ano'])}\n"
                    f"({fmt_num(int(maior_surto['pico_casos']))} pico)",
                    xy=(maior_surto["data_inicio"], maior_surto["pico_casos"]),
                    xytext=(maior_surto["data_inicio"],
                            maior_surto["pico_casos"] * 1.15),
                    fontsize=8, fontweight="bold",
                    arrowprops=dict(arrowstyle="->", color="black"),
                )

            ax.set_title("Surtos Epidêmicos Identificados – Dengue Campo Grande/MS",
                         fontweight="bold")
            ax.set_xlabel("Semana Epidemiológica")
            ax.set_ylabel("Casos / Semana")
            ax.legend(ncol=2, fontsize=8)
            ax.yaxis.set_major_formatter(mticker.FuncFormatter(
                lambda x, _: fmt_num(int(max(x, 0)))
            ))
            salvar_fig("surtos_grafico_identificados_cg")

        # ── Estatísticas dos surtos ───────────────────────────────────────────
        duracoes  = [s["duracao_sem"] for s in surtos]
        totais    = [s["total_casos"]  for s in surtos]
        rows_st   = [
            ["Número de surtos identificados", len(surtos)],
            ["Duração média (semanas)",         fmt_num(np.mean(duracoes), 1)],
            ["Duração máxima (semanas)",         max(duracoes)],
            ["Total de casos (maior surto)",     fmt_num(int(max(totais)))],
            ["Pico máximo (semana)",             fmt_num(int(max(s["pico_casos"] for s in surtos)))],
            ["Ano com mais surtos",              max(set(s["ano"] for s in surtos),
                                                   key=lambda a: sum(1 for s in surtos if s["ano"]==a))],
        ]
        tab_st = make_table(["Indicador","Valor"], rows_st, col_align=["l","r"])
        log.info(f"\n{tab_st}")
        salvar_txt(tab_st, "surtos_estatisticas_cg",
                   "Estatísticas dos Surtos – Campo Grande")

    # ── Gráfico de duração e magnitude dos surtos ────────────────────────────
    if len(surtos) >= 3:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        duracoes_s = [s["duracao_sem"] for s in surtos]
        totais_s   = [s["total_casos"]  for s in surtos]
        anos_s     = [s["ano"]          for s in surtos]

        axes[0].bar(range(1, len(surtos)+1), duracoes_s,
                    color=COR_SECUNDARIA, edgecolor="white")
        axes[0].set_title("Duração dos Surtos (semanas)", fontweight="bold")
        axes[0].set_xlabel("Surto #")
        axes[0].set_ylabel("Semanas")
        for j, (bar, ano) in enumerate(zip(axes[0].patches, anos_s)):
            axes[0].text(bar.get_x() + bar.get_width()/2,
                         bar.get_height() + 0.1,
                         str(ano), ha="center", fontsize=7)

        axes[1].bar(range(1, len(surtos)+1),
                    [t/1000 for t in totais_s],
                    color=COR_PRINCIPAL, alpha=0.7, edgecolor="white")
        axes[1].set_title("Magnitude dos Surtos (mil casos)", fontweight="bold")
        axes[1].set_xlabel("Surto #")
        axes[1].set_ylabel("Casos (×1000)")
        plt.suptitle("Análise dos Surtos – Dengue Campo Grande/MS",
                     fontsize=13, fontweight="bold")
        salvar_fig("surtos_duracao_magnitude_cg")

    log.info("  Análise de surtos concluída.")
    return resultados




In [83]:
# =============================================================================
# SEÇÃO 40 – CORRELAÇÃO ESPACIAL ENTRE MUNICÍPIOS DE MS


In [84]:
# =============================================================================

def correlacao_espacial_ms(df_ms: pd.DataFrame) -> dict:
    """
    Analisa correlações espaciais entre municípios de MS:
    - Correlação entre séries temporais anuais dos municípios
    - Heatmap de correlação inter-municipal
    - Identificação de grupos de municípios sincronizados
    - Correlação Campo Grande vs demais municípios
    """
    print_section("CORRELAÇÃO ESPACIAL – MUNICÍPIOS DE MS")
    resultados = {}

    if df_ms.empty or "municipio_nome" not in df_ms.columns:
        return resultados

    # Constrói matriz: colunas = municípios, linhas = ano-semana
    if "ANO" not in df_ms.columns or "SEMANA" not in df_ms.columns:
        return resultados

    pivot_ms = df_ms.pivot_table(
        index=["ANO","SEMANA"],
        columns="municipio_nome",
        values="casos",
        aggfunc="sum",
    ).fillna(0)

    # Mantém apenas municípios com ≥ 80% de semanas preenchidas
    threshold_pct = 0.80
    n_total = len(pivot_ms)
    cols_validos = [c for c in pivot_ms.columns
                    if (pivot_ms[c] > 0).sum() / n_total >= threshold_pct * 0.3]
    pivot_ms = pivot_ms[cols_validos]

    if pivot_ms.shape[1] < 5:
        log.warning("  Poucos municípios com dados suficientes para correlação espacial.")
        return resultados

    # ── Matriz de correlação de Pearson ──────────────────────────────────────
    corr_mat = pivot_ms.corr(method="pearson")

    fig, ax = plt.subplots(figsize=(14, 12))
    n_muns_corr = min(corr_mat.shape[0], 25)
    top_muns_corr = corr_mat.index[:n_muns_corr]
    sns.heatmap(
        corr_mat.loc[top_muns_corr, top_muns_corr],
        cmap="coolwarm", vmin=-1, vmax=1,
        linewidths=0.2, ax=ax,
        annot=(n_muns_corr <= 15),
        fmt=".1f", annot_kws={"size": 7},
    )
    ax.set_title(f"Correlação Semanal entre Municípios MS (top {n_muns_corr})",
                 fontsize=13, fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
    salvar_fig("espacial_corr_matricial_ms")

    # ── Correlação de Campo Grande com demais ────────────────────────────────
    if "Campo Grande" in corr_mat.columns:
        corr_cg = corr_mat["Campo Grande"].drop("Campo Grande").sort_values(
            ascending=False
        )
        log.info(f"  Top 10 municípios mais correlacionados com CG:")
        rows_corr_cg = []
        for mun, r in corr_cg.head(10).items():
            log.info(f"    {mun}: r={r:.4f}")
            rows_corr_cg.append([mun, fmt_num(r, 4)])
        log.info(f"  Bottom 5 (menos correlacionados):")
        for mun, r in corr_cg.tail(5).items():
            log.info(f"    {mun}: r={r:.4f}")

        tab_cg_corr = make_table(
            ["Município","Correlação com Campo Grande"],
            rows_corr_cg, col_align=["l","r"]
        )
        salvar_txt(tab_cg_corr, "espacial_corr_cg_vs_municipios",
                   "Correlação Campo Grande × Municípios MS")
        resultados["corr_cg"] = corr_cg

    # ── Gráfico: Top 10 mais correlacionados ──────────────────────────────────
    if "Campo Grande" in corr_mat.columns:
        top10_corr = corr_cg.head(10)
        fig, ax = plt.subplots(figsize=(10, 5))
        cores_c = [COR_VERDE if r > 0.7 else COR_ALERTA if r > 0.4 else COR_CINZA
                   for r in top10_corr.values]
        ax.barh(top10_corr.index[::-1], top10_corr.values[::-1],
                color=cores_c[::-1], edgecolor="white")
        ax.set_title("Top 10 Municípios Mais Correlacionados com Campo Grande",
                     fontweight="bold")
        ax.set_xlabel("Correlação de Pearson")
        ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
        salvar_fig("espacial_top10_corr_cg")

    log.info("  Correlação espacial concluída.")
    return resultados




In [85]:
# =============================================================================
# SEÇÃO 41 – BOOTSTRAP: INTERVALOS DE CONFIANÇA PARA MÉDIAS


In [86]:
# =============================================================================

def bootstrap_intervalos(df_cg: pd.DataFrame) -> dict:
    """
    Calcula intervalos de confiança (IC 95%) via bootstrap
    para indicadores-chave de Campo Grande:
    - Média semanal de casos
    - Taxa de incidência média
    - Rt médio
    - Proporção de semanas com nível 4
    """
    print_section("BOOTSTRAP – INTERVALOS DE CONFIANÇA 95%")
    resultados = {}

    if df_cg.empty or "casos" not in df_cg.columns:
        return resultados

    np.random.seed(42)
    N_BOOTSTRAP = 2000
    pop_cg = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942_140

    def bootstrap_ci(arr: np.ndarray, func=np.mean, n=N_BOOTSTRAP) -> Tuple[float, float, float]:
        stats_b = [func(np.random.choice(arr, size=len(arr), replace=True))
                   for _ in range(n)]
        return float(np.mean(stats_b)), float(np.percentile(stats_b, 2.5)), float(np.percentile(stats_b, 97.5))

    casos_arr = df_cg["casos"].dropna().values

    # Média semanal
    m, lo, hi = bootstrap_ci(casos_arr)
    resultados["media_semanal"] = (m, lo, hi)

    # Taxa de incidência média
    taxas = np.array([taxa_inc(c, pop_cg) for c in casos_arr])
    m_t, lo_t, hi_t = bootstrap_ci(taxas)
    resultados["taxa_inc_media"] = (m_t, lo_t, hi_t)

    # Rt médio
    rt_ci = None
    if "Rt" in df_cg.columns:
        rt_arr = df_cg["Rt"].dropna().values
        if len(rt_arr) > 0:
            m_r, lo_r, hi_r = bootstrap_ci(rt_arr)
            resultados["rt_medio"] = (m_r, lo_r, hi_r)
            rt_ci = (m_r, lo_r, hi_r)

    # Proporção semanas nível 4
    if "nivel" in df_cg.columns:
        nivel_arr = (df_cg["nivel"] == 4).astype(float).dropna().values
        m_n, lo_n, hi_n = bootstrap_ci(nivel_arr)
        resultados["prop_nivel4"] = (m_n, lo_n, hi_n)

    # Tabela de resultados
    rows_bs = [
        ["Média Semanal de Casos",
         fmt_num(m, 1),
         fmt_num(lo, 1), fmt_num(hi, 1)],
        ["Taxa de Incidência Média (/100k)",
         fmt_num(m_t, 2),
         fmt_num(lo_t, 2), fmt_num(hi_t, 2)],
    ]
    if rt_ci:
        rows_bs.append(["Rt Médio Histórico",
                        fmt_num(rt_ci[0], 4),
                        fmt_num(rt_ci[1], 4),
                        fmt_num(rt_ci[2], 4)])

    tab_bs = make_table(
        ["Indicador","Estimativa","IC 2.5%","IC 97.5%"],
        rows_bs, col_align=["l","r","r","r"]
    )
    log.info(f"\n{tab_bs}")
    salvar_txt(tab_bs, "bootstrap_ic_indicadores_cg",
               f"Bootstrap IC 95% (n={N_BOOTSTRAP}) – Campo Grande")

    # Gráfico: distribuição bootstrap da média semanal
    bs_means = [np.mean(np.random.choice(casos_arr, size=len(casos_arr), replace=True))
                for _ in range(N_BOOTSTRAP)]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(bs_means, bins=50, color=COR_SECUNDARIA, edgecolor="white",
            alpha=0.7, label="Bootstrap samples")
    ax.axvline(m, color=COR_PRINCIPAL, linewidth=2.5, label=f"Média = {m:.1f}")
    ax.axvline(lo, color=COR_ALERTA, linewidth=1.5, linestyle="--",
               label=f"IC 95%: [{lo:.1f}, {hi:.1f}]")
    ax.axvline(hi, color=COR_ALERTA, linewidth=1.5, linestyle="--")
    ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 100],
                     lo, hi, alpha=0.12, color=COR_ALERTA)
    ax.set_title("Bootstrap – Distribuição da Média Semanal de Casos – CG",
                 fontweight="bold")
    ax.set_xlabel("Média de Casos / Semana")
    ax.set_ylabel("Frequência")
    ax.legend()
    salvar_fig("bootstrap_distribuicao_media_cg")

    log.info(f"  Bootstrap concluído (n={N_BOOTSTRAP} reamostras).")
    return resultados




In [87]:
# =============================================================================
# SEÇÃO 42 – RELATÓRIO EPIDEMIOLÓGICO DETALHADO POR ANO


In [88]:
# =============================================================================

def relatorio_por_ano(df_cg: pd.DataFrame,
                       df_ms: pd.DataFrame) -> None:
    """
    Gera relatório epidemiológico detalhado para cada ano (2016–2025):
    - Indicadores CG por ano
    - Comparação com média do quinquênio
    - Classificação do ano epidemiológico
    - Tabela completa TXT + LOG
    """
    print_section("RELATÓRIO EPIDEMIOLÓGICO ANUAL – CAMPO GRANDE")

    if df_cg.empty or "ANO" not in df_cg.columns:
        return

    pop_cg = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942_140
    anos   = sorted([int(a) for a in df_cg["ANO"].unique() if 2016 <= int(a) <= 2025])

    # Médias históricas para comparação
    media_hist_casos = float(df_cg["casos"].mean())
    media_hist_rt    = float(df_cg["Rt"].mean()) if "Rt" in df_cg.columns else 1.0

    rows_anual = []
    for ano in anos:
        sub = df_cg[df_cg["ANO"] == ano]
        if sub.empty:
            continue

        total     = int(sub["casos"].sum())
        media_s   = float(sub["casos"].mean())
        max_s     = int(sub["casos"].max())
        taxa_a    = taxa_inc(total, pop_cg)
        rt_m      = float(sub["Rt"].mean()) if "Rt" in sub.columns else 0
        n4        = int((sub["nivel"] == 4).sum()) if "nivel" in sub.columns else 0
        n_trans   = int(sub["transmissao"].sum()) if "transmissao" in sub.columns else 0
        semana_pico = int(sub.loc[sub["casos"].idxmax(), "SEMANA"]) if "SEMANA" in sub.columns else 0

        # Classificação do ano
        if taxa_a >= 1000:
            classif = "CRÍTICO"
        elif taxa_a >= 500:
            classif = "MUITO ALTO"
        elif taxa_a >= 300:
            classif = "ALTO"
        elif taxa_a >= 100:
            classif = "MÉDIO"
        elif taxa_a >= 50:
            classif = "BAIXO"
        else:
            classif = "MUITO BAIXO"

        rows_anual.append([
            ano,
            fmt_num(total),
            fmt_num(taxa_a, 1),
            fmt_num(media_s, 1),
            fmt_num(max_s),
            semana_pico,
            fmt_num(rt_m, 3),
            n4,
            n_trans,
            classif,
        ])

    tab_anual = make_table(
        ["Ano","Total Casos","Taxa/100k","Méd/Sem","Pico","Sem Pico",
         "Rt Médio","N.4 Sems","Trans Ativa","Classificação"],
        rows_anual,
        col_align=["c","r","r","r","r","c","r","c","c","l"]
    )
    log.info(f"\n{tab_anual}")
    salvar_txt(tab_anual, "relatorio_epidemiologico_anual_cg",
               "Relatório Epidemiológico Anual – Campo Grande/MS")
    salvar_log_tabela(tab_anual, "relatorio_epidemiologico_anual_cg",
                      "Epidemiologia Anual – CG")

    # ── Gráfico: Perfil epidemiológico por ano ─────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Total de casos
    totais_anos = [int(df_cg[df_cg["ANO"]==a]["casos"].sum()) for a in anos]
    cores_a = [COR_PRINCIPAL if t == max(totais_anos) else "#AED6F1"
               for t in totais_anos]
    axes[0,0].bar(anos, totais_anos, color=cores_a, edgecolor="white")
    axes[0,0].set_title("Total de Casos Anuais", fontweight="bold")
    axes[0,0].set_ylabel("Casos")
    axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: fmt_num(int(max(x,0)))
    ))

    # Taxa de incidência
    taxas_anos = [taxa_inc(t, pop_cg) for t in totais_anos]
    axes[0,1].plot(anos, taxas_anos, marker="o", color=COR_ALERTA,
                   linewidth=2, markersize=7)
    axes[0,1].fill_between(anos, taxas_anos, alpha=0.2, color=COR_ALERTA)
    axes[0,1].axhline(300, color="red", linestyle="--", linewidth=1,
                      label="Limiar Alto (300/100k)")
    axes[0,1].axhline(100, color="orange", linestyle="--", linewidth=1,
                      label="Limiar Médio (100/100k)")
    axes[0,1].set_title("Taxa de Incidência / 100k hab", fontweight="bold")
    axes[0,1].set_ylabel("Taxa / 100k")
    axes[0,1].legend(fontsize=7)

    # Rt médio por ano
    if "Rt" in df_cg.columns:
        rt_anos = [float(df_cg[df_cg["ANO"]==a]["Rt"].mean()) for a in anos]
        axes[1,0].bar(anos, rt_anos, color=COR_SECUNDARIA, edgecolor="white")
        axes[1,0].axhline(1.0, color="red", linestyle="--", linewidth=1.5,
                          label="Rt = 1")
        axes[1,0].set_title("Rt Médio por Ano", fontweight="bold")
        axes[1,0].set_ylabel("Rt Estimado")
        axes[1,0].legend(fontsize=8)

    # Semanas em nível 4 por ano
    if "nivel" in df_cg.columns:
        n4_anos = [int((df_cg[df_cg["ANO"]==a]["nivel"]==4).sum()) for a in anos]
        axes[1,1].bar(anos, n4_anos, color=NIVEL_CORES[4], edgecolor="white",
                      alpha=0.8)
        axes[1,1].set_title("Semanas em Nível 4 (Alerta Vermelho)", fontweight="bold")
        axes[1,1].set_ylabel("Número de Semanas")

    for ax in axes.flatten():
        ax.set_xticks(anos)
        ax.set_xticklabels(anos, rotation=45, fontsize=8)

    plt.suptitle("Perfil Epidemiológico Anual – Dengue Campo Grande/MS (2016–2025)",
                 fontsize=14, fontweight="bold")
    salvar_fig("relatorio_perfil_anual_cg")

    log.info("  Relatório anual concluído.")




In [89]:
# =============================================================================
# SEÇÃO 43 – PERSISTÊNCIA DE MODELOS (SAVE / LOAD)


In [90]:
# =============================================================================

def salvar_modelos(resultados_ml: dict,
                   resultados_reg: dict,
                   resultados_dl: dict) -> None:
    """
    Persiste modelos treinados em disco para reutilização futura:
    - Modelos sklearn: pickle (.pkl)
    - Modelos TensorFlow/Keras: SavedModel (.h5)
    - Scalers e configurações: pickle + JSON
    """
    print_section("PERSISTÊNCIA DE MODELOS – SAVE")

    import pickle

    modelos_dir = OUTPUT_DIR / "modelos"
    modelos_dir.mkdir(parents=True, exist_ok=True)

    salvos = []

    # ── Modelos de classificação (sklearn) ─────────────────────────────────────
    for dataset_nome, res in resultados_ml.items():
        if "modelos" not in res:
            continue
        for nome_m, obj in res["modelos"].items():
            try:
                nome_arq = nome_m.lower().replace(" ", "_").replace("-", "_")
                pkl_path = modelos_dir / f"clf_{nome_arq}_{TIMESTAMP}.pkl"
                mdl_obj  = obj[0] if isinstance(obj, tuple) else obj
                with open(pkl_path, "wb") as f:
                    pickle.dump(mdl_obj, f)
                salvos.append(("Classificação", nome_m, str(pkl_path.name)))
                log.info(f"  [PKL] {pkl_path.name}")
            except Exception as e:
                log.warning(f"  Falha ao salvar {nome_m}: {e}")

    # ── Modelos de regressão ───────────────────────────────────────────────────
    if resultados_reg and "y_preds" in resultados_reg:
        # Salva scaler
        if "scaler" in resultados_reg:
            scaler_path = modelos_dir / f"scaler_regressao_{TIMESTAMP}.pkl"
            try:
                with open(scaler_path, "wb") as f:
                    pickle.dump(resultados_reg["scaler"], f)
                salvos.append(("Scaler", "StandardScaler Reg.", str(scaler_path.name)))
                log.info(f"  [PKL] {scaler_path.name}")
            except Exception as e:
                log.warning(f"  Scaler falhou: {e}")

        if "feat_cols" in resultados_reg:
            feat_path = modelos_dir / f"feat_cols_regressao_{TIMESTAMP}.json"
            with open(feat_path, "w") as f:
                json.dump(resultados_reg["feat_cols"], f, indent=2)
            log.info(f"  [JSON] {feat_path.name}")

    # ── Modelos Deep Learning (Keras .h5) ──────────────────────────────────────
    if resultados_dl and "modelos_dl" in resultados_dl and HAS_TF:
        for nome_m, mdl in resultados_dl["modelos_dl"].items():
            try:
                nome_arq = nome_m.lower().replace("-","_").replace(" ","_")
                h5_path  = modelos_dir / f"dl_{nome_arq}_{TIMESTAMP}.h5"
                mdl.save(str(h5_path))
                salvos.append(("Deep Learning", nome_m, str(h5_path.name)))
                log.info(f"  [H5]  {h5_path.name}")
            except Exception as e:
                log.warning(f"  Keras save {nome_m}: {e}")

        if "scaler_dl" in resultados_dl:
            scaler_dl_path = modelos_dir / f"scaler_dl_{TIMESTAMP}.pkl"
            try:
                import pickle
                with open(scaler_dl_path, "wb") as f:
                    pickle.dump(resultados_dl["scaler_dl"], f)
                log.info(f"  [PKL] {scaler_dl_path.name}")
            except Exception as e:
                log.warning(f"  Scaler DL falhou: {e}")

    # ── Manifesto de modelos salvos ────────────────────────────────────────────
    manifest_path = modelos_dir / f"manifesto_modelos_{TIMESTAMP}.json"
    manifest = {
        "timestamp": TIMESTAMP,
        "total_modelos": len(salvos),
        "modelos": [
            {"tipo": t, "nome": n, "arquivo": a}
            for t, n, a in salvos
        ],
        "params": PARAMS,
    }
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    log.info(f"  [JSON] {manifest_path.name}")

    if salvos:
        rows_s = [[t, n, a] for t, n, a in salvos]
        tab_s  = make_table(["Tipo","Modelo","Arquivo"], rows_s,
                             col_align=["l","l","l"])
        log.info(f"\n{tab_s}")
        salvar_txt(tab_s, "modelos_salvos_manifesto",
                   "Modelos Salvos em Disco")

    log.info(f"  {len(salvos)} modelos persistidos em {modelos_dir}")




In [91]:
# =============================================================================
# SEÇÃO 44 – SISTEMA DE ALERTA PRECOCE (NEXT-4-WEEKS FORECAST)


In [92]:
# =============================================================================

def sistema_alerta_precoce(df_cg: pd.DataFrame,
                            resultados_ts: dict,
                            resultados_dl: dict) -> dict:
    """
    Sistema de alerta precoce:
    Combina previsões dos melhores modelos (ensemble ponderado)
    para as próximas 4 semanas e emite boletim de alerta.
    Gera semáforo visual de risco para cada semana prevista.
    """
    print_section("SISTEMA DE ALERTA PRECOCE – PRÓXIMAS 4 SEMANAS")
    resultados = {}

    if df_cg.empty or "casos" not in df_cg.columns:
        return resultados

    pop_cg       = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942_140
    ultima_data  = df_cg["data_SE"].max() if "data_SE" in df_cg.columns else datetime.now()
    ultima_data  = pd.to_datetime(ultima_data)
    N_WEEKS      = 4

    datas_prev   = [ultima_data + timedelta(weeks=i+1) for i in range(N_WEEKS)]

    # Coleta previsões disponíveis
    previsoes_coletadas = {}

    # DL forecast
    if resultados_dl and "previsao_futura_dl" in resultados_dl:
        pf = resultados_dl["previsao_futura_dl"][:N_WEEKS]
        if pf:
            previsoes_coletadas["Deep Learning"] = [v for _, v in pf]

    # ARIMA forecast
    if resultados_ts and "arima_pred" in resultados_ts:
        ap = resultados_ts["arima_pred"]
        # Converte mensal → semanal (÷4)
        vals_arima = (ap["previsao"].values[:N_WEEKS] / 4).tolist()
        if vals_arima:
            previsoes_coletadas["ARIMA"] = vals_arima

    # Prophet forecast
    if resultados_ts and "prophet_pred" in resultados_ts:
        pp = resultados_ts["prophet_pred"]
        vals_prop = (pp["yhat"].values[:N_WEEKS] / 4).tolist()
        if vals_prop:
            previsoes_coletadas["Prophet"] = vals_prop

    # Holt-Winters
    if resultados_ts and "hw_pred" in resultados_ts:
        hw = resultados_ts["hw_pred"]
        vals_hw = (hw["previsao_hw"].values[:N_WEEKS] / 4).tolist()
        if vals_hw:
            previsoes_coletadas["Holt-Winters"] = vals_hw

    # Fallback: média móvel simples
    if not previsoes_coletadas:
        mm8 = float(df_cg["casos"].tail(8).mean())
        previsoes_coletadas["Média Móvel 8 sem"] = [mm8] * N_WEEKS

    # ── Ensemble ponderado ────────────────────────────────────────────────────
    # Pesos: DL=0.4, ARIMA=0.25, Prophet=0.20, HW=0.15
    pesos_modelos = {
        "Deep Learning":    0.40,
        "ARIMA":            0.25,
        "Prophet":          0.20,
        "Holt-Winters":     0.15,
        "Média Móvel 8 sem":1.00,
    }

    ensemble = []
    for i in range(N_WEEKS):
        vals_i = []
        pesos_i = []
        for nome_m, vals in previsoes_coletadas.items():
            if i < len(vals):
                vals_i.append(max(float(vals[i]), 0))
                pesos_i.append(pesos_modelos.get(nome_m, 0.25))
        if vals_i:
            total_peso = sum(pesos_i)
            ens_val    = sum(v * p for v, p in zip(vals_i, pesos_i)) / total_peso
            ensemble.append(max(ens_val, 0))
        else:
            ensemble.append(float(df_cg["casos"].tail(4).mean()))

    # ── Classificação de risco para cada semana ───────────────────────────────
    semaforo = []
    for val in ensemble:
        taxa  = taxa_inc(val, pop_cg)
        risco = classificar_risco(taxa)
        nivel = (4 if taxa >= 1000 else
                 3 if taxa >= 300  else
                 2 if taxa >= 100  else 1)
        semaforo.append((val, taxa, risco, nivel))

    # ── Boletim de alerta ─────────────────────────────────────────────────────
    rows_bol = []
    for i, (data, (val, taxa, risco, nivel)) in enumerate(zip(datas_prev, semaforo)):
        rows_bol.append([
            f"Semana {i+1}",
            data.strftime("%d/%m/%Y"),
            fmt_num(int(val)),
            fmt_num(taxa, 1),
            risco,
            NIVEL_NOMES.get(nivel, "?"),
        ])

    tab_bol = make_table(
        ["Período","Data","Casos Prev.","Taxa/100k","Risco","Nível Alerta"],
        rows_bol, col_align=["l","l","r","r","l","l"]
    )
    log.info(f"\n{tab_bol}")
    salvar_txt(tab_bol, "alerta_precoce_proximo_mes",
               "Boletim de Alerta Precoce – Próximas 4 Semanas")
    salvar_log_tabela(tab_bol, "alerta_precoce_proximo_mes",
                      "Alerta Precoce")

    # ── Gráfico semáforo ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Barra de previsão
    ax = axes[0]
    cores_sem = [NIVEL_CORES.get(s[3], "#999") for s in semaforo]
    bars = ax.bar(
        [d.strftime("%d/%m") for d in datas_prev],
        [s[0] for s in semaforo],
        color=cores_sem, edgecolor="white", linewidth=0.5
    )
    for bar, (val, taxa, risco, _) in zip(bars, semaforo):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max([s[0] for s in semaforo]) * 0.02,
                f"{int(val)}\n({risco})",
                ha="center", va="bottom", fontsize=8, fontweight="bold")

    # Histórico recente
    ult8 = df_cg.sort_values("data_SE").tail(8)
    ax.plot(
        [d.strftime("%d/%m") for d in datas_prev],
        [s[0] for s in semaforo],
        marker="o", color="black", linewidth=1.5, zorder=5
    )
    ax.set_title("Previsão Ensemble – Próximas 4 Semanas", fontweight="bold")
    ax.set_ylabel("Casos / Semana")
    ax.set_xlabel("Data")

    # Semáforo visual
    ax2 = axes[1]
    ax2.set_xlim(0, 4)
    ax2.set_ylim(0, 1)
    ax2.axis("off")
    ax2.set_title("Semáforo de Risco", fontweight="bold")

    for i, (data, (val, taxa, risco, nivel)) in enumerate(zip(datas_prev, semaforo)):
        cor = NIVEL_CORES.get(nivel, "#999")
        circle = plt.Circle((i * 0.9 + 0.45, 0.5), 0.35,
                              color=cor, zorder=3)
        ax2.add_patch(circle)
        ax2.text(i * 0.9 + 0.45, 0.5, f"S{i+1}\n{risco[:4]}",
                 ha="center", va="center", fontsize=7,
                 fontweight="bold", color="white", zorder=4)
        ax2.text(i * 0.9 + 0.45, 0.1, data.strftime("%d/%m"),
                 ha="center", va="bottom", fontsize=7)

    plt.suptitle("Sistema de Alerta Precoce – Dengue Campo Grande/MS",
                 fontsize=13, fontweight="bold")
    salvar_fig("alerta_precoce_semaforo_cg")

    # ── Tabela de contribuições por modelo ────────────────────────────────────
    if len(previsoes_coletadas) > 1:
        rows_mod = []
        for nome_m, vals in previsoes_coletadas.items():
            for i, v in enumerate(vals[:N_WEEKS]):
                if i == 0:
                    rows_mod.append([nome_m] + [fmt_num(int(max(v, 0)))
                                                 for v in vals[:N_WEEKS]])
                    break
        tab_mod = make_table(
            ["Modelo"] + [f"Sem {i+1}" for i in range(N_WEEKS)],
            rows_mod, col_align=["l"] + ["r"]*N_WEEKS
        )
        log.info(f"\n{tab_mod}")
        salvar_txt(tab_mod, "alerta_previsoes_por_modelo",
                   "Previsões por Modelo – Próximas 4 Semanas")

    resultados["ensemble"]  = ensemble
    resultados["semaforo"]  = semaforo
    resultados["datas_prev"] = datas_prev
    log.info("  Sistema de alerta precoce concluído.")
    return resultados




In [93]:
# =============================================================================
# SEÇÃO 45 – DASHBOARDS PLOTLY AVANÇADOS


In [94]:
# =============================================================================

def gerar_dashboards_avancados(df_cg: pd.DataFrame,
                                df_ms: pd.DataFrame,
                                df_cap: pd.DataFrame,
                                alerta: dict) -> None:
    """
    Dashboards avançados adicionais:
    1. Sunburst: hierarquia Região → UF → Capital
    2. Scatter geo com bolhas de incidência
    3. Gauge de risco atual (Campo Grande)
    4. Violin interativo por mês
    5. Waterfall de variação anual
    """
    print_section("DASHBOARDS AVANÇADOS – PLOTLY")

    if not HAS_PLOTLY:
        log.warning("  Plotly não disponível.")
        return

    # ── 45.1 Sunburst Região → Capital → Casos ─────────────────────────────────
    print_sub("45.1 Sunburst – Hierarquia Nacional")
    try:
        if not df_cap.empty and "municipio_nome" in df_cap.columns:
            tot_cap = df_cap.groupby("municipio_nome")["casos"].sum().reset_index()
            tot_cap["UF"]     = tot_cap["municipio_nome"].map(CAPITAIS_UF)
            tot_cap["REGIAO"] = tot_cap["UF"].map(REGIAO_UF)
            tot_cap["pop"]    = tot_cap["municipio_nome"].map(POP_CAPITAIS).fillna(1e6)
            tot_cap["taxa"]   = tot_cap.apply(
                lambda r: taxa_inc(r["casos"], r["pop"]), axis=1)

            ids, labels, parents, values, hover = [], [], [], [], []
            for reg in tot_cap["REGIAO"].dropna().unique():
                ids.append(reg); labels.append(reg)
                parents.append("Brasil"); values.append(0)
                hover.append(f"Região: {reg}")
            ids.append("Brasil"); labels.append("Brasil")
            parents.append(""); values.append(0)
            hover.append("Brasil")
            for _, r in tot_cap.iterrows():
                ids.append(r["municipio_nome"])
                labels.append(f"{r['municipio_nome']} ({r.get('UF','?')})")
                parents.append(r.get("REGIAO","Brasil"))
                values.append(int(r["casos"]))
                hover.append(f"Taxa: {r['taxa']:.1f}/100k")

            fig_sun = go.Figure(go.Sunburst(
                ids=ids, labels=labels, parents=parents,
                values=values, hovertext=hover,
                branchvalues="total",
                marker=dict(colorscale="YlOrRd"),
            ))
            fig_sun.update_layout(
                title="Sunburst – Casos de Dengue: Brasil → Região → Capital",
                title_font_size=14, height=650,
                template="plotly_white",
            )
            salvar_html(fig_sun, "dash_adv_sunburst_nacional", "dashboards")
            _inc("dashboards_gerados")
    except Exception as e:
        log.warning(f"  Sunburst falhou: {e}")

    # ── 45.2 Scatter geográfico – capitais ────────────────────────────────────
    print_sub("45.2 Scatter Geográfico – Capitais")
    try:
        if not df_cap.empty:
            tot_c = df_cap.groupby("municipio_nome")["casos"].sum().reset_index()
            tot_c["lat"]     = tot_c["municipio_nome"].map(
                lambda m: COORDS_CAPITAIS.get(m, (None, None))[0])
            tot_c["lon"]     = tot_c["municipio_nome"].map(
                lambda m: COORDS_CAPITAIS.get(m, (None, None))[1])
            tot_c["UF"]      = tot_c["municipio_nome"].map(CAPITAIS_UF)
            tot_c["pop"]     = tot_c["municipio_nome"].map(POP_CAPITAIS).fillna(1e6)
            tot_c["taxa"]    = tot_c.apply(
                lambda r: taxa_inc(r["casos"], r["pop"]), axis=1)
            tot_c["risco"]   = tot_c["taxa"].apply(classificar_risco)
            tot_c = tot_c.dropna(subset=["lat","lon"])

            fig_geo = px.scatter_geo(
                tot_c,
                lat="lat", lon="lon",
                size="taxa",
                color="risco",
                hover_name="municipio_nome",
                hover_data={"taxa": ":.1f", "casos": True, "UF": True,
                            "lat": False, "lon": False},
                color_discrete_map={
                    "Muito Baixo": "#2ECC71",
                    "Baixo":       "#82E0AA",
                    "Médio":       "#F0B27A",
                    "Alto":        "#E74C3C",
                    "Muito Alto":  "#8E44AD",
                    "Crítico":     "#4A235A",
                    "Sem Dados":   "#CCCCCC",
                },
                size_max=40,
                scope="south america",
                title="Mapa de Bolhas – Incidência de Dengue nas Capitais (2016–2025)",
            )
            fig_geo.update_layout(height=650, template="plotly_white")
            fig_geo.update_geos(
                showcountries=True, countrycolor="lightgray",
                showland=True, landcolor="#F8F8F8",
                showocean=True, oceancolor="#EAF2F8",
            )
            salvar_html(fig_geo, "dash_adv_geo_bolhas_capitais", "dashboards")
            _inc("dashboards_gerados")
    except Exception as e:
        log.warning(f"  Scatter geo falhou: {e}")

    # ── 45.3 Gauge de risco atual – Campo Grande ──────────────────────────────
    print_sub("45.3 Gauge – Risco Atual Campo Grande")
    try:
        if not df_cg.empty:
            # Usa últimas 4 semanas
            ult4 = df_cg.sort_values("data_SE").tail(4)
            casos_recentes = float(ult4["casos"].mean())
            pop_cg = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942_140
            taxa_recente   = taxa_inc(casos_recentes, pop_cg)
            rt_recente     = float(df_cg["Rt"].tail(4).mean()) if "Rt" in df_cg.columns else 1.0
            nivel_recente  = int(df_cg["nivel"].tail(4).mode()[0]) if "nivel" in df_cg.columns else 1

            fig_gauge = make_subplots(
                rows=1, cols=2,
                specs=[[{"type": "indicator"}, {"type": "indicator"}]],
            )
            fig_gauge.add_trace(
                go.Indicator(
                    mode="gauge+number+delta",
                    value=taxa_recente,
                    title={"text": "Taxa Inc./100k<br>(últimas 4 sem.)"},
                    delta={"reference": 100,
                           "increasing": {"color": "red"},
                           "decreasing": {"color": "green"}},
                    gauge={
                        "axis": {"range": [0, 1500]},
                        "bar": {"color": NIVEL_CORES.get(nivel_recente, "#999")},
                        "steps": [
                            {"range": [0,   50], "color": "#2ECC71"},
                            {"range": [50, 100], "color": "#82E0AA"},
                            {"range": [100,300], "color": "#F0B27A"},
                            {"range": [300,1000],"color": "#E74C3C"},
                            {"range": [1000,1500],"color":"#8E44AD"},
                        ],
                        "threshold": {
                            "line": {"color": "red", "width": 3},
                            "thickness": 0.75, "value": 300
                        },
                    },
                    number={"suffix": "/100k"},
                ),
                row=1, col=1
            )
            fig_gauge.add_trace(
                go.Indicator(
                    mode="gauge+number",
                    value=rt_recente,
                    title={"text": "Rt Médio<br>(últimas 4 sem.)"},
                    gauge={
                        "axis": {"range": [0, 3]},
                        "bar": {"color": COR_PRINCIPAL if rt_recente >= 1 else COR_VERDE},
                        "steps": [
                            {"range": [0,   1], "color": "#D5F5E3"},
                            {"range": [1, 1.5], "color": "#FCF3CF"},
                            {"range": [1.5, 3], "color": "#FADBD8"},
                        ],
                        "threshold": {
                            "line": {"color": "red", "width": 3},
                            "thickness": 0.75, "value": 1.0
                        },
                    },
                ),
                row=1, col=2
            )
            fig_gauge.update_layout(
                title_text=f"Painel de Risco Atual – Campo Grande/MS | "
                           f"Nível {nivel_recente}: "
                           f"{NIVEL_NOMES.get(nivel_recente, '?')}",
                height=400, template="plotly_white",
            )
            salvar_html(fig_gauge, "dash_adv_gauge_risco_cg", "dashboards")
            _inc("dashboards_gerados")
    except Exception as e:
        log.warning(f"  Gauge falhou: {e}")

    # ── 45.4 Waterfall – Variação Anual ──────────────────────────────────────
    print_sub("45.4 Waterfall – Variação Anual de Casos")
    try:
        if not df_cg.empty and "ANO" in df_cg.columns:
            por_ano = df_cg.groupby("ANO")["casos"].sum().reset_index()
            por_ano = por_ano[por_ano["ANO"].between(2016, 2025)].sort_values("ANO")
            variacoes = por_ano["casos"].diff().fillna(por_ano["casos"].iloc[0])

            measures = ["absolute"] + ["relative"] * (len(por_ano) - 1)
            text_vals = [f"+{int(v):,}" if v >= 0 else f"{int(v):,}"
                         for v in variacoes]

            fig_wf = go.Figure(go.Waterfall(
                name="Casos",
                orientation="v",
                measure=measures,
                x=por_ano["ANO"].astype(str).tolist(),
                y=variacoes.tolist(),
                text=text_vals,
                textposition="outside",
                increasing={"marker": {"color": COR_PRINCIPAL}},
                decreasing={"marker": {"color": COR_VERDE}},
                totals={"marker": {"color": COR_CINZA}},
            ))
            fig_wf.update_layout(
                title="Variação Anual de Casos – Campo Grande/MS (Waterfall)",
                yaxis_title="Variação de Casos",
                xaxis_title="Ano",
                height=450, template="plotly_white",
                waterfallgap=0.2,
            )
            salvar_html(fig_wf, "dash_adv_waterfall_anual_cg", "dashboards")
            _inc("dashboards_gerados")
    except Exception as e:
        log.warning(f"  Waterfall falhou: {e}")

    # ── 45.5 Box plot interativo por mês ──────────────────────────────────────
    print_sub("45.5 Box Plot Interativo – Casos por Mês")
    try:
        if not df_cg.empty and "MES" in df_cg.columns:
            df_box = df_cg.copy()
            df_box["MES_NOME"] = df_box["MES"].map(MESES_PT)
            ordem_meses = [MESES_PT[m] for m in range(1, 13)
                           if m in df_box["MES"].values]

            fig_box = px.box(
                df_box, x="MES_NOME", y="casos",
                category_orders={"MES_NOME": ordem_meses},
                color="MES_NOME",
                points="outliers",
                title="Distribuição de Casos por Mês – Campo Grande/MS (2016–2025)",
                labels={"MES_NOME": "Mês", "casos": "Casos / Semana"},
                template="plotly_white",
            )
            fig_box.update_layout(
                height=500, showlegend=False,
                xaxis_tickangle=-30,
            )
            salvar_html(fig_box, "dash_adv_boxplot_mes_cg", "dashboards")
            _inc("dashboards_gerados")
    except Exception as e:
        log.warning(f"  Box plot interativo falhou: {e}")

    # ── 45.6 Dashboard de previsão de alerta ─────────────────────────────────
    print_sub("45.6 Dashboard Alerta Precoce")
    try:
        if alerta and "ensemble" in alerta:
            ens    = alerta["ensemble"]
            datas  = [d.strftime("%d/%m/%Y") for d in alerta["datas_prev"]]
            riscos = [s[2] for s in alerta["semaforo"]]
            niveis = [s[3] for s in alerta["semaforo"]]
            cores_al = [NIVEL_CORES.get(n, "#999") for n in niveis]

            hist_casos = df_cg.sort_values("data_SE").tail(52)["casos"].values
            hist_datas = df_cg.sort_values("data_SE").tail(52)["data_SE"].dt.strftime(
                "%d/%m/%Y").values

            fig_al = make_subplots(
                rows=2, cols=1,
                shared_xaxes=False,
                subplot_titles=[
                    "Histórico (último ano) + Previsão 4 semanas",
                    "Semáforo de Risco",
                ],
                row_heights=[0.7, 0.3],
            )
            fig_al.add_trace(
                go.Bar(x=list(hist_datas), y=list(hist_casos),
                       name="Histórico", marker_color="rgba(41,128,185,0.5)"),
                row=1, col=1
            )
            fig_al.add_trace(
                go.Bar(x=datas, y=ens, name="Previsão",
                       marker_color=cores_al, opacity=0.85),
                row=1, col=1
            )
            fig_al.add_trace(
                go.Bar(x=datas, y=[1]*4,
                       marker_color=cores_al,
                       text=[f"Sem.{i+1}<br>{r}" for i, r in enumerate(riscos)],
                       textposition="inside",
                       showlegend=False),
                row=2, col=1
            )
            fig_al.update_yaxes(showticklabels=False, row=2, col=1)
            fig_al.update_layout(
                title_text="Sistema de Alerta Precoce – Dengue Campo Grande/MS",
                height=650, template="plotly_white",
            )
            salvar_html(fig_al, "dash_adv_alerta_precoce_cg", "dashboards")
            _inc("dashboards_gerados")
    except Exception as e:
        log.warning(f"  Dashboard alerta falhou: {e}")

    log.info("  Dashboards avançados concluídos.")




In [95]:
# =============================================================================
# SEÇÃO 46 – FICHAS MUNICIPAIS: TOP 10 MS


In [96]:
# =============================================================================

def fichas_municipais(df_ms: pd.DataFrame) -> None:
    """
    Gera ficha epidemiológica individual para os 10 municípios
    de maior incidência em MS, com:
    - Série temporal
    - Sazonalidade
    - Distribuição de alertas
    - Indicadores síntese em tabela
    """
    print_section("FICHAS MUNICIPAIS – TOP 10 MS")

    if df_ms.empty or "municipio_nome" not in df_ms.columns:
        return

    total_mun = df_ms.groupby("municipio_nome")["casos"].sum()
    top10_muns = total_mun.nlargest(10).index.tolist()
    if "Campo Grande" not in top10_muns:
        top10_muns = ["Campo Grande"] + top10_muns[:9]

    for mun in top10_muns:
        df_m = df_ms[df_ms["municipio_nome"] == mun].sort_values("data_SE") \
               if "data_SE" in df_ms.columns \
               else df_ms[df_ms["municipio_nome"] == mun]

        if df_m.empty:
            continue

        pop = POP_MUNICIPIOS_MS.get(mun, 50_000)

        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        nome_arq = mun.lower().replace(" ","_").replace("/","_")

        # Série temporal
        if "data_SE" in df_m.columns:
            mm8 = df_m["casos"].rolling(8, min_periods=1).mean()
            axes[0,0].bar(df_m["data_SE"], df_m["casos"],
                          color="#AED6F1", alpha=0.5)
            axes[0,0].plot(df_m["data_SE"], mm8,
                           color=COR_PRINCIPAL, linewidth=2)
        axes[0,0].set_title(f"Série Temporal – {mun}", fontweight="bold", fontsize=9)
        axes[0,0].set_ylabel("Casos")
        axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: fmt_num(int(max(x,0)))
        ))

        # Sazonalidade mensal
        if "MES" in df_m.columns:
            sem_m = df_m.groupby("MES")["casos"].mean()
            axes[0,1].bar(sem_m.index, sem_m.values,
                          color=[COR_PRINCIPAL if m in {1,2,3,10,11,12}
                                 else COR_SECUNDARIA for m in sem_m.index])
            axes[0,1].set_xticks(range(1,13))
            axes[0,1].set_xticklabels([MESES_ABREV[i] for i in range(1,13)],
                                       fontsize=7)
            axes[0,1].set_title("Sazonalidade Mensal (média)", fontweight="bold",
                                 fontsize=9)
            axes[0,1].set_ylabel("Casos Médios")

        # Distribuição por nível de alerta
        if "nivel" in df_m.columns:
            dist_n = df_m["nivel"].value_counts().sort_index()
            cores_n = [NIVEL_CORES.get(int(n), "#999") for n in dist_n.index]
            axes[1,0].bar([NIVEL_NOMES.get(int(n), f"N{int(n)}")
                           for n in dist_n.index],
                          dist_n.values, color=cores_n)
            axes[1,0].set_title("Semanas por Nível de Alerta", fontweight="bold",
                                 fontsize=9)
            axes[1,0].set_ylabel("Semanas")
            axes[1,0].set_xticklabels(axes[1,0].get_xticklabels(),
                                       rotation=20, ha="right", fontsize=7)

        # Rt temporal
        if "Rt" in df_m.columns and "data_SE" in df_m.columns:
            df_rt = df_m[df_m["Rt"] > 0]
            axes[1,1].plot(df_rt["data_SE"], df_rt["Rt"],
                           color=COR_ALERTA, linewidth=0.8)
            axes[1,1].axhline(1.0, color="red", linestyle="--", linewidth=1)
            axes[1,1].fill_between(df_rt["data_SE"], df_rt["Rt"],
                                   where=df_rt["Rt"] >= 1,
                                   color=COR_PRINCIPAL, alpha=0.2)
            axes[1,1].set_ylim(0, min(df_rt["Rt"].max() * 1.2, 8))
            axes[1,1].set_title("Rt – Número Reprodutivo", fontweight="bold",
                                  fontsize=9)
            axes[1,1].set_ylabel("Rt")

        # Indicadores síntese como texto
        total_casos = int(df_m["casos"].sum())
        taxa_t      = taxa_inc(total_casos, pop)
        rt_m        = float(df_m["Rt"].mean()) if "Rt" in df_m.columns else 0
        fig.text(0.5, 0.01,
                 f"Total: {fmt_num(total_casos)} casos | "
                 f"Taxa: {fmt_num(taxa_t,1)}/100k | "
                 f"Rt médio: {rt_m:.3f} | "
                 f"Pop.: {fmt_num(pop)}",
                 ha="center", fontsize=9, color="#444")

        plt.suptitle(f"Ficha Epidemiológica – {mun} / MS (2016–2025)",
                     fontsize=13, fontweight="bold")
        salvar_fig(f"ficha_municipal_{nome_arq}")

        # Tabela síntese TXT
        rows_fi = [
            ["Município",              mun],
            ["População (IBGE 2022)",  fmt_num(pop)],
            ["Total de casos",         fmt_num(total_casos)],
            ["Taxa histórica (100k)",  fmt_num(taxa_t, 1)],
            ["Rt médio",               fmt_num(rt_m, 3)],
            ["Pico semanal",           fmt_num(int(df_m["casos"].max()))],
            ["Semanas nível 4",        fmt_num(int((df_m["nivel"]==4).sum()))
             if "nivel" in df_m.columns else "–"],
            ["Anos analisados",        f"{int(df_m['ANO'].min())}–{int(df_m['ANO'].max())}"
             if "ANO" in df_m.columns else "?"],
        ]
        tab_fi = make_table(["Indicador","Valor"], rows_fi,
                             col_align=["l","r"])
        salvar_txt(tab_fi, f"ficha_municipal_{nome_arq}",
                   f"Ficha Epidemiológica – {mun}")

    log.info(f"  Fichas municipais geradas para {len(top10_muns)} municípios.")




In [97]:
# =============================================================================
# SEÇÃO 47 – RELATÓRIO PDF EXPANDIDO (PÁGINAS ADICIONAIS)


In [98]:
# =============================================================================

def complementar_pdf(df_cg: pd.DataFrame,
                     df_ms: pd.DataFrame,
                     df_cap: pd.DataFrame,
                     resultados_tendencia: dict,
                     alerta: dict) -> Optional[Path]:
    """
    Gera arquivo PDF complementar com:
    - Análise de tendência (tabela de projeção)
    - Boletim de alerta precoce
    - Ranking estadual completo
    - Ranking nacional completo
    """
    print_section("PDF COMPLEMENTAR – TENDÊNCIA + ALERTA + RANKINGS")

    if not HAS_FPDF:
        log.warning("  fpdf2 não disponível.")
        return None

    try:
        pdf = FPDF()
        pdf.set_auto_page_break(auto=True, margin=15)

        # ── Capa compacta ─────────────────────────────────────────────────────
        pdf.add_page()
        pdf.set_fill_color(41, 128, 185)
        pdf.set_text_color(255, 255, 255)
        pdf.set_font("Helvetica", "B", 16)
        pdf.rect(0, 0, 210, 30, "F")
        pdf.set_y(8)
        pdf.cell(0, 14,
                 "SIPREV – Relatório Complementar de Análise",
                 align="C", ln=True)
        pdf.set_text_color(0, 0, 0)
        pdf.set_y(38)
        pdf.set_font("Helvetica", "", 10)
        pdf.multi_cell(0, 6,
            f"Campo Grande/MS | InfoDengue 2016-2025\n"
            f"Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M')}\n"
            f"Timestamp: {TIMESTAMP}"
        )

        # ── Análise de Tendência ──────────────────────────────────────────────
        pdf.add_page()
        pdf.set_fill_color(192, 57, 43)
        pdf.set_text_color(255, 255, 255)
        pdf.set_font("Helvetica", "B", 12)
        pdf.cell(0, 9, "ANÁLISE DE TENDÊNCIA (2016–2025)", fill=True, ln=True)
        pdf.set_text_color(0, 0, 0)
        pdf.ln(4)
        pdf.set_font("Helvetica", "", 10)

        if "tendencia_linear" in resultados_tendencia:
            tl = resultados_tendencia["tendencia_linear"]
            pdf.multi_cell(0, 6,
                f"Regressao linear sobre serie anual (2016-2025):\n"
                f"  Variacao media por ano  : {tl['slope']:+.1f} casos/ano\n"
                f"  Coeficiente R2          : {tl['r2']:.4f}\n"
                f"  Significancia (p-value) : {tl['p']:.6f}\n"
                f"  Tendencia               : {'CRESCENTE' if tl['slope']>0 else 'DECRESCENTE'}\n"
            )
        pdf.ln(3)

        # Tabela de projeção 2026-2030
        pop_cg = float(df_cg["pop"].median()) if not df_cg.empty and "pop" in df_cg.columns else 942_140
        if "tendencia_linear" in resultados_tendencia:
            tl    = resultados_tendencia["tendencia_linear"]
            anos_p = range(2026, 2031)
            pdf.set_font("Helvetica", "B", 10)
            pdf.cell(0, 7, "Projecao 2026-2030:", ln=True)
            pdf.set_font("Helvetica", "", 9)
            for ap in anos_p:
                proj_v = max(tl["slope"] * ap + tl["intercept"], 0)
                taxa_p = taxa_inc(proj_v, pop_cg)
                pdf.cell(0, 6,
                    f"  {ap}: {fmt_num(int(proj_v))} casos estimados "
                    f"(taxa {taxa_p:.1f}/100k — {classificar_risco(taxa_p)})",
                    ln=True)

        # ── Boletim de Alerta Precoce ─────────────────────────────────────────
        pdf.add_page()
        pdf.set_fill_color(230, 126, 34)
        pdf.set_text_color(255, 255, 255)
        pdf.set_font("Helvetica", "B", 12)
        pdf.cell(0, 9, "BOLETIM DE ALERTA PRECOCE – PRÓXIMAS 4 SEMANAS",
                 fill=True, ln=True)
        pdf.set_text_color(0, 0, 0)
        pdf.ln(4)
        pdf.set_font("Helvetica", "", 10)

        if alerta and "semaforo" in alerta:
            for i, (data, (val, taxa, risco, nivel)) in enumerate(
                zip(alerta.get("datas_prev", []),
                    alerta.get("semaforo", []))
            ):
                cor_hex = NIVEL_CORES.get(nivel, "#999")
                pdf.multi_cell(0, 6,
                    f"  Semana {i+1} ({data.strftime('%d/%m/%Y')}): "
                    f"{fmt_num(int(val))} casos previstos | "
                    f"Taxa: {fmt_num(taxa,1)}/100k | "
                    f"Risco: {risco} | {NIVEL_NOMES.get(nivel,'?')}"
                )
        else:
            pdf.multi_cell(0, 6, "  Previsao nao disponivel nesta execucao.")

        # ── Ranking MS Completo ───────────────────────────────────────────────
        pdf.add_page()
        pdf.set_fill_color(39, 174, 96)
        pdf.set_text_color(255, 255, 255)
        pdf.set_font("Helvetica", "B", 12)
        pdf.cell(0, 9, "RANKING COMPLETO – MUNICÍPIOS MS", fill=True, ln=True)
        pdf.set_text_color(0, 0, 0)
        pdf.ln(3)
        pdf.set_font("Helvetica", "", 8)

        if not df_ms.empty:
            r_ms = df_ms.groupby("municipio_nome")["casos"].sum().reset_index()
            r_ms = r_ms.sort_values("casos", ascending=False).reset_index(drop=True)
            r_ms["pop"]  = r_ms["municipio_nome"].map(POP_MUNICIPIOS_MS).fillna(50_000)
            r_ms["taxa"] = r_ms.apply(lambda r: taxa_inc(r["casos"], r["pop"]), axis=1)
            for rank, (_, row) in enumerate(r_ms.head(50).iterrows(), 1):
                pdf.cell(0, 5,
                    f"  {rank:3d}. {row['municipio_nome']:<28} "
                    f"{fmt_num(int(row['casos'])):>10} casos | "
                    f"{fmt_num(row['taxa'],1):>8}/100k | "
                    f"{classificar_risco(row['taxa'])}",
                    ln=True)

        # ── Ranking Nacional ──────────────────────────────────────────────────
        pdf.add_page()
        pdf.set_fill_color(142, 68, 173)
        pdf.set_text_color(255, 255, 255)
        pdf.set_font("Helvetica", "B", 12)
        pdf.cell(0, 9, "RANKING NACIONAL – CAPITAIS BRASILEIRAS",
                 fill=True, ln=True)
        pdf.set_text_color(0, 0, 0)
        pdf.ln(3)
        pdf.set_font("Helvetica", "", 9)

        if not df_cap.empty:
            r_cap = df_cap.groupby("municipio_nome")["casos"].sum().reset_index()
            r_cap["pop"]  = r_cap["municipio_nome"].map(POP_CAPITAIS).fillna(1_000_000)
            r_cap["taxa"] = r_cap.apply(lambda r: taxa_inc(r["casos"], r["pop"]), axis=1)
            r_cap["UF"]   = r_cap["municipio_nome"].map(CAPITAIS_UF)
            r_cap = r_cap.sort_values("taxa", ascending=False).reset_index(drop=True)
            for rank, (_, row) in enumerate(r_cap.iterrows(), 1):
                destaque = " ◀ CG" if row["municipio_nome"] == "Campo Grande" else ""
                pdf.cell(0, 6,
                    f"  {rank:3d}. {row['municipio_nome']:<20} ({row.get('UF','?')}) "
                    f"{fmt_num(int(row['casos'])):>12} casos | "
                    f"{fmt_num(row['taxa'],1):>8}/100k{destaque}",
                    ln=True)

        pdf_path = OUTPUT_DIR / "pdf" / f"SIPREV_Complementar_{TIMESTAMP}.pdf"
        pdf.output(str(pdf_path))
        _inc("relatorios_gerados")
        log.info(f"  [PDF] {pdf_path.name}")
        return pdf_path

    except Exception as e:
        log.error(f"  PDF complementar falhou: {e}")
        traceback.print_exc()
        return None




In [99]:
# =============================================================================
# SEÇÃO 48 – XLSX AVANÇADO COM FORMATAÇÃO E GRÁFICOS EMBUTIDOS


In [100]:
# =============================================================================

def exportar_xlsx_avancado(df_cg: pd.DataFrame,
                            df_ms: pd.DataFrame,
                            resultados_tendencia: dict,
                            alerta: dict) -> Optional[Path]:
    """
    Gera XLSX avançado com formatação condicional, gráficos embutidos
    e abas adicionais para análises complementares.
    """
    print_section("XLSX AVANÇADO – FORMATAÇÃO E GRÁFICOS")

    if not HAS_OPENPYXL:
        log.warning("  openpyxl não disponível.")
        return None

    from openpyxl import Workbook
    from openpyxl.styles import (PatternFill, Font, Alignment,
                                   Border, Side, numbers)
    from openpyxl.formatting.rule import ColorScaleRule, DataBarRule
    from openpyxl.utils import get_column_letter
    from openpyxl.chart import BarChart, LineChart, Reference

    xlsx_path = OUTPUT_DIR / "dados" / f"SIPREV_Avancado_{TIMESTAMP}.xlsx"
    wb = Workbook()

    # ── Helpers de estilo ─────────────────────────────────────────────────────
    COR_HEADER = "C0392B"
    COR_SUBHEADER = "2980B9"
    COR_ROW_PAR = "F2F2F2"

    def _header_cell(ws, row, col, text, bold=True, bg=COR_HEADER, fg="FFFFFF"):
        c = ws.cell(row=row, column=col, value=text)
        c.font = Font(bold=bold, color=fg, size=10)
        c.fill = PatternFill("solid", fgColor=bg)
        c.alignment = Alignment(horizontal="center", vertical="center",
                                 wrap_text=True)
        c.border = Border(
            left=Side(style="thin"), right=Side(style="thin"),
            top=Side(style="thin"), bottom=Side(style="thin"),
        )
        return c

    def _data_cell(ws, row, col, value, fmt=None):
        c = ws.cell(row=row, column=col, value=value)
        if (row % 2) == 0:
            c.fill = PatternFill("solid", fgColor=COR_ROW_PAR)
        c.alignment = Alignment(horizontal="center", vertical="center")
        if fmt:
            c.number_format = fmt
        return c

    # ── Aba 1: Resumo Executivo ────────────────────────────────────────────────
    ws_res = wb.active
    ws_res.title = "Resumo Executivo"
    ws_res.column_dimensions["A"].width = 35
    ws_res.column_dimensions["B"].width = 22

    _header_cell(ws_res, 1, 1, "SIPREV – Resumo Executivo")
    _header_cell(ws_res, 1, 2, datetime.now().strftime("%d/%m/%Y %H:%M"))

    ws_res.merge_cells("A1:B1")
    ws_res.row_dimensions[1].height = 20

    indicadores_res = []
    if not df_cg.empty and "casos" in df_cg.columns:
        pop_cg = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942_140
        indicadores_res = [
            ("Total de casos CG (2016-2025)",    int(df_cg["casos"].sum())),
            ("Média semanal CG",                 round(float(df_cg["casos"].mean()), 1)),
            ("Pico semanal CG",                  int(df_cg["casos"].max())),
            ("Taxa histórica média (/100k)",
             round(taxa_inc(float(df_cg["casos"].mean()), pop_cg), 2)),
            ("Rt médio histórico",
             round(float(df_cg["Rt"].mean()), 4) if "Rt" in df_cg.columns else "N/A"),
            ("Semanas em Nível 4",
             int((df_cg["nivel"]==4).sum()) if "nivel" in df_cg.columns else 0),
            ("Municípios MS analisados",
             int(df_ms["municipio_nome"].nunique()) if not df_ms.empty else 0),
            ("Ambiente de execução",
             "Google Colab" if IS_COLAB else "Local"),
            ("TensorFlow",                       TF_VERSION),
            ("Timestamp",                        TIMESTAMP),
        ]

    for i, (ind, val) in enumerate(indicadores_res, start=2):
        _header_cell(ws_res, i, 1, ind, bg=COR_SUBHEADER)
        _data_cell(ws_res, i, 2, val)

    # ── Aba 2: Série Anual CG ─────────────────────────────────────────────────
    ws_anual = wb.create_sheet("CG Anual Detalhado")
    headers_a = ["Ano","Total Casos","Taxa/100k","Rt Médio",
                  "Nível Máx","Semanas N4","Semana Pico","Cresc. %"]
    for col, h in enumerate(headers_a, 1):
        _header_cell(ws_anual, 1, col, h)
        ws_anual.column_dimensions[get_column_letter(col)].width = 14

    if not df_cg.empty and "ANO" in df_cg.columns:
        pop_cg = float(df_cg["pop"].median()) if "pop" in df_cg.columns else 942_140
        totais_prev = None
        for row_i, ano in enumerate(sorted([int(a) for a in df_cg["ANO"].unique()
                                             if 2016 <= int(a) <= 2025]), start=2):
            sub = df_cg[df_cg["ANO"] == ano]
            tot = int(sub["casos"].sum())
            cresc = round((tot - totais_prev) / totais_prev * 100, 2) \
                    if totais_prev and totais_prev > 0 else None
            totais_prev = tot
            row_data = [
                ano, tot,
                round(taxa_inc(tot, pop_cg), 2),
                round(float(sub["Rt"].mean()), 4) if "Rt" in sub.columns else "",
                int(sub["nivel"].max()) if "nivel" in sub.columns else "",
                int((sub["nivel"]==4).sum()) if "nivel" in sub.columns else 0,
                int(sub.loc[sub["casos"].idxmax(), "SEMANA"])
                if "SEMANA" in sub.columns else "",
                cresc,
            ]
            for col_i, val in enumerate(row_data, 1):
                c = _data_cell(ws_anual, row_i, col_i, val)
                if col_i == 8 and val is not None:
                    c.font = Font(
                        color="C0392B" if (val or 0) > 0 else "27AE60",
                        bold=True, size=10
                    )

        # Formatação condicional (escala de cores) na coluna B (Total Casos)
        last_row = 2 + len([a for a in df_cg["ANO"].unique() if 2016 <= int(a) <= 2025]) - 1
        ws_anual.conditional_formatting.add(
            f"B2:B{last_row}",
            ColorScaleRule(
                start_type="min", start_color="27AE60",
                mid_type="percentile", mid_value=50, mid_color="F1C40F",
                end_type="max", end_color="C0392B",
            )
        )

        # Gráfico de barras embutido
        chart_a = BarChart()
        chart_a.title  = "Total de Casos Anuais – Campo Grande"
        chart_a.y_axis.title = "Casos"
        chart_a.x_axis.title = "Ano"
        chart_a.style  = 10
        chart_a.width  = 18
        chart_a.height = 10

        data_ref = Reference(ws_anual, min_col=2, min_row=1,
                              max_row=last_row)
        cats_ref = Reference(ws_anual, min_col=1, min_row=2,
                              max_row=last_row)
        chart_a.add_data(data_ref, titles_from_data=True)
        chart_a.set_categories(cats_ref)
        ws_anual.add_chart(chart_a, "J2")

    # ── Aba 3: Projeção 2026-2030 ─────────────────────────────────────────────
    ws_proj = wb.create_sheet("Projeção 2026-2030")
    headers_p = ["Ano","Casos Projetados","Taxa/100k","Risco Estimado",
                  "IC Inferior (−30%)","IC Superior (+30%)"]
    for col, h in enumerate(headers_p, 1):
        _header_cell(ws_proj, 1, col, h)
        ws_proj.column_dimensions[get_column_letter(col)].width = 18

    if "tendencia_linear" in resultados_tendencia:
        tl   = resultados_tendencia["tendencia_linear"]
        pop_proj = float(df_cg["pop"].median()) if not df_cg.empty and "pop" in df_cg.columns else 942_140
        for row_i, ano_p in enumerate(range(2026, 2031), start=2):
            proj_v = max(tl["slope"] * ano_p + tl["intercept"], 0)
            taxa_p = taxa_inc(proj_v, pop_proj)
            risco_p = classificar_risco(taxa_p)
            row_data = [
                ano_p, int(proj_v), round(taxa_p, 2), risco_p,
                int(proj_v * 0.7), int(proj_v * 1.3),
            ]
            for col_i, val in enumerate(row_data, 1):
                c = _data_cell(ws_proj, row_i, col_i, val)
                if col_i == 4:
                    cor_r = (COR_HEADER if risco_p in {"Alto","Muito Alto","Crítico"}
                             else "27AE60")
                    c.fill = PatternFill("solid", fgColor=cor_r)
                    c.font = Font(color="FFFFFF", bold=True, size=10)

    # ── Aba 4: Alerta Precoce ─────────────────────────────────────────────────
    ws_al = wb.create_sheet("Alerta Precoce")
    headers_al = ["Semana","Data","Casos Previstos","Taxa/100k","Risco","Nível Alerta"]
    for col, h in enumerate(headers_al, 1):
        _header_cell(ws_al, 1, col, h, bg="E67E22")
        ws_al.column_dimensions[get_column_letter(col)].width = 18

    if alerta and "semaforo" in alerta:
        pop_cg_al = float(df_cg["pop"].median()) if not df_cg.empty and "pop" in df_cg.columns else 942_140
        for row_i, (data, (val, taxa, risco, nivel)) in enumerate(
            zip(alerta.get("datas_prev", []),
                alerta.get("semaforo", [])), start=2
        ):
            row_data = [
                f"Semana {row_i-1}",
                data.strftime("%d/%m/%Y"),
                int(val), round(taxa, 2), risco,
                NIVEL_NOMES.get(nivel, "?"),
            ]
            for col_i, v in enumerate(row_data, 1):
                c = _data_cell(ws_al, row_i, col_i, v)
                nivel_cor = NIVEL_CORES.get(nivel, "#999999").replace("#","")
                if col_i >= 5:
                    c.fill = PatternFill("solid", fgColor=nivel_cor)
                    c.font = Font(color="FFFFFF", bold=True, size=10)

    wb.save(str(xlsx_path))
    log.info(f"  [XLSX] {xlsx_path.name}")
    return xlsx_path




In [101]:
# =============================================================================
# SEÇÃO 49 – RELATÓRIO DE COMPARAÇÃO EPIDEMIOLÓGICA REGIONAL


In [102]:
# =============================================================================

def comparacao_regional_detalhada(df_cap: pd.DataFrame) -> None:
    """
    Análise comparativa detalhada por região brasileira:
    - Sazonalidade por região
    - Distribuição de alertas por região
    - Tabela de posição de CG dentro do Centro-Oeste
    - Evolução temporal regionalizada
    """
    print_section("COMPARAÇÃO REGIONAL DETALHADA – CAPITAIS")

    if df_cap.empty or "municipio_nome" not in df_cap.columns:
        return

    df_cap_r = df_cap.copy()
    df_cap_r["UF"]     = df_cap_r["municipio_nome"].map(CAPITAIS_UF)
    df_cap_r["REGIAO"] = df_cap_r["UF"].map(REGIAO_UF)

    regioes = ["Norte","Nordeste","Centro-Oeste","Sudeste","Sul"]

    # ── Sazonalidade por região ────────────────────────────────────────────────
    print_sub("49.1 Sazonalidade por Região")
    if "MES" in df_cap_r.columns:
        fig, axes = plt.subplots(1, len(regioes), figsize=(20, 5), sharey=True)
        for ax, reg in zip(axes, regioes):
            sub_r = df_cap_r[df_cap_r["REGIAO"] == reg]
            if sub_r.empty:
                ax.set_title(reg, fontsize=9)
                continue
            perfil = sub_r.groupby("MES")["casos"].mean()
            cores_r = [COR_PRINCIPAL if m in {1,2,3,10,11,12}
                       else "#AED6F1" for m in perfil.index]
            ax.bar(perfil.index, perfil.values, color=cores_r)
            ax.set_title(reg, fontweight="bold", fontsize=9)
            ax.set_xticks(range(1, 13))
            ax.set_xticklabels([MESES_ABREV[m] for m in range(1, 13)],
                                fontsize=6, rotation=45)
            ax.set_xlabel("Mês")
        axes[0].set_ylabel("Casos Médios / Semana")
        plt.suptitle("Sazonalidade Mensal por Região Brasileira",
                     fontsize=13, fontweight="bold")
        salvar_fig("regional_sazonalidade_por_regiao")

    # ── Evolução anual por região ─────────────────────────────────────────────
    print_sub("49.2 Evolução Anual por Região")
    if "ANO" in df_cap_r.columns:
        evol_reg = df_cap_r.groupby(["ANO","REGIAO"])["casos"].sum().reset_index()
        fig, ax  = plt.subplots(figsize=(13, 5))
        palette  = plt.cm.get_cmap("tab10", len(regioes))
        for i, reg in enumerate(regioes):
            sub_r = evol_reg[evol_reg["REGIAO"] == reg].sort_values("ANO")
            if sub_r.empty:
                continue
            ax.plot(sub_r["ANO"].astype(int), sub_r["casos"],
                    marker="o", markersize=5, linewidth=2,
                    color=palette(i), label=reg)
        ax.set_title("Evolução Anual de Casos por Região (2016–2025)",
                     fontweight="bold")
        ax.set_xlabel("Ano")
        ax.set_ylabel("Casos")
        ax.legend(ncol=2, fontsize=8)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: fmt_num(int(max(x,0)))
        ))
        salvar_fig("regional_evolucao_anual_regioes")

    # ── Tabela comparativa Centro-Oeste ────────────────────────────────────────
    print_sub("49.3 Centro-Oeste – Detalhamento")
    caps_co = ["Campo Grande","Goiânia","Cuiabá","Brasília"]
    df_co   = df_cap_r[df_cap_r["municipio_nome"].isin(caps_co)]

    if not df_co.empty:
        rows_co = []
        for cap in caps_co:
            sub_c = df_co[df_co["municipio_nome"] == cap]
            if sub_c.empty:
                continue
            pop = POP_CAPITAIS.get(cap, 1_000_000)
            tot = int(sub_c["casos"].sum())
            taxa_t = taxa_inc(tot, pop)
            rt_m   = float(sub_c["Rt"].mean()) if "Rt" in sub_c.columns else 0
            nivel_m = int(sub_c["nivel"].mean()) if "nivel" in sub_c.columns else 0
            rows_co.append([
                cap,
                fmt_num(pop),
                fmt_num(tot),
                fmt_num(taxa_t, 1),
                fmt_num(rt_m, 3),
                fmt_num(nivel_m, 2),
                classificar_risco(taxa_t),
            ])
        tab_co = make_table(
            ["Capital","Pop.","Total Casos","Taxa/100k","Rt Médio",
             "Nível Médio","Risco"],
            rows_co, col_align=["l","r","r","r","r","r","l"]
        )
        log.info(f"\n{tab_co}")
        salvar_txt(tab_co, "regional_centro_oeste_detalhado",
                   "Centro-Oeste – Comparativo das Capitais")

    # ── Heatmap: Regiões × Anos ────────────────────────────────────────────────
    print_sub("49.4 Heatmap Regiões × Anos")
    if "ANO" in df_cap_r.columns:
        pivot_reg = df_cap_r.groupby(["REGIAO","ANO"])["casos"].sum().unstack(fill_value=0)
        pivot_reg = pivot_reg.loc[[r for r in regioes if r in pivot_reg.index]]
        fig, ax   = plt.subplots(figsize=(13, 5))
        sns.heatmap(pivot_reg, annot=True, fmt=".0f", cmap="YlOrRd",
                    linewidths=0.3, ax=ax, cbar_kws={"label":"Casos"},
                    annot_kws={"size": 8})
        ax.set_title("Casos por Região × Ano (Capitais Brasileiras)",
                     fontsize=13, fontweight="bold")
        ax.set_xlabel("Ano")
        ax.set_ylabel("Região")
        salvar_fig("regional_heatmap_regiao_ano")

    # ── Tabela síntese por região ─────────────────────────────────────────────
    rows_reg_s = []
    for reg in regioes:
        sub_r = df_cap_r[df_cap_r["REGIAO"] == reg]
        if sub_r.empty:
            continue
        caps_r  = sub_r["municipio_nome"].unique().tolist()
        tot_r   = int(sub_r["casos"].sum())
        pop_r   = sum(POP_CAPITAIS.get(c, 1_000_000) for c in caps_r)
        taxa_r  = taxa_inc(tot_r, pop_r)
        rows_reg_s.append([
            reg, len(caps_r), fmt_num(tot_r),
            fmt_num(taxa_r, 1), classificar_risco(taxa_r),
        ])
    tab_reg_s = make_table(
        ["Região","Capitais","Total Casos","Taxa/100k","Risco"],
        rows_reg_s, col_align=["l","c","r","r","l"]
    )
    log.info(f"\n{tab_reg_s}")
    salvar_txt(tab_reg_s, "regional_sintese_por_regiao",
               "Síntese Regional – Capitais Brasileiras")

    log.info("  Comparação regional detalhada concluída.")




In [103]:
# =============================================================================
# SEÇÃO 50 – ANÁLISE DE VARIÁVEIS CLIMÁTICAS AVANÇADA


In [104]:
# =============================================================================

def analise_climatica_avancada(df_cg: pd.DataFrame) -> dict:
    """
    Análise avançada do impacto climático na dengue:
    - Defasagem (lag) entre variáveis climáticas e casos
    - Correlação cruzada (CCF)
    - Regressão clima → casos com polinomiais
    - Identificação das condições climáticas críticas
    """
    print_section("ANÁLISE CLIMÁTICA AVANÇADA")
    resultados = {}

    if df_cg.empty or "casos" not in df_cg.columns:
        return resultados

    vars_clima = [c for c in ["tempmin","tempmed","tempmax",
                               "umidmin","umidmed","umidmax"]
                  if c in df_cg.columns]

    if not vars_clima:
        log.warning("  Variáveis climáticas não disponíveis.")
        return resultados

    df_c = df_cg.sort_values("data_SE").copy() if "data_SE" in df_cg.columns \
           else df_cg.copy()

    # ── 50.1 Correlação cruzada com lag ──────────────────────────────────────
    print_sub("50.1 Correlação Cruzada (CCF) – Lag 0 a 12 semanas")
    casos = df_c["casos"].fillna(0).values
    rows_ccf = []

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for i, var in enumerate(vars_clima[:6]):
        clima = df_c[var].fillna(df_c[var].median()).values
        lags  = range(0, 13)
        ccf_vals = []
        for lag in lags:
            if lag == 0:
                c_arr, k_arr = casos, clima
            else:
                c_arr, k_arr = casos[lag:], clima[:-lag]
            if len(c_arr) > 10:
                r, _ = pearsonr(k_arr, c_arr)
            else:
                r = 0
            ccf_vals.append(r)

        best_lag = int(np.argmax(np.abs(ccf_vals)))
        best_r   = ccf_vals[best_lag]
        rows_ccf.append([var, best_lag, fmt_num(best_r, 4)])

        ax = axes[i]
        cores_ccf = [COR_PRINCIPAL if abs(v) == max(abs(c) for c in ccf_vals)
                     else COR_SECUNDARIA for v in ccf_vals]
        ax.bar(list(lags), ccf_vals, color=cores_ccf)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.axvline(best_lag, color="red", linestyle="--", linewidth=1.5,
                   label=f"Melhor lag={best_lag} (r={best_r:.3f})")
        ax.set_title(f"CCF: {var} → Casos", fontsize=9, fontweight="bold")
        ax.set_xlabel("Lag (semanas)")
        ax.set_ylabel("Correlação de Pearson")
        ax.legend(fontsize=7)
        ax.set_xticks(list(lags))

    plt.suptitle("Correlação Cruzada: Variáveis Climáticas → Casos – CG/MS",
                 fontsize=13, fontweight="bold")
    salvar_fig("clima_ccf_lag_variaveis_cg")

    tab_ccf = make_table(
        ["Variável","Melhor Lag (sem)","Correlação (r)"],
        rows_ccf, col_align=["l","c","r"]
    )
    log.info(f"\n{tab_ccf}")
    salvar_txt(tab_ccf, "clima_ccf_resultados",
               "Correlação Cruzada – Variáveis Climáticas")
    resultados["ccf"] = rows_ccf

    # ── 50.2 Condições climáticas de risco ────────────────────────────────────
    print_sub("50.2 Condições Climáticas Críticas")
    if "tempmed" in df_c.columns and "umidmed" in df_c.columns:
        df_c2 = df_c[["tempmed","umidmed","casos","nivel"]].dropna()

        # Quartis de temperatura e umidade
        q75_temp = df_c2["tempmed"].quantile(0.75)
        q75_umid = df_c2["umidmed"].quantile(0.75)
        q25_temp = df_c2["tempmed"].quantile(0.25)
        q25_umid = df_c2["umidmed"].quantile(0.25)

        cond_critica = (df_c2["tempmed"] >= q75_temp) & (df_c2["umidmed"] >= q75_umid)
        cond_baixa   = (df_c2["tempmed"] <= q25_temp) | (df_c2["umidmed"] <= q25_umid)

        m_critica = df_c2[cond_critica]["casos"].mean()
        m_baixa   = df_c2[cond_baixa]["casos"].mean()
        m_total   = df_c2["casos"].mean()

        log.info(f"  Média casos – Condições críticas (T≥P75 e U≥P75): {m_critica:.1f}")
        log.info(f"  Média casos – Condições favoráveis (T≤P25 ou U≤P25): {m_baixa:.1f}")
        log.info(f"  Média geral: {m_total:.1f}")

        rows_cond = [
            ["Condições Críticas (T≥Q75 e U≥Q75)",    fmt_num(m_critica, 1),
             f"{int(cond_critica.sum())} semanas"],
            ["Condições Favoráveis (T≤Q25 ou U≤Q25)", fmt_num(m_baixa, 1),
             f"{int(cond_baixa.sum())} semanas"],
            ["Média Geral",                            fmt_num(m_total, 1),
             f"{len(df_c2)} semanas"],
            ["Razão Crítica/Favorável",
             fmt_num(m_critica/m_baixa if m_baixa > 0 else 0, 2) + "x", ""],
        ]
        tab_cond = make_table(
            ["Condição","Média de Casos","Semanas"],
            rows_cond, col_align=["l","r","l"]
        )
        log.info(f"\n{tab_cond}")
        salvar_txt(tab_cond, "clima_condicoes_criticas",
                   "Condições Climáticas Críticas – Campo Grande")

        # Scatter temperatura × umidade colorido por casos
        fig, ax = plt.subplots(figsize=(10, 7))
        sc = ax.scatter(
            df_c2["tempmed"], df_c2["umidmed"],
            c=df_c2["casos"], cmap="YlOrRd",
            s=20, alpha=0.6, edgecolors="none"
        )
        plt.colorbar(sc, ax=ax, label="Casos / Semana")
        ax.axvline(q75_temp, color="red", linestyle="--", linewidth=1,
                   label=f"Q75 Temp={q75_temp:.1f}°C")
        ax.axhline(q75_umid, color="blue", linestyle="--", linewidth=1,
                   label=f"Q75 Umid={q75_umid:.1f}%")
        ax.set_title("Temperatura × Umidade (colorido por Casos) – CG/MS",
                     fontweight="bold")
        ax.set_xlabel("Temperatura Média (°C)")
        ax.set_ylabel("Umidade Relativa Média (%)")
        ax.legend(fontsize=8)
        salvar_fig("clima_scatter_temp_umid_casos_cg")

    log.info("  Análise climática avançada concluída.")
    return resultados




In [105]:
# =============================================================================
# SEÇÃO 51 – RELATÓRIO FINAL EXPANDIDO (TXT / LOG)


In [106]:
# =============================================================================

def relatorio_final_expandido(df_cg: pd.DataFrame,
                               df_ms: pd.DataFrame,
                               df_cap: pd.DataFrame,
                               resultados_tendencia: dict,
                               alerta: dict,
                               bootstrap_res: dict) -> None:
    """
    Relatório textual final com todas as seções integradas,
    indicadores de execução e resumo dos principais resultados.
    """
    print_section("RELATÓRIO FINAL EXPANDIDO")

    pop_cg = float(df_cg["pop"].median()) if not df_cg.empty and "pop" in df_cg.columns else 942_140

    linhas = [
        "=" * 78,
        "SIPREV – RELATÓRIO FINAL EXPANDIDO",
        f"Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}",
        f"Timestamp : {TIMESTAMP}",
        f"Ambiente  : {'Google Colab' if IS_COLAB else 'Máquina Local'}",
        "=" * 78, "",
    ]

    # Seção 1: Campo Grande
    if not df_cg.empty and "casos" in df_cg.columns:
        total_cg  = int(df_cg["casos"].sum())
        media_cg  = float(df_cg["casos"].mean())
        max_cg    = int(df_cg["casos"].max())
        taxa_med  = taxa_inc(media_cg, pop_cg)
        rt_med    = float(df_cg["Rt"].mean()) if "Rt" in df_cg.columns else 0
        n_nivel4  = int((df_cg["nivel"]==4).sum()) if "nivel" in df_cg.columns else 0

        linhas += [
            "1. CAMPO GRANDE / MATO GROSSO DO SUL",
            "-" * 60,
            f"   Total de casos (2016-2025)         : {fmt_num(total_cg)}",
            f"   Média semanal                       : {media_cg:.1f} casos/semana",
            f"   Pico semanal absoluto               : {fmt_num(max_cg)} casos",
            f"   Taxa incidência média (/100k)        : {taxa_med:.2f}",
            f"   Rt médio histórico                  : {rt_med:.4f}",
            f"   Semanas em Nível 4 (Vermelho)        : {fmt_num(n_nivel4)}",
        ]
        if "ANO" in df_cg.columns:
            por_ano = df_cg.groupby("ANO")["casos"].sum()
            linhas += [
                f"   Pior ano epidêmico                  : {int(por_ano.idxmax())} "
                f"({fmt_num(int(por_ano.max()))} casos)",
                f"   Melhor ano epidêmico                : {int(por_ano.idxmin())} "
                f"({fmt_num(int(por_ano.min()))} casos)",
            ]

        # Bootstrap IC
        if "media_semanal" in bootstrap_res:
            m, lo, hi = bootstrap_res["media_semanal"]
            linhas.append(
                f"   IC 95% Bootstrap (média sem.)       : "
                f"[{lo:.1f} – {hi:.1f}] casos"
            )
        linhas.append("")

    # Seção 2: Tendência
    if "tendencia_linear" in resultados_tendencia:
        tl = resultados_tendencia["tendencia_linear"]
        linhas += [
            "2. ANÁLISE DE TENDÊNCIA",
            "-" * 60,
            f"   Variação linear                     : {tl['slope']:+.2f} casos/ano",
            f"   R²                                  : {tl['r2']:.4f}",
            f"   Tendência                           : "
            f"{'CRESCENTE ↑' if tl['slope'] > 0 else 'DECRESCENTE ↓'}",
            "",
        ]

    # Seção 3: Alerta precoce
    if alerta and "semaforo" in alerta:
        linhas += [
            "3. BOLETIM DE ALERTA PRECOCE – PRÓXIMAS 4 SEMANAS",
            "-" * 60,
        ]
        for i, (data, (val, taxa, risco, nivel)) in enumerate(
            zip(alerta.get("datas_prev",[]), alerta.get("semaforo",[])), 1
        ):
            linhas.append(
                f"   Semana {i} ({data.strftime('%d/%m/%Y')}): "
                f"{fmt_num(int(val))} casos | {taxa:.1f}/100k | {risco}"
            )
        linhas.append("")

    # Seção 4: Resumo MS
    if not df_ms.empty:
        top5_ms = df_ms.groupby("municipio_nome")["casos"].sum().nlargest(5)
        linhas += [
            "4. MATO GROSSO DO SUL – TOP 5 MUNICÍPIOS",
            "-" * 60,
        ]
        for rank, (mun, casos) in enumerate(top5_ms.items(), 1):
            pop = POP_MUNICIPIOS_MS.get(mun, 50_000)
            taxa_t = taxa_inc(casos, pop)
            linhas.append(
                f"   {rank}. {mun:<28}: {fmt_num(int(casos))} casos | "
                f"{taxa_t:.1f}/100k"
            )
        linhas.append("")

    # Seção 5: Nacional
    if not df_cap.empty:
        top5_cap = df_cap.groupby("municipio_nome")["casos"].sum().nlargest(5)
        linhas += [
            "5. RANKING NACIONAL – TOP 5 CAPITAIS",
            "-" * 60,
        ]
        for rank, (cap, casos) in enumerate(top5_cap.items(), 1):
            pop = POP_CAPITAIS.get(cap, 1_000_000)
            taxa_t = taxa_inc(casos, pop)
            dest   = " ← Campo Grande" if cap == "Campo Grande" else ""
            linhas.append(
                f"   {rank}. {cap:<22}: {fmt_num(int(casos))} casos | "
                f"{taxa_t:.1f}/100k{dest}"
            )
        linhas.append("")

    # Seção 6: Execução
    linhas += [
        "6. ESTATÍSTICAS DE EXECUÇÃO",
        "-" * 60,
        f"   Registros lidos      : {fmt_num(_stats['registros_lidos'])}",
        f"   Registros válidos    : {fmt_num(_stats['registros_validos'])}",
        f"   Gráficos gerados     : {_stats['graficos_gerados']}",
        f"   Mapas gerados        : {_stats['mapas_gerados']}",
        f"   Dashboards gerados   : {_stats['dashboards_gerados']}",
        f"   Modelos treinados    : {_stats['modelos_treinados']}",
        f"   Relatórios gerados   : {_stats['relatorios_gerados']}",
        f"   Diretório de saída   : {OUTPUT_DIR}",
        "",
        "=" * 78,
        "FIM DO RELATÓRIO SIPREV",
        "=" * 78,
    ]

    conteudo = "\n".join(linhas)
    salvar_txt(conteudo, f"relatorio_final_expandido_{TIMESTAMP}",
               "Relatório Final Expandido SIPREV")
    salvar_log_tabela(conteudo, f"relatorio_final_expandido_{TIMESTAMP}",
                      "Relatório Final")
    log.info("  Relatório final expandido concluído.")




In [107]:
# =============================================================================
# SEÇÃO 52 – MAIN EXPANDIDO (INTEGRA TODAS AS SEÇÕES)


In [108]:
# =============================================================================

def main():
    """
    Pipeline principal do SIPREV – versão expandida.
    Orquestra todas as 52 seções sequencialmente.
    """
    t_inicio = datetime.now()
    _banner()

    # ── BLOCO A: DADOS ────────────────────────────────────────────────────────
    df_cg, df_ms, df_cap = carregar_tudo()

    for nome, df in [("Campo Grande",   df_cg),
                     ("MS-Municípios",  df_ms),
                     ("Capitais-Brasil",df_cap)]:
        if not df.empty:
            relatorio_qualidade(df, nome)

    # ── BLOCO B: EDA ──────────────────────────────────────────────────────────
    eda_visao_geral(df_cg, df_ms, df_cap)
    resultados_cg   = analise_campo_grande(df_cg, df_ms)
    df_mun_ano      = analise_municipal_ms(df_ms)
    rank_capitais   = analise_capitais(df_cap)
    rankings_consolidados(df_cg, df_ms, df_cap)

    # ── BLOCO C: ANÁLISES AVANÇADAS ──────────────────────────────────────────
    df_cg_feat, df_ms_feat = engenharia_features(df_cg, df_ms)
    resultados_estat       = testes_estatisticos(df_cg, df_ms, df_cap)
    resultados_tendencia   = analise_tendencia(df_cg)
    df_indice_risco        = indice_risco_municipal(df_ms)
    resultados_sazon       = analise_sazonalidade_avancada(df_cg, df_cap)
    resultados_surtos      = analise_surtos(df_cg)
    resultados_corr_esp    = correlacao_espacial_ms(df_ms)
    bootstrap_res          = bootstrap_intervalos(df_cg)
    resultados_clima       = analise_climatica_avancada(df_cg)
    relatorio_por_ano(df_cg, df_ms)
    comparacao_regional_detalhada(df_cap)

    # ── BLOCO D: MACHINE LEARNING ─────────────────────────────────────────────
    df_clusters    = ml_clusterizacao(df_ms)
    resultados_ml  = ml_classificacao_risco(df_cg, df_ms)
    resultados_reg = ml_regressao_casos(df_cg)
    resultados_reg_av = ml_regressao_avancada(df_cg)
    resultados_cv  = validacao_cruzada_temporal(df_cg)
    df_anomalias   = deteccao_anomalias(df_cg)

    # ── BLOCO E: SÉRIES TEMPORAIS ─────────────────────────────────────────────
    resultados_ts  = series_temporais(df_cg)

    # ── BLOCO F: DEEP LEARNING ────────────────────────────────────────────────
    resultados_dl  = deep_learning_lstm_gru(df_cg)
    resultados_nn  = redes_neurais_avancadas(df_cg, df_ms)

    # ── BLOCO G: ALERTA PRECOCE ───────────────────────────────────────────────
    alerta = sistema_alerta_precoce(df_cg, resultados_ts, resultados_dl)

    # ── BLOCO H: VISUALIZAÇÕES ────────────────────────────────────────────────
    gerar_mapas(df_cg, df_ms, df_cap)
    gerar_dashboards(df_cg, df_ms, df_cap)
    gerar_dashboards_avancados(df_cg, df_ms, df_cap, alerta)
    fichas_municipais(df_ms)

    # ── BLOCO I: EXPORTAÇÕES ──────────────────────────────────────────────────
    gerar_relatorio_pdf(df_cg, df_ms, df_cap)
    complementar_pdf(df_cg, df_ms, df_cap, resultados_tendencia, alerta)
    exportar_xlsx(df_cg, df_ms, df_cap)
    exportar_xlsx_avancado(df_cg, df_ms, resultados_tendencia, alerta)
    exportar_parquet_json(df_cg, df_ms, df_cap)

    # ── BLOCO J: RELATÓRIOS TEXTUAIS ─────────────────────────────────────────
    relatorio_txt_consolidado(df_cg, df_ms, df_cap)
    relatorio_modelos(resultados_ml, resultados_ts, resultados_dl)
    relatorio_final_expandido(df_cg, df_ms, df_cap,
                               resultados_tendencia, alerta, bootstrap_res)

    # ── BLOCO K: PERSISTÊNCIA E ENCERRAMENTO ─────────────────────────────────
    salvar_modelos(resultados_ml, resultados_reg, resultados_dl)
    sumario_final(t_inicio)

    # ── BLOCO L: ANÁLISES COMPLEMENTARES (Seções 53–60) ─────────────────────
    try:
        _resultados_bl = _executar_bloco_l(
            df_cg, df_ms, df_cap,
            resultados_ml=resultados_ml,
            resultados_reg=resultados_reg,
            resultados_dl=resultados_dl,
            resultados_ts=resultados_ts,
            alerta=alerta,
        )
    except Exception as _e_bl:
        log_warn(f"Bloco L ignorado: {_e_bl}")
        _resultados_bl = {}

    # -- BLOCO M: Validacao, CCF e Metadados (Secoes 61-63)
    try:
        _executar_bloco_m(
            df_cg, df_ms, df_cap,
            resultados_ml=resultados_ml,
            resultados_ts=resultados_ts,
            resultados_dl=resultados_dl,
            alerta=alerta,
        )
    except Exception as _e_bm:
        log_warn(f"Bloco M ignorado: {_e_bm}")

    compactar_resultados()

    return {
        "df_cg":               df_cg,
        "df_ms":               df_ms,
        "df_cap":              df_cap,
        "df_cg_feat":          df_cg_feat,
        "df_ms_feat":          df_ms_feat,
        "resultados_cg":       resultados_cg,
        "resultados_ml":       resultados_ml,
        "resultados_reg":      resultados_reg,
        "resultados_reg_av":   resultados_reg_av,
        "resultados_cv":       resultados_cv,
        "resultados_ts":       resultados_ts,
        "resultados_dl":       resultados_dl,
        "resultados_nn":       resultados_nn,
        "resultados_estat":    resultados_estat,
        "resultados_tendencia":resultados_tendencia,
        "resultados_sazon":    resultados_sazon,
        "resultados_surtos":   resultados_surtos,
        "resultados_clima":    resultados_clima,
        "df_clusters":         df_clusters,
        "df_anomalias":        df_anomalias,
        "df_indice_risco":     df_indice_risco,
        "alerta":              alerta,
        "bootstrap_res":       bootstrap_res,
    }





In [109]:
# =============================================================================
# SIPREV — PARTE 8: Seções 53–60 — Análises Complementares e Finalização


In [110]:
# =============================================================================



In [111]:
# =============================================================================
# SEÇÃO 53: Análise STL e Decomposição Espectral Avançada


In [112]:
# =============================================================================

def analise_stl_espectral(df_cg: pd.DataFrame) -> dict:
    """
    Decomposição STL (Seasonal-Trend-Loess) e análise espectral
    completa para a série temporal de dengue em Campo Grande.
    Inclui periodograma, wavelets simplificados e análise de ruído.
    """
    resultado = {}
    log_section("53 — STL e Análise Espectral Avançada")

    if df_cg.empty:
        log_warn("Seção 53: df_cg vazio — ignorado.")
        return resultado

    try:
        serie = df_cg.set_index("data_SE")["casos"].asfreq("W-SUN").fillna(0)
        n = len(serie)
        resultado["n_semanas"] = n

        # ── Decomposição STL via statsmodels (se disponível)
        if HAS_STATSMODELS:
            from statsmodels.tsa.seasonal import STL
            stl = STL(serie, period=52, robust=True)
            res_stl = stl.fit()

            trend_vals    = res_stl.trend.values
            seasonal_vals = res_stl.seasonal.values
            resid_vals    = res_stl.resid.values

            var_total    = np.var(serie.values)
            var_trend    = np.var(trend_vals)
            var_seasonal = np.var(seasonal_vals)
            var_resid    = np.var(resid_vals)

            resultado["stl_var_trend_pct"]    = round(100 * var_trend    / var_total, 2) if var_total else 0
            resultado["stl_var_seasonal_pct"] = round(100 * var_seasonal / var_total, 2) if var_total else 0
            resultado["stl_var_resid_pct"]    = round(100 * var_resid    / var_total, 2) if var_total else 0

            log_info(f"  STL — Tendência: {resultado['stl_var_trend_pct']:.1f}% | "
                     f"Sazonalidade: {resultado['stl_var_seasonal_pct']:.1f}% | "
                     f"Resíduo: {resultado['stl_var_resid_pct']:.1f}%")

            # Gráfico STL
            fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
            fig.suptitle("Decomposição STL — Dengue Campo Grande/MS", fontsize=14, fontweight="bold")
            axes[0].plot(serie.index, serie.values, color="#2196F3", lw=1.2)
            axes[0].set_ylabel("Observado")
            axes[1].plot(serie.index, trend_vals, color="#E91E63", lw=1.5)
            axes[1].set_ylabel("Tendência")
            axes[2].plot(serie.index, seasonal_vals, color="#4CAF50", lw=1.0)
            axes[2].set_ylabel("Sazonalidade")
            axes[3].plot(serie.index, resid_vals, color="#FF9800", lw=0.8, alpha=0.7)
            axes[3].axhline(0, color="black", lw=0.5, ls="--")
            axes[3].set_ylabel("Resíduo")
            axes[3].set_xlabel("Semana Epidemiológica")
            plt.tight_layout()
            _salvar_figura(fig, "stl_decomposicao")
            plt.close(fig)

            # Periodograma dos resíduos (FFT)
            fft_vals = np.fft.rfft(resid_vals)
            freqs    = np.fft.rfftfreq(n, d=1)  # ciclos/semana
            power    = np.abs(fft_vals) ** 2
            # períodos dominantes (semanas)
            periods  = 1.0 / (freqs[1:] + 1e-10)
            idx_sort = np.argsort(power[1:])[::-1][:5]
            dom_periods = [round(periods[i], 1) for i in idx_sort]
            resultado["periodograma_resid_top5_semanas"] = dom_periods
            log_info(f"  Períodos dominantes no resíduo: {dom_periods}")

            fig2, ax2 = plt.subplots(figsize=(12, 4))
            ax2.semilogy(freqs[1:], power[1:], color="#673AB7", lw=1.0, alpha=0.8)
            ax2.set_xlabel("Frequência (ciclos/semana)")
            ax2.set_ylabel("Potência Espectral (log)")
            ax2.set_title("Periodograma dos Resíduos STL — Campo Grande/MS")
            ax2.grid(True, alpha=0.3)
            plt.tight_layout()
            _salvar_figura(fig2, "periodograma_residuos_stl")
            plt.close(fig2)

        # ── Periodograma da série bruta (FFT)
        y = serie.values - serie.mean()
        fft_raw = np.fft.rfft(y)
        freqs_r = np.fft.rfftfreq(n, d=1)
        power_r = np.abs(fft_raw) ** 2
        periods_r = 1.0 / (freqs_r[1:] + 1e-10)
        idx_r = np.argsort(power_r[1:])[::-1][:10]
        resultado["periodograma_top10_semanas"] = [round(periods_r[i], 1) for i in idx_r]
        log_info(f"  Top-10 períodos série bruta: {resultado['periodograma_top10_semanas']}")

        fig3, ax3 = plt.subplots(figsize=(12, 4))
        ax3.semilogy(freqs_r[1:], power_r[1:], color="#009688", lw=1.0)
        ax3.axvline(1/52, color="red",    ls="--", lw=1.0, label="Ciclo anual (52 sem)")
        ax3.axvline(1/26, color="orange", ls="--", lw=1.0, label="Ciclo semestral (26 sem)")
        ax3.set_xlabel("Frequência (ciclos/semana)")
        ax3.set_ylabel("Potência Espectral (log)")
        ax3.set_title("Periodograma — Série de Casos Dengue — Campo Grande/MS")
        ax3.legend(fontsize=9)
        ax3.grid(True, alpha=0.3)
        plt.tight_layout()
        _salvar_figura(fig3, "periodograma_serie_bruta")
        plt.close(fig3)

        # ── Análise de ruído (distribuição dos resíduos STL)
        if HAS_STATSMODELS and "stl_var_resid_pct" in resultado:
            fig4, axes4 = plt.subplots(1, 2, figsize=(12, 4))
            axes4[0].hist(resid_vals, bins=40, color="#FF5722", edgecolor="white", alpha=0.8)
            axes4[0].set_title("Distribuição dos Resíduos STL")
            axes4[0].set_xlabel("Resíduo"); axes4[0].set_ylabel("Frequência")
            from scipy import stats as sp_stats
            (osm, osr), (slope, intercept, r) = sp_stats.probplot(resid_vals, dist="norm")
            axes4[1].scatter(osm, osr, s=8, color="#3F51B5", alpha=0.6)
            axes4[1].plot([osm[0], osm[-1]],
                          [slope * osm[0] + intercept, slope * osm[-1] + intercept],
                          "r-", lw=1.5)
            axes4[1].set_title("Q-Q Plot dos Resíduos STL")
            axes4[1].set_xlabel("Quantis Teóricos"); axes4[1].set_ylabel("Quantis Observados")
            plt.tight_layout()
            _salvar_figura(fig4, "residuos_stl_qqplot")
            plt.close(fig4)

        log_ok("Seção 53 concluída.")

    except Exception as exc:
        log_warn(f"Seção 53 erro: {exc}")

    return resultado




In [113]:
# =============================================================================
# SEÇÃO 54: Análise de Clusters Temporais (K-Means por Semana Epidemiológica)


In [114]:
# =============================================================================

def clusters_temporais_semanais(df_cg: pd.DataFrame, df_ms: pd.DataFrame) -> dict:
    """
    Agrupa semanas epidemiológicas em padrões comportamentais usando K-Means.
    Identifica clusters de semanas de alto risco, transição e baixa endemia.
    """
    resultado = {}
    log_section("54 — Clusters Temporais Semanais")

    if df_cg.empty or not HAS_SKLEARN:
        log_warn("Seção 54: dados insuficientes ou sklearn ausente.")
        return resultado

    try:
        from sklearn.cluster import KMeans
        from sklearn.preprocessing import StandardScaler
        from sklearn.metrics import silhouette_score

        # Features por semana epidemiológica (1-52)
        df_tmp = df_cg.copy()
        df_tmp["semana"] = df_tmp["SE"].astype(str).str[-2:].astype(int)
        feat = df_tmp.groupby("semana").agg(
            casos_mean   = ("casos",     "mean"),
            casos_std    = ("casos",     "std"),
            inc_mean     = ("p_inc100k", "mean"),
            rt_mean      = ("Rt",        "mean"),
            nivel_mean   = ("nivel",     "mean"),
            casos_max    = ("casos",     "max"),
        ).fillna(0).reset_index()

        X = feat[["casos_mean", "casos_std", "inc_mean", "rt_mean", "nivel_mean", "casos_max"]].values
        sc = StandardScaler()
        Xs = sc.fit_transform(X)

        # Escolha de k via silhouette
        sil_scores = {}
        for k in range(2, 7):
            km = KMeans(n_clusters=k, random_state=42, n_init=10)
            labels = km.fit_predict(Xs)
            sil_scores[k] = round(silhouette_score(Xs, labels), 4)

        best_k = max(sil_scores, key=sil_scores.get)
        resultado["silhouette_scores"] = sil_scores
        resultado["best_k"] = best_k
        log_info(f"  Melhor k={best_k} (silhouette={sil_scores[best_k]:.4f})")

        km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
        feat["cluster"] = km_final.fit_predict(Xs)

        # Ordenar clusters por casos_mean
        order = feat.groupby("cluster")["casos_mean"].mean().sort_values(ascending=False)
        label_map = {old: new for new, (old, _) in enumerate(order.items())}
        feat["cluster_ord"] = feat["cluster"].map(label_map)
        resultado["semanas_por_cluster"] = feat.groupby("cluster_ord")["semana"].apply(list).to_dict()

        cluster_names = {0: "Alto Risco", 1: "Transição", 2: "Baixa Endemia",
                         3: "Muito Baixo", 4: "Mínimo"}
        feat["cluster_nome"] = feat["cluster_ord"].map(cluster_names)

        # Gráfico circular das semanas por cluster
        colors_cl = ["#F44336", "#FF9800", "#FFC107", "#4CAF50", "#2196F3"]
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle("Clusters Temporais — Semanas Epidemiológicas / Campo Grande", fontsize=13, fontweight="bold")

        # Polar plot
        theta = np.linspace(0, 2 * np.pi, 53)
        ax_pol = axes[0]
        for _, row in feat.iterrows():
            sem = int(row["semana"])
            cl  = int(row["cluster_ord"])
            ang = theta[sem - 1]
            r   = row["casos_mean"] / (feat["casos_mean"].max() + 1e-6)
            col = colors_cl[cl % len(colors_cl)]
            ax_pol.barh(r, width=0.1, left=ang, height=0.04, color=col, alpha=0.8)
        ax_pol.set_aspect("equal")
        ax_pol.set_title("Distribuição Circular de Risco")
        ax_pol.axis("off")

        # Scatter semanas × casos_mean colorido por cluster
        scatter_data = feat.copy()
        for cl_id in sorted(scatter_data["cluster_ord"].unique()):
            sub = scatter_data[scatter_data["cluster_ord"] == cl_id]
            col = colors_cl[cl_id % len(colors_cl)]
            nome = cluster_names.get(cl_id, f"Cluster {cl_id}")
            axes[1].scatter(sub["semana"], sub["casos_mean"], c=col, label=nome, s=80, zorder=3)
        axes[1].set_xlabel("Semana Epidemiológica")
        axes[1].set_ylabel("Média de Casos")
        axes[1].set_title("Clusters por Semana Epidemiológica")
        axes[1].legend(fontsize=9)
        axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        _salvar_figura(fig, "clusters_temporais_semanais")
        plt.close(fig)

        # Heatmap cluster × ano
        df_tmp["ano"] = df_tmp["data_SE"].dt.year
        df_merged = df_tmp.merge(feat[["semana", "cluster_ord", "cluster_nome"]], on="semana", how="left")
        pivot_cl = df_merged.pivot_table(index="semana", columns="ano", values="casos", aggfunc="mean").fillna(0)

        fig2, ax2 = plt.subplots(figsize=(14, 8))
        sns.heatmap(pivot_cl, cmap="YlOrRd", ax=ax2, linewidths=0.1, linecolor="white",
                    cbar_kws={"label": "Média Casos"})
        ax2.set_title("Heatmap Semana × Ano — Casos Dengue — Campo Grande/MS", fontsize=12)
        ax2.set_xlabel("Ano"); ax2.set_ylabel("Semana Epidemiológica")
        plt.tight_layout()
        _salvar_figura(fig2, "heatmap_semana_ano_clusters")
        plt.close(fig2)

        log_ok("Seção 54 concluída.")

    except Exception as exc:
        log_warn(f"Seção 54 erro: {exc}")

    return resultado




In [115]:
# =============================================================================
# SEÇÃO 55: Análise de Impacto Socioeconômico Estimado


In [116]:
# =============================================================================

def analise_impacto_socioeconomico(df_cg: pd.DataFrame, df_ms: pd.DataFrame) -> dict:
    """
    Estima impacto socioeconômico da dengue em Campo Grande e Mato Grosso do Sul.
    Usa parâmetros da literatura: custo por caso ambulatorial, hospitalar, óbito estimado.
    Calcula anos de vida perdidos ajustados por incapacidade (AVAI simplificado).
    """
    resultado = {}
    log_section("55 — Impacto Socioeconômico Estimado")

    # Parâmetros baseados em literatura brasileira (Siqueira et al., 2022; PAHO, 2023)
    CUSTO_AMB    = 1_200.0   # R$ por caso ambulatorial (2024)
    CUSTO_HOSP   = 8_500.0   # R$ por caso hospitalizado
    TAXA_HOSP    = 0.054      # 5,4% dos casos confirmados hospitalizam
    CUSTO_OBITO  = 180_000.0  # R$ (salário futuro perdido + custos funerários)
    TAXA_OBITO   = 0.000_8    # 0,08% letalidade
    AVAI_POR_CASO= 0.018      # anos de vida ajustados por incapacidade por caso
    SALARIO_MEDIO= 3_200.0    # R$/mês

    try:
        if df_cg.empty:
            raise ValueError("df_cg vazio")

        anos = sorted(df_cg["data_SE"].dt.year.unique())
        rows = []
        for ano in anos:
            sub = df_cg[df_cg["data_SE"].dt.year == ano]
            casos_total = int(sub["casos"].sum())
            casos_hosp  = int(casos_total * TAXA_HOSP)
            obitos_est  = round(casos_total * TAXA_OBITO, 2)
            custo_amb   = round((casos_total - casos_hosp) * CUSTO_AMB)
            custo_hosp  = round(casos_hosp * CUSTO_HOSP)
            custo_obito = round(obitos_est * CUSTO_OBITO)
            custo_total = custo_amb + custo_hosp + custo_obito
            avai        = round(casos_total * AVAI_POR_CASO, 1)
            dias_perdidos = round(casos_total * 7)  # ~7 dias afastamento médio
            perda_prod  = round(dias_perdidos / 22 * SALARIO_MEDIO)  # dias úteis/mês
            rows.append({
                "Ano": ano, "Casos": casos_total, "Hosp_Est": casos_hosp,
                "Obitos_Est": obitos_est,
                "Custo_Amb_R$": custo_amb, "Custo_Hosp_R$": custo_hosp,
                "Custo_Obito_R$": custo_obito, "Custo_Total_R$": custo_total,
                "AVAI": avai, "Dias_Perdidos": dias_perdidos, "Perda_Prod_R$": perda_prod,
            })

        df_imp = pd.DataFrame(rows)
        resultado["df_impacto_cg"] = df_imp
        resultado["custo_total_acumulado"] = int(df_imp["Custo_Total_R$"].sum())
        resultado["avai_total"]            = round(df_imp["AVAI"].sum(), 1)
        resultado["ano_mais_oneroso"]      = int(df_imp.loc[df_imp["Custo_Total_R$"].idxmax(), "Ano"])

        log_info(f"  Custo total acumulado CG: R$ {resultado['custo_total_acumulado']:,.0f}")
        log_info(f"  AVAI total: {resultado['avai_total']:.1f} anos")
        log_info(f"  Ano mais oneroso: {resultado['ano_mais_oneroso']}")

        # Gráfico: custo por componente + AVAI
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle("Impacto Socioeconômico Estimado — Dengue Campo Grande/MS", fontsize=13, fontweight="bold")

        anos_str = df_imp["Ano"].astype(str)
        width = 0.6
        b1 = axes[0].bar(anos_str, df_imp["Custo_Amb_R$"]   / 1e6, width, label="Ambulatorial", color="#42A5F5")
        b2 = axes[0].bar(anos_str, df_imp["Custo_Hosp_R$"]  / 1e6, width,
                         bottom=df_imp["Custo_Amb_R$"] / 1e6, label="Hospitalar", color="#EF5350")
        b3 = axes[0].bar(anos_str, df_imp["Custo_Obito_R$"] / 1e6, width,
                         bottom=(df_imp["Custo_Amb_R$"] + df_imp["Custo_Hosp_R$"]) / 1e6,
                         label="Óbito Estimado", color="#AB47BC")
        axes[0].set_ylabel("Custo (R$ milhões)")
        axes[0].set_title("Custo Econômico por Ano")
        axes[0].legend(fontsize=8)
        axes[0].tick_params(axis="x", rotation=45)
        axes[0].grid(axis="y", alpha=0.3)

        axes[1].bar(anos_str, df_imp["AVAI"], color="#FF7043", edgecolor="white")
        axes[1].set_ylabel("AVAI (Anos de Vida Perdidos)")
        axes[1].set_title("Carga de Doença — AVAI por Ano")
        axes[1].tick_params(axis="x", rotation=45)
        axes[1].grid(axis="y", alpha=0.3)
        plt.tight_layout()
        _salvar_figura(fig, "impacto_socioeconomico_cg")
        plt.close(fig)

        # ── Tabela Texttable
        if HAS_TEXTTABLE:
            tt = texttable.Texttable(max_width=110)
            tt.set_deco(texttable.Texttable.HEADER | texttable.Texttable.VLINES)
            tt.header(["Ano", "Casos", "Hosp", "Óbitos", "Custo Total (R$)", "AVAI", "Dias Perdidos"])
            tt.set_cols_dtype(["i", "i", "i", "f", "i", "f", "i"])
            tt.set_cols_align(["c", "r", "r", "r", "r", "r", "r"])
            for _, rw in df_imp.iterrows():
                tt.add_row([rw.Ano, rw.Casos, rw.Hosp_Est, rw.Obitos_Est,
                            rw["Custo_Total_R$"], rw.AVAI, rw.Dias_Perdidos])
            arq = OUTPUT_DIR / f"tabela_impacto_socioeconomico_{TIMESTAMP}.txt"
            with open(arq, "w", encoding="utf-8") as fh:
                fh.write("SIPREV — IMPACTO SOCIOECONÔMICO ESTIMADO — CAMPO GRANDE/MS\n")
                fh.write("=" * 80 + "\n")
                fh.write(tt.draw())
                fh.write(f"\n\nCusto Total Acumulado: R$ {resultado['custo_total_acumulado']:,.2f}\n")
                fh.write(f"AVAI Total: {resultado['avai_total']:.1f} anos\n")
            log_ok(f"  Tabela impacto salva: {arq.name}")

        log_ok("Seção 55 concluída.")

    except Exception as exc:
        log_warn(f"Seção 55 erro: {exc}")

    return resultado




In [117]:
# =============================================================================
# SEÇÃO 56: Análise de Vulnerabilidade e Capacidade de Resposta (Score)


In [118]:
# =============================================================================

def analise_vulnerabilidade_resposta(df_ms: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula um Score de Vulnerabilidade composto para os municípios do MS,
    combinando: tendência de casos, nível médio de alerta, receptividade,
    transmissibilidade, proporção epidêmica histórica e volatilidade (CV).
    """
    log_section("56 — Score de Vulnerabilidade e Resposta Municipal")

    df_vul = pd.DataFrame()
    if df_ms.empty:
        log_warn("Seção 56: df_ms vazio.")
        return df_vul

    try:
        from sklearn.preprocessing import MinMaxScaler

        munis = df_ms["municipio_geocodigo"].unique() if "municipio_geocodigo" in df_ms.columns else []
        if len(munis) == 0 and "Localidade_id" in df_ms.columns:
            munis = df_ms["Localidade_id"].unique()
        col_id = "municipio_geocodigo" if "municipio_geocodigo" in df_ms.columns else "Localidade_id"

        rows = []
        for muni in munis:
            sub = df_ms[df_ms[col_id] == muni].copy()
            if len(sub) < 10:
                continue
            nome = sub["municipio_nome"].iloc[0] if "municipio_nome" in sub.columns else str(muni)
            casos = sub["casos"].fillna(0).values
            inc   = sub["p_inc100k"].fillna(0).values
            niv   = sub["nivel"].fillna(1).values
            rec   = sub["receptivo"].fillna(0).values if "receptivo" in sub.columns else np.zeros(len(sub))
            tra   = sub["transmissao"].fillna(0).values if "transmissao" in sub.columns else np.zeros(len(sub))
            rt    = sub["Rt"].fillna(1).values

            mean_casos = np.mean(casos)
            cv_casos   = np.std(casos) / (mean_casos + 1e-6)
            mean_niv   = np.mean(niv)
            pct_rec    = np.mean(rec)
            pct_tra    = np.mean(tra)
            mean_rt    = np.mean(rt)
            # Tendência (slope normalizado)
            if len(casos) > 10:
                from scipy.stats import linregress
                slope, *_ = linregress(np.arange(len(casos)), casos)
                tend_norm = slope / (mean_casos + 1e-6)
            else:
                tend_norm = 0.0
            pct_ep4    = np.mean(niv >= 4)

            rows.append({
                "municipio": nome, col_id: muni,
                "mean_casos": round(mean_casos, 1),
                "cv_casos": round(cv_casos, 4),
                "mean_nivel": round(mean_niv, 3),
                "pct_receptivo": round(pct_rec, 3),
                "pct_transmissao": round(pct_tra, 3),
                "mean_rt": round(mean_rt, 3),
                "tend_normalizada": round(tend_norm, 6),
                "pct_nivel4": round(pct_ep4, 4),
            })

        if not rows:
            log_warn("Seção 56: nenhum município com dados suficientes.")
            return df_vul

        df_vul = pd.DataFrame(rows)
        feats = ["mean_nivel", "pct_receptivo", "pct_transmissao", "mean_rt",
                 "tend_normalizada", "pct_nivel4"]
        weights = np.array([0.25, 0.15, 0.20, 0.20, 0.10, 0.10])

        sc2 = MinMaxScaler()
        X_sc = sc2.fit_transform(df_vul[feats].fillna(0))
        df_vul["score_vulnerabilidade"] = np.round((X_sc * weights).sum(axis=1), 4)

        # Classificação
        def classif_vul(s):
            if s >= 0.75: return "Crítico"
            if s >= 0.55: return "Muito Alto"
            if s >= 0.40: return "Alto"
            if s >= 0.25: return "Moderado"
            return "Baixo"

        df_vul["classe_vulnerabilidade"] = df_vul["score_vulnerabilidade"].apply(classif_vul)
        df_vul = df_vul.sort_values("score_vulnerabilidade", ascending=False).reset_index(drop=True)

        log_info(f"  {len(df_vul)} municípios avaliados")
        log_info(f"  Top-5 vulneráveis: {df_vul['municipio'].head(5).tolist()}")

        # Gráfico top-20
        top20 = df_vul.head(20)
        cores_vul = {"Crítico": "#B71C1C", "Muito Alto": "#E53935", "Alto": "#FB8C00",
                     "Moderado": "#FDD835", "Baixo": "#43A047"}
        colors = [cores_vul.get(c, "#9E9E9E") for c in top20["classe_vulnerabilidade"]]

        fig, ax = plt.subplots(figsize=(12, 7))
        bars = ax.barh(top20["municipio"][::-1], top20["score_vulnerabilidade"][::-1],
                       color=colors[::-1], edgecolor="white")
        ax.set_xlabel("Score de Vulnerabilidade (0–1)")
        ax.set_title("Top-20 Municípios por Vulnerabilidade à Dengue — Mato Grosso do Sul", fontsize=12)
        ax.set_xlim(0, 1)
        ax.grid(axis="x", alpha=0.3)
        for bar, val in zip(bars, top20["score_vulnerabilidade"][::-1]):
            ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                    f"{val:.3f}", va="center", fontsize=8)
        handles = [plt.Rectangle((0,0),1,1, color=v) for v in cores_vul.values()]
        ax.legend(handles, cores_vul.keys(), title="Classe", fontsize=8, loc="lower right")
        plt.tight_layout()
        _salvar_figura(fig, "score_vulnerabilidade_ms")
        plt.close(fig)

        log_ok("Seção 56 concluída.")

    except Exception as exc:
        log_warn(f"Seção 56 erro: {exc}")

    return df_vul




In [119]:
# =============================================================================
# SEÇÃO 57: Análise de Tendência de Longo Prazo e Projeções de Incidência


In [120]:
# =============================================================================

def tendencia_longo_prazo(df_cg: pd.DataFrame, df_cap: pd.DataFrame) -> dict:
    """
    Analisa a tendência de longo prazo (2016-2025) e projeta para 2026-2030
    usando múltiplos modelos: linear, exponencial, polinomial grau-2.
    Inclui comparação com capitais brasileiras.
    """
    resultado = {}
    log_section("57 — Tendência de Longo Prazo e Projeções 2026–2030")

    if df_cg.empty:
        log_warn("Seção 57: df_cg vazio.")
        return resultado

    try:
        from scipy.stats import linregress

        # Dados anuais CG
        df_cg_ano = df_cg.groupby(df_cg["data_SE"].dt.year)["casos"].sum().reset_index()
        df_cg_ano.columns = ["ano", "casos"]
        anos  = df_cg_ano["ano"].values.astype(float)
        casos = df_cg_ano["casos"].values.astype(float)
        t     = anos - anos[0]   # centrado

        # Modelo 1: Linear
        slope, intercept, r_lin, p_lin, _ = linregress(t, casos)
        resultado["linear_slope"]     = round(slope, 2)
        resultado["linear_r2"]        = round(r_lin ** 2, 4)
        resultado["linear_pvalue"]    = round(p_lin, 6)

        # Modelo 2: Exponencial (log)
        casos_log = np.log(casos + 1)
        sl_e, ic_e, r_e, p_e, _ = linregress(t, casos_log)
        resultado["exp_r2"]  = round(r_e ** 2, 4)

        # Modelo 3: Polinomial grau-2
        coef2 = np.polyfit(t, casos, 2)
        cas_pred_poly = np.polyval(coef2, t)
        ss_res = np.sum((casos - cas_pred_poly) ** 2)
        ss_tot = np.sum((casos - casos.mean()) ** 2)
        resultado["poly2_r2"] = round(1 - ss_res / (ss_tot + 1e-10), 4)

        log_info(f"  Linear R²={resultado['linear_r2']} | Exp R²={resultado['exp_r2']} | Poly2 R²={resultado['poly2_r2']}")

        # Projeções 2026–2030
        anos_proj  = np.arange(2026, 2031)
        t_proj     = anos_proj - anos[0]
        proj_lin   = slope * t_proj + intercept
        proj_exp   = np.exp(sl_e * t_proj + ic_e) - 1
        proj_poly  = np.polyval(coef2, t_proj)

        # Garantir não-negativo
        proj_lin  = np.maximum(proj_lin,  0)
        proj_exp  = np.maximum(proj_exp,  0)
        proj_poly = np.maximum(proj_poly, 0)

        df_proj = pd.DataFrame({
            "ano": anos_proj,
            "proj_linear": np.round(proj_lin).astype(int),
            "proj_exp":    np.round(proj_exp).astype(int),
            "proj_poly2":  np.round(proj_poly).astype(int),
        })
        resultado["df_projecoes"] = df_proj
        log_info("  Projeções 2026-2030:\n" + df_proj.to_string(index=False))

        # Gráfico
        fig, ax = plt.subplots(figsize=(13, 6))
        ax.bar(df_cg_ano["ano"], df_cg_ano["casos"], color="#B0BEC5", alpha=0.6,
               label="Histórico", zorder=2)
        ax.plot(df_cg_ano["ano"], slope * t + intercept,
                "b--", lw=1.5, label=f"Linear (R²={resultado['linear_r2']:.3f})")
        ax.plot(df_cg_ano["ano"], cas_pred_poly,
                "g-.", lw=1.5, label=f"Polinomial-2 (R²={resultado['poly2_r2']:.3f})")

        ax.plot(df_proj["ano"], df_proj["proj_linear"],  "bo--", ms=7, lw=1.5, label="Proj. Linear")
        ax.plot(df_proj["ano"], df_proj["proj_exp"],     "rs--", ms=7, lw=1.5, label="Proj. Exponencial")
        ax.plot(df_proj["ano"], df_proj["proj_poly2"],   "g^--", ms=7, lw=1.5, label="Proj. Polinomial-2")

        ax.axvline(2025.5, color="gray", ls=":", lw=1, label="Projeção →")
        ax.set_xlabel("Ano"); ax.set_ylabel("Total de Casos")
        ax.set_title("Tendência e Projeções de Longo Prazo — Dengue Campo Grande/MS (2016–2030)", fontsize=12)
        ax.legend(fontsize=8, ncol=2)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        _salvar_figura(fig, "tendencia_longo_prazo_proj2030")
        plt.close(fig)

        # ── Capitais: comparação de tendências
        if not df_cap.empty and "municipio_nome" in df_cap.columns:
            caps_selecionadas = ["Campo Grande", "Cuiabá", "Goiânia", "Brasília", "Manaus", "Belo Horizonte"]
            fig2, ax2 = plt.subplots(figsize=(13, 6))
            ax2.set_title("Evolução Anual de Casos — Capitais Selecionadas (2016–2025)", fontsize=12)
            pal = sns.color_palette("tab10", len(caps_selecionadas))
            for idx, cap in enumerate(caps_selecionadas):
                sub_c = df_cap[df_cap["municipio_nome"].str.contains(cap, case=False, na=False)]
                if sub_c.empty:
                    continue
                evo = sub_c.groupby(sub_c["data_SE"].dt.year)["casos"].sum()
                ax2.plot(evo.index, evo.values, marker="o", ms=5, lw=1.5,
                         color=pal[idx], label=cap)
            ax2.set_xlabel("Ano"); ax2.set_ylabel("Total de Casos")
            ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
            plt.tight_layout()
            _salvar_figura(fig2, "tendencia_capitais_selecionadas")
            plt.close(fig2)

        log_ok("Seção 57 concluída.")

    except Exception as exc:
        log_warn(f"Seção 57 erro: {exc}")

    return resultado




In [121]:
# =============================================================================
# SEÇÃO 58: Mapa de Calor Climático-Epidemiológico


In [122]:
# =============================================================================

def mapa_calor_climatico_epidemiologico(df_cg: pd.DataFrame) -> None:
    """
    Gera mapa de calor bivariado temperatura × umidade com overlay de incidência,
    boxplot mensal de temperatura e casos, e análise de condições críticas.
    """
    log_section("58 — Mapa de Calor Climático-Epidemiológico")

    if df_cg.empty:
        log_warn("Seção 58: df_cg vazio.")
        return

    try:
        col_temp = "tempmed" if "tempmed" in df_cg.columns else "tempmax"
        col_umid = "umidmed" if "umidmed" in df_cg.columns else "umidmax"

        if col_temp not in df_cg.columns or col_umid not in df_cg.columns:
            log_warn("Seção 58: colunas climáticas ausentes.")
            return

        df_c = df_cg[[col_temp, col_umid, "casos", "p_inc100k", "data_SE"]].dropna()
        if len(df_c) < 20:
            log_warn("Seção 58: dados insuficientes.")
            return

        df_c = df_c.copy()
        df_c["mes"] = df_c["data_SE"].dt.month
        df_c["ano"] = df_c["data_SE"].dt.year

        # ── Scatter hex binning temperatura × umidade
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle("Análise Climático-Epidemiológica — Campo Grande/MS", fontsize=13, fontweight="bold")

        sc = axes[0].hexbin(df_c[col_temp], df_c[col_umid], C=df_c["casos"],
                            gridsize=20, cmap="YlOrRd", reduce_C_function=np.mean, mincnt=1)
        plt.colorbar(sc, ax=axes[0], label="Média de Casos")
        axes[0].set_xlabel(f"Temperatura Média (°C)")
        axes[0].set_ylabel("Umidade Relativa Média (%)")
        axes[0].set_title("Casos Médios por Célula Temp×Umid")

        # Contornos de incidência
        try:
            from scipy.stats import gaussian_kde
            xy   = np.vstack([df_c[col_temp], df_c[col_umid]])
            kde  = gaussian_kde(xy)
            xg   = np.linspace(df_c[col_temp].min(), df_c[col_temp].max(), 50)
            yg   = np.linspace(df_c[col_umid].min(), df_c[col_umid].max(), 50)
            XX, YY = np.meshgrid(xg, yg)
            ZZ   = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
            axes[1].contourf(XX, YY, ZZ, levels=10, cmap="Blues", alpha=0.5)
            axes[1].contour( XX, YY, ZZ, levels=10, colors="navy", linewidths=0.5, alpha=0.4)
        except Exception:
            pass

        scatter_c = axes[1].scatter(df_c[col_temp], df_c[col_umid],
                                    c=df_c["p_inc100k"], cmap="plasma",
                                    s=25, alpha=0.5, edgecolors="none")
        plt.colorbar(scatter_c, ax=axes[1], label="Incidência/100k hab")
        axes[1].set_xlabel("Temperatura Média (°C)")
        axes[1].set_ylabel("Umidade Relativa Média (%)")
        axes[1].set_title("Incidência por Condição Climática")
        plt.tight_layout()
        _salvar_figura(fig, "mapa_calor_climatico_epidemiologico")
        plt.close(fig)

        # ── Boxplot mensal
        fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
        fig2.suptitle("Padrão Mensal — Temperatura e Casos Dengue — Campo Grande/MS", fontsize=12)
        meses_nomes = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]

        data_temp = [df_c[df_c["mes"] == m][col_temp].values for m in range(1, 13)]
        data_cas  = [df_c[df_c["mes"] == m]["casos"].values   for m in range(1, 13)]

        bp1 = axes2[0].boxplot(data_temp, patch_artist=True,
                               boxprops=dict(facecolor="#FFCDD2", color="#C62828"),
                               medianprops=dict(color="#B71C1C", lw=2))
        axes2[0].set_xticklabels(meses_nomes, rotation=45, fontsize=9)
        axes2[0].set_ylabel("Temperatura Média (°C)"); axes2[0].set_title("Distribuição Mensal — Temperatura")
        axes2[0].grid(axis="y", alpha=0.3)

        bp2 = axes2[1].boxplot(data_cas, patch_artist=True,
                               boxprops=dict(facecolor="#BBDEFB", color="#0D47A1"),
                               medianprops=dict(color="#1565C0", lw=2))
        axes2[1].set_xticklabels(meses_nomes, rotation=45, fontsize=9)
        axes2[1].set_ylabel("Casos Semanais"); axes2[1].set_title("Distribuição Mensal — Casos")
        axes2[1].grid(axis="y", alpha=0.3)
        plt.tight_layout()
        _salvar_figura(fig2, "boxplot_mensal_temp_casos")
        plt.close(fig2)

        # ── Análise de condições críticas (T≥Q75 E U≥Q75)
        q75_t = df_c[col_temp].quantile(0.75)
        q75_u = df_c[col_umid].quantile(0.75)
        df_crit = df_c[(df_c[col_temp] >= q75_t) & (df_c[col_umid] >= q75_u)]
        pct_crit = len(df_crit) / len(df_c) * 100
        media_casos_crit = df_crit["casos"].mean() if len(df_crit) else 0
        media_casos_norm = df_c[~((df_c[col_temp] >= q75_t) & (df_c[col_umid] >= q75_u))]["casos"].mean()

        log_info(f"  Condições críticas (T≥Q75 e U≥Q75): {pct_crit:.1f}% das semanas")
        log_info(f"  Média casos crítico: {media_casos_crit:.1f} vs normal: {media_casos_norm:.1f}")

        log_ok("Seção 58 concluída.")

    except Exception as exc:
        log_warn(f"Seção 58 erro: {exc}")




In [123]:
# =============================================================================
# SEÇÃO 59: Análise de Distribuição Espacial por Mesorregiões do MS


In [124]:
# =============================================================================

def analise_mesorregioes_ms(df_ms: pd.DataFrame) -> dict:
    """
    Agrupa municípios do MS por mesorregião (Pantanais Sul-Mato-Grossenses,
    Centro-Norte de Mato Grosso do Sul, Leste de Mato Grosso do Sul) e
    analisa padrões epidemiológicos por região administrativa.
    """
    resultado = {}
    log_section("59 — Análise por Mesorregiões do Mato Grosso do Sul")

    # Distribuição de municípios por mesorregião (IBGE)
    MESORREGIOES = {
        "Pantanais Sul-Mato-Grossenses": [
            "Corumbá", "Ladário", "Porto Murtinho", "Aquidauana", "Anastácio",
            "Miranda", "Bodoquena", "Bonito",
        ],
        "Centro-Norte de Mato Grosso do Sul": [
            "Campo Grande", "Jaraguari", "Ribas do Rio Pardo", "Rochedo",
            "Sidrolândia", "Terenos", "Bandeirantes", "Camapuã", "Corguinho",
            "Costa Rica", "Coxim", "Pedro Gomes", "Rio Verde de Mato Grosso",
            "Sonora", "Dois Irmãos do Buriti", "Figueirão",
        ],
        "Leste de Mato Grosso do Sul": [
            "Dourados", "Ponta Porã", "Naviraí", "Nova Andradina", "Três Lagoas",
            "Aparecida do Taboado", "Bataguassu", "Água Clara", "Brasilândia",
            "Inocência", "Paranaíba", "Selvíria",
        ],
        "Sudoeste de Mato Grosso do Sul": [
            "Jardim", "Bela Vista", "Caracol", "Guia Lopes da Laguna",
            "Nioaque", "Maracaju", "Piraputanga",
        ],
    }

    if df_ms.empty or "municipio_nome" not in df_ms.columns:
        log_warn("Seção 59: df_ms vazio ou sem coluna municipio_nome.")
        return resultado

    try:
        # Mapear cada linha ao mesorregião
        def get_meso(nome):
            for meso, municipios in MESORREGIOES.items():
                for m in municipios:
                    if m.lower() in str(nome).lower() or str(nome).lower() in m.lower():
                        return meso
            return "Outros"

        df_ms2 = df_ms.copy()
        df_ms2["mesorregiao"] = df_ms2["municipio_nome"].apply(get_meso)

        # Agregação por mesorregião e ano
        df_ms2["ano"] = df_ms2["data_SE"].dt.year
        agg = df_ms2.groupby(["mesorregiao", "ano"]).agg(
            casos_total = ("casos",     "sum"),
            inc_media   = ("p_inc100k", "mean"),
            rt_medio    = ("Rt",        "mean"),
            nivel_medio = ("nivel",     "mean"),
        ).reset_index()

        resultado["df_mesorregioes"] = agg

        # Gráfico: evolução anual por mesorregião
        mesorregioes = agg["mesorregiao"].unique()
        cores_meso = {"Pantanais Sul-Mato-Grossenses": "#1565C0",
                      "Centro-Norte de Mato Grosso do Sul": "#E53935",
                      "Leste de Mato Grosso do Sul": "#2E7D32",
                      "Sudoeste de Mato Grosso do Sul": "#F57F17",
                      "Outros": "#78909C"}

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle("Padrão Epidemiológico por Mesorregião — Mato Grosso do Sul", fontsize=13, fontweight="bold")

        for meso in mesorregioes:
            sub_m = agg[agg["mesorregiao"] == meso]
            col   = cores_meso.get(meso, "#78909C")
            lw    = 2.0 if meso == "Centro-Norte de Mato Grosso do Sul" else 1.2
            axes[0].plot(sub_m["ano"], sub_m["casos_total"] / 1e3, marker="o",
                         ms=5, lw=lw, color=col, label=meso)
            axes[1].plot(sub_m["ano"], sub_m["inc_media"], marker="s",
                         ms=5, lw=lw, color=col, label=meso)

        axes[0].set_ylabel("Total de Casos (milhares)"); axes[0].set_title("Casos por Mesorregião")
        axes[0].legend(fontsize=7); axes[0].grid(True, alpha=0.3)
        axes[1].set_ylabel("Incidência Média /100k"); axes[1].set_title("Incidência por Mesorregião")
        axes[1].legend(fontsize=7); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        _salvar_figura(fig, "mesorregioes_ms_evolucao")
        plt.close(fig)

        # Boxplot incidência por mesorregião
        fig2, ax2 = plt.subplots(figsize=(11, 5))
        data_box  = [df_ms2[df_ms2["mesorregiao"] == m]["p_inc100k"].dropna().values
                     for m in sorted(df_ms2["mesorregiao"].unique())]
        labels_box = sorted(df_ms2["mesorregiao"].unique())
        bp = ax2.boxplot(data_box, patch_artist=True, notch=False)
        for patch, label in zip(bp["boxes"], labels_box):
            patch.set_facecolor(cores_meso.get(label, "#9E9E9E"))
            patch.set_alpha(0.75)
        ax2.set_xticklabels(labels_box, rotation=25, ha="right", fontsize=9)
        ax2.set_ylabel("Incidência /100k hab")
        ax2.set_title("Distribuição da Incidência por Mesorregião — MS")
        ax2.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        _salvar_figura(fig2, "boxplot_incidencia_mesorregioes")
        plt.close(fig2)

        log_ok("Seção 59 concluída.")

    except Exception as exc:
        log_warn(f"Seção 59 erro: {exc}")

    return resultado




In [125]:
# =============================================================================
# SEÇÃO 60: Sumário Executivo Final e Metadados de Entrega


In [126]:
# =============================================================================

def sumario_executivo_final(
    df_cg: pd.DataFrame,
    df_ms: pd.DataFrame,
    df_cap: pd.DataFrame,
    resultados_ml: dict,
    resultados_reg: dict,
    resultados_dl: dict,
    resultados_ts: dict,
    alerta: dict,
    df_vul: pd.DataFrame,
    df_imp: dict,
) -> None:
    """
    Gera o sumário executivo final consolidado em TXT/LOG, incluindo:
    - Estatísticas globais do projeto
    - Principais achados por seção
    - Desempenho dos modelos de ML/DL
    - Sistema de alerta atual
    - Recomendações prioritárias
    - Índice de arquivos gerados
    """
    log_section("60 — Sumário Executivo Final e Metadados de Entrega")

    try:
        linhas = []
        sep  = "=" * 100
        sep2 = "-" * 100

        linhas.append(sep)
        linhas.append("SIPREV — SISTEMA INTELIGENTE DE PREVISÃO EPIDEMIOLÓGICA DE DENGUE")
        linhas.append("Análise Organizacional e Soluções Tecnológicas | Ciência dos Dados | Módulo 3")
        linhas.append(f"Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
        linhas.append(f"Ambiente: {'Google Colab' if IS_COLAB else 'Local'}")
        linhas.append(sep)
        linhas.append("")

        # ── 1. Escopo
        linhas.append("1. ESCOPO DA ANÁLISE")
        linhas.append(sep2)
        n_cg  = len(df_cg)
        anos_cg = sorted(df_cg["data_SE"].dt.year.unique()) if not df_cg.empty else []
        n_ms  = df_ms["municipio_geocodigo"].nunique() if "municipio_geocodigo" in df_ms.columns else 0
        n_cap = df_cap["municipio_nome"].nunique() if "municipio_nome" in df_cap.columns else 0
        linhas.append(f"  Campo Grande/MS : {n_cg:,} registros semanais | {anos_cg[0] if anos_cg else 'N/A'}–{anos_cg[-1] if anos_cg else 'N/A'}")
        linhas.append(f"  Mato Grosso do Sul: {n_ms} municípios | {len(df_ms):,} registros")
        linhas.append(f"  Capitais Brasileiras: {n_cap} capitais | {len(df_cap):,} registros")
        linhas.append(f"  Fonte dos dados: InfoDengue (FGV/EMAp/FIOCRUZ)")
        linhas.append("")

        # ── 2. Resumo Epidemiológico CG
        linhas.append("2. RESUMO EPIDEMIOLÓGICO — CAMPO GRANDE/MS")
        linhas.append(sep2)
        if not df_cg.empty:
            casos_total = int(df_cg["casos"].sum())
            inc_media   = round(df_cg["p_inc100k"].mean(), 2)
            rt_medio    = round(df_cg["Rt"].mean(), 3)
            pct_v4      = round(100 * (df_cg["nivel"] == 4).mean(), 2)
            ano_pico    = int(df_cg.groupby(df_cg["data_SE"].dt.year)["casos"].sum().idxmax())
            linhas.append(f"  Total de casos (estimados): {casos_total:,}")
            linhas.append(f"  Incidência média: {inc_media:.2f}/100k hab")
            linhas.append(f"  Rt médio: {rt_medio:.3f}")
            linhas.append(f"  Semanas em alerta vermelho (nível 4): {pct_v4:.1f}%")
            linhas.append(f"  Ano de maior incidência: {ano_pico}")
        linhas.append("")

        # ── 3. Modelos de ML/DL
        linhas.append("3. DESEMPENHO DOS MODELOS PREDITIVOS")
        linhas.append(sep2)
        if resultados_ml:
            linhas.append("  [ML — Regressão]")
            for nome, mets in resultados_ml.items():
                rmse = mets.get("rmse", "N/A")
                r2   = mets.get("r2",   "N/A")
                linhas.append(f"    {nome:30s} RMSE={rmse}  R²={r2}")
        if resultados_dl:
            linhas.append("  [DL — Redes Neurais]")
            for nome, mets in resultados_dl.items():
                rmse = mets.get("rmse", "N/A")
                linhas.append(f"    {nome:30s} RMSE={rmse}")
        if resultados_ts:
            linhas.append("  [Séries Temporais]")
            for nome, mets in resultados_ts.items():
                if isinstance(mets, dict):
                    rmse = mets.get("rmse", "N/A")
                    linhas.append(f"    {nome:30s} RMSE={rmse}")
        linhas.append("")

        # ── 4. Sistema de Alerta
        linhas.append("4. SISTEMA DE ALERTA PRECOCE (PRÓXIMAS 4 SEMANAS)")
        linhas.append(sep2)
        if alerta:
            sinal = alerta.get("sinal_atual", "N/A")
            cor   = alerta.get("cor_semaforo", "N/A")
            linhas.append(f"  Sinal atual: {sinal}")
            linhas.append(f"  Semáforo:    {cor}")
            prev = alerta.get("previsoes_4sem", [])
            if prev:
                linhas.append(f"  Previsões (casos): {[round(p, 0) for p in prev]}")
        linhas.append("")

        # ── 5. Vulnerabilidade MS
        linhas.append("5. TOP-10 MUNICÍPIOS MAIS VULNERÁVEIS — MS")
        linhas.append(sep2)
        if isinstance(df_vul, pd.DataFrame) and not df_vul.empty and "score_vulnerabilidade" in df_vul.columns:
            top10_vul = df_vul.head(10)[["municipio", "score_vulnerabilidade", "classe_vulnerabilidade"]]
            for _, row in top10_vul.iterrows():
                linhas.append(f"  {row['municipio']:30s}  Score={row['score_vulnerabilidade']:.4f}  [{row['classe_vulnerabilidade']}]")
        linhas.append("")

        # ── 6. Impacto Socioeconômico
        linhas.append("6. IMPACTO SOCIOECONÔMICO ESTIMADO — CAMPO GRANDE/MS")
        linhas.append(sep2)
        if isinstance(df_imp, dict) and "custo_total_acumulado" in df_imp:
            linhas.append(f"  Custo total acumulado: R$ {df_imp['custo_total_acumulado']:,.2f}")
            linhas.append(f"  AVAI total: {df_imp.get('avai_total', 'N/A')} anos")
            linhas.append(f"  Ano mais oneroso: {df_imp.get('ano_mais_oneroso', 'N/A')}")
        linhas.append("")

        # ── 7. Recomendações
        linhas.append("7. RECOMENDAÇÕES PRIORITÁRIAS")
        linhas.append(sep2)
        recs = [
            "Intensificar ações de vigilância entomológica nas semanas epidemiológicas 5–15 (pico histórico).",
            "Priorizar municípios com score de vulnerabilidade > 0.55 para campanhas preventivas direcionadas.",
            "Manter monitoramento contínuo do Rt semanal; acionar protocolo de emergência se Rt > 1,5 por 3 semanas consecutivas.",
            "Ampliar capacidade hospitalar nos municípios de alto risco nos meses de janeiro a abril.",
            "Implementar campanhas de educação em saúde nos períodos de condições climáticas críticas (T≥Q75 e U≥Q75).",
            "Fortalecer o sistema de notificação para reduzir subnotificação (estimada em 30–60%).",
            "Utilizar modelos ensemble (DL+ARIMA+Prophet) para previsão semanal e alimentar dashboard em tempo real.",
            "Desenvolver protocolo de resposta rápida baseado no sistema de semáforo de 4 cores implementado.",
        ]
        for i, rec in enumerate(recs, 1):
            linhas.append(f"  {i}. {rec}")
        linhas.append("")

        # ── 8. Índice de arquivos
        linhas.append("8. ÍNDICE DE ARQUIVOS GERADOS")
        linhas.append(sep2)
        try:
            arquivos = sorted(OUTPUT_DIR.glob(f"*{TIMESTAMP}*"))
            tipos = {}
            for arq in arquivos:
                ext = arq.suffix.lower()
                tipos.setdefault(ext, []).append(arq.name)
            for ext, nomes in sorted(tipos.items()):
                linhas.append(f"  {ext.upper():8s}: {len(nomes):3d} arquivo(s)")
            linhas.append(f"\n  Total: {len(arquivos)} arquivo(s) em {OUTPUT_DIR}")
        except Exception:
            pass
        linhas.append("")

        # ── 9. Metadados técnicos
        linhas.append("9. METADADOS TÉCNICOS")
        linhas.append(sep2)
        import sys
        linhas.append(f"  Python: {sys.version.split()[0]}")
        linhas.append(f"  TensorFlow: {'disponível' if HAS_TF else 'ausente'}")
        linhas.append(f"  Scikit-learn: {'disponível' if HAS_SKLEARN else 'ausente'}")
        linhas.append(f"  XGBoost: {'disponível' if HAS_XGB else 'ausente'}")
        linhas.append(f"  LightGBM: {'disponível' if HAS_LGB else 'ausente'}")
        linhas.append(f"  CatBoost: {'disponível' if HAS_CAT else 'ausente'}")
        linhas.append(f"  Plotly: {'disponível' if HAS_PLOTLY else 'ausente'}")
        linhas.append(f"  Folium: {'disponível' if HAS_FOLIUM else 'ausente'}")
        linhas.append(f"  Prophet: {'disponível' if HAS_PROPHET else 'ausente'}")
        linhas.append(f"  Statsmodels: {'disponível' if HAS_STATSMODELS else 'ausente'}")
        linhas.append(f"  SHAP: {'disponível' if HAS_SHAP else 'ausente'}")
        linhas.append(f"  pmdarima: {'disponível' if HAS_PMDARIMA else 'ausente'}")
        linhas.append("")
        linhas.append(sep)
        linhas.append("FIM DO SUMÁRIO EXECUTIVO — SIPREV v1.0")
        linhas.append(sep)

        # Salvar TXT
        arq_sumario = OUTPUT_DIR / f"sumario_executivo_final_{TIMESTAMP}.txt"
        with open(arq_sumario, "w", encoding="utf-8") as fh:
            fh.write("\n".join(linhas))
        log_ok(f"  Sumário executivo salvo: {arq_sumario.name}")

        # Exibir no console
        for ln in linhas:
            print(ln)

    except Exception as exc:
        log_warn(f"Seção 60 erro: {exc}")




In [127]:
# =============================================================================
# ATUALIZAÇÃO DO main() — Bloco L: Seções 53–60


In [128]:
# =============================================================================
# Nota: O main() completo está na Seção 52 (part7.py). Aqui adicionamos
# um bloco complementar que é chamado DENTRO do main() existente via
# patch de execução no bloco __main__. O main() da Seção 52 já foi
# escrito para aceitar extensão via _executar_bloco_l().

def _executar_bloco_l(
    df_cg, df_ms, df_cap,
    resultados_ml=None, resultados_reg=None, resultados_dl=None,
    resultados_ts=None, alerta=None,
):
    """
    Bloco L — Análises Complementares (Seções 53–60).
    Chamado ao final do main() se os dados estiverem disponíveis.
    """
    resultados_ml  = resultados_ml  or {}
    resultados_reg = resultados_reg or {}
    resultados_dl  = resultados_dl  or {}
    resultados_ts  = resultados_ts  or {}
    alerta         = alerta         or {}

    log_section("BLOCO L — Análises Complementares (Seções 53–60)")

    r53  = analise_stl_espectral(df_cg)
    r54  = clusters_temporais_semanais(df_cg, df_ms)
    r55  = analise_impacto_socioeconomico(df_cg, df_ms)
    r56  = analise_vulnerabilidade_resposta(df_ms)
    r57  = tendencia_longo_prazo(df_cg, df_cap)
    mapa_calor_climatico_epidemiologico(df_cg)          # Seção 58
    r59  = analise_mesorregioes_ms(df_ms)

    df_imp_dict = r55 if isinstance(r55, dict) else {}

    sumario_executivo_final(
        df_cg, df_ms, df_cap,
        resultados_ml, resultados_reg, resultados_dl,
        resultados_ts, alerta, r56, df_imp_dict,
    )

    log_ok("Bloco L concluído — Seções 53–60.")
    return {
        "stl_espectral": r53,
        "clusters_temporais": r54,
        "impacto_socioeconomico": r55,
        "vulnerabilidade": r56,
        "tendencia_lp": r57,
        "mesorregioes": r59,
    }




In [129]:
# =============================================================================
# PONTO DE ENTRADA


In [130]:
# =============================================================================




In [131]:
# =============================================================================
# SIPREV - PARTE 9: Secoes 61-63 - Validacao, CCF e Metadados


In [132]:
# =============================================================================

def validacao_qualidade_dados(df_cg, df_ms, df_cap):
    resultado = {}
    log_section("61 -- Validacao de Qualidade dos Dados")
    def _rep(df, nome):
        if df is None or df.empty:
            return {"nome": nome, "status": "vazio"}
        r = {"nome": nome, "n_linhas": len(df),
             "missing_pct": round(df.isnull().mean().mean()*100, 2),
             "duplicatas": int(df.duplicated().sum())}
        if "data_SE" in df.columns:
            d0, d1 = pd.to_datetime(df["data_SE"].min()), pd.to_datetime(df["data_SE"].max())
            n_esp = max(1, (d1 - d0).days // 7 + 1)
            n_pre = df["data_SE"].nunique()
            r["cobertura_pct"] = round(100 * n_pre / n_esp, 2)
        if "casos" in df.columns:
            q1, q3 = df["casos"].quantile(0.25), df["casos"].quantile(0.75)
            iqr = q3 - q1
            r["outliers_3iqr"] = int(((df["casos"] < q1 - 3*iqr) | (df["casos"] > q3 + 3*iqr)).sum())
            r["casos_max"] = int(df["casos"].max())
        return r
    r_cg = _rep(df_cg, "Campo Grande")
    r_ms = _rep(df_ms, "Mato Grosso do Sul")
    r_cap = _rep(df_cap, "Capitais Brasileiras")
    resultado.update({"cg": r_cg, "ms": r_ms, "cap": r_cap})
    for r in [r_cg, r_ms, r_cap]:
        if r.get("status") == "vazio":
            log_warn(f"  {r['nome']}: vazio")
        else:
            log_info(f"  {r['nome']}: {r['n_linhas']:,} linhas | "
                     f"missing={r['missing_pct']}% | dup={r['duplicatas']} | "
                     f"cobertura={r.get('cobertura_pct', 'N/A')}%")
    if HAS_TEXTTABLE:
        tt = texttable.Texttable(max_width=100)
        tt.set_deco(texttable.Texttable.HEADER | texttable.Texttable.VLINES)
        tt.header(["Dataset", "Linhas", "Missing%", "Dup.", "Cobertura%", "Out3IQR"])
        tt.set_cols_dtype(["t", "i", "f", "i", "f", "i"])
        for r in [r_cg, r_ms, r_cap]:
            if r.get("status") == "vazio":
                continue
            tt.add_row([r["nome"], r["n_linhas"], r["missing_pct"], r["duplicatas"],
                        r.get("cobertura_pct", 0), r.get("outliers_3iqr", 0)])
        arq = OUTPUT_DIR / f"data_quality_report_{TIMESTAMP}.txt"
        with open(arq, "w", encoding="utf-8") as fh:
            fh.write("SIPREV -- DATA QUALITY REPORT\n" + "="*80 + "\n" + tt.draw())
        log_ok(f"  DQ salvo: {arq.name}")
    log_ok("Secao 61 concluida.")
    return resultado


def ccf_capitais_campo_grande(df_cg, df_cap, max_lag=12):
    resultado = {}
    log_section("62 -- CCF Capitais vs Campo Grande")
    if df_cg.empty or df_cap.empty or "municipio_nome" not in df_cap.columns:
        log_warn("Secao 62: dados insuficientes.")
        return resultado
    try:
        cg_s = df_cg.set_index("data_SE")["casos"].resample("W-SUN").sum().fillna(0)
        caps_alvo = ["Cuiaba", "Goiania", "Brasilia", "Manaus",
                     "Belo Horizonte", "Sao Paulo", "Rio de Janeiro"]
        fig, axes = plt.subplots(len(caps_alvo), 1, figsize=(12, 3.5 * len(caps_alvo)), sharex=True)
        fig.suptitle("CCF -- Capitais vs Campo Grande (lag 0-12 sem)", fontsize=13, fontweight="bold")
        if len(caps_alvo) == 1:
            axes = [axes]
        for i, cap in enumerate(caps_alvo):
            sub = df_cap[df_cap["municipio_nome"].str.contains(cap, case=False, na=False)]
            if sub.empty:
                axes[i].set_title(f"{cap} -- sem dados")
                continue
            cap_s = sub.set_index("data_SE")["casos"].resample("W-SUN").sum().fillna(0)
            idx_c = cg_s.index.intersection(cap_s.index)
            if len(idx_c) < 20:
                axes[i].set_title(f"{cap} -- insuficiente")
                continue
            x = cg_s.loc[idx_c].values
            y = cap_s.loc[idx_c].values
            xn = (x - x.mean()) / (x.std() + 1e-9)
            yn = (y - y.mean()) / (y.std() + 1e-9)
            ccf_v = [np.corrcoef(xn[lag:], yn[:len(xn)-lag])[0, 1]
                     if lag < len(xn) - 5 else 0
                     for lag in range(max_lag + 1)]
            resultado[cap] = {"ccf": [round(v, 4) for v in ccf_v],
                               "max_lag": int(np.argmax(np.abs(ccf_v)))}
            cols_bar = ["#2196F3" if v >= 0 else "#F44336" for v in ccf_v]
            axes[i].bar(range(max_lag + 1), ccf_v, color=cols_bar, alpha=0.8, edgecolor="white")
            axes[i].axhline(0, color="black", lw=0.8)
            conf = 1.96 / np.sqrt(len(idx_c))
            axes[i].axhline(conf,  color="gray", lw=1, ls="--", alpha=0.6)
            axes[i].axhline(-conf, color="gray", lw=1, ls="--", alpha=0.6)
            axes[i].set_ylabel("Corr")
            axes[i].set_title(f"{cap} (max lag={resultado[cap]['max_lag']})")
            axes[i].grid(axis="y", alpha=0.3)
        axes[-1].set_xlabel("Lag (semanas)")
        plt.tight_layout()
        _salvar_figura(fig, "ccf_capitais_campo_grande")
        plt.close(fig)
        log_ok("Secao 62 concluida.")
    except Exception as exc:
        log_warn(f"Secao 62 erro: {exc}")
    return resultado


def exportar_metadados_json_final(df_cg, df_ms, df_cap,
                                   resultados_ml=None, resultados_ts=None,
                                   resultados_dl=None, alerta=None):
    resultados_ml = resultados_ml or {}
    resultados_ts = resultados_ts or {}
    resultados_dl = resultados_dl or {}
    alerta        = alerta        or {}
    log_section("63 -- Metadados JSON Final")
    try:
        meta = {
            "siprev_version": "1.0",
            "timestamp": TIMESTAMP,
            "ambiente": "colab" if IS_COLAB else "local",
            "data_execucao": datetime.now().isoformat(),
            "datasets": {
                "campo_grande": {
                    "n_registros": len(df_cg),
                    "total_casos": int(df_cg["casos"].sum()) if not df_cg.empty else 0,
                },
                "mato_grosso_sul": {
                    "n_registros": len(df_ms),
                    "n_municipios": (int(df_ms["municipio_geocodigo"].nunique())
                                    if "municipio_geocodigo" in df_ms.columns else 0),
                },
                "capitais_brasil": {
                    "n_registros": len(df_cap),
                    "n_capitais": (int(df_cap["municipio_nome"].nunique())
                                   if "municipio_nome" in df_cap.columns else 0),
                },
            },
            "alerta_atual": alerta.get("sinal_atual", "N/A"),
            "cor_semaforo": alerta.get("cor_semaforo", "N/A"),
        }
        def _best(d):
            if not d:
                return {}
            def _r(v):
                return v.get("rmse", float("inf")) if isinstance(v, dict) else float("inf")
            k = min(d, key=lambda x: _r(d[x]))
            return {"nome": k, "rmse": _r(d[k])}
        meta["melhor_ml"] = _best(resultados_ml)
        meta["melhor_ts"] = _best(resultados_ts)
        meta["melhor_dl"] = _best(resultados_dl)
        try:
            arqs = [f.name for f in sorted(OUTPUT_DIR.glob(f"*{TIMESTAMP}*"))]
            meta["arquivos_gerados"] = arqs
            meta["n_arquivos"] = len(arqs)
        except Exception:
            pass
        arq = OUTPUT_DIR / f"metadados_siprev_final_{TIMESTAMP}.json"
        with open(arq, "w", encoding="utf-8") as fh:
            json.dump(meta, fh, ensure_ascii=False, indent=2, default=str)
        log_ok(f"  JSON final: {arq.name}")
    except Exception as exc:
        log_warn(f"Secao 63 erro: {exc}")


def _executar_bloco_m(df_cg, df_ms, df_cap,
                       resultados_ml=None, resultados_ts=None,
                       resultados_dl=None, alerta=None):
    resultados_ml = resultados_ml or {}
    resultados_ts = resultados_ts or {}
    resultados_dl = resultados_dl or {}
    alerta        = alerta        or {}
    log_section("BLOCO M -- Validacao, CCF e Metadados (Secoes 61-63)")
    validacao_qualidade_dados(df_cg, df_ms, df_cap)
    ccf_capitais_campo_grande(df_cg, df_cap)
    exportar_metadados_json_final(df_cg, df_ms, df_cap,
                                   resultados_ml, resultados_ts, resultados_dl, alerta)
    log_ok("Bloco M concluido.")

if __name__ == "__main__":
    _resultados = main()




2026-05-31 16:46:18 [INFO] ==============================================================================


2026-05-31 16:46:18 [INFO]   SIPREV – Sistema Inteligente de Previsão Epidemiológica de Dengue


2026-05-31 16:46:18 [INFO]   Início  : 31/05/2026 16:46:18


2026-05-31 16:46:18 [INFO]   Ambiente: Máquina Local


2026-05-31 16:46:18 [INFO]   Python  : 3.14.5  |  Pandas: 2.3.3  |  NumPy: 2.3.5


...

2026-05-31 16:47:50 [INFO] 


2026-05-31 16:47:50 [INFO] ==============================================================================


2026-05-31 16:47:50 [INFO]   EXPORTAÇÃO FINAL – ZIP


2026-05-31 16:47:50 [INFO] ==============================================================================


2026-05-31 16:47:55 [INFO]   [ZIP] EpiAnalysis_DENG_20260531_164616.zip (53.6 MB)


In [133]:
# =============================================================================
# FIM DO MÓDULO SIPREV v1.0
# Disciplina: Análise Organizacional e Soluções Tecnológicas — Ciência dos Dados
# Módulo 3 — Previsão Epidemiológica de Dengue
# Fonte de Dados: InfoDengue (FGV/EMAp/FIOCRUZ)
# Cobertura: Campo Grande/MS | Mato Grosso do Sul (79 municípios) | 27 Capitais
# Ambiente: Google Colab / Python Local


In [134]:
# =============================================================================

In [135]:
# =============================================================================
# EXECUCAO PRINCIPAL — chama o pipeline completo
# =============================================================================
if __name__ == '__main__' or True:   # True para executar no Jupyter
    resultado = main()
    print('Pipeline SIPREV concluido com sucesso!')


2026-05-31 16:47:55 [INFO] ==============================================================================


2026-05-31 16:47:55 [INFO]   SIPREV – Sistema Inteligente de Previsão Epidemiológica de Dengue


2026-05-31 16:47:55 [INFO]   Início  : 31/05/2026 16:47:55


2026-05-31 16:47:55 [INFO]   Ambiente: Máquina Local


2026-05-31 16:47:55 [INFO]   Python  : 3.14.5  |  Pandas: 2.3.3  |  NumPy: 2.3.5


...

2026-05-31 16:49:21 [INFO] ==============================================================================


2026-05-31 16:49:21 [INFO]   EXPORTAÇÃO FINAL – ZIP


2026-05-31 16:49:21 [INFO] ==============================================================================


2026-05-31 16:49:26 [INFO]   [ZIP] EpiAnalysis_DENG_20260531_164616.zip (53.6 MB)


Pipeline SIPREV concluido com sucesso!
